# 14 · Extracción de comentarios de ForoCoches

Muestra de contenido **público e indexado**, sin inicio de sesión. El universo no equivale a todo ForoCoches: los buscadores externos deciden qué hilos indexan. Se estudian inmigración y LGTBI en 2015, 2016, 2019 y 2023.

In [1]:
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

import hashlib
import re
import time
import html

import pandas as pd
import requests

from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [4]:
# ============================
# RUTAS
# ============================

from pathlib import Path

BASE = Path(
    r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13"
)

RUTA_RAW = (
    BASE
    / "01_data"
    / "01_raw"
    / "rrss"
    / "forocoches"
)

RUTA_PROCESADOS = (
    BASE
    / "01_data"
    / "02_processed"
    / "rrss"
    / "forocoches"
)

RUTA_COMENTARIOS_POR_HILO = (
    RUTA_RAW
    / "COMENTARIOS_POR_HILO"
)

# Alias usados al final del notebook
RUTA_RAW_FOROCOCHES = RUTA_RAW
RUTA_PROCESADOS_FOROCOCHES = RUTA_PROCESADOS

for ruta in [
    RUTA_RAW,
    RUTA_PROCESADOS,
    RUTA_COMENTARIOS_POR_HILO
]:
    ruta.mkdir(
        parents=True,
        exist_ok=True
    )

print("Proyecto:", BASE)
print("RAW:", RUTA_RAW)
print("PROCESADOS:", RUTA_PROCESADOS)

Proyecto: C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13
RAW: C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\01_raw\rrss\forocoches
PROCESADOS: C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\rrss\forocoches


In [5]:
# Parámetros del estudio

ANIOS_OBJETIVO = [
    2015,
    2016,
    2019,
    2023
]

TEMAS_OBJETIVO = [
    "inmigracion",
    "lgtbi"
]

print("Años objetivo:", ANIOS_OBJETIVO)
print("Temas objetivo:", TEMAS_OBJETIVO)

Años objetivo: [2015, 2016, 2019, 2023]
Temas objetivo: ['inmigracion', 'lgtbi']


### Configuración de la conexión

In [6]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9",
}


def crear_sesion():
    """
    Crea una sesión HTTP con reintentos automáticos.

    No inicia sesión en ForoCoches y solamente trabaja
    con páginas accesibles públicamente.
    """
    
    sesion = requests.Session()
    sesion.headers.update(HEADERS)

    reintentos = Retry(
        total=4,
        connect=4,
        read=4,
        status=4,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )

    adaptador = HTTPAdapter(max_retries=reintentos)

    sesion.mount("https://", adaptador)
    sesion.mount("http://", adaptador)

    return sesion


sesion = crear_sesion()

In [ ]:
#Comprobación

print("Ruta del proyecto:")
print(BASE)

print("\nCarpeta RAW:")
print(RUTA_RAW)

print("\nCarpeta PROCESADOS:")
print(RUTA_PROCESADOS)

print("\nAños objetivo:", ANIOS_OBJETIVO)
print("Temas objetivo:", TEMAS_OBJETIVO)

Ruta del proyecto:
c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13

Carpeta RAW:
c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\RAW

Carpeta PROCESADOS:
c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\PROCESADOS

Años objetivo: [2015, 2016, 2019, 2023]
Temas objetivo: ['inmigracion', 'lgtbi']


### Términos para encontrar hilos de inmigración y LGTBI

In [8]:
TERMINOS_BUSQUEDA = {
    "inmigracion": [
        "inmigración",
        "inmigrantes",
        "inmigración ilegal",
        "refugiados",
        "crisis migratoria",
        "pateras",
        "menas",
    ],
    
    "lgtbi": [
        "LGTBI",
        "LGBT",
        "homosexualidad",
        "homofobia",
        "gay",
        "lesbianas",
        "transexualidad",
        "ley trans",
        "orgullo gay",
    ],
}


def construir_consultas(terminos, anios):
    
    filas = []

    for tema, palabras in terminos.items():
        for anio in anios:
            for termino in palabras:

                consulta = (
                    'site:forocoches.com/foro/showthread.php '
                    f'"{termino}" "{anio}"'
                )

                filas.append({
                    "tema": tema,
                    "anio_objetivo": anio,
                    "termino_busqueda": termino,
                    "consulta": consulta,
                })

    return pd.DataFrame(filas)


consultas_forocoches = construir_consultas(
    TERMINOS_BUSQUEDA,
    ANIOS_OBJETIVO,
)


print(
    f"Consultas generadas: "
    f"{len(consultas_forocoches):,}"
)

display(consultas_forocoches.head(15))

Consultas generadas: 64


,tema,anio_objetivo,termino_busqueda,consulta
0,inmigracion,2015,inmigración,"site:forocoches.com/foro/showthread.php ""inmig..."
1,inmigracion,2015,inmigrantes,"site:forocoches.com/foro/showthread.php ""inmig..."
2,inmigracion,2015,inmigración ilegal,"site:forocoches.com/foro/showthread.php ""inmig..."
3,inmigracion,2015,refugiados,"site:forocoches.com/foro/showthread.php ""refug..."
4,inmigracion,2015,crisis migratoria,"site:forocoches.com/foro/showthread.php ""crisi..."
5,inmigracion,2015,pateras,"site:forocoches.com/foro/showthread.php ""pater..."
6,inmigracion,2015,menas,"site:forocoches.com/foro/showthread.php ""menas..."
7,inmigracion,2016,inmigración,"site:forocoches.com/foro/showthread.php ""inmig..."
8,inmigracion,2016,inmigrantes,"site:forocoches.com/foro/showthread.php ""inmig..."
9,inmigracion,2016,inmigración ilegal,"site:forocoches.com/foro/showthread.php ""inmig..."


### Buscador externo

In [ ]:
from urllib.parse import quote_plus
import xml.etree.ElementTree as ET


def buscar_en_bing_rss(consulta, sesion, max_resultados=50):
    """
    Ejecuta una consulta en Bing mediante su salida RSS.

    Devuelve:
    - estado HTTP
    - tipo de contenido
    - lista de resultados
    """

    url_busqueda = (
        "https://www.bing.com/search"
        f"?q={quote_plus(consulta)}"
        f"&count={max_resultados}"
        "&format=rss"
        "&setlang=es"
    )

    respuesta = sesion.get(
        url_busqueda,
        timeout=30,
    )

    estado_http = respuesta.status_code
    tipo_contenido = respuesta.headers.get("Content-Type", "")

    respuesta.raise_for_status()

    resultados = []

    try:
        raiz = ET.fromstring(respuesta.content)

        for item in raiz.findall(".//item"):
            titulo = item.findtext("title", default="").strip()
            enlace = item.findtext("link", default="").strip()
            descripcion = item.findtext(
                "description",
                default=""
            ).strip()

            resultados.append({
                "titulo_resultado": titulo,
                "url_resultado": enlace,
                "descripcion_resultado": descripcion,
            })

    except ET.ParseError as error:
        print("La respuesta no es un RSS válido:")
        print(error)

    return {
        "estado_http": estado_http,
        "tipo_contenido": tipo_contenido,
        "url_busqueda": url_busqueda,
        "resultados": resultados,
    }

In [10]:
#Consulta de prueba

fila_prueba = consultas_forocoches.iloc[0]

print("Consulta de prueba:")
print(fila_prueba["consulta"])

prueba_bing = buscar_en_bing_rss(
    consulta=fila_prueba["consulta"],
    sesion=sesion,
    max_resultados=50,
)


Consulta de prueba:
site:forocoches.com/foro/showthread.php "inmigración" "2015"


In [11]:
print("\nEstado HTTP:")
print(prueba_bing["estado_http"])

print("\nTipo de contenido:")
print(prueba_bing["tipo_contenido"])

print("\nResultados recibidos:")
print(len(prueba_bing["resultados"]))


resultados_prueba = pd.DataFrame(
    prueba_bing["resultados"]
)

if resultados_prueba.empty:
    print("\nBing no ha devuelto resultados para esta consulta.")
else:
    display(
        resultados_prueba[
            [
                "titulo_resultado",
                "url_resultado",
            ]
        ].head(20)
    )


Estado HTTP:
200

Tipo de contenido:
text/xml; charset=utf-8

Resultados recibidos:
10


,titulo_resultado,url_resultado
0,"Home - Migraciones - Ministerio de Inclusión, ...",https://www.inclusion.gob.es/web/migraciones
1,Extranjería - Administraciones Públicas,https://sede.administracionespublicas.gob.es/p...
2,Inmigración en EL PAÍS,https://elpais.com/noticias/inmigracion/
3,"Inmigración - Wikipedia, la enciclopedia libre",https://es.m.wikipedia.org/wiki/Inmigraci%C3%B3n
4,Inmigración - elperiodico,https://www.elperiodico.com/es/temas/inmigraci...
5,"Inmigración, últimas noticias - ABC",https://www.abc.es/noticias/inmigracion/
6,"Inmigración - Concepto, causas y países con má...",https://concepto.de/inmigracion/
7,Ocho datos para entender lo que está pasando e...,https://www.elmundo.es/espana/2026/02/01/697d1...
8,Inmigración - La Razón,https://www.larazon.es/tags/inmigracion
9,El mapa de la inmigración en España: América g...,https://www.elmundo.es/espana/2026/02/26/6998c...


### COMPARACIÓN DE FORMAS DE CONSULTA

In [12]:
consultas_prueba_bing = [
    "site:forocoches.com inmigración 2015",
    "site:forocoches.com/foro inmigración 2015",
    "inmigración 2015 forocoches",
    '"forocoches.com/foro/showthread.php" inmigración 2015',
]


def es_resultado_forocoches(url):
    """
    Comprueba si una URL pertenece a un hilo de ForoCoches.
    """
    
    url = str(url).lower()

    return (
        "forocoches.com" in url
        and "showthread.php" in url
    )


resumen_pruebas = []
resultados_comparacion = []

for numero, consulta in enumerate(
    consultas_prueba_bing,
    start=1,
):
    print(
        f"Probando {numero}/{len(consultas_prueba_bing)}: "
        f"{consulta}"
    )

    prueba = buscar_en_bing_rss(
        consulta=consulta,
        sesion=sesion,
        max_resultados=50,
    )

    resultados = pd.DataFrame(
        prueba["resultados"]
    )

    if resultados.empty:
        total_resultados = 0
        total_forocoches = 0

    else:
        resultados["es_forocoches"] = (
            resultados["url_resultado"]
            .apply(es_resultado_forocoches)
        )

        total_resultados = len(resultados)
        total_forocoches = int(
            resultados["es_forocoches"].sum()
        )

        resultados["consulta_prueba"] = consulta
        resultados_comparacion.append(resultados)

    resumen_pruebas.append({
        "consulta": consulta,
        "estado_http": prueba["estado_http"],
        "resultados_totales": total_resultados,
        "hilos_forocoches": total_forocoches,
    })

    time.sleep(1)


resumen_pruebas = pd.DataFrame(
    resumen_pruebas
)

display(resumen_pruebas)

Probando 1/4: site:forocoches.com inmigración 2015
Probando 2/4: site:forocoches.com/foro inmigración 2015
Probando 3/4: inmigración 2015 forocoches
Probando 4/4: "forocoches.com/foro/showthread.php" inmigración 2015


,consulta,estado_http,resultados_totales,hilos_forocoches
0,site:forocoches.com inmigración 2015,200,10,0
1,site:forocoches.com/foro inmigración 2015,200,10,0
2,inmigración 2015 forocoches,200,10,0
3,"""forocoches.com/foro/showthread.php"" inmigraci...",200,7,0


In [13]:
if resultados_comparacion:
    resultados_comparacion = pd.concat(
        resultados_comparacion,
        ignore_index=True,
    )

    hilos_forocoches_prueba = (
        resultados_comparacion.loc[
            resultados_comparacion["es_forocoches"]
        ]
        .drop_duplicates("url_resultado")
        .reset_index(drop=True)
    )

    print(
        "Hilos de ForoCoches encontrados:",
        len(hilos_forocoches_prueba),
    )

    if not hilos_forocoches_prueba.empty:
        display(
            hilos_forocoches_prueba[
                [
                    "titulo_resultado",
                    "url_resultado",
                    "consulta_prueba",
                ]
            ]
        )

else:
    print("No se recibió ningún resultado.")

Hilos de ForoCoches encontrados: 0


### Con Google

In [14]:
from urllib.parse import (
    parse_qs,
    quote_plus,
    unquote,
    urlparse,
)


def extraer_url_google(href):
    """
    Convierte los enlaces intermedios de Google en la URL final.
    """
    
    if not href:
        return None

    href = html.unescape(str(href))

    # Formato habitual: /url?q=https://...
    if href.startswith("/url?"):
        parametros = parse_qs(
            urlparse(href).query
        )

        destino = parametros.get("q", [None])[0]

        if destino:
            return unquote(destino)

    # Algunas respuestas incluyen directamente la URL.
    if href.startswith("http"):
        return href

    return None


def buscar_en_google(consulta, sesion, max_resultados=100):
    """
    Ejecuta una consulta pública en Google y extrae los enlaces
    visibles de la página de resultados.
    """
    
    url_busqueda = (
        "https://www.google.com/search"
        f"?q={quote_plus(consulta)}"
        f"&num={max_resultados}"
        "&hl=es"
        "&filter=0"
    )

    respuesta = sesion.get(
        url_busqueda,
        timeout=30,
    )

    contenido_minusculas = respuesta.text.lower()

    posible_bloqueo = any(
        expresion in contenido_minusculas
        for expresion in [
            "nuestros sistemas han detectado",
            "unusual traffic",
            "recaptcha",
            "before you continue to google",
            "consent.google",
        ]
    )

    resultados = []

    if respuesta.status_code == 200 and not posible_bloqueo:
        soup = BeautifulSoup(
            respuesta.text,
            "html.parser",
        )

        for enlace in soup.select("a[href]"):
            url_final = extraer_url_google(
                enlace.get("href")
            )

            if not url_final:
                continue

            titulo = enlace.get_text(
                " ",
                strip=True,
            )

            resultados.append({
                "titulo_resultado": titulo,
                "url_resultado": url_final,
            })

    return {
        "estado_http": respuesta.status_code,
        "tipo_contenido": respuesta.headers.get(
            "Content-Type",
            "",
        ),
        "posible_bloqueo": posible_bloqueo,
        "longitud_respuesta": len(respuesta.text),
        "url_busqueda": url_busqueda,
        "resultados": resultados,
    }


# ------------------------------------------------------------
# Una única consulta de prueba
# ------------------------------------------------------------

consulta_google_prueba = (
    'site:forocoches.com/foro/showthread.php '
    '"inmigración" "2015"'
)

print("Consulta:")
print(consulta_google_prueba)

prueba_google = buscar_en_google(
    consulta=consulta_google_prueba,
    sesion=sesion,
    max_resultados=100,
)


# ------------------------------------------------------------
# Diagnóstico de la respuesta
# ------------------------------------------------------------

print("\nEstado HTTP:")
print(prueba_google["estado_http"])

print("\nTipo de contenido:")
print(prueba_google["tipo_contenido"])

print("\n¿Posible bloqueo?")
print(prueba_google["posible_bloqueo"])

print("\nTamaño de la respuesta:")
print(
    f'{prueba_google["longitud_respuesta"]:,} caracteres'
)

print("\nEnlaces extraídos:")
print(len(prueba_google["resultados"]))

Consulta:
site:forocoches.com/foro/showthread.php "inmigración" "2015"

Estado HTTP:
200

Tipo de contenido:
text/html; charset=UTF-8

¿Posible bloqueo?
False

Tamaño de la respuesta:
91,449 caracteres

Enlaces extraídos:
1


In [15]:
resultados_google_prueba = pd.DataFrame(
    prueba_google["resultados"]
)

if resultados_google_prueba.empty:
    print(
        "\nNo se pudieron extraer resultados."
    )

else:
    resultados_google_prueba["es_hilo_forocoches"] = (
        resultados_google_prueba["url_resultado"]
        .apply(es_resultado_forocoches)
    )

    hilos_google_prueba = (
        resultados_google_prueba.loc[
            resultados_google_prueba[
                "es_hilo_forocoches"
            ]
        ]
        .drop_duplicates("url_resultado")
        .reset_index(drop=True)
    )

    print(
        "\nHilos de ForoCoches encontrados:",
        len(hilos_google_prueba),
    )

    display(
        hilos_google_prueba[
            [
                "titulo_resultado",
                "url_resultado",
            ]
        ].head(20)
    )


Hilos de ForoCoches encontrados: 0


,titulo_resultado,url_resultado


In [16]:
# ============================================================
# CELDA 6 — PRUEBA DE DESCUBRIMIENTO CON DUCKDUCKGO
# ============================================================

from urllib.parse import parse_qs, unquote, urlparse


def extraer_url_duckduckgo(href):
    """
    Extrae la URL real de un resultado de DuckDuckGo.
    """
    
    if not href:
        return None

    href = html.unescape(str(href))

    # DuckDuckGo puede devolver un enlace intermedio con uddg.
    parametros = parse_qs(
        urlparse(href).query
    )

    if "uddg" in parametros:
        return unquote(
            parametros["uddg"][0]
        )

    if href.startswith("http"):
        return href

    return None


def buscar_en_duckduckgo(consulta, sesion):
    """
    Ejecuta una consulta en la versión HTML de DuckDuckGo.
    """
    
    url_busqueda = (
        "https://html.duckduckgo.com/html/"
        f"?q={quote_plus(consulta)}"
    )

    respuesta = sesion.get(
        url_busqueda,
        timeout=30,
    )

    contenido_minusculas = respuesta.text.lower()

    posible_bloqueo = any(
        expresion in contenido_minusculas
        for expresion in [
            "anomaly detected",
            "captcha",
            "bots use duckduckgo",
            "automated requests",
        ]
    )

    resultados = []

    if respuesta.status_code == 200 and not posible_bloqueo:
        soup = BeautifulSoup(
            respuesta.text,
            "html.parser",
        )

        for enlace in soup.select(
            "a.result__a[href]"
        ):
            url_final = extraer_url_duckduckgo(
                enlace.get("href")
            )

            if not url_final:
                continue

            resultados.append({
                "titulo_resultado": enlace.get_text(
                    " ",
                    strip=True,
                ),
                "url_resultado": url_final,
            })

    return {
        "estado_http": respuesta.status_code,
        "tipo_contenido": respuesta.headers.get(
            "Content-Type",
            "",
        ),
        "posible_bloqueo": posible_bloqueo,
        "longitud_respuesta": len(respuesta.text),
        "resultados": resultados,
        "url_busqueda": url_busqueda,
    }


# ------------------------------------------------------------
# Consulta de prueba
# ------------------------------------------------------------

consulta_duckduckgo_prueba = (
    'site:forocoches.com/foro/showthread.php '
    '"inmigración" "2015"'
)

print("Consulta:")
print(consulta_duckduckgo_prueba)

prueba_duckduckgo = buscar_en_duckduckgo(
    consulta=consulta_duckduckgo_prueba,
    sesion=sesion,
)


# ------------------------------------------------------------
# Diagnóstico
# ------------------------------------------------------------

print("\nEstado HTTP:")
print(prueba_duckduckgo["estado_http"])

print("\nTipo de contenido:")
print(prueba_duckduckgo["tipo_contenido"])

print("\n¿Posible bloqueo?")
print(prueba_duckduckgo["posible_bloqueo"])

print("\nTamaño de la respuesta:")
print(
    f'{prueba_duckduckgo["longitud_respuesta"]:,} caracteres'
)

print("\nResultados extraídos:")
print(len(prueba_duckduckgo["resultados"]))

Consulta:
site:forocoches.com/foro/showthread.php "inmigración" "2015"

Estado HTTP:
202

Tipo de contenido:
text/html; charset=UTF-8

¿Posible bloqueo?
True

Tamaño de la respuesta:
14,309 caracteres

Resultados extraídos:
0


In [17]:
resultados_duckduckgo_prueba = pd.DataFrame(
    prueba_duckduckgo["resultados"]
)

if resultados_duckduckgo_prueba.empty:
    print(
        "\nDuckDuckGo no ha devuelto resultados aprovechables."
    )

else:
    resultados_duckduckgo_prueba[
        "es_hilo_forocoches"
    ] = (
        resultados_duckduckgo_prueba[
            "url_resultado"
        ].apply(es_resultado_forocoches)
    )

    hilos_duckduckgo_prueba = (
        resultados_duckduckgo_prueba.loc[
            resultados_duckduckgo_prueba[
                "es_hilo_forocoches"
            ]
        ]
        .drop_duplicates("url_resultado")
        .reset_index(drop=True)
    )

    print(
        "\nHilos de ForoCoches encontrados:",
        len(hilos_duckduckgo_prueba),
    )

    if not hilos_duckduckgo_prueba.empty:
        display(
            hilos_duckduckgo_prueba[
                [
                    "titulo_resultado",
                    "url_resultado",
                ]
            ]
        )


DuckDuckGo no ha devuelto resultados aprovechables.


In [18]:
# ============================================================
# CELDA 7 — CATÁLOGO INICIAL DE HILOS CANDIDATOS
# ============================================================

hilos_candidatos = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4514261",
        "titulo_resultado": "El problema que hemos creado",
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4514261"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4730394",
        "titulo_resultado": (
            "Un asunto delicado el tema "
            "INMIGRACIÓN y delincuencia. ¿Qué eliges?"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4730394"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7092240",
        "titulo_resultado": (
            "ALERTA anti-INMIGRACIÓN ILEGAL "
            "+TemaSerio"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7092240"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2023,
        "id_hilo": "9541304",
        "titulo_resultado": (
            "Estoy a favor de la inmigración ilegal"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=9541304"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2015,
        "id_hilo": "4407356",
        "titulo_resultado": "Mañana voy a Chueca",
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4407356"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2016,
        "id_hilo": "4932029",
        "titulo_resultado": (
            "Árbitro deja el arbitraje por recibir "
            "continuos insultos homófobos"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4932029"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2019,
        "id_hilo": "7306066",
        "titulo_resultado": (
            "A favor o en contra de que se hagan "
            "cursos LGTB en las escuelas"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7306066"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2023,
        "id_hilo": "9472060",
        "titulo_resultado": (
            "Las empresas con más de 50 empleados "
            "tienen un año para implantar un plan LGTBI"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=9472060"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
])


# ------------------------------------------------------------
# Eliminamos posibles duplicados
# ------------------------------------------------------------

hilos_candidatos = (
    hilos_candidatos
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Comprobamos la cobertura inicial
# ------------------------------------------------------------

cobertura_inicial = (
    hilos_candidatos
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .size()
    .rename("hilos_candidatos")
    .reset_index()
)


print(
    "Hilos candidatos:",
    len(hilos_candidatos),
)

display(cobertura_inicial)

display(
    hilos_candidatos[
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "titulo_resultado",
        ]
    ]
)

Hilos candidatos: 8


,tema,anio_objetivo,hilos_candidatos
0,inmigracion,2015,1
1,inmigracion,2016,1
2,inmigracion,2019,1
3,inmigracion,2023,1
4,lgtbi,2015,1
5,lgtbi,2016,1
6,lgtbi,2019,1
7,lgtbi,2023,1


,tema,anio_objetivo,id_hilo,titulo_resultado
0,inmigracion,2015,4514261,El problema que hemos creado
1,inmigracion,2016,4730394,Un asunto delicado el tema INMIGRACIÓN y delin...
2,inmigracion,2019,7092240,ALERTA anti-INMIGRACIÓN ILEGAL +TemaSerio
3,inmigracion,2023,9541304,Estoy a favor de la inmigración ilegal
4,lgtbi,2015,4407356,Mañana voy a Chueca
5,lgtbi,2016,4932029,Árbitro deja el arbitraje por recibir continuo...
6,lgtbi,2019,7306066,A favor o en contra de que se hagan cursos LGT...
7,lgtbi,2023,9472060,Las empresas con más de 50 empleados tienen un...


### Hilo de prueba

In [19]:
# ============================================================
# CELDA 8 — DESCARGA DE UNA PÁGINA DE PRUEBA
# ============================================================

# Seleccionamos el hilo de inmigración de 2016.
fila_hilo_prueba = (
    hilos_candidatos.loc[
        (hilos_candidatos["tema"] == "inmigracion")
        & (hilos_candidatos["anio_objetivo"] == 2016)
    ]
    .iloc[0]
)


url_hilo_prueba = fila_hilo_prueba["url_hilo"]

print("Hilo seleccionado:")
print(fila_hilo_prueba["titulo_resultado"])

print("\nURL:")
print(url_hilo_prueba)


# ------------------------------------------------------------
# Descargamos la primera página
# ------------------------------------------------------------

respuesta_hilo_prueba = sesion.get(
    url_hilo_prueba,
    timeout=40,
)


print("\nEstado HTTP:")
print(respuesta_hilo_prueba.status_code)

print("\nURL final:")
print(respuesta_hilo_prueba.url)

print("\nTamaño de la respuesta:")
print(
    f"{len(respuesta_hilo_prueba.text):,} caracteres"
)

print("\nTipo de contenido:")
print(
    respuesta_hilo_prueba.headers.get(
        "Content-Type",
        "",
    )
)

Hilo seleccionado:
Un asunto delicado el tema INMIGRACIÓN y delincuencia. ¿Qué eliges?

URL:
https://forocoches.com/foro/showthread.php?t=4730394

Estado HTTP:
200

URL final:
https://forocoches.com/foro/showthread.php?t=4730394

Tamaño de la respuesta:
212,185 caracteres

Tipo de contenido:
text/html; charset=UTF-8


In [20]:
# ------------------------------------------------------------
# Parseamos el HTML
# ------------------------------------------------------------

respuesta_hilo_prueba.raise_for_status()

soup_hilo_prueba = BeautifulSoup(
    respuesta_hilo_prueba.text,
    "html.parser",
)


# Los mensajes tienen identificadores como:
# post_message_123456789

mensajes_html_prueba = soup_hilo_prueba.select(
    'div[id^="post_message_"]'
)


# Buscamos posibles enlaces de paginación.
enlaces_paginas_prueba = soup_hilo_prueba.select(
    'a[href*="showthread.php"][href*="page="]'
)


# Comprobamos si aparece una pantalla de acceso.
texto_pagina_minusculas = (
    soup_hilo_prueba
    .get_text(" ", strip=True)
    .lower()
)

posible_pantalla_acceso = (
    len(mensajes_html_prueba) == 0
    and any(
        expresion in texto_pagina_minusculas
        for expresion in [
            "iniciar sesión",
            "debes estar registrado",
            "no tienes permisos",
            "registrarse",
        ]
    )
)


print(
    "Mensajes reconocidos en la página:",
    len(mensajes_html_prueba),
)

print(
    "Enlaces de paginación encontrados:",
    len(enlaces_paginas_prueba),
)

print(
    "¿Posible pantalla de acceso?",
    posible_pantalla_acceso,
)

Mensajes reconocidos en la página: 9
Enlaces de paginación encontrados: 0
¿Posible pantalla de acceso? False


In [21]:
# ------------------------------------------------------------
# Vista previa del primer mensaje
# ------------------------------------------------------------

if mensajes_html_prueba:
    primer_mensaje_html = mensajes_html_prueba[0]

    id_html_primer_mensaje = (
        primer_mensaje_html.get("id", "")
    )

    id_primer_mensaje = (
        id_html_primer_mensaje
        .replace("post_message_", "")
    )

    texto_primer_mensaje = (
        primer_mensaje_html
        .get_text(
            "\n",
            strip=True,
        )
    )

    print("ID del primer mensaje:")
    print(id_primer_mensaje)

    print("\nVista previa del texto:")
    print(texto_primer_mensaje[:1000])

else:
    print(
        "No se encontró ningún mensaje "
        "con la estructura esperada."
    )

ID del primer mensaje:
217029942

Vista previa del texto:
Es lo que están acostumbrados en sus países, si no mirad lo que pasa en Guatemala, El Salvador, Honduras...
Y países musulmanes, con guerras civiles, decapitaciones, crímenes sangrientos, torturas, atentados cada semana...
Es lo que han vivido en sus países y eso no podemos cambiarlo. Lo llevan en la sangre. Hay países que jamás tendrán solución, pero no por su política, si no por sus propios habitantes.
Para esa gente, nuestra sociedad y costumbres son el paraíso, pueden hacer lo mismo que en sus países sin apenas repercusión. Robar, violar, matar, etc. Y no sufren si tienen que enfrentarse a la policía con armas o lo que sea, están acostumbrados a eso, no tienen NADA que perder.
La única solución que tenemos aquí en Europa es endurecer las leyes, no hay otra. Pero incluso así, esta gente seguirá delinquiendo, porque más duras que son las leyes en sus países y se las pasan por el forro de los huevos.
Es un asunto muy delicado t

In [23]:
# ============================================================
# CELDA 9 — PARSER DE COMENTARIOS DE UNA PÁGINA
# ============================================================

MESES_ES = {
    "ene": 1,
    "feb": 2,
    "mar": 3,
    "abr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "ago": 8,
    "sep": 9,
    "oct": 10,
    "nov": 11,
    "dic": 12,
}


def convertir_fecha_forocoches(texto_fecha):
    """
    Convierte fechas como:

        07-ene-2016 10:01

    en un Timestamp de pandas.

    Se conserva la hora mostrada públicamente por ForoCoches,
    sin asumir que sea UTC.
    """
    
    texto_fecha = (
        str(texto_fecha)
        .replace("\xa0", " ")
        .strip()
        .lower()
    )

    texto_fecha = " ".join(
        texto_fecha.split()
    )

    coincidencia = re.search(
        r"(\d{1,2})-([a-záéíóú]{3})-(\d{4})"
        r"\s+(\d{1,2}):(\d{2})",
        texto_fecha,
    )

    if not coincidencia:
        return pd.NaT

    dia, mes_texto, anio, hora, minuto = (
        coincidencia.groups()
    )

    numero_mes = MESES_ES.get(
        mes_texto[:3]
    )

    if numero_mes is None:
        return pd.NaT

    try:
        return pd.Timestamp(
            year=int(anio),
            month=numero_mes,
            day=int(dia),
            hour=int(hora),
            minute=int(minuto),
        )

    except ValueError:
        return pd.NaT

### Se le da a cada usuario un identificador único

In [24]:
def crear_hash_usuario(nombre_usuario):
    
    nombre_normalizado = (
        str(nombre_usuario)
        .strip()
        .casefold()
    )

    if not nombre_normalizado:
        return None

    return hashlib.sha256(
        nombre_normalizado.encode("utf-8")
    ).hexdigest()[:16]

In [25]:
def extraer_comentarios_pagina(
    soup,
    tema,
    anio_objetivo,
    id_hilo,
    titulo_hilo,
    url_hilo,
    pagina=1,
):
    """
    Extrae todos los mensajes reconocibles de una página.
    """
    
    filas = []

    mensajes_html = soup.select(
        'div[id^="post_message_"]'
    )

    for mensaje_html in mensajes_html:

        # ----------------------------------------------------
        # ID del mensaje
        # ----------------------------------------------------

        id_html = mensaje_html.get("id", "")

        coincidencia_id = re.fullmatch(
            r"post_message_(\d+)",
            id_html,
        )

        if not coincidencia_id:
            continue

        id_mensaje = coincidencia_id.group(1)


        # ----------------------------------------------------
        # Contenedor que incluye autor, fecha y número
        # ----------------------------------------------------

        contenedor = mensaje_html.find_parent(
            "div",
            class_="postbit_wrapper",
        )

        if contenedor is None:
            continue


        # ----------------------------------------------------
        # Usuario
        # ----------------------------------------------------

        selector_usuario = (
            f'#postmenu_{id_mensaje} '
            'a[href*="member.php?u="]'
        )

        nodo_usuario = contenedor.select_one(
            selector_usuario
        )

        if nodo_usuario is not None:
            nombre_usuario = nodo_usuario.get_text(
                " ",
                strip=True,
            )
        else:
            nombre_usuario = ""

        usuario_hash = crear_hash_usuario(
            nombre_usuario
        )


        # ----------------------------------------------------
        # Fecha
        # ----------------------------------------------------

        nodo_fecha = contenedor.select_one(
            "span.postdate.old"
        )

        if nodo_fecha is not None:
            fecha_texto = nodo_fecha.get_text(
                " ",
                strip=True,
            )
        else:
            fecha_texto = ""

        fecha_comentario = (
            convertir_fecha_forocoches(
                fecha_texto
            )
        )


        # ----------------------------------------------------
        # Número del mensaje: #1, #2...
        # ----------------------------------------------------

        numero_mensaje = None

        textos_numero = contenedor.find_all(
            string=re.compile(r"^\s*#\d+\s*$")
        )

        for texto_numero in textos_numero:
            coincidencia_numero = re.fullmatch(
                r"\s*#(\d+)\s*",
                str(texto_numero),
            )

            if coincidencia_numero:
                numero_mensaje = int(
                    coincidencia_numero.group(1)
                )
                break


        # ----------------------------------------------------
        # Texto
        # ----------------------------------------------------

        texto_original = mensaje_html.get_text(
            "\n",
            strip=True,
        )

        lineas_limpias = [
            linea.strip()
            for linea in texto_original.splitlines()
            if linea.strip()
        ]

        texto_original = "\n".join(
            lineas_limpias
        )


        # ----------------------------------------------------
        # Fila final
        # ----------------------------------------------------

        filas.append({
            "tema": tema,
            "anio_objetivo": int(anio_objetivo),
            "id_hilo": str(id_hilo),
            "titulo_hilo": titulo_hilo,
            "url_hilo": url_hilo,
            "pagina": int(pagina),
            "id_mensaje": str(id_mensaje),
            "numero_mensaje": numero_mensaje,
            "usuario_hash": usuario_hash,
            "fecha_comentario": fecha_comentario,
            "anio_comentario": (
                int(fecha_comentario.year)
                if not pd.isna(fecha_comentario)
                else None
            ),
            "es_mensaje_inicial": (
                numero_mensaje == 1
            ),
            "texto_original": texto_original,
        })

    return pd.DataFrame(filas)

In [26]:
# ------------------------------------------------------------
# Aplicamos el parser al hilo de prueba
# ------------------------------------------------------------

comentarios_pagina_prueba = (
    extraer_comentarios_pagina(
        soup=soup_hilo_prueba,
        tema=fila_hilo_prueba["tema"],
        anio_objetivo=fila_hilo_prueba[
            "anio_objetivo"
        ],
        id_hilo=fila_hilo_prueba["id_hilo"],
        titulo_hilo=fila_hilo_prueba[
            "titulo_resultado"
        ],
        url_hilo=fila_hilo_prueba["url_hilo"],
        pagina=1,
    )
)


print(
    "Comentarios extraídos:",
    len(comentarios_pagina_prueba),
)

print(
    "IDs únicos:",
    comentarios_pagina_prueba[
        "id_mensaje"
    ].nunique(),
)

print(
    "Fechas reconocidas:",
    comentarios_pagina_prueba[
        "fecha_comentario"
    ].notna().sum(),
)

print(
    "Usuarios seudonimizados:",
    comentarios_pagina_prueba[
        "usuario_hash"
    ].notna().sum(),
)

Comentarios extraídos: 9
IDs únicos: 9
Fechas reconocidas: 9
Usuarios seudonimizados: 9


In [27]:
display(
    comentarios_pagina_prueba[
        [
            "numero_mensaje",
            "id_mensaje",
            "usuario_hash",
            "fecha_comentario",
            "anio_comentario",
            "es_mensaje_inicial",
            "texto_original",
        ]
    ].head(10)
)

,numero_mensaje,id_mensaje,usuario_hash,fecha_comentario,anio_comentario,es_mensaje_inicial,texto_original
0,1,217029942,adf0a55d9756d742,2016-01-07 10:01:00,2016,True,"Es lo que están acostumbrados en sus países, s..."
1,2,217029965,ed380416ab43785f,2016-01-07 10:01:00,2016,False,"Ya te lo dice esta pole, cada vecino en su par..."
2,3,217029968,0b812141842fc387,2016-01-07 10:01:00,2016,False,Lo que hay que hacer es dar una patada en el c...
3,4,217029973,f11f0d951c6afdb6,2016-01-07 10:01:00,2016,False,Menudo ladrillo a estas horas
4,5,217030090,48ba2627a46ec5d1,2016-01-07 10:05:00,2016,False,La segunda y cuarta opción son lo mismo.
5,6,217030142,a981d720abc6d45c,2016-01-07 10:06:00,2016,False,Cita de\nlongboard\n..... Si evitas su llegada...
6,7,217030151,1ebeec4ca7fecd0c,2016-01-07 10:06:00,2016,False,Esta claro que hay que endurecer las leyes. Pe...
7,8,217188436,adf0a55d9756d742,2016-01-08 20:21:00,2016,False,Cita de\nBocasecaman69\nEsta claro que hay que...
8,9,217190462,8df124bf02253be3,2016-01-08 20:42:00,2016,False,Que se vayan todos y de paso de lleven a los g...


### Citas ForoCoche

In [28]:
# ============================================================
# CELDA 10 — DIAGNÓSTICO DE LA ESTRUCTURA DE LAS CITAS
# ============================================================

def resumir_nodo_html(nodo):
    """
    Resume un elemento HTML sin imprimir todo su contenido.
    """
    
    if nodo is None:
        return None

    clases = nodo.get("class", [])
    
    if isinstance(clases, list):
        clases = " ".join(clases)

    texto = nodo.get_text(
        " ",
        strip=True,
    )

    return {
        "etiqueta": nodo.name,
        "id": nodo.get("id"),
        "clases": clases,
        "style": nodo.get("style"),
        "texto_inicial": texto[:250],
    }


mensaje_con_cita = None
nodo_texto_cita = None

for mensaje_html in mensajes_html_prueba:
    
    nodo_encontrado = mensaje_html.find(
        string=re.compile(
            r"cita\s+de",
            flags=re.IGNORECASE,
        )
    )

    if nodo_encontrado is not None:
        mensaje_con_cita = mensaje_html
        nodo_texto_cita = nodo_encontrado
        break


if mensaje_con_cita is None:
    print(
        "No se encontró una cita en la página de prueba."
    )

else:
    print("Mensaje con cita:")
    print(
        mensaje_con_cita.get(
            "id",
            "",
        )
    )

    print("\nTexto donde se detectó la cita:")
    print(
        str(nodo_texto_cita).strip()
    )

    print("\nEstructura ascendente:")

Mensaje con cita:
post_message_217030142

Texto donde se detectó la cita:
Cita de

Estructura ascendente:


In [29]:
    estructura_cita = []

    nodo_actual = nodo_texto_cita.parent

    for nivel in range(8):
        if nodo_actual is None:
            break

        resumen = resumir_nodo_html(
            nodo_actual
        )

        resumen["nivel"] = nivel

        estructura_cita.append(
            resumen
        )

        if nodo_actual == mensaje_con_cita:
            break

        nodo_actual = nodo_actual.parent


    estructura_cita = pd.DataFrame(
        estructura_cita
    )

    display(
        estructura_cita[
            [
                "nivel",
                "etiqueta",
                "id",
                "clases",
                "style",
                "texto_inicial",
            ]
        ]
    )

,nivel,etiqueta,id,clases,style,texto_inicial
0,0,div,None,,display: inline-block,Cita de longboard
1,1,div,None,,display: flex; flex-direction: row; align-item...,Cita de longboard
2,2,div,None,quote,None,"Cita de longboard ..... Si evitas su llegada, ..."
3,3,div,None,squote,None,"Cita de longboard ..... Si evitas su llegada, ..."
4,4,td,None,,font-size: 1rem; width: 100%,"Cita de longboard ..... Si evitas su llegada, ..."
5,5,tr,None,,width: 100%,"Cita de longboard ..... Si evitas su llegada, ..."
6,6,table,None,,width: 100%,"Cita de longboard ..... Si evitas su llegada, ..."
7,7,div,post_message_217030142,,letter-spacing: 0.02px; line-height: 1.2;,"Cita de longboard ..... Si evitas su llegada, ..."


In [30]:
# ------------------------------------------------------------
# Selectores habituales encontrados dentro del mensaje
# ------------------------------------------------------------

if mensaje_con_cita is not None:

    selectores_cita_posibles = [
        "blockquote",
        ".quote",
        ".quotecontainer",
        ".bbcode_quote",
        "table",
    ]

    resumen_selectores = []

    for selector in selectores_cita_posibles:
        nodos = mensaje_con_cita.select(
            selector
        )

        resumen_selectores.append({
            "selector": selector,
            "elementos_encontrados": len(nodos),
            "primer_texto": (
                nodos[0].get_text(
                    " ",
                    strip=True,
                )[:300]
                if nodos
                else ""
            ),
        })

    display(
        pd.DataFrame(
            resumen_selectores
        )
    )

,selector,elementos_encontrados,primer_texto
0,blockquote,0,
1,.quote,1,"Cita de longboard ..... Si evitas su llegada, ..."
2,.quotecontainer,0,
3,.bbcode_quote,0,
4,table,1,"Cita de longboard ..... Si evitas su llegada, ..."


In [31]:
# ============================================================
# CELDA 11 — ELIMINACIÓN CONTROLADA DE CITAS
# ============================================================

def normalizar_texto_elemento(elemento):
    """
    Convierte un elemento HTML en texto limpio,
    conservando los saltos entre fragmentos.
    """
    
    texto = elemento.get_text(
        "\n",
        strip=True,
    )

    lineas = [
        linea.strip()
        for linea in texto.splitlines()
        if linea.strip()
    ]

    return "\n".join(lineas)


def extraer_texto_sin_citas(mensaje_html):
    """
    Crea una copia del mensaje y elimina únicamente
    los elementos identificados con la clase .quote.
    
    El HTML original no se modifica.
    """
    
    copia_soup = BeautifulSoup(
        str(mensaje_html),
        "html.parser",
    )

    mensaje_copia = copia_soup.select_one(
        'div[id^="post_message_"]'
    )

    if mensaje_copia is None:
        return ""

    # Eliminamos todas las citas contenidas en el mensaje.
    for cita in mensaje_copia.select(".quote"):
        cita.decompose()

    return normalizar_texto_elemento(
        mensaje_copia
    )

In [32]:
def extraer_comentarios_pagina_limpios(
    soup,
    tema,
    anio_objetivo,
    id_hilo,
    titulo_hilo,
    url_hilo,
    pagina=1,
):
    """
    Ejecuta el parser estructurado y añade la versión
    del texto sin citas.
    """
    
    comentarios = extraer_comentarios_pagina(
        soup=soup,
        tema=tema,
        anio_objetivo=anio_objetivo,
        id_hilo=id_hilo,
        titulo_hilo=titulo_hilo,
        url_hilo=url_hilo,
        pagina=pagina,
    )

    textos_sin_citas = {}

    for mensaje_html in soup.select(
        'div[id^="post_message_"]'
    ):
        id_html = mensaje_html.get(
            "id",
            "",
        )

        coincidencia = re.fullmatch(
            r"post_message_(\d+)",
            id_html,
        )

        if not coincidencia:
            continue

        id_mensaje = coincidencia.group(1)

        textos_sin_citas[id_mensaje] = (
            extraer_texto_sin_citas(
                mensaje_html
            )
        )

    comentarios["texto_sin_citas"] = (
        comentarios["id_mensaje"]
        .map(textos_sin_citas)
        .fillna("")
    )

    comentarios["contiene_cita"] = (
        comentarios["texto_original"]
        != comentarios["texto_sin_citas"]
    )

    comentarios["texto_propio_vacio"] = (
        comentarios["texto_sin_citas"]
        .str.strip()
        .eq("")
    )

    return comentarios

In [33]:
comentarios_pagina_prueba_limpios = (
    extraer_comentarios_pagina_limpios(
        soup=soup_hilo_prueba,
        tema=fila_hilo_prueba["tema"],
        anio_objetivo=fila_hilo_prueba[
            "anio_objetivo"
        ],
        id_hilo=fila_hilo_prueba["id_hilo"],
        titulo_hilo=fila_hilo_prueba[
            "titulo_resultado"
        ],
        url_hilo=fila_hilo_prueba["url_hilo"],
        pagina=1,
    )
)


print(
    "Comentarios:",
    len(comentarios_pagina_prueba_limpios),
)

print(
    "Comentarios con cita:",
    comentarios_pagina_prueba_limpios[
        "contiene_cita"
    ].sum(),
)

print(
    "Comentarios sin texto propio:",
    comentarios_pagina_prueba_limpios[
        "texto_propio_vacio"
    ].sum(),
)

Comentarios: 9
Comentarios con cita: 2
Comentarios sin texto propio: 0


In [34]:
display(
    comentarios_pagina_prueba_limpios.loc[
        comentarios_pagina_prueba_limpios[
            "contiene_cita"
        ],
        [
            "numero_mensaje",
            "texto_original",
            "texto_sin_citas",
            "texto_propio_vacio",
        ],
    ]
)

,numero_mensaje,texto_original,texto_sin_citas,texto_propio_vacio
5,6,Cita de\nlongboard\n..... Si evitas su llegada...,qué libertad?\nes que acaso tú puedes ir al pa...,False
7,8,Cita de\nBocasecaman69\nEsta claro que hay que...,"Es un dilema bastante grande shur, muy delicad...",False


In [35]:
# ============================================================
# CELDA 12 — DETECCIÓN DE PÁGINAS DE UN HILO
# ============================================================

def obtener_id_hilo_desde_url(url_hilo):
    """
    Extrae el parámetro t de una URL de ForoCoches.
    """
    
    parametros = parse_qs(
        urlparse(url_hilo).query
    )

    id_hilo = parametros.get(
        "t",
        [None],
    )[0]

    if id_hilo is None:
        return None

    return str(id_hilo)


def obtener_paginas_hilo(
    soup,
    url_hilo,
):
    """
    Obtiene los números de página presentes en la navegación.

    Siempre incluye la página 1.
    """
    
    id_hilo = obtener_id_hilo_desde_url(
        url_hilo
    )

    paginas = {1}

    enlaces = soup.select(
        'a[href*="showthread.php"]'
    )

    for enlace in enlaces:
        href = enlace.get(
            "href",
            "",
        )

        url_completa = urljoin(
            url_hilo,
            href,
        )

        parametros = parse_qs(
            urlparse(url_completa).query
        )

        id_enlace = parametros.get(
            "t",
            [None],
        )[0]

        numero_pagina = parametros.get(
            "page",
            [None],
        )[0]

        # Ignoramos enlaces pertenecientes a otros hilos.
        if str(id_enlace) != str(id_hilo):
            continue

        if (
            numero_pagina is not None
            and str(numero_pagina).isdigit()
        ):
            paginas.add(
                int(numero_pagina)
            )

    return sorted(paginas)

In [36]:
paginas_hilo_prueba = obtener_paginas_hilo(
    soup=soup_hilo_prueba,
    url_hilo=url_hilo_prueba,
)


print("Páginas detectadas:")
print(paginas_hilo_prueba)

print(
    "\nNúmero total de páginas:",
    len(paginas_hilo_prueba),
)

print(
    "Última página:",
    max(paginas_hilo_prueba),
)

Páginas detectadas:
[1]

Número total de páginas: 1
Última página: 1


In [37]:
def construir_url_pagina(
    url_hilo,
    numero_pagina,
):
    """
    Construye la URL de una página concreta conservando
    únicamente el identificador del hilo.
    """
    
    id_hilo = obtener_id_hilo_desde_url(
        url_hilo
    )

    if id_hilo is None:
        raise ValueError(
            f"No se encontró el ID del hilo: {url_hilo}"
        )

    url_base = (
        "https://forocoches.com/foro/"
        "showthread.php"
    )

    return (
        f"{url_base}?t={id_hilo}"
        f"&page={int(numero_pagina)}"
    )


urls_paginas_prueba = [
    construir_url_pagina(
        url_hilo_prueba,
        numero_pagina,
    )
    for numero_pagina in paginas_hilo_prueba
]


display(
    pd.DataFrame({
        "pagina": paginas_hilo_prueba,
        "url_pagina": urls_paginas_prueba,
    })
)

,pagina,url_pagina
0,1,https://forocoches.com/foro/showthread.php?t=4...


In [38]:
# ============================================================
# CELDA 13 — DESCARGA COMPLETA DE UN HILO
# ============================================================

def obtener_titulo_real_hilo(
    soup,
    titulo_respaldo="",
):
    """
    Intenta obtener el título directamente del HTML.
    """
    
    # En ForoCoches, la etiqueta keywords suele comenzar
    # con el título del hilo.
    meta_keywords = soup.select_one(
        'meta[name="keywords"]'
    )

    if meta_keywords is not None:
        contenido = meta_keywords.get(
            "content",
            "",
        ).strip()

        if contenido:
            titulo = re.split(
                r",\s*coches\s*,\s*motor\s*,\s*foro",
                contenido,
                maxsplit=1,
                flags=re.IGNORECASE,
            )[0].strip()

            if titulo:
                return titulo

    # Alternativa: encabezado principal.
    encabezado = soup.select_one("h1")

    if encabezado is not None:
        titulo = encabezado.get_text(
            " ",
            strip=True,
        )

        if titulo:
            return titulo

    return str(titulo_respaldo).strip()

In [39]:
def descargar_hilo_completo(
    fila_hilo,
    sesion,
    espera_segundos=1.5,
    mostrar_progreso=True,
):
    """
    Descarga y procesa todas las páginas de un hilo candidato.

    Devuelve:
    - DataFrame de comentarios;
    - diccionario de diagnóstico.
    """
    
    tema = fila_hilo["tema"]
    anio_objetivo = int(
        fila_hilo["anio_objetivo"]
    )
    id_hilo = str(
        fila_hilo["id_hilo"]
    )
    url_hilo = fila_hilo["url_hilo"]

    titulo_respaldo = fila_hilo.get(
        "titulo_resultado",
        "",
    )

    comentarios_paginas = []
    diagnostico_paginas = []


    # --------------------------------------------------------
    # Primera página
    # --------------------------------------------------------

    url_primera_pagina = construir_url_pagina(
        url_hilo,
        1,
    )

    respuesta = sesion.get(
        url_primera_pagina,
        timeout=40,
    )

    respuesta.raise_for_status()

    soup_primera_pagina = BeautifulSoup(
        respuesta.text,
        "html.parser",
    )

    titulo_real = obtener_titulo_real_hilo(
        soup_primera_pagina,
        titulo_respaldo=titulo_respaldo,
    )

    paginas_detectadas = obtener_paginas_hilo(
        soup=soup_primera_pagina,
        url_hilo=url_hilo,
    )

    ultima_pagina = max(
        paginas_detectadas
    )

    # Aunque la navegación muestre solamente algunas páginas,
    # recorremos todos los números hasta la última.
    numeros_paginas = list(
        range(1, ultima_pagina + 1)
    )

    if mostrar_progreso:
        print(
            f"Hilo {id_hilo}: "
            f"{ultima_pagina} página(s)"
        )


    # --------------------------------------------------------
    # Recorremos las páginas
    # --------------------------------------------------------

    for posicion, numero_pagina in enumerate(
        numeros_paginas,
        start=1,
    ):
        url_pagina = construir_url_pagina(
            url_hilo,
            numero_pagina,
        )

        # Reutilizamos la primera respuesta.
        if numero_pagina == 1:
            respuesta_pagina = respuesta
            soup_pagina = soup_primera_pagina

        else:
            time.sleep(
                espera_segundos
            )

            respuesta_pagina = sesion.get(
                url_pagina,
                timeout=40,
            )

            respuesta_pagina.raise_for_status()

            soup_pagina = BeautifulSoup(
                respuesta_pagina.text,
                "html.parser",
            )


        comentarios_pagina = (
            extraer_comentarios_pagina_limpios(
                soup=soup_pagina,
                tema=tema,
                anio_objetivo=anio_objetivo,
                id_hilo=id_hilo,
                titulo_hilo=titulo_real,
                url_hilo=url_hilo,
                pagina=numero_pagina,
            )
        )

        comentarios_paginas.append(
            comentarios_pagina
        )

        diagnostico_paginas.append({
            "id_hilo": id_hilo,
            "pagina": numero_pagina,
            "estado_http": (
                respuesta_pagina.status_code
            ),
            "comentarios_extraidos": len(
                comentarios_pagina
            ),
            "url_pagina": url_pagina,
        })

        if mostrar_progreso:
            print(
                f"  Página {posicion}/"
                f"{len(numeros_paginas)}: "
                f"{len(comentarios_pagina)} "
                "comentarios"
            )


    # --------------------------------------------------------
    # Consolidación
    # --------------------------------------------------------

    if comentarios_paginas:
        comentarios_hilo = pd.concat(
            comentarios_paginas,
            ignore_index=True,
        )

    else:
        comentarios_hilo = pd.DataFrame()

    if comentarios_hilo.empty:
        raise ValueError(
            f"No se extrajeron comentarios del hilo {id_hilo}"
        )

    numero_antes_deduplicar = len(
        comentarios_hilo
    )

    comentarios_hilo = (
        comentarios_hilo
        .drop_duplicates(
            subset=["id_mensaje"],
            keep="first",
        )
        .sort_values(
            [
                "pagina",
                "numero_mensaje",
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    duplicados_eliminados = (
        numero_antes_deduplicar
        - len(comentarios_hilo)
    )


    # --------------------------------------------------------
    # Validación del año real del hilo
    # --------------------------------------------------------

    mensajes_iniciales = comentarios_hilo.loc[
        comentarios_hilo[
            "es_mensaje_inicial"
        ]
    ]

    if mensajes_iniciales.empty:
        anio_real_hilo = None
        anio_validado = False

    else:
        fecha_inicial = mensajes_iniciales.iloc[
            0
        ]["fecha_comentario"]

        if pd.isna(fecha_inicial):
            anio_real_hilo = None
            anio_validado = False

        else:
            anio_real_hilo = int(
                fecha_inicial.year
            )

            anio_validado = (
                anio_real_hilo
                == anio_objetivo
            )


    diagnostico = {
        "tema": tema,
        "anio_objetivo": anio_objetivo,
        "anio_real_hilo": anio_real_hilo,
        "anio_validado": anio_validado,
        "id_hilo": id_hilo,
        "titulo_hilo": titulo_real,
        "url_hilo": url_hilo,
        "paginas": ultima_pagina,
        "comentarios": len(
            comentarios_hilo
        ),
        "duplicados_eliminados": (
            duplicados_eliminados
        ),
        "detalle_paginas": pd.DataFrame(
            diagnostico_paginas
        ),
    }

    return comentarios_hilo, diagnostico

In [40]:
comentarios_hilo_prueba, diagnostico_hilo_prueba = (
    descargar_hilo_completo(
        fila_hilo=fila_hilo_prueba,
        sesion=sesion,
        espera_segundos=1.5,
        mostrar_progreso=True,
    )
)


resumen_hilo_prueba = {
    clave: valor
    for clave, valor
    in diagnostico_hilo_prueba.items()
    if clave != "detalle_paginas"
}


display(
    pd.DataFrame(
        [resumen_hilo_prueba]
    )
)

display(
    diagnostico_hilo_prueba[
        "detalle_paginas"
    ]
)

Hilo 4730394: 1 página(s)
  Página 1/1: 9 comentarios


,tema,anio_objetivo,anio_real_hilo,anio_validado,id_hilo,titulo_hilo,url_hilo,paginas,comentarios,duplicados_eliminados
0,inmigracion,2016,2016,True,4730394,Un asunto delicado el tema INMIGRACIÓN y delin...,https://forocoches.com/foro/showthread.php?t=4...,1,9,0


,id_hilo,pagina,estado_http,comentarios_extraidos,url_pagina
0,4730394,1,200,9,https://forocoches.com/foro/showthread.php?t=4...


In [41]:
display(
    comentarios_hilo_prueba[
        [
            "numero_mensaje",
            "fecha_comentario",
            "usuario_hash",
            "contiene_cita",
            "texto_sin_citas",
        ]
    ]
)

,numero_mensaje,fecha_comentario,usuario_hash,contiene_cita,texto_sin_citas
0,1,2016-01-07 10:01:00,adf0a55d9756d742,False,"Es lo que están acostumbrados en sus países, s..."
1,2,2016-01-07 10:01:00,ed380416ab43785f,False,"Ya te lo dice esta pole, cada vecino en su par..."
2,3,2016-01-07 10:01:00,0b812141842fc387,False,Lo que hay que hacer es dar una patada en el c...
3,4,2016-01-07 10:01:00,f11f0d951c6afdb6,False,Menudo ladrillo a estas horas
4,5,2016-01-07 10:05:00,48ba2627a46ec5d1,False,La segunda y cuarta opción son lo mismo.
5,6,2016-01-07 10:06:00,a981d720abc6d45c,True,qué libertad?\nes que acaso tú puedes ir al pa...
6,7,2016-01-07 10:06:00,1ebeec4ca7fecd0c,False,Esta claro que hay que endurecer las leyes. Pe...
7,8,2016-01-08 20:21:00,adf0a55d9756d742,True,"Es un dilema bastante grande shur, muy delicad..."
8,9,2016-01-08 20:42:00,8df124bf02253be3,False,Que se vayan todos y de paso de lleven a los g...


In [42]:
# ============================================================
# CELDA 14 — DESCARGA DEL CATÁLOGO INICIAL
# ============================================================

comentarios_catalogo = [
    comentarios_hilo_prueba.copy()
]

diagnosticos_catalogo = [
    {
        clave: valor
        for clave, valor
        in diagnostico_hilo_prueba.items()
        if clave != "detalle_paginas"
    }
]

errores_catalogo = []


id_hilo_ya_procesado = str(
    fila_hilo_prueba["id_hilo"]
)


for posicion, (_, fila_hilo) in enumerate(
    hilos_candidatos.iterrows(),
    start=1,
):
    id_hilo_actual = str(
        fila_hilo["id_hilo"]
    )

    # Ya descargamos este hilo en la celda anterior.
    if id_hilo_actual == id_hilo_ya_procesado:
        print(
            f"\n[{posicion}/{len(hilos_candidatos)}] "
            f"Hilo {id_hilo_actual}: ya procesado"
        )
        continue

    print("\n" + "=" * 70)

    print(
        f"[{posicion}/{len(hilos_candidatos)}] "
        f"{fila_hilo['tema']} | "
        f"{fila_hilo['anio_objetivo']} | "
        f"hilo {id_hilo_actual}"
    )

    print(
        fila_hilo["titulo_resultado"]
    )

    try:
        comentarios_hilo, diagnostico_hilo = (
            descargar_hilo_completo(
                fila_hilo=fila_hilo,
                sesion=sesion,
                espera_segundos=1.5,
                mostrar_progreso=True,
            )
        )

        comentarios_catalogo.append(
            comentarios_hilo
        )

        diagnosticos_catalogo.append({
            clave: valor
            for clave, valor
            in diagnostico_hilo.items()
            if clave != "detalle_paginas"
        })

        print(
            "Resultado:",
            f"{len(comentarios_hilo)} comentarios",
        )

        print(
            "Año validado:",
            diagnostico_hilo["anio_validado"],
        )

    except Exception as error:
        print(
            "ERROR:",
            type(error).__name__,
            str(error),
        )

        errores_catalogo.append({
            "tema": fila_hilo["tema"],
            "anio_objetivo": int(
                fila_hilo["anio_objetivo"]
            ),
            "id_hilo": id_hilo_actual,
            "url_hilo": fila_hilo["url_hilo"],
            "tipo_error": type(error).__name__,
            "mensaje_error": str(error),
        })

    # Pausa adicional entre hilos.
    time.sleep(2)


[1/8] inmigracion | 2015 | hilo 4514261
El problema que hemos creado
Hilo 4514261: 5 página(s)
  Página 1/5: 30 comentarios
  Página 2/5: 30 comentarios
  Página 3/5: 30 comentarios
  Página 4/5: 30 comentarios
  Página 5/5: 4 comentarios
Resultado: 124 comentarios
Año validado: True

[2/8] Hilo 4730394: ya procesado

[3/8] inmigracion | 2019 | hilo 7092240
ALERTA anti-INMIGRACIÓN ILEGAL +TemaSerio
Hilo 7092240: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 30 comentarios
  Página 3/3: 2 comentarios
Resultado: 62 comentarios
Año validado: True

[4/8] inmigracion | 2023 | hilo 9541304
Estoy a favor de la inmigración ilegal
Hilo 9541304: 11 página(s)
  Página 1/11: 30 comentarios
  Página 2/11: 30 comentarios
  Página 3/11: 30 comentarios
  Página 4/11: 30 comentarios
  Página 5/11: 30 comentarios
  Página 6/11: 30 comentarios
  Página 7/11: 30 comentarios
  Página 8/11: 30 comentarios
  Página 9/11: 30 comentarios
  Página 10/11: 30 comentarios
  Página 11/11: 2 comentarios
Re

In [43]:
# ------------------------------------------------------------
# Consolidación provisional
# ------------------------------------------------------------

if comentarios_catalogo:
    comentarios_forocoches_inicial = (
        pd.concat(
            comentarios_catalogo,
            ignore_index=True,
        )
        .drop_duplicates(
            subset=["id_mensaje"],
            keep="first",
        )
        .reset_index(drop=True)
    )

else:
    comentarios_forocoches_inicial = pd.DataFrame()


diagnosticos_forocoches_inicial = pd.DataFrame(
    diagnosticos_catalogo
)

errores_forocoches_inicial = pd.DataFrame(
    errores_catalogo
)


print(
    "Comentarios únicos descargados:",
    len(comentarios_forocoches_inicial),
)

print(
    "Hilos procesados:",
    len(diagnosticos_forocoches_inicial),
)

print(
    "Hilos con error:",
    len(errores_forocoches_inicial),
)

Comentarios únicos descargados: 805
Hilos procesados: 7
Hilos con error: 1


In [44]:
columnas_diagnostico = [
    "tema",
    "anio_objetivo",
    "anio_real_hilo",
    "anio_validado",
    "id_hilo",
    "paginas",
    "comentarios",
    "duplicados_eliminados",
    "titulo_hilo",
]

display(
    diagnosticos_forocoches_inicial[
        columnas_diagnostico
    ]
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

,tema,anio_objetivo,anio_real_hilo,anio_validado,id_hilo,paginas,comentarios,duplicados_eliminados,titulo_hilo
0,inmigracion,2015,2015,True,4514261,5,124,0,El problema que hemos creado
1,inmigracion,2016,2016,True,4730394,1,9,0,Un asunto delicado el tema INMIGRACIÓN y delin...
2,inmigracion,2019,2019,True,7092240,3,62,0,🔊 ALERTA anti-INMIGRACIÓN ILEGAL ❗+TemaSerio
3,inmigracion,2023,2023,True,9541304,11,302,0,Estoy a favor de la inmigración ilegal
4,lgtbi,2016,2016,True,4932029,4,99,0,Árbitro deja el arbitraje por recibir continuo...
5,lgtbi,2019,2019,True,7306066,7,188,0,A favor o en contra de que se hagan cursos LGT...
6,lgtbi,2023,2023,True,9472060,1,21,0,Las empresas con más de 50 empleados tienen un...


In [45]:
cobertura_comentarios_inicial = (
    comentarios_forocoches_inicial
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .agg(
        hilos=("id_hilo", "nunique"),
        comentarios=("id_mensaje", "nunique"),
        usuarios=("usuario_hash", "nunique"),
        fecha_minima=(
            "fecha_comentario",
            "min",
        ),
        fecha_maxima=(
            "fecha_comentario",
            "max",
        ),
    )
    .reset_index()
)


display(
    cobertura_comentarios_inicial
)

,tema,anio_objetivo,hilos,comentarios,usuarios,fecha_minima,fecha_maxima
0,inmigracion,2015,1,124,68,2015-09-07 20:38:00,2016-07-19 12:11:00
1,inmigracion,2016,1,9,8,2016-01-07 10:01:00,2016-01-08 20:42:00
2,inmigracion,2019,1,62,41,2019-04-03 01:17:00,2019-04-08 13:52:00
3,inmigracion,2023,1,302,207,2023-05-12 23:26:00,2023-05-14 23:59:00
4,lgtbi,2016,1,99,80,2016-05-11 19:26:00,2016-05-12 20:43:00
5,lgtbi,2019,1,188,150,2019-07-15 10:05:00,2020-07-01 20:18:00
6,lgtbi,2023,1,21,19,2023-03-10 20:29:00,2023-03-10 20:47:00


In [46]:
if errores_forocoches_inicial.empty:
    print(
        "No se produjeron errores."
    )
else:
    display(
        errores_forocoches_inicial
    )

,tema,anio_objetivo,id_hilo,url_hilo,tipo_error,mensaje_error
0,lgtbi,2015,4407356,https://forocoches.com/foro/showthread.php?t=4...,KeyError,'id_mensaje'


In [47]:
# ============================================================
# CELDA 15 — DIAGNÓSTICO DEL HILO LGTBI DE 2015
# ============================================================

fila_lgtbi_2015 = (
    hilos_candidatos.loc[
        (hilos_candidatos["tema"] == "lgtbi")
        & (hilos_candidatos["anio_objetivo"] == 2015)
    ]
    .iloc[0]
)


url_lgtbi_2015 = fila_lgtbi_2015[
    "url_hilo"
]

print("ID del hilo:")
print(fila_lgtbi_2015["id_hilo"])

print("\nTítulo candidato:")
print(fila_lgtbi_2015["titulo_resultado"])

print("\nURL:")
print(url_lgtbi_2015)

ID del hilo:
4407356

Título candidato:
Mañana voy a Chueca

URL:
https://forocoches.com/foro/showthread.php?t=4407356


In [48]:
respuesta_lgtbi_2015 = sesion.get(
    url_lgtbi_2015,
    timeout=40,
)


print("Estado HTTP:")
print(respuesta_lgtbi_2015.status_code)

print("\nURL final:")
print(respuesta_lgtbi_2015.url)

print("\nTamaño:")
print(
    f"{len(respuesta_lgtbi_2015.text):,} caracteres"
)

print("\nApariciones de 'post_message_' en el HTML:")
print(
    respuesta_lgtbi_2015.text.count(
        "post_message_"
    )
)

Estado HTTP:
200

URL final:
https://forocoches.com/foro/misc.php?do=page&template=Info&tid=4407356

Tamaño:
109,551 caracteres

Apariciones de 'post_message_' en el HTML:
0


In [50]:
respuesta_lgtbi_2015.raise_for_status()

soup_lgtbi_2015 = BeautifulSoup(
    respuesta_lgtbi_2015.text,
    "html.parser",
)


selectores_diagnostico = {
    "div_post_message": (
        'div[id^="post_message_"]'
    ),
    "cualquier_post_message": (
        '[id^="post_message_"]'
    ),
    "postbit_wrapper": (
        ".postbit_wrapper"
    ),
    "posts": (
        "#posts"
    ),
    "article": (
        "article"
    ),
}


resumen_estructura_2015 = []

for nombre, selector in (
    selectores_diagnostico.items()
):
    nodos = soup_lgtbi_2015.select(
        selector
    )

    resumen_estructura_2015.append({
        "estructura": nombre,
        "selector": selector,
        "elementos": len(nodos),
    })


display(
    pd.DataFrame(
        resumen_estructura_2015
    )
)

,estructura,selector,elementos
0,div_post_message,"div[id^=""post_message_""]",0
1,cualquier_post_message,"[id^=""post_message_""]",0
2,postbit_wrapper,.postbit_wrapper,0
3,posts,#posts,0
4,article,article,0


In [51]:
texto_lgtbi_2015 = (
    soup_lgtbi_2015
    .get_text(
        " ",
        strip=True,
    )
)

texto_lgtbi_2015_minusculas = (
    texto_lgtbi_2015.lower()
)


avisos_posibles = [
    "no tienes permisos",
    "tema especificado no existe",
    "tema ha sido borrado",
    "debes estar registrado",
    "iniciar sesión",
    "registrarse",
    "ningún tema especificado",
]


avisos_detectados = [
    aviso
    for aviso in avisos_posibles
    if aviso in texto_lgtbi_2015_minusculas
]


print("Avisos detectados:")
print(avisos_detectados)

print("\nPrimeros 1.000 caracteres visibles:")
print(texto_lgtbi_2015[:1000])

Avisos detectados:
['iniciar sesión', 'registrarse']

Primeros 1.000 caracteres visibles:
Forocoches - Información Registrarse Iniciar sesión Tus notificaciones Mensajes privados Menciones Citas Invitaciones ¿No tienes invitaciones? Descubre como conseguirlas. Modo día Modo noche Modo  automático Dar feedback del nuevo diseño Ver mensajes propios Suscribirse Información EL TEMA QUE NECESITAS VER ESTÁ DISPONIBLE PARA USUARIOS REGISTRADOS CON INVITACIÓN Mañana voy a Chueca +Hd [ Autor: ManMatao ] [ 32 Mensajes ] [ 1.582 Visualizaciones ] [ Formas de conseguir una Invitación para ForoCoches ] [ Iniciar Sesión ] ⌃ Legal Privacidad Cookies Config Cookies ⚙ Contacto


In [52]:
titulo_html_2015 = soup_lgtbi_2015.select_one(
    "title"
)

if titulo_html_2015 is not None:
    print("Título de la página:")
    print(
        titulo_html_2015.get_text(
            " ",
            strip=True,
        )
    )
else:
    print(
        "La página no contiene una etiqueta title."
    )

Título de la página:
Forocoches - Información


In [53]:
# ============================================================
# CELDA 16 — SUSTITUCIÓN DEL HILO LGTBI 2015
# ============================================================

mascara_lgtbi_2015 = (
    (hilos_candidatos["tema"] == "lgtbi")
    & (
        hilos_candidatos["anio_objetivo"]
        == 2015
    )
)


hilos_candidatos.loc[
    mascara_lgtbi_2015,
    "id_hilo",
] = "4677567"

hilos_candidatos.loc[
    mascara_lgtbi_2015,
    "titulo_resultado",
] = (
    "Un hombre de 52 años se transforma "
    "en una niña de seis"
)

hilos_candidatos.loc[
    mascara_lgtbi_2015,
    "url_hilo",
] = (
    "https://forocoches.com/foro/"
    "showthread.php?t=4677567"
)

hilos_candidatos.loc[
    mascara_lgtbi_2015,
    "fuente_descubrimiento",
] = "buscador_externo"


fila_lgtbi_2015_nueva = (
    hilos_candidatos.loc[
        mascara_lgtbi_2015
    ]
    .iloc[0]
)


display(
    fila_lgtbi_2015_nueva.to_frame(
        name="valor"
    )
)

,valor
tema,lgtbi
anio_objetivo,2015
id_hilo,4677567
titulo_resultado,Un hombre de 52 años se transforma en una niña...
url_hilo,https://forocoches.com/foro/showthread.php?t=4...
fuente_descubrimiento,buscador_externo


In [56]:
comentarios_lgtbi_2015, diagnostico_lgtbi_2015 = (
    descargar_hilo_completo(
        fila_hilo=fila_lgtbi_2015_nueva,
        sesion=sesion,
        espera_segundos=1.5,
        mostrar_progreso=True,
    )
)

Hilo 4677567: 21 página(s)
  Página 1/21: 30 comentarios
  Página 2/21: 30 comentarios
  Página 3/21: 30 comentarios
  Página 4/21: 30 comentarios
  Página 5/21: 30 comentarios
  Página 6/21: 30 comentarios
  Página 7/21: 30 comentarios
  Página 8/21: 30 comentarios
  Página 9/21: 30 comentarios
  Página 10/21: 30 comentarios
  Página 11/21: 30 comentarios
  Página 12/21: 30 comentarios
  Página 13/21: 30 comentarios
  Página 14/21: 30 comentarios
  Página 15/21: 30 comentarios
  Página 16/21: 30 comentarios
  Página 17/21: 30 comentarios
  Página 18/21: 30 comentarios
  Página 19/21: 30 comentarios
  Página 20/21: 30 comentarios
  Página 21/21: 4 comentarios


In [57]:
resumen_lgtbi_2015 = {
    clave: valor
    for clave, valor
    in diagnostico_lgtbi_2015.items()
    if clave != "detalle_paginas"
}


display(
    pd.DataFrame(
        [resumen_lgtbi_2015]
    )
)


display(
    diagnostico_lgtbi_2015[
        "detalle_paginas"
    ]
)

,tema,anio_objetivo,anio_real_hilo,anio_validado,id_hilo,titulo_hilo,url_hilo,paginas,comentarios,duplicados_eliminados
0,lgtbi,2015,2015,True,4677567,Un hombre de 52 años se transforma en una niña...,https://forocoches.com/foro/showthread.php?t=4...,21,604,0


,id_hilo,pagina,estado_http,comentarios_extraidos,url_pagina
0,4677567,1,200,30,https://forocoches.com/foro/showthread.php?t=4...
1,4677567,2,200,30,https://forocoches.com/foro/showthread.php?t=4...
2,4677567,3,200,30,https://forocoches.com/foro/showthread.php?t=4...
3,4677567,4,200,30,https://forocoches.com/foro/showthread.php?t=4...
4,4677567,5,200,30,https://forocoches.com/foro/showthread.php?t=4...
5,4677567,6,200,30,https://forocoches.com/foro/showthread.php?t=4...
6,4677567,7,200,30,https://forocoches.com/foro/showthread.php?t=4...
7,4677567,8,200,30,https://forocoches.com/foro/showthread.php?t=4...
8,4677567,9,200,30,https://forocoches.com/foro/showthread.php?t=4...
9,4677567,10,200,30,https://forocoches.com/foro/showthread.php?t=4...


In [58]:
if diagnostico_lgtbi_2015["anio_validado"]:

    comentarios_catalogo.append(
        comentarios_lgtbi_2015
    )

    diagnosticos_catalogo.append(
        resumen_lgtbi_2015
    )

    print(
        "Hilo LGTBI 2015 incorporado correctamente."
    )

else:
    print(
        "El hilo no se incorporó porque su año "
        "real no coincide con 2015."
    )

Hilo LGTBI 2015 incorporado correctamente.


In [59]:
comentarios_forocoches_inicial = (
    pd.concat(
        comentarios_catalogo,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


diagnosticos_forocoches_inicial = (
    pd.DataFrame(
        diagnosticos_catalogo
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

In [60]:
cobertura_actualizada = (
    comentarios_forocoches_inicial
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .agg(
        hilos=("id_hilo", "nunique"),
        comentarios=(
            "id_mensaje",
            "nunique",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
        fecha_minima=(
            "fecha_comentario",
            "min",
        ),
        fecha_maxima=(
            "fecha_comentario",
            "max",
        ),
    )
    .reset_index()
)


print(
    "Combinaciones presentes:",
    len(cobertura_actualizada),
)

display(
    cobertura_actualizada
)

Combinaciones presentes: 8


,tema,anio_objetivo,hilos,comentarios,usuarios,fecha_minima,fecha_maxima
0,inmigracion,2015,1,124,68,2015-09-07 20:38:00,2016-07-19 12:11:00
1,inmigracion,2016,1,9,8,2016-01-07 10:01:00,2016-01-08 20:42:00
2,inmigracion,2019,1,62,41,2019-04-03 01:17:00,2019-04-08 13:52:00
3,inmigracion,2023,1,302,207,2023-05-12 23:26:00,2023-05-14 23:59:00
4,lgtbi,2015,1,604,531,2015-12-11 16:00:00,2016-02-10 01:05:00
5,lgtbi,2016,1,99,80,2016-05-11 19:26:00,2016-05-12 20:43:00
6,lgtbi,2019,1,188,150,2019-07-15 10:05:00,2020-07-01 20:18:00
7,lgtbi,2023,1,21,19,2023-03-10 20:29:00,2023-03-10 20:47:00


In [61]:
# ============================================================
# CELDA 17 — FILTRO POR AÑO DEL COMENTARIO
# ============================================================

# Conservamos una copia completa como conjunto bruto.
comentarios_forocoches_bruto = (
    comentarios_forocoches_inicial.copy()
)


# ------------------------------------------------------------
# Comprobamos si cada comentario pertenece al año objetivo
# ------------------------------------------------------------

comentarios_forocoches_bruto[
    "pertenece_anio_objetivo"
] = (
    comentarios_forocoches_bruto[
        "anio_comentario"
    ]
    == comentarios_forocoches_bruto[
        "anio_objetivo"
    ]
)


# ------------------------------------------------------------
# Conjunto analítico anual
# ------------------------------------------------------------

comentarios_forocoches_anual = (
    comentarios_forocoches_bruto.loc[
        comentarios_forocoches_bruto[
            "pertenece_anio_objetivo"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Comentarios brutos:",
    len(comentarios_forocoches_bruto),
)

print(
    "Comentarios del año objetivo:",
    len(comentarios_forocoches_anual),
)

print(
    "Comentarios posteriores excluidos:",
    (
        len(comentarios_forocoches_bruto)
        - len(comentarios_forocoches_anual)
    ),
)

Comentarios brutos: 1409
Comentarios del año objetivo: 1349
Comentarios posteriores excluidos: 60


In [62]:
comprobacion_anios = (
    comentarios_forocoches_anual[
        "anio_comentario"
    ]
    == comentarios_forocoches_anual[
        "anio_objetivo"
    ]
).all()


print(
    "¿Todos los comentarios pertenecen "
    "al año objetivo?",
    comprobacion_anios,
)


display(
    comentarios_forocoches_anual
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .size()
    .rename("comentarios_anuales")
    .reset_index()
)

¿Todos los comentarios pertenecen al año objetivo? True


,tema,anio_objetivo,comentarios_anuales
0,inmigracion,2015,77
1,inmigracion,2016,9
2,inmigracion,2019,62
3,inmigracion,2023,302
4,lgtbi,2015,602
5,lgtbi,2016,99
6,lgtbi,2019,177
7,lgtbi,2023,21


## Ampliar el catálogo

In [63]:
# ============================================================
# CELDA 18 — AMPLIACIÓN DEL CATÁLOGO DE HILOS
# ============================================================

nuevos_hilos_candidatos = pd.DataFrame([
    # --------------------------------------------------------
    # INMIGRACIÓN — 2015
    # --------------------------------------------------------
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4508934",
        "titulo_resultado": (
            "Reflexión sobre inmigración y política "
            "(TEMA SERIO)"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4508934"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4343322",
        "titulo_resultado": (
            "Récord de saltos en la valla de "
            "Ceuta y Melilla en 2014"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4343322"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # INMIGRACIÓN — 2016
    # --------------------------------------------------------
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4840201",
        "titulo_resultado": (
            "Refugees vienen, ven que España está "
            "hecho mierdas y se van desengañados"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4840201"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4930389",
        "titulo_resultado": (
            "Según El Jueves Europa se llena "
            "de peligrosos nancys"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4930389"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # INMIGRACIÓN — 2019
    # --------------------------------------------------------
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7602070",
        "titulo_resultado": (
            "Los extranjeros aportan el 10 % a la "
            "Seguridad Social y solo reciben el 0,9 %"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7602070"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7281127",
        "titulo_resultado": (
            "Descubierta la mentira de las pateras"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7281127"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # INMIGRACIÓN — 2023
    # --------------------------------------------------------
    {
        "tema": "inmigracion",
        "anio_objetivo": 2023,
        "id_hilo": "9572133",
        "titulo_resultado": (
            "Uno de los problemas del precio de la "
            "vivienda es la inmigración"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=9572133"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2023,
        "id_hilo": "9735179",
        "titulo_resultado": "Duda inmigración ilegal",
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=9735179"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # LGTBI — 2015
    # --------------------------------------------------------
    {
        "tema": "lgtbi",
        "anio_objetivo": 2015,
        "id_hilo": "4323181",
        "titulo_resultado": (
            "It's a Trap: cuando ella es él "
            "y él es ella"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4323181"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # LGTBI — 2016
    # --------------------------------------------------------
    {
        "tema": "lgtbi",
        "anio_objetivo": 2016,
        "id_hilo": "4900779",
        "titulo_resultado": (
            "La que me han liado por decir que los "
            "mariquitas no son normales"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4900779"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2016,
        "id_hilo": "4784732",
        "titulo_resultado": (
            "Nueva agresión homófoba en la capital"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4784732"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # LGTBI — 2019
    # --------------------------------------------------------
    {
        "tema": "lgtbi",
        "anio_objetivo": 2019,
        "id_hilo": "7293798",
        "titulo_resultado": (
            "¿VOX es un partido homófobo?"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7293798"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2019,
        "id_hilo": "7268857",
        "titulo_resultado": (
            "La obsesión homófoba de VOX "
            "empieza a ser enfermiza"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7268857"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },

    # --------------------------------------------------------
    # LGTBI — 2023
    # --------------------------------------------------------
    {
        "tema": "lgtbi",
        "anio_objetivo": 2023,
        "id_hilo": "9677912",
        "titulo_resultado": (
            "Podemos quiere que Irene Montero siga "
            "como ministra de Igualdad"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=9677912"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
    {
        "tema": "lgtbi",
        "anio_objetivo": 2023,
        "id_hilo": "9763018",
        "titulo_resultado": (
            "Ángela Pam: recopilación de políticas "
            "de Igualdad y avances LGTBI"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=9763018"
        ),
        "fuente_descubrimiento": "buscador_externo",
    },
])

### Se incorpora a los anteriores

In [64]:
hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevos_hilos_candidatos,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )
    .reset_index(drop=True)
)


cobertura_catalogo_ampliado = (
    hilos_candidatos
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .size()
    .rename("hilos_candidatos")
    .reset_index()
)


print(
    "Total de hilos candidatos:",
    len(hilos_candidatos),
)

display(
    cobertura_catalogo_ampliado
)

Total de hilos candidatos: 23


,tema,anio_objetivo,hilos_candidatos
0,inmigracion,2015,3
1,inmigracion,2016,3
2,inmigracion,2019,3
3,inmigracion,2023,3
4,lgtbi,2015,2
5,lgtbi,2016,3
6,lgtbi,2019,3
7,lgtbi,2023,3


## Procesamos los candidatos nuevos

In [65]:
# ============================================================
# CELDA 19 — PROCESAMIENTO DE LOS NUEVOS CANDIDATOS
# ============================================================

def comprobar_primera_pagina_hilo(
    fila_hilo,
    sesion,
):
    """
    Comprueba que la primera página contiene mensajes
    reconocibles antes de ejecutar el parser completo.
    """
    
    url_hilo = fila_hilo["url_hilo"]

    respuesta = sesion.get(
        construir_url_pagina(
            url_hilo,
            1,
        ),
        timeout=40,
    )

    respuesta.raise_for_status()

    soup = BeautifulSoup(
        respuesta.text,
        "html.parser",
    )

    mensajes = soup.select(
        'div[id^="post_message_"]'
    )

    titulo_nodo = soup.select_one(
        "title"
    )

    titulo_pagina = (
        titulo_nodo.get_text(
            " ",
            strip=True,
        )
        if titulo_nodo is not None
        else ""
    )

    return {
        "es_publico": len(mensajes) > 0,
        "numero_mensajes_primera_pagina": (
            len(mensajes)
        ),
        "estado_http": respuesta.status_code,
        "titulo_pagina": titulo_pagina,
        "url_final": respuesta.url,
    }

In [66]:
# IDs que ya tenemos descargados.
ids_ya_procesados = set(
    comentarios_forocoches_bruto[
        "id_hilo"
    ].astype(str)
)


comentarios_ampliacion = []
diagnosticos_ampliacion = []
incidencias_ampliacion = []


nuevos_pendientes = (
    nuevos_hilos_candidatos.loc[
        ~nuevos_hilos_candidatos[
            "id_hilo"
        ].astype(str).isin(
            ids_ya_procesados
        )
    ]
    .reset_index(drop=True)
)


print(
    "Candidatos nuevos pendientes:",
    len(nuevos_pendientes),
)

Candidatos nuevos pendientes: 15


In [67]:
for posicion, (_, fila_hilo) in enumerate(
    nuevos_pendientes.iterrows(),
    start=1,
):
    id_hilo = str(
        fila_hilo["id_hilo"]
    )

    print("\n" + "=" * 70)

    print(
        f"[{posicion}/{len(nuevos_pendientes)}] "
        f"{fila_hilo['tema']} | "
        f"{fila_hilo['anio_objetivo']} | "
        f"hilo {id_hilo}"
    )

    print(
        fila_hilo["titulo_resultado"]
    )

    try:
        # ----------------------------------------------------
        # Comprobación previa
        # ----------------------------------------------------

        comprobacion = (
            comprobar_primera_pagina_hilo(
                fila_hilo=fila_hilo,
                sesion=sesion,
            )
        )

        if not comprobacion["es_publico"]:
            incidencias_ampliacion.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "estado": "no_publico",
                "anio_real_hilo": None,
                "estado_http": comprobacion[
                    "estado_http"
                ],
                "titulo_pagina": comprobacion[
                    "titulo_pagina"
                ],
                "detalle": (
                    "La primera página no contiene "
                    "mensajes reconocibles"
                ),
            })

            print(
                "Descartado: el hilo no contiene "
                "mensajes públicos."
            )

            time.sleep(2)
            continue


        # ----------------------------------------------------
        # Descarga completa
        # ----------------------------------------------------

        comentarios_hilo, diagnostico_hilo = (
            descargar_hilo_completo(
                fila_hilo=fila_hilo,
                sesion=sesion,
                espera_segundos=1.5,
                mostrar_progreso=True,
            )
        )


        # ----------------------------------------------------
        # Validación del año
        # ----------------------------------------------------

        if not diagnostico_hilo[
            "anio_validado"
        ]:
            incidencias_ampliacion.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "estado": "anio_no_coincide",
                "anio_real_hilo": diagnostico_hilo[
                    "anio_real_hilo"
                ],
                "estado_http": 200,
                "titulo_pagina": diagnostico_hilo[
                    "titulo_hilo"
                ],
                "detalle": (
                    "El año del mensaje #1 no coincide "
                    "con el año objetivo"
                ),
            })

            print(
                "Descartado por año:",
                diagnostico_hilo[
                    "anio_real_hilo"
                ],
            )

            time.sleep(2)
            continue


        # ----------------------------------------------------
        # Incorporación en memoria
        # ----------------------------------------------------

        comentarios_ampliacion.append(
            comentarios_hilo
        )

        diagnosticos_ampliacion.append({
            clave: valor
            for clave, valor
            in diagnostico_hilo.items()
            if clave != "detalle_paginas"
        })

        print(
            "Incorporado:",
            len(comentarios_hilo),
            "comentarios",
        )

    except Exception as error:
        incidencias_ampliacion.append({
            "tema": fila_hilo["tema"],
            "anio_objetivo": int(
                fila_hilo["anio_objetivo"]
            ),
            "id_hilo": id_hilo,
            "estado": "error",
            "anio_real_hilo": None,
            "estado_http": None,
            "titulo_pagina": "",
            "detalle": (
                f"{type(error).__name__}: {error}"
            ),
        })

        print(
            "ERROR:",
            type(error).__name__,
            str(error),
        )

    time.sleep(2)


[1/15] inmigracion | 2015 | hilo 4508934
Reflexión sobre inmigración y política (TEMA SERIO)
Hilo 4508934: 1 página(s)
  Página 1/1: 5 comentarios
Incorporado: 5 comentarios

[2/15] inmigracion | 2015 | hilo 4343322
Récord de saltos en la valla de Ceuta y Melilla en 2014
Hilo 4343322: 1 página(s)
  Página 1/1: 2 comentarios
Incorporado: 2 comentarios

[3/15] inmigracion | 2016 | hilo 4840201
Refugees vienen, ven que España está hecho mierdas y se van desengañados
Hilo 4840201: 7 página(s)
  Página 1/7: 30 comentarios
  Página 2/7: 30 comentarios
  Página 3/7: 30 comentarios
  Página 4/7: 30 comentarios
  Página 5/7: 30 comentarios
  Página 6/7: 30 comentarios
  Página 7/7: 3 comentarios
Incorporado: 183 comentarios

[4/15] inmigracion | 2016 | hilo 4930389
Según El Jueves Europa se llena de peligrosos nancys
Hilo 4930389: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 22 comentarios
Incorporado: 52 comentarios

[5/15] inmigracion | 2019 | hilo 7602070
Los extranjeros aportan e

In [68]:
# ------------------------------------------------------------
# Consolidación ampliada
# ------------------------------------------------------------

partes_comentarios = [
    comentarios_forocoches_bruto.drop(
        columns=["pertenece_anio_objetivo"],
        errors="ignore",
    )
]

partes_comentarios.extend(
    comentarios_ampliacion
)


comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        partes_comentarios,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


partes_diagnosticos = [
    diagnosticos_forocoches_inicial
]

if diagnosticos_ampliacion:
    partes_diagnosticos.append(
        pd.DataFrame(
            diagnosticos_ampliacion
        )
    )


diagnosticos_forocoches_ampliado = (
    pd.concat(
        partes_diagnosticos,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)


incidencias_forocoches_ampliacion = (
    pd.DataFrame(
        incidencias_ampliacion
    )
)

In [69]:
comentarios_forocoches_bruto_ampliado[
    "pertenece_anio_objetivo"
] = (
    comentarios_forocoches_bruto_ampliado[
        "anio_comentario"
    ]
    == comentarios_forocoches_bruto_ampliado[
        "anio_objetivo"
    ]
)


comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "pertenece_anio_objetivo"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

In [70]:
cobertura_ampliada = (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .agg(
        hilos=("id_hilo", "nunique"),
        comentarios=(
            "id_mensaje",
            "nunique",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
    )
    .reset_index()
)


print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)

display(
    cobertura_ampliada
)

Hilos válidos totales: 23
Comentarios anuales totales: 2608


,tema,anio_objetivo,hilos,comentarios,usuarios
0,inmigracion,2015,3,84,47
1,inmigracion,2016,3,244,189
2,inmigracion,2019,3,408,352
3,inmigracion,2023,3,368,251
4,lgtbi,2015,2,644,566
5,lgtbi,2016,3,183,141
6,lgtbi,2019,3,485,362
7,lgtbi,2023,3,192,166


In [71]:
if incidencias_forocoches_ampliacion.empty:
    print(
        "Todos los nuevos candidatos se incorporaron."
    )
else:
    display(
        incidencias_forocoches_ampliacion
        .sort_values(
            [
                "tema",
                "anio_objetivo",
                "id_hilo",
            ]
        )
        .reset_index(drop=True)
    )

Todos los nuevos candidatos se incorporaron.


In [72]:
# ============================================================
# CELDA 20 — CONTROL DE CALIDAD
# ============================================================

df_calidad = (
    comentarios_forocoches_anual_ampliado
    .copy()
)


# ------------------------------------------------------------
# Longitudes del texto
# ------------------------------------------------------------

df_calidad["texto_sin_citas"] = (
    df_calidad["texto_sin_citas"]
    .fillna("")
    .astype(str)
)


df_calidad["longitud_caracteres"] = (
    df_calidad["texto_sin_citas"]
    .str.strip()
    .str.len()
)


df_calidad["numero_palabras"] = (
    df_calidad["texto_sin_citas"]
    .str.findall(r"\b\w+\b")
    .str.len()
)

In [73]:
control_general = pd.DataFrame([
    {
        "comprobacion": "Filas totales",
        "resultado": len(df_calidad),
    },
    {
        "comprobacion": "IDs de mensaje únicos",
        "resultado": df_calidad[
            "id_mensaje"
        ].nunique(),
    },
    {
        "comprobacion": "IDs de mensaje duplicados",
        "resultado": df_calidad[
            "id_mensaje"
        ].duplicated().sum(),
    },
    {
        "comprobacion": "Fechas ausentes",
        "resultado": df_calidad[
            "fecha_comentario"
        ].isna().sum(),
    },
    {
        "comprobacion": "Usuarios hash ausentes",
        "resultado": df_calidad[
            "usuario_hash"
        ].isna().sum(),
    },
    {
        "comprobacion": "Textos originales vacíos",
        "resultado": (
            df_calidad["texto_original"]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        ),
    },
    {
        "comprobacion": "Textos sin citas vacíos",
        "resultado": (
            df_calidad["longitud_caracteres"]
            == 0
        ).sum(),
    },
    {
        "comprobacion": "Comentarios con cita",
        "resultado": df_calidad[
            "contiene_cita"
        ].sum(),
    },
    {
        "comprobacion": "Años incorrectos",
        "resultado": (
            df_calidad["anio_comentario"]
            != df_calidad["anio_objetivo"]
        ).sum(),
    },
])


display(control_general)

,comprobacion,resultado
0,Filas totales,2608
1,IDs de mensaje únicos,2608
2,IDs de mensaje duplicados,0
3,Fechas ausentes,0
4,Usuarios hash ausentes,7
5,Textos originales vacíos,99
6,Textos sin citas vacíos,143
7,Comentarios con cita,952
8,Años incorrectos,0


In [74]:
mensajes_iniciales_por_hilo = (
    df_calidad
    .groupby(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )["es_mensaje_inicial"]
    .sum()
    .rename("mensajes_iniciales")
    .reset_index()
)


hilos_con_problema_inicial = (
    mensajes_iniciales_por_hilo.loc[
        mensajes_iniciales_por_hilo[
            "mensajes_iniciales"
        ]
        != 1
    ]
)


print(
    "Hilos sin exactamente un mensaje #1:",
    len(hilos_con_problema_inicial),
)


if not hilos_con_problema_inicial.empty:
    display(
        hilos_con_problema_inicial
    )

Hilos sin exactamente un mensaje #1: 0


In [75]:
resumen_longitudes = (
    df_calidad
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .agg(
        comentarios=(
            "id_mensaje",
            "nunique",
        ),
        caracteres_mediana=(
            "longitud_caracteres",
            "median",
        ),
        palabras_mediana=(
            "numero_palabras",
            "median",
        ),
        textos_vacios=(
            "texto_propio_vacio",
            "sum",
        ),
    )
    .reset_index()
)


resumen_longitudes[
    "caracteres_mediana"
] = (
    resumen_longitudes[
        "caracteres_mediana"
    ]
    .round(1)
)


resumen_longitudes[
    "palabras_mediana"
] = (
    resumen_longitudes[
        "palabras_mediana"
    ]
    .round(1)
)


display(resumen_longitudes)

,tema,anio_objetivo,comentarios,caracteres_mediana,palabras_mediana,textos_vacios
0,inmigracion,2015,84,68.0,13.5,4
1,inmigracion,2016,244,80.5,15.0,5
2,inmigracion,2019,408,62.0,12.0,5
3,inmigracion,2023,368,133.5,24.0,3
4,lgtbi,2015,644,35.0,6.0,97
5,lgtbi,2016,183,129.0,25.0,7
6,lgtbi,2019,485,99.0,18.0,12
7,lgtbi,2023,192,63.0,11.0,10


In [76]:
textos_muy_cortos = (
    df_calidad.loc[
        df_calidad["numero_palabras"] <= 2,
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "numero_mensaje",
            "id_mensaje",
            "numero_palabras",
            "texto_sin_citas",
        ],
    ]
    .sort_values(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "numero_mensaje",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Comentarios con dos palabras o menos:",
    len(textos_muy_cortos),
)


display(
    textos_muy_cortos.head(30)
)

Comentarios con dos palabras o menos: 369


,tema,anio_objetivo,id_hilo,numero_mensaje,id_mensaje,numero_palabras,texto_sin_citas
0,inmigracion,2015,4514261,2,205813862,0,
1,inmigracion,2015,4514261,4,205816543,1,@\nAnaknas
2,inmigracion,2015,4514261,10,205843166,0,
3,inmigracion,2015,4514261,20,205870565,0,
4,inmigracion,2015,4514261,26,205921508,2,mis dies
5,inmigracion,2015,4514261,35,205968520,1,Upppp....
6,inmigracion,2015,4514261,37,206016526,1,Upeo
7,inmigracion,2015,4514261,50,206788617,1,Upeando.
8,inmigracion,2015,4514261,53,206823413,2,Pillo lectura
9,inmigracion,2015,4514261,54,206823555,1,opiniones?


In [77]:
# ============================================================
# CELDA 21 — PREPARACIÓN DEL CONJUNTO TEXTUAL
# ============================================================

PATRONES_CODIFICACION_ROTA = [
    "Ã",
    "Â",
    "â€",
    "â€™",
    "â€œ",
    "â€˜",
    "ðŸ",
    "�",
]


def puntuacion_codificacion_rota(texto):
    """
    Cuenta indicadores habituales de texto UTF-8
    interpretado como Latin-1 o Windows-1252.
    """
    
    texto = str(texto)

    return sum(
        texto.count(patron)
        for patron in PATRONES_CODIFICACION_ROTA
    )


def reparar_codificacion_texto(texto):
    """
    Prueba reparaciones conservadoras y solo acepta una
    versión si reduce los indicadores de codificación rota.
    """
    
    if pd.isna(texto):
        return texto

    original = str(texto)
    puntuacion_original = (
        puntuacion_codificacion_rota(original)
    )

    if puntuacion_original == 0:
        return original

    candidatos = [original]

    for codificacion_origen in [
        "latin-1",
        "cp1252",
    ]:
        try:
            candidato = (
                original
                .encode(codificacion_origen)
                .decode("utf-8")
            )

            candidatos.append(candidato)

        except (
            UnicodeEncodeError,
            UnicodeDecodeError,
        ):
            pass

    mejor_candidato = min(
        candidatos,
        key=puntuacion_codificacion_rota,
    )

    if (
        puntuacion_codificacion_rota(
            mejor_candidato
        )
        < puntuacion_original
    ):
        return mejor_candidato

    return original

In [78]:
comentarios_forocoches_textual = (
    comentarios_forocoches_anual_ampliado
    .copy()
)


columnas_texto_reparables = [
    "titulo_hilo",
    "texto_original",
    "texto_sin_citas",
]


for columna in columnas_texto_reparables:
    columna_antes = (
        comentarios_forocoches_textual[
            columna
        ]
        .fillna("")
        .astype(str)
    )

    columna_despues = (
        columna_antes.apply(
            reparar_codificacion_texto
        )
    )

    comentarios_forocoches_textual[
        f"{columna}_reparado"
    ] = (
        columna_antes
        != columna_despues
    )

    comentarios_forocoches_textual[
        columna
    ] = columna_despues

In [79]:
resumen_reparacion = pd.DataFrame([
    {
        "columna": columna,
        "filas_reparadas": (
            comentarios_forocoches_textual[
                f"{columna}_reparado"
            ]
            .sum()
        ),
    }
    for columna in columnas_texto_reparables
])


display(resumen_reparacion)

,columna,filas_reparadas
0,titulo_hilo,0
1,texto_original,0
2,texto_sin_citas,0


In [80]:
comentarios_forocoches_textual[
    "texto_sin_citas"
] = (
    comentarios_forocoches_textual[
        "texto_sin_citas"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


comentarios_forocoches_textual[
    "longitud_caracteres"
] = (
    comentarios_forocoches_textual[
        "texto_sin_citas"
    ]
    .str.len()
)


comentarios_forocoches_textual[
    "numero_palabras"
] = (
    comentarios_forocoches_textual[
        "texto_sin_citas"
    ]
    .str.findall(r"\b\w+\b")
    .str.len()
)

In [81]:
# Solamente excluimos los mensajes sin ninguna palabra.
# Conservamos respuestas reales como "+1", "sí" o "pole".

comentarios_forocoches_nlp = (
    comentarios_forocoches_textual.loc[
        comentarios_forocoches_textual[
            "numero_palabras"
        ]
        > 0
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Comentarios anuales:",
    len(comentarios_forocoches_textual),
)

print(
    "Comentarios con texto analizable:",
    len(comentarios_forocoches_nlp),
)

print(
    "Mensajes sin texto excluidos del NLP:",
    (
        len(comentarios_forocoches_textual)
        - len(comentarios_forocoches_nlp)
    ),
)

Comentarios anuales: 2608
Comentarios con texto analizable: 2461
Mensajes sin texto excluidos del NLP: 147


In [82]:
filas_reparadas = (
    comentarios_forocoches_textual.loc[
        comentarios_forocoches_textual[
            "texto_sin_citas_reparado"
        ],
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "id_mensaje",
            "texto_sin_citas",
        ],
    ]
)


print(
    "Textos sin citas reparados:",
    len(filas_reparadas),
)


display(
    filas_reparadas.head(20)
)

Textos sin citas reparados: 0


,tema,anio_objetivo,id_hilo,id_mensaje,texto_sin_citas


In [83]:
# ============================================================
# CELDA 22 — GUARDADO DE LA EXTRACCIÓN
# ============================================================

RUTA_CATALOGO_FOROCOCHES = (
    RUTA_RAW
    / "forocoches_hilos_candidatos.csv"
)

RUTA_COMENTARIOS_BRUTOS = (
    RUTA_RAW
    / "forocoches_comentarios_bruto.csv"
)

RUTA_DIAGNOSTICOS = (
    RUTA_RAW
    / "forocoches_diagnosticos_hilos.csv"
)

RUTA_INCIDENCIAS = (
    RUTA_RAW
    / "forocoches_incidencias.csv"
)

RUTA_COMENTARIOS_ANUALES = (
    RUTA_PROCESADOS
    / "forocoches_comentarios_anuales.csv"
)

RUTA_RESUMEN_HILOS = (
    RUTA_PROCESADOS
    / "forocoches_resumen_hilos.csv"
)

In [84]:
COLUMNAS_EXTRACCION = [
    "tema",
    "anio_objetivo",
    "id_hilo",
    "titulo_hilo",
    "url_hilo",
    "pagina",
    "id_mensaje",
    "numero_mensaje",
    "usuario_hash",
    "fecha_comentario",
    "anio_comentario",
    "es_mensaje_inicial",
    "texto_original",
    "texto_sin_citas",
    "contiene_cita",
    "texto_propio_vacio",
]

In [85]:
comentarios_brutos_guardar = (
    comentarios_forocoches_bruto_ampliado[
        COLUMNAS_EXTRACCION
    ]
    .copy()
    .sort_values(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "pagina",
            "numero_mensaje",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


comentarios_anuales_guardar = (
    comentarios_forocoches_anual_ampliado[
        COLUMNAS_EXTRACCION
    ]
    .copy()
    .sort_values(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "pagina",
            "numero_mensaje",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

In [86]:
resumen_hilos_guardar = (
    comentarios_anuales_guardar
    .groupby(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "titulo_hilo",
            "url_hilo",
        ],
        as_index=False,
    )
    .agg(
        paginas=(
            "pagina",
            "max",
        ),
        comentarios_anuales=(
            "id_mensaje",
            "nunique",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
        comentarios_con_cita=(
            "contiene_cita",
            "sum",
        ),
        textos_propios_vacios=(
            "texto_propio_vacio",
            "sum",
        ),
        fecha_minima=(
            "fecha_comentario",
            "min",
        ),
        fecha_maxima=(
            "fecha_comentario",
            "max",
        ),
    )
)

In [87]:
hilos_candidatos.to_csv(
    RUTA_CATALOGO_FOROCOCHES,
    index=False,
    encoding="utf-8-sig",
)


comentarios_brutos_guardar.to_csv(
    RUTA_COMENTARIOS_BRUTOS,
    index=False,
    encoding="utf-8-sig",
)


comentarios_anuales_guardar.to_csv(
    RUTA_COMENTARIOS_ANUALES,
    index=False,
    encoding="utf-8-sig",
)


diagnosticos_forocoches_ampliado.to_csv(
    RUTA_DIAGNOSTICOS,
    index=False,
    encoding="utf-8-sig",
)


resumen_hilos_guardar.to_csv(
    RUTA_RESUMEN_HILOS,
    index=False,
    encoding="utf-8-sig",
)


incidencias_forocoches_ampliacion.to_csv(
    RUTA_INCIDENCIAS,
    index=False,
    encoding="utf-8-sig",
)

In [88]:
archivos_guardados = [
    RUTA_CATALOGO_FOROCOCHES,
    RUTA_COMENTARIOS_BRUTOS,
    RUTA_COMENTARIOS_ANUALES,
    RUTA_DIAGNOSTICOS,
    RUTA_RESUMEN_HILOS,
    RUTA_INCIDENCIAS,
]


comprobacion_guardado = pd.DataFrame([
    {
        "archivo": ruta.name,
        "carpeta": str(ruta.parent),
        "existe": ruta.exists(),
        "tamano_kb": (
            round(
                ruta.stat().st_size / 1024,
                2,
            )
            if ruta.exists()
            else None
        ),
    }
    for ruta in archivos_guardados
])


display(comprobacion_guardado)

,archivo,carpeta,existe,tamano_kb
0,forocoches_hilos_candidatos.csv,c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESP...,True,3.42
1,forocoches_comentarios_bruto.csv,c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESP...,True,1658.31
2,forocoches_comentarios_anuales.csv,c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESP...,True,1604.03
3,forocoches_diagnosticos_hilos.csv,c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESP...,True,3.52
4,forocoches_resumen_hilos.csv,c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESP...,True,4.36
5,forocoches_incidencias.csv,c:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESP...,True,0.00


In [89]:
prueba_lectura_anual = pd.read_csv(
    RUTA_COMENTARIOS_ANUALES,
    dtype={
        "id_hilo": str,
        "id_mensaje": str,
        "usuario_hash": str,
    },
)


print(
    "Comentarios anuales guardados:",
    len(prueba_lectura_anual),
)

print(
    "Hilos guardados:",
    prueba_lectura_anual[
        "id_hilo"
    ].nunique(),
)

print(
    "Combinaciones tema-año:",
    prueba_lectura_anual[
        [
            "tema",
            "anio_objetivo",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)

Comentarios anuales guardados: 2608
Hilos guardados: 23
Combinaciones tema-año: 8


## Ampliamos los hilos por tema y año 

In [90]:
# ============================================================
# CELDA 23 — OBJETIVO FINAL DE COBERTURA
# ============================================================

OBJETIVO_HILOS_POR_GRUPO = 20

NUMERO_GRUPOS = (
    len(TEMAS_OBJETIVO)
    * len(ANIOS_OBJETIVO)
)

OBJETIVO_HILOS_TOTAL = (
    OBJETIVO_HILOS_POR_GRUPO
    * NUMERO_GRUPOS
)


print(
    "Objetivo por tema y año:",
    OBJETIVO_HILOS_POR_GRUPO,
)

print(
    "Número de grupos:",
    NUMERO_GRUPOS,
)

print(
    "Objetivo total:",
    OBJETIVO_HILOS_TOTAL,
)

Objetivo por tema y año: 20
Número de grupos: 8
Objetivo total: 160


In [91]:
hilos_validos_actuales = (
    diagnosticos_forocoches_ampliado.loc[
        diagnosticos_forocoches_ampliado[
            "anio_validado"
        ]
        .fillna(False)
    ]
    .copy()
)


cobertura_objetivo = (
    pd.MultiIndex.from_product(
        [
            TEMAS_OBJETIVO,
            ANIOS_OBJETIVO,
        ],
        names=[
            "tema",
            "anio_objetivo",
        ],
    )
    .to_frame(index=False)
)


hilos_por_grupo_actuales = (
    hilos_validos_actuales
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )["id_hilo"]
    .nunique()
    .rename("hilos_validos")
    .reset_index()
)


cobertura_objetivo = (
    cobertura_objetivo.merge(
        hilos_por_grupo_actuales,
        on=[
            "tema",
            "anio_objetivo",
        ],
        how="left",
    )
)


cobertura_objetivo[
    "hilos_validos"
] = (
    cobertura_objetivo[
        "hilos_validos"
    ]
    .fillna(0)
    .astype(int)
)


cobertura_objetivo[
    "objetivo"
] = OBJETIVO_HILOS_POR_GRUPO


cobertura_objetivo[
    "hilos_pendientes"
] = (
    cobertura_objetivo["objetivo"]
    - cobertura_objetivo["hilos_validos"]
).clip(lower=0)


cobertura_objetivo[
    "porcentaje_completado"
] = (
    100
    * cobertura_objetivo[
        "hilos_validos"
    ]
    / cobertura_objetivo[
        "objetivo"
    ]
).round(1)


display(cobertura_objetivo)

,tema,anio_objetivo,hilos_validos,objetivo,hilos_pendientes,porcentaje_completado
0,inmigracion,2015,3,20,17,15.0
1,inmigracion,2016,3,20,17,15.0
2,inmigracion,2019,3,20,17,15.0
3,inmigracion,2023,3,20,17,15.0
4,lgtbi,2015,2,20,18,10.0
5,lgtbi,2016,3,20,17,15.0
6,lgtbi,2019,3,20,17,15.0
7,lgtbi,2023,3,20,17,15.0


In [92]:
hilos_validos_total = (
    hilos_validos_actuales[
        "id_hilo"
    ]
    .astype(str)
    .nunique()
)


hilos_pendientes_total = (
    cobertura_objetivo[
        "hilos_pendientes"
    ]
    .sum()
)


print(
    "Hilos válidos actuales:",
    hilos_validos_total,
)

print(
    "Hilos todavía necesarios:",
    hilos_pendientes_total,
)

print(
    "Progreso total:",
    f"{100 * hilos_validos_total / OBJETIVO_HILOS_TOTAL:.1f}%",
)

Hilos válidos actuales: 23
Hilos todavía necesarios: 137
Progreso total: 14.4%


In [93]:
# ============================================================
# CELDA 24 — NUEVOS CANDIDATOS: INMIGRACIÓN 2015
# ============================================================

nuevos_candidatos_inmigracion_2015 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4085624",
        "titulo_resultado": "La inmigración, un evidente problema social",
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4085624",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "inmigración problema social 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": "El tema central declarado es la inmigración"
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4096364",
        "titulo_resultado": (
            "Reflexión sobre los musulmanes. "
            "Punto de vista diferente. TEMA SERIO"
        ),
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4096364",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "musulmanes inmigración 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Debate explícito sobre inmigración musulmana, "
            "fronteras y deportación"
        )
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4509903",
        "titulo_resultado": (
            "Sobre una intervención de la OTAN en Siria "
            "a raíz de la crisis de refugiados"
        ),
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4509903",
        "fuente_descubrimiento": "enlace_desde_otro_hilo",
        "consulta_descubrimiento": "crisis de refugiados Siria 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Crisis de refugiados vinculada a la guerra de Siria"
        )
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4554014",
        "titulo_resultado": (
            "¿Por qué ForoCoches no es como la sociedad "
            "española en general?"
        ),
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4554014",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "ForoCoches refugiados 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Compara la opinión de ForoCoches sobre la acogida "
            "de refugiados con la sociedad española"
        )
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4598366",
        "titulo_resultado": (
            "¿Estarías a favor de expulsar a todos "
            "los musulmanes de España?"
        ),
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4598366",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "expulsión musulmanes España 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Debate sobre expulsión, integración y población migrante"
        )
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4605288",
        "titulo_resultado": (
            "Programa de Podemos para las generales "
            "respecto a la inmigración"
        ),
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4605288",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "programa político inmigración 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Políticas migratorias, CIE, deportaciones "
            "y derechos políticos"
        )
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4634310",
        "titulo_resultado": (
            "Apuntando alto: quiénes están detrás "
            "de la inmigración (pruebas dentro)"
        ),
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4634310",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "inmigración masiva política 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Discusión centrada en las causas políticas "
            "de la inmigración"
        )
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4661741",
        "titulo_resultado": "UPyD e inmigración",
        "url_hilo": "https://forocoches.com/foro/showthread.php?t=4661741",
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": "UPyD inmigración 2015",
        "lote_descubrimiento": "ampliacion_02",
        "criterio_inclusion": (
            "Debate sobre propuestas políticas relacionadas "
            "con inmigración"
        )
    }
])

# Aseguramos que los identificadores sean cadenas
hilos_candidatos["id_hilo"] = hilos_candidatos["id_hilo"].astype(str)
nuevos_candidatos_inmigracion_2015["id_hilo"] = (
    nuevos_candidatos_inmigracion_2015["id_hilo"].astype(str)
)

# Comprobamos cuáles son realmente nuevos
ids_ya_presentes = set(hilos_candidatos["id_hilo"])

candidatos_realmente_nuevos = (
    nuevos_candidatos_inmigracion_2015[
        ~nuevos_candidatos_inmigracion_2015["id_hilo"].isin(
            ids_ya_presentes
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Incorporación al catálogo general
hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos
        ],
        ignore_index=True,
        sort=False
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en este lote:",
    len(candidatos_realmente_nuevos)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos de inmigración–2015:",
    len(
        hilos_candidatos[
            (hilos_candidatos["tema"] == "inmigracion")
            & (hilos_candidatos["anio_objetivo"] == 2015)
        ]
    )
)

display(
    candidatos_realmente_nuevos[
        [
            "id_hilo",
            "titulo_resultado",
            "criterio_inclusion"
        ]
    ]
)

Candidatos incorporados en este lote: 8
Candidatos totales en el catálogo: 31
Candidatos de inmigración–2015: 11


,id_hilo,titulo_resultado,criterio_inclusion
0,4085624,"La inmigración, un evidente problema social",El tema central declarado es la inmigración
1,4096364,Reflexión sobre los musulmanes. Punto de vista...,"Debate explícito sobre inmigración musulmana, ..."
2,4509903,Sobre una intervención de la OTAN en Siria a r...,Crisis de refugiados vinculada a la guerra de ...
3,4554014,¿Por qué ForoCoches no es como la sociedad esp...,Compara la opinión de ForoCoches sobre la acog...
4,4598366,¿Estarías a favor de expulsar a todos los musu...,"Debate sobre expulsión, integración y població..."
5,4605288,Programa de Podemos para las generales respect...,"Políticas migratorias, CIE, deportaciones y de..."
6,4634310,Apuntando alto: quiénes están detrás de la inm...,Discusión centrada en las causas políticas de ...
7,4661741,UPyD e inmigración,Debate sobre propuestas políticas relacionadas...


In [94]:
# ============================================================
# CELDA 25 — PREPARACIÓN DEL LOTE DE VALIDACIÓN
# ============================================================

# Hilos que ya forman parte del corpus válido actual
ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

# Seleccionamos exclusivamente los candidatos del lote nuevo
pendientes_lote_02 = (
    candidatos_realmente_nuevos.loc[
        ~candidatos_realmente_nuevos[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

# Contenedores exclusivos de este lote
comentarios_lote_02 = []
diagnosticos_lote_02 = []
incidencias_lote_02 = []

print(
    "Hilos válidos antes del lote:",
    len(ids_hilos_validos_actuales)
)

print(
    "Candidatos pendientes del lote 02:",
    len(pendientes_lote_02)
)

display(
    pendientes_lote_02[
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "titulo_resultado"
        ]
    ]
)

Hilos válidos antes del lote: 23
Candidatos pendientes del lote 02: 8


,tema,anio_objetivo,id_hilo,titulo_resultado
0,inmigracion,2015,4085624,"La inmigración, un evidente problema social"
1,inmigracion,2015,4096364,Reflexión sobre los musulmanes. Punto de vista...
2,inmigracion,2015,4509903,Sobre una intervención de la OTAN en Siria a r...
3,inmigracion,2015,4554014,¿Por qué ForoCoches no es como la sociedad esp...
4,inmigracion,2015,4598366,¿Estarías a favor de expulsar a todos los musu...
5,inmigracion,2015,4605288,Programa de Podemos para las generales respect...
6,inmigracion,2015,4634310,Apuntando alto: quiénes están detrás de la inm...
7,inmigracion,2015,4661741,UPyD e inmigración


In [96]:
# ============================================================
# CELDA 26 — VALIDACIÓN Y DESCARGA DEL LOTE 02
# ============================================================

for posicion, (_, fila_hilo) in enumerate(
    pendientes_lote_02.iterrows(),
    start=1,
):
    id_hilo = str(
        fila_hilo["id_hilo"]
    )

    print("\n" + "=" * 70)

    print(
        f"[{posicion}/{len(pendientes_lote_02)}] "
        f"{fila_hilo['tema']} | "
        f"{fila_hilo['anio_objetivo']} | "
        f"hilo {id_hilo}"
    )

    print(
        fila_hilo["titulo_resultado"]
    )

    try:
        # ----------------------------------------------------
        # 1. Comprobar que el hilo es realmente público
        # ----------------------------------------------------

        comprobacion = (
            comprobar_primera_pagina_hilo(
                fila_hilo=fila_hilo,
                sesion=sesion,
            )
        )

        if not comprobacion["es_publico"]:
            incidencias_lote_02.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "no_publico",
                "anio_real_hilo": None,
                "estado_http": comprobacion[
                    "estado_http"
                ],
                "titulo_pagina": comprobacion[
                    "titulo_pagina"
                ],
                "detalle": (
                    "La primera página no contiene "
                    "mensajes reconocibles"
                ),
                "lote": "ampliacion_02",
            })

            print(
                "Descartado: no contiene mensajes "
                "públicos reconocibles."
            )

            time.sleep(2)
            continue

        print(
            "Primera página pública:",
            comprobacion[
                "numero_mensajes_primera_pagina"
            ],
            "mensajes detectados"
        )

        # ----------------------------------------------------
        # 2. Descargar el hilo completo
        # ----------------------------------------------------

        comentarios_hilo, diagnostico_hilo = (
            descargar_hilo_completo(
                fila_hilo=fila_hilo,
                sesion=sesion,
                espera_segundos=1.5,
                mostrar_progreso=True,
            )
        )

        # ----------------------------------------------------
        # 3. Validar el año del mensaje inicial
        # ----------------------------------------------------

        if not diagnostico_hilo["anio_validado"]:
            incidencias_lote_02.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "anio_no_coincide",
                "anio_real_hilo": diagnostico_hilo[
                    "anio_real_hilo"
                ],
                "estado_http": 200,
                "titulo_pagina": diagnostico_hilo[
                    "titulo_hilo"
                ],
                "detalle": (
                    "El año del mensaje inicial no "
                    "coincide con el año objetivo"
                ),
                "lote": "ampliacion_02",
            })

            print(
                "Descartado por año. Año real:",
                diagnostico_hilo[
                    "anio_real_hilo"
                ],
            )

            time.sleep(2)
            continue

        # ----------------------------------------------------
        # 4. Comprobar que existen comentarios del año objetivo
        # ----------------------------------------------------

        numero_comentarios_anuales = int(
            (
                comentarios_hilo["anio_comentario"]
                == int(fila_hilo["anio_objetivo"])
            ).sum()
        )

        if numero_comentarios_anuales == 0:
            incidencias_lote_02.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "sin_comentarios_anuales",
                "anio_real_hilo": diagnostico_hilo[
                    "anio_real_hilo"
                ],
                "estado_http": 200,
                "titulo_pagina": diagnostico_hilo[
                    "titulo_hilo"
                ],
                "detalle": (
                    "El hilo no contiene comentarios "
                    "del año objetivo"
                ),
                "lote": "ampliacion_02",
            })

            print(
                "Descartado: no contiene comentarios "
                "del año objetivo."
            )

            time.sleep(2)
            continue

        # ----------------------------------------------------
        # 5. Guardar provisionalmente el hilo válido
        # ----------------------------------------------------

        comentarios_hilo = comentarios_hilo.copy()
        comentarios_hilo["lote_extraccion"] = (
            "ampliacion_02"
        )

        comentarios_lote_02.append(
            comentarios_hilo
        )

        diagnostico_resumido = {
            clave: valor
            for clave, valor
            in diagnostico_hilo.items()
            if clave != "detalle_paginas"
        }

        diagnostico_resumido[
            "comentarios_anio_objetivo"
        ] = numero_comentarios_anuales

        diagnostico_resumido[
            "lote_extraccion"
        ] = "ampliacion_02"

        diagnosticos_lote_02.append(
            diagnostico_resumido
        )

        print(
            "Candidato válido:",
            len(comentarios_hilo),
            "comentarios totales |",
            numero_comentarios_anuales,
            "comentarios de 2015"
        )

    except Exception as error:
        incidencias_lote_02.append({
            "tema": fila_hilo["tema"],
            "anio_objetivo": int(
                fila_hilo["anio_objetivo"]
            ),
            "id_hilo": id_hilo,
            "url_hilo": fila_hilo["url_hilo"],
            "estado": "error",
            "anio_real_hilo": None,
            "estado_http": None,
            "titulo_pagina": "",
            "detalle": (
                f"{type(error).__name__}: {error}"
            ),
            "lote": "ampliacion_02",
        })

        print(
            "ERROR:",
            type(error).__name__,
            str(error),
        )

    # Pausa responsable entre hilos
    time.sleep(2)


print("\n" + "=" * 70)

print(
    "Hilos válidos en el lote:",
    len(diagnosticos_lote_02)
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_02)
)


[1/8] inmigracion | 2015 | hilo 4085624
La inmigración, un evidente problema social
Primera página pública: 30 mensajes detectados
Hilo 4085624: 5 página(s)
  Página 1/5: 30 comentarios
  Página 2/5: 30 comentarios
  Página 3/5: 30 comentarios
  Página 4/5: 24 comentarios
  Página 5/5: 24 comentarios
Candidato válido: 114 comentarios totales | 114 comentarios de 2015

[2/8] inmigracion | 2015 | hilo 4096364
Reflexión sobre los musulmanes. Punto de vista diferente. TEMA SERIO
Primera página pública: 15 mensajes detectados
Hilo 4096364: 1 página(s)
  Página 1/1: 15 comentarios
Candidato válido: 15 comentarios totales | 15 comentarios de 2015

[3/8] inmigracion | 2015 | hilo 4509903
Sobre una intervención de la OTAN en Siria a raíz de la crisis de refugiados
Descartado: no contiene mensajes públicos reconocibles.

[4/8] inmigracion | 2015 | hilo 4554014
¿Por qué ForoCoches no es como la sociedad española en general?
Primera página pública: 24 mensajes detectados
Hilo 4554014: 1 página(s)

In [100]:
# ============================================================
# CELDA 27 — CONSOLIDACIÓN DEL LOTE 02
# ============================================================

# ------------------------------------------------------------
# 1. Consolidar comentarios brutos
# ------------------------------------------------------------
OBJETIVO_HILOS_POR_GRUPO = 20

OBJETIVO_TOTAL_HILOS = (
    OBJETIVO_HILOS_POR_GRUPO
    * len(TEMAS_OBJETIVO)
    * len(ANIOS_OBJETIVO)
)

partes_comentarios_lote_02 = [
    comentarios_forocoches_bruto_ampliado
]

partes_comentarios_lote_02.extend(
    comentarios_lote_02
)

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        partes_comentarios_lote_02,
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Consolidar diagnósticos de hilos válidos
# ------------------------------------------------------------

partes_diagnosticos_lote_02 = [
    diagnosticos_forocoches_ampliado
]

if diagnosticos_lote_02:
    partes_diagnosticos_lote_02.append(
        pd.DataFrame(
            diagnosticos_lote_02
        )
    )

diagnosticos_forocoches_ampliado = (
    pd.concat(
        partes_diagnosticos_lote_02,
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Consolidar incidencias
# ------------------------------------------------------------

nuevas_incidencias_lote_02 = pd.DataFrame(
    incidencias_lote_02
)

if (
    "incidencias_forocoches_ampliacion" in globals()
    and not incidencias_forocoches_ampliacion.empty
):
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                nuevas_incidencias_lote_02,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo", "estado"],
            keep="last",
        )
        .reset_index(drop=True)
    )
else:
    incidencias_forocoches_ampliacion = (
        nuevas_incidencias_lote_02.copy()
    )

# ------------------------------------------------------------
# 4. Reconstruir corpus limitado a los años objetivo
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Recalcular cobertura
# ------------------------------------------------------------

cobertura_actualizada = (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        hilos=(
            "id_hilo",
            "nunique",
        ),
        comentarios=(
            "id_mensaje",
            "nunique",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Progreso respecto al objetivo
# ------------------------------------------------------------

hilos_validos_actualizados = int(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

hilos_pendientes_actualizados = max(
    OBJETIVO_TOTAL_HILOS
    - hilos_validos_actualizados,
    0,
)

progreso_actualizado = (
    hilos_validos_actualizados
    / OBJETIVO_TOTAL_HILOS
    * 100
)

print(
    "Hilos válidos actuales:",
    hilos_validos_actualizados
)

print(
    "Comentarios anuales actuales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Hilos todavía necesarios:",
    hilos_pendientes_actualizados
)

print(
    "Progreso total:",
    f"{progreso_actualizado:.1f}%"
)

display(
    cobertura_actualizada
)

Hilos válidos actuales: 30
Comentarios anuales actuales: 3178
Hilos todavía necesarios: 130
Progreso total: 18.8%


,tema,anio_objetivo,hilos,comentarios,usuarios
0,inmigracion,2015,10,654,455
1,inmigracion,2016,3,244,189
2,inmigracion,2019,3,408,352
3,inmigracion,2023,3,368,251
4,lgtbi,2015,2,644,566
5,lgtbi,2016,3,183,141
6,lgtbi,2019,3,485,362
7,lgtbi,2023,3,192,166


In [101]:
# ============================================================
# CELDA 28 — CANDIDATOS DEL LOTE 03: INMIGRACIÓN 2015
# ============================================================

nuevos_candidatos_lote_03 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4502429",
        "titulo_resultado": (
            "Quiero datos que confirmen que la "
            "inmigración africana es mala"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4502429"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigración africana datos 2015"
        ),
        "lote_descubrimiento": "ampliacion_03",
        "criterio_inclusion": (
            "Debate explícito sobre inmigración africana, "
            "delincuencia y evidencia estadística"
        ),
        "subtema_preliminar": (
            "inmigracion_delincuencia"
        ),
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4520069",
        "titulo_resultado": (
            "Más inmigración = Mayor delincuencia. "
            "TEMA SERIO"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4520069"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "extranjeros delincuencia inmigración 2015"
        ),
        "lote_descubrimiento": "ampliacion_03",
        "criterio_inclusion": (
            "Discusión central sobre la relación entre "
            "inmigración, delincuencia y control migratorio"
        ),
        "subtema_preliminar": (
            "inmigracion_delincuencia"
        ),
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4608044",
        "titulo_resultado": (
            "Migración: Volver a ninguna parte"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4608044"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "migración retorno inmigrantes 2015"
        ),
        "lote_descubrimiento": "ampliacion_03",
        "criterio_inclusion": (
            "Experiencias migratorias, adaptación, retorno "
            "y permanencia de población inmigrante"
        ),
        "subtema_preliminar": (
            "experiencia_migratoria"
        ),
    },
])

# Normalización de identificadores
hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"].astype(str)
)

nuevos_candidatos_lote_03["id_hilo"] = (
    nuevos_candidatos_lote_03[
        "id_hilo"
    ].astype(str)
)

# Eliminar candidatos que ya estuvieran catalogados
ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_realmente_nuevos_lote_03 = (
    nuevos_candidatos_lote_03.loc[
        ~nuevos_candidatos_lote_03[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

# Incorporación al catálogo
hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos_lote_03,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 03:",
    len(candidatos_realmente_nuevos_lote_03)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos catalogados de inmigración–2015:",
    len(
        hilos_candidatos.loc[
            (
                hilos_candidatos["tema"]
                == "inmigracion"
            )
            & (
                hilos_candidatos["anio_objetivo"]
                == 2015
            )
        ]
    )
)

display(
    candidatos_realmente_nuevos_lote_03[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "criterio_inclusion",
        ]
    ]
)

Candidatos incorporados en el lote 03: 3
Candidatos totales en el catálogo: 34
Candidatos catalogados de inmigración–2015: 14


,id_hilo,titulo_resultado,subtema_preliminar,criterio_inclusion
0,4502429,Quiero datos que confirmen que la inmigración ...,inmigracion_delincuencia,"Debate explícito sobre inmigración africana, d..."
1,4520069,Más inmigración = Mayor delincuencia. TEMA SERIO,inmigracion_delincuencia,Discusión central sobre la relación entre inmi...
2,4608044,Migración: Volver a ninguna parte,experiencia_migratoria,"Experiencias migratorias, adaptación, retorno ..."


In [102]:
# ============================================================
# CELDA 29 — PREPARACIÓN DEL LOTE 03
# ============================================================

ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

pendientes_lote_03 = (
    candidatos_realmente_nuevos_lote_03.loc[
        ~candidatos_realmente_nuevos_lote_03[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_lote_03 = []
diagnosticos_lote_03 = []
incidencias_lote_03 = []

print(
    "Hilos válidos antes del lote:",
    len(ids_hilos_validos_actuales)
)

print(
    "Candidatos pendientes del lote 03:",
    len(pendientes_lote_03)
)

display(
    pendientes_lote_03[
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
        ]
    ]
)

Hilos válidos antes del lote: 30
Candidatos pendientes del lote 03: 3


,tema,anio_objetivo,id_hilo,titulo_resultado,subtema_preliminar
0,inmigracion,2015,4502429,Quiero datos que confirmen que la inmigración ...,inmigracion_delincuencia
1,inmigracion,2015,4520069,Más inmigración = Mayor delincuencia. TEMA SERIO,inmigracion_delincuencia
2,inmigracion,2015,4608044,Migración: Volver a ninguna parte,experiencia_migratoria


In [103]:
# ============================================================
# CELDA 30 — VALIDACIÓN Y DESCARGA DEL LOTE 03
# ============================================================

for posicion, (_, fila_hilo) in enumerate(
    pendientes_lote_03.iterrows(),
    start=1,
):
    id_hilo = str(
        fila_hilo["id_hilo"]
    )

    print("\n" + "=" * 70)

    print(
        f"[{posicion}/{len(pendientes_lote_03)}] "
        f"{fila_hilo['tema']} | "
        f"{fila_hilo['anio_objetivo']} | "
        f"hilo {id_hilo}"
    )

    print(
        fila_hilo["titulo_resultado"]
    )

    try:
        # ----------------------------------------------------
        # 1. Comprobación de acceso público
        # ----------------------------------------------------

        comprobacion = (
            comprobar_primera_pagina_hilo(
                fila_hilo=fila_hilo,
                sesion=sesion,
            )
        )

        if not comprobacion["es_publico"]:
            incidencias_lote_03.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "no_publico",
                "anio_real_hilo": None,
                "estado_http": comprobacion[
                    "estado_http"
                ],
                "titulo_pagina": comprobacion[
                    "titulo_pagina"
                ],
                "detalle": (
                    "La primera página no contiene "
                    "mensajes reconocibles"
                ),
                "lote": "ampliacion_03",
            })

            print(
                "Descartado: no contiene mensajes "
                "públicos reconocibles."
            )

            time.sleep(2)
            continue

        print(
            "Primera página pública:",
            comprobacion[
                "numero_mensajes_primera_pagina"
            ],
            "mensajes detectados"
        )

        # ----------------------------------------------------
        # 2. Descarga completa
        # ----------------------------------------------------

        comentarios_hilo, diagnostico_hilo = (
            descargar_hilo_completo(
                fila_hilo=fila_hilo,
                sesion=sesion,
                espera_segundos=1.5,
                mostrar_progreso=True,
            )
        )

        # ----------------------------------------------------
        # 3. Validación temporal
        # ----------------------------------------------------

        if not diagnostico_hilo["anio_validado"]:
            incidencias_lote_03.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "anio_no_coincide",
                "anio_real_hilo": diagnostico_hilo[
                    "anio_real_hilo"
                ],
                "estado_http": 200,
                "titulo_pagina": diagnostico_hilo[
                    "titulo_hilo"
                ],
                "detalle": (
                    "El año del mensaje inicial no "
                    "coincide con el año objetivo"
                ),
                "lote": "ampliacion_03",
            })

            print(
                "Descartado por año. Año real:",
                diagnostico_hilo[
                    "anio_real_hilo"
                ],
            )

            time.sleep(2)
            continue

        # ----------------------------------------------------
        # 4. Comentarios pertenecientes al año objetivo
        # ----------------------------------------------------

        numero_comentarios_anuales = int(
            (
                comentarios_hilo["anio_comentario"]
                == int(fila_hilo["anio_objetivo"])
            ).sum()
        )

        if numero_comentarios_anuales == 0:
            incidencias_lote_03.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": int(
                    fila_hilo["anio_objetivo"]
                ),
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "sin_comentarios_anuales",
                "anio_real_hilo": diagnostico_hilo[
                    "anio_real_hilo"
                ],
                "estado_http": 200,
                "titulo_pagina": diagnostico_hilo[
                    "titulo_hilo"
                ],
                "detalle": (
                    "No contiene comentarios del "
                    "año objetivo"
                ),
                "lote": "ampliacion_03",
            })

            print(
                "Descartado: no contiene comentarios "
                "del año objetivo."
            )

            time.sleep(2)
            continue

        # ----------------------------------------------------
        # 5. Almacenamiento provisional
        # ----------------------------------------------------

        comentarios_hilo = comentarios_hilo.copy()

        comentarios_hilo[
            "lote_extraccion"
        ] = "ampliacion_03"

        comentarios_lote_03.append(
            comentarios_hilo
        )

        diagnostico_resumido = {
            clave: valor
            for clave, valor
            in diagnostico_hilo.items()
            if clave != "detalle_paginas"
        }

        diagnostico_resumido[
            "comentarios_anio_objetivo"
        ] = numero_comentarios_anuales

        diagnostico_resumido[
            "lote_extraccion"
        ] = "ampliacion_03"

        diagnosticos_lote_03.append(
            diagnostico_resumido
        )

        print(
            "Candidato técnicamente válido:",
            len(comentarios_hilo),
            "comentarios totales |",
            numero_comentarios_anuales,
            "comentarios de 2015"
        )

    except Exception as error:
        incidencias_lote_03.append({
            "tema": fila_hilo["tema"],
            "anio_objetivo": int(
                fila_hilo["anio_objetivo"]
            ),
            "id_hilo": id_hilo,
            "url_hilo": fila_hilo["url_hilo"],
            "estado": "error",
            "anio_real_hilo": None,
            "estado_http": None,
            "titulo_pagina": "",
            "detalle": (
                f"{type(error).__name__}: {error}"
            ),
            "lote": "ampliacion_03",
        })

        print(
            "ERROR:",
            type(error).__name__,
            str(error),
        )

    time.sleep(2)


print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    len(diagnosticos_lote_03)
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_03)
)


[1/3] inmigracion | 2015 | hilo 4502429
Quiero datos que confirmen que la inmigración africana es mala
Primera página pública: 20 mensajes detectados
Hilo 4502429: 1 página(s)
  Página 1/1: 20 comentarios
Candidato técnicamente válido: 20 comentarios totales | 20 comentarios de 2015

[2/3] inmigracion | 2015 | hilo 4520069
Más inmigración = Mayor delincuencia. TEMA SERIO
Primera página pública: 30 mensajes detectados
Hilo 4520069: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 13 comentarios
Candidato técnicamente válido: 43 comentarios totales | 26 comentarios de 2015

[3/3] inmigracion | 2015 | hilo 4608044
Migración: Volver a ninguna parte
Primera página pública: 19 mensajes detectados
Hilo 4608044: 1 página(s)
  Página 1/1: 19 comentarios
Candidato técnicamente válido: 19 comentarios totales | 19 comentarios de 2015

Hilos técnicamente válidos: 3
Hilos descartados o con error: 0


In [104]:
# ============================================================
# CELDA 31 — REVISIÓN TEMÁTICA DEL HILO 4608044
# ============================================================

id_hilo_revision = "4608044"

comentarios_revision = (
    pd.concat(
        comentarios_lote_03,
        ignore_index=True,
        sort=False,
    )
    .loc[
        lambda df:
        df["id_hilo"].astype(str)
        == id_hilo_revision
    ]
    .copy()
    .sort_values(
        "numero_mensaje"
    )
    .reset_index(drop=True)
)

# Texto que utilizaremos para el diagnóstico.
# Seguimos trabajando en extracción/QC, no en el NLP definitivo.
columna_texto_revision = (
    "texto_sin_citas"
    if "texto_sin_citas" in comentarios_revision.columns
    else "texto_original"
)

texto_revision_normalizado = (
    comentarios_revision[
        columna_texto_revision
    ]
    .fillna("")
    .astype(str)
    .str.lower()
)

# Indicadores de inmigración hacia España
patron_inmigracion = (
    r"\binmigr"
    r"|\bextranj"
    r"|dominican"
    r"|latinoamer"
    r"|lleg(?:a|an|ó|aron)\s+a\s+españa"
    r"|venir\s+a\s+españa"
    r"|residencia\s+en\s+españa"
)

# Indicadores de emigración o retorno de españoles
patron_emigracion = (
    r"\bemigr"
    r"|\bexpatriad"
    r"|\bretorn"
    r"|\bregres"
    r"|\bvolver"
    r"|españoles\s+en\s+el\s+extranjero"
    r"|vivir\s+fuera"
    r"|irse\s+de\s+españa"
)

comentarios_revision[
    "menciona_inmigracion"
] = texto_revision_normalizado.str.contains(
    patron_inmigracion,
    regex=True,
    na=False,
)

comentarios_revision[
    "menciona_emigracion_retorno"
] = texto_revision_normalizado.str.contains(
    patron_emigracion,
    regex=True,
    na=False,
)

resumen_revision_tematica = pd.DataFrame({
    "indicador": [
        "comentarios_totales",
        "mencionan_inmigracion",
        "mencionan_emigracion_retorno",
    ],
    "numero_comentarios": [
        len(comentarios_revision),
        int(
            comentarios_revision[
                "menciona_inmigracion"
            ].sum()
        ),
        int(
            comentarios_revision[
                "menciona_emigracion_retorno"
            ].sum()
        ),
    ],
})

display(
    resumen_revision_tematica
)

# Mostramos todos los comentarios porque solamente son 19
with pd.option_context(
    "display.max_colwidth",
    500,
    "display.max_rows",
    30,
):
    display(
        comentarios_revision[
            [
                "numero_mensaje",
                "menciona_inmigracion",
                "menciona_emigracion_retorno",
                columna_texto_revision,
            ]
        ]
    )

,indicador,numero_comentarios
0,comentarios_totales,19
1,mencionan_inmigracion,2
2,mencionan_emigracion_retorno,11


,numero_mensaje,menciona_inmigracion,menciona_emigracion_retorno,texto_sin_citas
0,1,False,True,http://politica.elpais.com/politica/...56_087923.html\nQué opinan los foreros emigrantes? Tema más que manido pero que cae en una ristra de tópicos...
1,2,False,True,"Yo ahora vivo fuera (de nuevo) y en fin, como tantas otras cosas creo que cada caso es un mundo (obviedad inside).\nDeterminado sobretodo por la personalidad del sujeto en sí, y su ""environment"" en el país en el que emigre. Si esa persona se echa una pareja fuera con cierta edad va a ser bastante complicado que vuelva.\nPor otra parte, el pestilente panorama laboral español cada día es peor. Además cada vez más concentrado únicamente en 2 polos: Madrid y Barcelona. País en el que más vale te..."
2,3,False,True,"Llevo un par de meses en Madrid. Después del erasmus, creo que es lo más cerca que voy a estar de mi ciudad natal, en Asturias. Simplemente ya no puedo volver. Lo que una vez me pareció mi hogar, es cada vez más un lugar gris en mi memoria.\nIncluso Madrid no es más que una escala en mi viaje. Espero ver la primavera de 2017 en Escocia, Irlanda, Francia o algo más ""exótico"". Tengo 27 años recién cumplidos, y siento que mi país se muere de tristeza poco a poco, llevándose a mi generación por ..."
3,4,False,True,Yo lo que no entiendo es esa gente que regresa a España para quejarse... cuál es el punto?
4,5,False,False,Algunos se ahogan en un puto vaso de agua.
5,6,False,False,"Aro, shur. Ha mí me a pasao lo mismo. Ahora paseo por el pueblo y me voy dando cuenta de que está lleno de gilipollas. Bueno, eso ya lo decía antes de irme, pero ahora queda más reshulón porque añado la coletilla ""pues en América atan a los perros con longaniza"".\n--\n¡Ay, amigo! El que se crea que España está tan mal, es que no ha llegado a conocer el país en el que ha vivido."
6,7,True,True,"A mi no me parece raro que los hijos no quieran volver a la República Dominicana. Si viven en un país europeo como España, volver al tercer mundo tiene que ser un estado de shock.\nA mi lo que me sorprende de esta gente tan crítica con España en el artículo, es que si en su país de acogida estaban tan bien y es tan maravilloso... para qué regresaron?"
7,8,False,False,"Lazos familiares, amistades y en algunois casos, por lo que vi en algunas personas, incluso amor patrio y las ganas de hacer algo en y por el pais en el que naciste."
8,9,False,False,"Eso es un tópico muy manido... además esta gente va a España a criticar por temas bastante estúpidos a mi modo de ver (que si la gente grita mucho, que si la educación)... no veo el amor por la patria."
9,10,False,True,"El que sea manido no quiere decir que no sea una realidad para muchos retornados.\nEn cuanto al amor por la patria, tengo amigos cercanos que no es la primera vez que me preguntan cuando voy a volver a ""casa"" y a trabajar para empresas espanyolas por el hecho de ser espanyol. De hecho, alguno busco la oportunidad hasta que la encontro y volvio.\nEn lo de que las criticas estupidas y sin sentido, estamos totalmente de acuerdo."


In [105]:
# ============================================================
# CELDA 32 — EXCLUSIÓN POR FALTA DE RELEVANCIA TEMÁTICA
# ============================================================

id_hilo_excluido = "4608044"

# Retirarlo de los comentarios provisionales del lote
comentarios_lote_03 = [
    dataframe_hilo
    for dataframe_hilo in comentarios_lote_03
    if (
        dataframe_hilo["id_hilo"]
        .astype(str)
        .iloc[0]
        != id_hilo_excluido
    )
]

# Retirarlo de los diagnósticos válidos provisionales
diagnosticos_lote_03 = [
    diagnostico
    for diagnostico in diagnosticos_lote_03
    if str(
        diagnostico["id_hilo"]
    ) != id_hilo_excluido
]

# Registrar expresamente la exclusión
incidencias_lote_03.append({
    "tema": "inmigracion",
    "anio_objetivo": 2015,
    "id_hilo": id_hilo_excluido,
    "url_hilo": (
        "https://forocoches.com/foro/"
        "showthread.php?t=4608044"
    ),
    "estado": "excluido_relevancia_tematica",
    "anio_real_hilo": 2015,
    "estado_http": 200,
    "titulo_pagina": (
        "Migración: Volver a ninguna parte"
    ),
    "detalle": (
        "El hilo está dominado por emigración y retorno "
        "de españoles: 11 de 19 comentarios presentan "
        "indicadores de emigración/retorno y solamente "
        "2 presentan indicadores de inmigración."
    ),
    "lote": "ampliacion_03",
})

print(
    "Hilos temáticamente válidos en el lote 03:",
    len(diagnosticos_lote_03)
)

print(
    "Hilos excluidos o con incidencias:",
    len(incidencias_lote_03)
)

display(
    pd.DataFrame(
        incidencias_lote_03
    )[
        [
            "id_hilo",
            "estado",
            "detalle",
        ]
    ]
)

Hilos temáticamente válidos en el lote 03: 2
Hilos excluidos o con incidencias: 1


,id_hilo,estado,detalle
0,4608044,excluido_relevancia_tematica,El hilo está dominado por emigración y retorno...


In [106]:
# ============================================================
# CELDA 33 — CONSOLIDACIÓN DEL LOTE 03
# ============================================================

# ------------------------------------------------------------
# 1. Incorporar comentarios válidos
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            *comentarios_lote_03,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Incorporar diagnósticos válidos
# ------------------------------------------------------------

if diagnosticos_lote_03:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                pd.DataFrame(
                    diagnosticos_lote_03
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo"],
            keep="last",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Incorporar incidencias
# ------------------------------------------------------------

nuevas_incidencias_lote_03 = pd.DataFrame(
    incidencias_lote_03
)

incidencias_forocoches_ampliacion = (
    pd.concat(
        [
            incidencias_forocoches_ampliacion,
            nuevas_incidencias_lote_03,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo", "estado"],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Reconstruir el corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Actualizar cobertura
# ------------------------------------------------------------

cobertura_actualizada = (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        hilos=(
            "id_hilo",
            "nunique",
        ),
        comentarios=(
            "id_mensaje",
            "nunique",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Calcular progreso global
# ------------------------------------------------------------

OBJETIVO_HILOS_POR_GRUPO = 20

OBJETIVO_TOTAL_HILOS = (
    OBJETIVO_HILOS_POR_GRUPO
    * len(TEMAS_OBJETIVO)
    * len(ANIOS_OBJETIVO)
)

hilos_validos_actualizados = int(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

hilos_pendientes_actualizados = max(
    OBJETIVO_TOTAL_HILOS
    - hilos_validos_actualizados,
    0,
)

progreso_actualizado = (
    hilos_validos_actualizados
    / OBJETIVO_TOTAL_HILOS
    * 100
)

print(
    "Hilos válidos actuales:",
    hilos_validos_actualizados
)

print(
    "Comentarios anuales actuales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Hilos todavía necesarios:",
    hilos_pendientes_actualizados
)

print(
    "Progreso total:",
    f"{progreso_actualizado:.1f}%"
)

display(
    cobertura_actualizada
)

Hilos válidos actuales: 32
Comentarios anuales actuales: 3224
Hilos todavía necesarios: 128
Progreso total: 20.0%


,tema,anio_objetivo,hilos,comentarios,usuarios
0,inmigracion,2015,12,700,487
1,inmigracion,2016,3,244,189
2,inmigracion,2019,3,408,352
3,inmigracion,2023,3,368,251
4,lgtbi,2015,2,644,566
5,lgtbi,2016,3,183,141
6,lgtbi,2019,3,485,362
7,lgtbi,2023,3,192,166


In [107]:
# ============================================================
# CELDA 34 — CONCENTRACIÓN DEL ESTRATO INMIGRACIÓN–2015
# ============================================================

estrato_inmigracion_2015 = (
    comentarios_forocoches_anual_ampliado.loc[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ] == "inmigracion"
        )
        & (
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ] == 2015
        )
    ]
    .copy()
)

distribucion_hilos_2015 = (
    estrato_inmigracion_2015
    .groupby(
        [
            "id_hilo",
            "titulo_hilo",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        comentarios=(
            "id_mensaje",
            "nunique",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
    )
)

total_comentarios_2015 = int(
    distribucion_hilos_2015[
        "comentarios"
    ].sum()
)

distribucion_hilos_2015[
    "porcentaje_comentarios"
] = (
    distribucion_hilos_2015[
        "comentarios"
    ]
    / total_comentarios_2015
    * 100
)

distribucion_hilos_2015 = (
    distribucion_hilos_2015
    .sort_values(
        "comentarios",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Índice de concentración HHI:
# cuanto más próximo a 1, más concentrado está el corpus.
proporciones_hilos = (
    distribucion_hilos_2015[
        "comentarios"
    ]
    / total_comentarios_2015
)

indice_hhi = float(
    (proporciones_hilos ** 2).sum()
)

# Número efectivo de hilos:
# traduce la concentración al equivalente de hilos
# perfectamente equilibrados.
numero_efectivo_hilos = (
    1 / indice_hhi
    if indice_hhi > 0
    else 0
)

mayor_porcentaje_hilo = float(
    distribucion_hilos_2015[
        "porcentaje_comentarios"
    ].max()
)

cumple_volumen = (
    total_comentarios_2015 >= 500
)

cumple_numero_hilos = (
    len(distribucion_hilos_2015) >= 12
)

cumple_concentracion = (
    mayor_porcentaje_hilo <= 25
)

cumple_hilos_efectivos = (
    numero_efectivo_hilos >= 6
)

print(
    "Comentarios:",
    total_comentarios_2015
)

print(
    "Hilos reales:",
    len(distribucion_hilos_2015)
)

print(
    "Usuarios únicos:",
    estrato_inmigracion_2015[
        "usuario_hash"
    ].nunique()
)

print(
    "Mayor aportación de un solo hilo:",
    f"{mayor_porcentaje_hilo:.1f}%"
)

print(
    "Índice HHI:",
    f"{indice_hhi:.3f}"
)

print(
    "Número efectivo de hilos:",
    f"{numero_efectivo_hilos:.1f}"
)

print(
    "Estrato suficiente según volumen y diversidad:",
    all([
        cumple_volumen,
        cumple_numero_hilos,
        cumple_concentracion,
        cumple_hilos_efectivos,
    ])
)

display(
    distribucion_hilos_2015
)

Comentarios: 700
Hilos reales: 12
Usuarios únicos: 487
Mayor aportación de un solo hilo: 29.4%
Índice HHI: 0.173
Número efectivo de hilos: 5.8
Estrato suficiente según volumen y diversidad: False


,id_hilo,titulo_hilo,comentarios,usuarios,porcentaje_comentarios
0,4598366,¿Estarias a favor de expulsar a todos los musu...,206,165,29.428571
1,4634310,Apuntando alto: quiénes están detrás de la inm...,136,106,19.428571
2,4085624,"La inmigración, un evidente problema social.",114,73,16.285714
3,4514261,El problema que hemos creado,77,42,11.000000
4,4605288,Programa de podemos para las generales respect...,42,32,6.000000
5,4661741,"UPyD e INMIGRACIÓN,",33,21,4.714286
6,4520069,Mas inmigracion = Mayor delincuencia. TEMA SERIO.,26,23,3.714286
7,4554014,Por que Forocoches no es como la sociedad espa...,24,20,3.428571
8,4502429,Quiero datos que confirmen que la inmigración ...,20,16,2.857143
9,4096364,Reflexión sobre los musulmanes. Punto de vista...,15,12,2.142857


In [108]:
# ============================================================
# CELDA 35 — LOTE 04: AMPLIACIÓN ADAPTATIVA INMIGRACIÓN–2015
# ============================================================

nuevos_candidatos_lote_04 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4411585",
        "titulo_resultado": (
            "Si un ciudadano comunitario quiere trabajar "
            "en España, ¿qué necesita?"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4411585"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigrantes trabajo España 2015"
        ),
        "lote_descubrimiento": "ampliacion_04",
        "criterio_inclusion": (
            "Movilidad laboral hacia España, NIE, "
            "residencia y requisitos administrativos"
        ),
        "subtema_preliminar": (
            "migracion_laboral_comunitaria"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4527461",
        "titulo_resultado": (
            "Anuncio de EDUCO: pobreza infantil"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4527461"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "refugiados ayudas España septiembre 2015"
        ),
        "lote_descubrimiento": "ampliacion_04",
        "criterio_inclusion": (
            "El mensaje inicial contrapone pobreza nacional, "
            "ayudas públicas y acogida de refugiados"
        ),
        "subtema_preliminar": (
            "refugiados_competencia_bienestar"
        ),
        "requiere_revision_tematica": True,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4675906",
        "titulo_resultado": (
            "Conseguir visado de trabajo en España "
            "[Tema Serio]"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4675906"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "visado trabajo España diciembre 2015"
        ),
        "lote_descubrimiento": "ampliacion_04",
        "criterio_inclusion": (
            "Migración legal hacia España, permisos de "
            "residencia, trabajo e integración administrativa"
        ),
        "subtema_preliminar": (
            "visados_residencia_trabajo"
        ),
        "requiere_revision_tematica": False,
    },
])

# Normalización
hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"].astype(str)
)

nuevos_candidatos_lote_04["id_hilo"] = (
    nuevos_candidatos_lote_04[
        "id_hilo"
    ].astype(str)
)

# Evitar duplicados en el catálogo
ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_realmente_nuevos_lote_04 = (
    nuevos_candidatos_lote_04.loc[
        ~nuevos_candidatos_lote_04[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos_lote_04,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 04:",
    len(candidatos_realmente_nuevos_lote_04)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos catalogados de inmigración–2015:",
    len(
        hilos_candidatos.loc[
            (
                hilos_candidatos["tema"]
                == "inmigracion"
            )
            & (
                hilos_candidatos["anio_objetivo"]
                == 2015
            )
        ]
    )
)

display(
    candidatos_realmente_nuevos_lote_04[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 04: 3
Candidatos totales en el catálogo: 37
Candidatos catalogados de inmigración–2015: 17


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,4411585,Si un ciudadano comunitario quiere trabajar en...,migracion_laboral_comunitaria,False
1,4527461,Anuncio de EDUCO: pobreza infantil,refugiados_competencia_bienestar,True
2,4675906,Conseguir visado de trabajo en España [Tema Se...,visados_residencia_trabajo,False


In [109]:
# ============================================================
# CELDA 36 — PREPARACIÓN DEL LOTE 04
# ============================================================

ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

pendientes_lote_04 = (
    candidatos_realmente_nuevos_lote_04.loc[
        ~candidatos_realmente_nuevos_lote_04[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_lote_04 = []
diagnosticos_lote_04 = []
incidencias_lote_04 = []

print(
    "Hilos válidos antes del lote:",
    len(ids_hilos_validos_actuales)
)

print(
    "Candidatos pendientes del lote 04:",
    len(pendientes_lote_04)
)

display(
    pendientes_lote_04[
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Hilos válidos antes del lote: 32
Candidatos pendientes del lote 04: 3


,tema,anio_objetivo,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,inmigracion,2015,4411585,Si un ciudadano comunitario quiere trabajar en...,migracion_laboral_comunitaria,False
1,inmigracion,2015,4527461,Anuncio de EDUCO: pobreza infantil,refugiados_competencia_bienestar,True
2,inmigracion,2015,4675906,Conseguir visado de trabajo en España [Tema Se...,visados_residencia_trabajo,False


In [110]:
# ============================================================
# CELDA 37 — PROCESADOR REUTILIZABLE DE LOTES
# ============================================================

def procesar_lote_forocoches(
    candidatos_pendientes,
    nombre_lote,
    sesion,
    espera_pagina=1.5,
    espera_hilo=2,
):
    """
    Valida y descarga un lote de candidatos de ForoCoches.

    Devuelve:
        comentarios_validos
        diagnosticos_validos
        incidencias
    """

    comentarios_validos = []
    diagnosticos_validos = []
    incidencias = []

    for posicion, (_, fila_hilo) in enumerate(
        candidatos_pendientes.iterrows(),
        start=1,
    ):
        id_hilo = str(
            fila_hilo["id_hilo"]
        )

        anio_objetivo = int(
            fila_hilo["anio_objetivo"]
        )

        print("\n" + "=" * 70)

        print(
            f"[{posicion}/{len(candidatos_pendientes)}] "
            f"{fila_hilo['tema']} | "
            f"{anio_objetivo} | "
            f"hilo {id_hilo}"
        )

        print(
            fila_hilo["titulo_resultado"]
        )

        try:
            # ------------------------------------------------
            # 1. Acceso público
            # ------------------------------------------------

            comprobacion = (
                comprobar_primera_pagina_hilo(
                    fila_hilo=fila_hilo,
                    sesion=sesion,
                )
            )

            if not comprobacion["es_publico"]:
                incidencias.append({
                    "tema": fila_hilo["tema"],
                    "anio_objetivo": anio_objetivo,
                    "id_hilo": id_hilo,
                    "url_hilo": fila_hilo["url_hilo"],
                    "estado": "no_publico",
                    "anio_real_hilo": None,
                    "estado_http": comprobacion[
                        "estado_http"
                    ],
                    "titulo_pagina": comprobacion[
                        "titulo_pagina"
                    ],
                    "detalle": (
                        "La primera página no contiene "
                        "mensajes reconocibles"
                    ),
                    "lote": nombre_lote,
                })

                print(
                    "Descartado: no contiene mensajes "
                    "públicos reconocibles."
                )

                time.sleep(espera_hilo)
                continue

            print(
                "Primera página pública:",
                comprobacion[
                    "numero_mensajes_primera_pagina"
                ],
                "mensajes detectados"
            )

            # ------------------------------------------------
            # 2. Descarga completa
            # ------------------------------------------------

            comentarios_hilo, diagnostico_hilo = (
                descargar_hilo_completo(
                    fila_hilo=fila_hilo,
                    sesion=sesion,
                    espera_segundos=espera_pagina,
                    mostrar_progreso=True,
                )
            )

            # ------------------------------------------------
            # 3. Año del mensaje inicial
            # ------------------------------------------------

            if not diagnostico_hilo["anio_validado"]:
                incidencias.append({
                    "tema": fila_hilo["tema"],
                    "anio_objetivo": anio_objetivo,
                    "id_hilo": id_hilo,
                    "url_hilo": fila_hilo["url_hilo"],
                    "estado": "anio_no_coincide",
                    "anio_real_hilo": diagnostico_hilo[
                        "anio_real_hilo"
                    ],
                    "estado_http": 200,
                    "titulo_pagina": diagnostico_hilo[
                        "titulo_hilo"
                    ],
                    "detalle": (
                        "El año del mensaje inicial no "
                        "coincide con el año objetivo"
                    ),
                    "lote": nombre_lote,
                })

                print(
                    "Descartado por año. Año real:",
                    diagnostico_hilo[
                        "anio_real_hilo"
                    ],
                )

                time.sleep(espera_hilo)
                continue

            # ------------------------------------------------
            # 4. Comentarios del año objetivo
            # ------------------------------------------------

            numero_comentarios_anuales = int(
                (
                    comentarios_hilo[
                        "anio_comentario"
                    ] == anio_objetivo
                ).sum()
            )

            if numero_comentarios_anuales == 0:
                incidencias.append({
                    "tema": fila_hilo["tema"],
                    "anio_objetivo": anio_objetivo,
                    "id_hilo": id_hilo,
                    "url_hilo": fila_hilo["url_hilo"],
                    "estado": "sin_comentarios_anuales",
                    "anio_real_hilo": diagnostico_hilo[
                        "anio_real_hilo"
                    ],
                    "estado_http": 200,
                    "titulo_pagina": diagnostico_hilo[
                        "titulo_hilo"
                    ],
                    "detalle": (
                        "No contiene comentarios del "
                        "año objetivo"
                    ),
                    "lote": nombre_lote,
                })

                print(
                    "Descartado: no contiene comentarios "
                    "del año objetivo."
                )

                time.sleep(espera_hilo)
                continue

            # ------------------------------------------------
            # 5. Almacenamiento provisional
            # ------------------------------------------------

            comentarios_hilo = (
                comentarios_hilo.copy()
            )

            comentarios_hilo[
                "lote_extraccion"
            ] = nombre_lote

            comentarios_validos.append(
                comentarios_hilo
            )

            diagnostico_resumido = {
                clave: valor
                for clave, valor
                in diagnostico_hilo.items()
                if clave != "detalle_paginas"
            }

            diagnostico_resumido[
                "comentarios_anio_objetivo"
            ] = numero_comentarios_anuales

            diagnostico_resumido[
                "lote_extraccion"
            ] = nombre_lote

            diagnosticos_validos.append(
                diagnostico_resumido
            )

            print(
                "Candidato técnicamente válido:",
                len(comentarios_hilo),
                "comentarios totales |",
                numero_comentarios_anuales,
                f"comentarios de {anio_objetivo}"
            )

        except Exception as error:
            incidencias.append({
                "tema": fila_hilo["tema"],
                "anio_objetivo": anio_objetivo,
                "id_hilo": id_hilo,
                "url_hilo": fila_hilo["url_hilo"],
                "estado": "error",
                "anio_real_hilo": None,
                "estado_http": None,
                "titulo_pagina": "",
                "detalle": (
                    f"{type(error).__name__}: {error}"
                ),
                "lote": nombre_lote,
            })

            print(
                "ERROR:",
                type(error).__name__,
                str(error),
            )

        time.sleep(espera_hilo)

    print("\n" + "=" * 70)

    print(
        "Hilos técnicamente válidos:",
        len(diagnosticos_validos)
    )

    print(
        "Hilos descartados o con error:",
        len(incidencias)
    )

    return (
        comentarios_validos,
        diagnosticos_validos,
        incidencias,
    )


# ------------------------------------------------------------
# Procesamiento del lote 04
# ------------------------------------------------------------

(
    comentarios_lote_04,
    diagnosticos_lote_04,
    incidencias_lote_04,
) = procesar_lote_forocoches(
    candidatos_pendientes=pendientes_lote_04,
    nombre_lote="ampliacion_04",
    sesion=sesion,
)


[1/3] inmigracion | 2015 | hilo 4411585
Si un ciudadano comunitario quiere trabajar en España, ¿qué necesita?
Primera página pública: 30 mensajes detectados
Hilo 4411585: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 4 comentarios
Candidato técnicamente válido: 34 comentarios totales | 33 comentarios de 2015

[2/3] inmigracion | 2015 | hilo 4527461
Anuncio de EDUCO: pobreza infantil
Primera página pública: 19 mensajes detectados
Hilo 4527461: 1 página(s)
  Página 1/1: 19 comentarios
Candidato técnicamente válido: 19 comentarios totales | 19 comentarios de 2015

[3/3] inmigracion | 2015 | hilo 4675906
Conseguir visado de trabajo en España [Tema Serio]
Primera página pública: 25 mensajes detectados
Hilo 4675906: 1 página(s)
  Página 1/1: 25 comentarios
Candidato técnicamente válido: 25 comentarios totales | 24 comentarios de 2015

Hilos técnicamente válidos: 3
Hilos descartados o con error: 0


In [111]:
# ============================================================
# CELDA 38 — REVISIÓN TEMÁTICA DEL HILO 4527461
# ============================================================

id_hilo_revision = "4527461"

comentarios_revision_4527461 = (
    pd.concat(
        comentarios_lote_04,
        ignore_index=True,
        sort=False,
    )
    .loc[
        lambda df:
        df["id_hilo"].astype(str)
        == id_hilo_revision
    ]
    .copy()
    .sort_values(
        "numero_mensaje"
    )
    .reset_index(drop=True)
)

columna_texto_revision = (
    "texto_sin_citas"
    if (
        "texto_sin_citas"
        in comentarios_revision_4527461.columns
    )
    else "texto_original"
)

texto_revision_normalizado = (
    comentarios_revision_4527461[
        columna_texto_revision
    ]
    .fillna("")
    .astype(str)
    .str.lower()
)

# Inmigración, refugiados y distribución de ayudas
patron_inmigracion_refugiados = (
    r"\binmigr"
    r"|\brefugiad"
    r"|\bextranjer"
    r"|\bmor[oa]s?\b"
    r"|\bsiri[oa]s?\b"
    r"|acoger"
    r"|acogida"
    r"|traer\s+a"
    r"|gente\s+de\s+fuera"
)

# Pobreza nacional, infancia y organizaciones sociales
patron_pobreza_ong = (
    r"\bpobrez"
    r"|\binfantil"
    r"|\bniñ"
    r"|\babuela"
    r"|\bong"
    r"|\beduco"
    r"|\bcáritas"
    r"|\bdonaci"
    r"|\bhambre"
)

comentarios_revision_4527461[
    "menciona_inmigracion_refugiados"
] = texto_revision_normalizado.str.contains(
    patron_inmigracion_refugiados,
    regex=True,
    na=False,
)

comentarios_revision_4527461[
    "menciona_pobreza_ong"
] = texto_revision_normalizado.str.contains(
    patron_pobreza_ong,
    regex=True,
    na=False,
)

comentarios_revision_4527461[
    "menciona_ambos_encuadres"
] = (
    comentarios_revision_4527461[
        "menciona_inmigracion_refugiados"
    ]
    & comentarios_revision_4527461[
        "menciona_pobreza_ong"
    ]
)

resumen_revision_4527461 = pd.DataFrame({
    "indicador": [
        "comentarios_totales",
        "mencionan_inmigracion_refugiados",
        "mencionan_pobreza_ong",
        "mencionan_ambos_encuadres",
    ],
    "numero_comentarios": [
        len(comentarios_revision_4527461),
        int(
            comentarios_revision_4527461[
                "menciona_inmigracion_refugiados"
            ].sum()
        ),
        int(
            comentarios_revision_4527461[
                "menciona_pobreza_ong"
            ].sum()
        ),
        int(
            comentarios_revision_4527461[
                "menciona_ambos_encuadres"
            ].sum()
        ),
    ],
})

display(
    resumen_revision_4527461
)

with pd.option_context(
    "display.max_colwidth",
    500,
    "display.max_rows",
    30,
):
    display(
        comentarios_revision_4527461[
            [
                "numero_mensaje",
                "menciona_inmigracion_refugiados",
                "menciona_pobreza_ong",
                "menciona_ambos_encuadres",
                columna_texto_revision,
            ]
        ]
    )

,indicador,numero_comentarios
0,comentarios_totales,19
1,mencionan_inmigracion_refugiados,8
2,mencionan_pobreza_ong,8
3,mencionan_ambos_encuadres,4


,numero_mensaje,menciona_inmigracion_refugiados,menciona_pobreza_ong,menciona_ambos_encuadres,texto_sin_citas
0,1,True,True,True,"Niño: Abuela por que no comes tu tambien?\nAbuela: No tengo hambre hijo tranquilo come tu.\nProgres: Fachas! Racistas! inhumanos! hay que traer a toda esa gente de fuera que vive en la miseria para darles paguita,viviendas,comida,libros escolares y la vagina de vuestras hijas de menos de 12 años, pero a la gente que se ha dejado la piel por este pais ellos y sus padres abuelos son todos fascistas y paletos que no tienen derecho a nada."
1,2,False,False,False,Joder por mas que busco no encuentro el puto anuncio
2,3,False,False,False,
3,4,False,False,False,Edito: arriba una versión más larga
4,5,False,False,False,"Gracias churs, la verdad que impacta.\nQue rabia."
5,6,False,True,False,"puto asco que me dan ya los políticos y las ong´s. Ya estaba hasta los cojones, pero cuando vi el anuncio de la vieja ¨no, yo no tengo hambre¨ ya me he decidido en cancelar las suscripciones que me cobran ong´s varias.\nPopulistas y oportunistas como nadie, cuando todos sabemos que funcionan como empresas, muchas de ellas verdaderas multinacionales."
6,7,True,False,False,"es fuerte el anuncio.......y hay algo que consigue que es ...no pensar en refugiados , si es que me cago en to pensar en otros cuando eso pasa aqui"
7,8,False,False,False,Lo malo es que esto es cierto a día de hoy. Lo se por casos cercanos.
8,9,False,False,False,Puse el mismo post la semana pasada y preguntaba ¿cómo es posible que hayamos llegado a esa situación en España?
9,10,True,True,True,"Pero eh! metamos a mas inmigrantes, a mas pobres y muertos de hambre.\nY que con mis impuestos mantenga todo este chiringuito...me hierve la sangre."


In [112]:
# ============================================================
# CELDA 39 — CONSOLIDACIÓN Y NUEVA PRUEBA DE CONCENTRACIÓN
# ============================================================

# ------------------------------------------------------------
# 1. Etiquetar la revisión temática
# ------------------------------------------------------------

for dataframe_hilo in comentarios_lote_04:
    id_hilo_actual = str(
        dataframe_hilo["id_hilo"].iloc[0]
    )

    dataframe_hilo[
        "revision_tematica"
    ] = (
        "valido_mixto"
        if id_hilo_actual == "4527461"
        else "valido_directo"
    )

for diagnostico in diagnosticos_lote_04:
    id_hilo_actual = str(
        diagnostico["id_hilo"]
    )

    diagnostico[
        "revision_tematica"
    ] = (
        "valido_mixto"
        if id_hilo_actual == "4527461"
        else "valido_directo"
    )

# ------------------------------------------------------------
# 2. Consolidar comentarios
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            *comentarios_lote_04,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Consolidar diagnósticos
# ------------------------------------------------------------

diagnosticos_forocoches_ampliado = (
    pd.concat(
        [
            diagnosticos_forocoches_ampliado,
            pd.DataFrame(
                diagnosticos_lote_04
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Consolidar incidencias, si existieran
# ------------------------------------------------------------

if incidencias_lote_04:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                pd.DataFrame(
                    incidencias_lote_04
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo", "estado"],
            keep="last",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Reconstruir corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Cobertura general
# ------------------------------------------------------------

cobertura_actualizada = (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        hilos=("id_hilo", "nunique"),
        comentarios=("id_mensaje", "nunique"),
        usuarios=("usuario_hash", "nunique"),
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Nueva concentración de inmigración–2015
# ------------------------------------------------------------

estrato_inmigracion_2015 = (
    comentarios_forocoches_anual_ampliado.loc[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ] == "inmigracion"
        )
        & (
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ] == 2015
        )
    ]
    .copy()
)

distribucion_hilos_2015 = (
    estrato_inmigracion_2015
    .groupby(
        [
            "id_hilo",
            "titulo_hilo",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        comentarios=("id_mensaje", "nunique"),
        usuarios=("usuario_hash", "nunique"),
    )
)

total_comentarios_2015 = int(
    distribucion_hilos_2015[
        "comentarios"
    ].sum()
)

distribucion_hilos_2015[
    "porcentaje_comentarios"
] = (
    distribucion_hilos_2015[
        "comentarios"
    ]
    / total_comentarios_2015
    * 100
)

distribucion_hilos_2015 = (
    distribucion_hilos_2015
    .sort_values(
        "comentarios",
        ascending=False,
    )
    .reset_index(drop=True)
)

proporciones_hilos = (
    distribucion_hilos_2015[
        "comentarios"
    ]
    / total_comentarios_2015
)

indice_hhi = float(
    (proporciones_hilos ** 2).sum()
)

numero_efectivo_hilos = (
    1 / indice_hhi
    if indice_hhi > 0
    else 0
)

mayor_porcentaje_hilo = float(
    distribucion_hilos_2015[
        "porcentaje_comentarios"
    ].max()
)

estrato_suficiente = all([
    total_comentarios_2015 >= 500,
    len(distribucion_hilos_2015) >= 12,
    mayor_porcentaje_hilo <= 25,
    numero_efectivo_hilos >= 6,
])

# ------------------------------------------------------------
# 8. Resumen
# ------------------------------------------------------------

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Hilos de inmigración–2015:",
    len(distribucion_hilos_2015)
)

print(
    "Comentarios de inmigración–2015:",
    total_comentarios_2015
)

print(
    "Usuarios de inmigración–2015:",
    estrato_inmigracion_2015[
        "usuario_hash"
    ].nunique()
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_porcentaje_hilo:.1f}%"
)

print(
    "Índice HHI:",
    f"{indice_hhi:.3f}"
)

print(
    "Número efectivo de hilos:",
    f"{numero_efectivo_hilos:.1f}"
)

print(
    "Estrato suficiente:",
    estrato_suficiente
)

display(
    distribucion_hilos_2015
)

Hilos válidos totales: 35
Comentarios anuales totales: 3300
Hilos de inmigración–2015: 15
Comentarios de inmigración–2015: 776
Usuarios de inmigración–2015: 528
Mayor aportación de un hilo: 26.5%
Índice HHI: 0.144
Número efectivo de hilos: 7.0
Estrato suficiente: False


,id_hilo,titulo_hilo,comentarios,usuarios,porcentaje_comentarios
0,4598366,¿Estarias a favor de expulsar a todos los musu...,206,165,26.546392
1,4634310,Apuntando alto: quiénes están detrás de la inm...,136,106,17.525773
2,4085624,"La inmigración, un evidente problema social.",114,73,14.690722
3,4514261,El problema que hemos creado,77,42,9.922680
4,4605288,Programa de podemos para las generales respect...,42,32,5.412371
5,4661741,"UPyD e INMIGRACIÓN,",33,21,4.252577
6,4411585,Si un ciudadano comunitario quiere trabajar en...,33,15,4.252577
7,4520069,Mas inmigracion = Mayor delincuencia. TEMA SERIO.,26,23,3.350515
8,4554014,Por que Forocoches no es como la sociedad espa...,24,20,3.092784
9,4675906,Conseguir visado de trabajo en España [Tema Se...,24,13,3.092784


In [114]:
# ============================================================
# CELDA 40 — CANDIDATO FINAL ADAPTATIVO: INMIGRACIÓN–2015
# ============================================================

nuevo_candidato_lote_05 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2015,
        "id_hilo": "4236180",
        "titulo_resultado": (
            "He vivido en Suecia y respondo preguntas "
            "sobre inmigración y sociedad"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4236180"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigración sociedad integración 2015"
        ),
        "lote_descubrimiento": "ampliacion_05",
        "criterio_inclusion": (
            "Experiencia personal sobre inmigración, "
            "integración, multiculturalidad, discriminación "
            "y convivencia en Suecia"
        ),
        "subtema_preliminar": (
            "integracion_multiculturalidad"
        ),
        "requiere_revision_tematica": False,
    }
])

nuevo_candidato_lote_05["id_hilo"] = (
    nuevo_candidato_lote_05[
        "id_hilo"
    ].astype(str)
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos[
        "id_hilo"
    ].astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_realmente_nuevos_lote_05 = (
    nuevo_candidato_lote_05.loc[
        ~nuevo_candidato_lote_05[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos_lote_05,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

pendientes_lote_05 = (
    candidatos_realmente_nuevos_lote_05.loc[
        ~candidatos_realmente_nuevos_lote_05[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados:",
    len(candidatos_realmente_nuevos_lote_05)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos pendientes del lote 05:",
    len(pendientes_lote_05)
)

display(
    pendientes_lote_05[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "criterio_inclusion",
        ]
    ]
)

Candidatos incorporados: 1
Candidatos totales en el catálogo: 38
Candidatos pendientes del lote 05: 1


,id_hilo,titulo_resultado,subtema_preliminar,criterio_inclusion
0,4236180,He vivido en Suecia y respondo preguntas sobre...,integracion_multiculturalidad,"Experiencia personal sobre inmigración, integr..."


In [115]:
# ============================================================
# CELDA 41 — DESCARGA DEL LOTE 05
# ============================================================

(
    comentarios_lote_05,
    diagnosticos_lote_05,
    incidencias_lote_05,
) = procesar_lote_forocoches(
    candidatos_pendientes=pendientes_lote_05,
    nombre_lote="ampliacion_05",
    sesion=sesion,
)


[1/1] inmigracion | 2015 | hilo 4236180
He vivido en Suecia y respondo preguntas sobre inmigración y sociedad
Primera página pública: 30 mensajes detectados
Hilo 4236180: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 30 comentarios
  Página 3/3: 16 comentarios
Candidato técnicamente válido: 76 comentarios totales | 76 comentarios de 2015

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [116]:
# ============================================================
# CELDA 42 — CONSOLIDACIÓN DEL LOTE 05 Y ESTADO DE ESTRATOS
# ============================================================

# ------------------------------------------------------------
# 1. Etiqueta de revisión temática
# ------------------------------------------------------------

for dataframe_hilo in comentarios_lote_05:
    dataframe_hilo[
        "revision_tematica"
    ] = "valido_directo"

for diagnostico in diagnosticos_lote_05:
    diagnostico[
        "revision_tematica"
    ] = "valido_directo"

# ------------------------------------------------------------
# 2. Consolidar comentarios
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            *comentarios_lote_05,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Consolidar diagnósticos
# ------------------------------------------------------------

diagnosticos_forocoches_ampliado = (
    pd.concat(
        [
            diagnosticos_forocoches_ampliado,
            pd.DataFrame(
                diagnosticos_lote_05
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Incidencias, si las hubiera
# ------------------------------------------------------------

if incidencias_lote_05:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                pd.DataFrame(
                    incidencias_lote_05
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo", "estado"],
            keep="last",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Reconstruir corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Función de calidad por estrato
# ------------------------------------------------------------

def evaluar_calidad_estrato(dataframe_estrato):
    distribucion = (
        dataframe_estrato
        .groupby(
            "id_hilo",
            as_index=False,
        )
        .agg(
            comentarios=(
                "id_mensaje",
                "nunique",
            )
        )
    )

    total_comentarios = int(
        distribucion[
            "comentarios"
        ].sum()
    )

    total_hilos = int(
        distribucion[
            "id_hilo"
        ].nunique()
    )

    total_usuarios = int(
        dataframe_estrato[
            "usuario_hash"
        ].nunique()
    )

    if total_comentarios > 0:
        proporciones = (
            distribucion["comentarios"]
            / total_comentarios
        )

        mayor_porcentaje = float(
            proporciones.max() * 100
        )

        indice_hhi = float(
            (proporciones ** 2).sum()
        )

        hilos_efectivos = float(
            1 / indice_hhi
        )

    else:
        mayor_porcentaje = 0
        indice_hhi = 0
        hilos_efectivos = 0

    estrato_suficiente = all([
        total_comentarios >= 500,
        total_hilos >= 12,
        mayor_porcentaje <= 25,
        hilos_efectivos >= 6,
    ])

    return pd.Series({
        "hilos": total_hilos,
        "comentarios": total_comentarios,
        "usuarios": total_usuarios,
        "mayor_hilo_pct": round(
            mayor_porcentaje,
            1,
        ),
        "hhi": round(
            indice_hhi,
            3,
        ),
        "hilos_efectivos": round(
            hilos_efectivos,
            1,
        ),
        "estrato_suficiente": (
            estrato_suficiente
        ),
    })

# ------------------------------------------------------------
# 7. Evaluar las ocho combinaciones
# ------------------------------------------------------------

filas_estado_estratos = []

for (
    tema_actual,
    anio_actual,
), dataframe_estrato in (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
):
    metricas = evaluar_calidad_estrato(
        dataframe_estrato
    )

    filas_estado_estratos.append({
        "tema": tema_actual,
        "anio_objetivo": int(
            anio_actual
        ),
        **metricas.to_dict(),
    })

estado_estratos = (
    pd.DataFrame(
        filas_estado_estratos
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Estratos suficientes:",
    int(
        estado_estratos[
            "estrato_suficiente"
        ].sum()
    ),
    "de",
    len(estado_estratos)
)

display(
    estado_estratos
)

Hilos válidos totales: 36
Comentarios anuales totales: 3376
Estratos suficientes: 1 de 8


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos,estrato_suficiente
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,True
1,inmigracion,2016,3,244,189,75.0,0.609,1.6,False
2,inmigracion,2019,3,408,352,55.1,0.415,2.4,False
3,inmigracion,2023,3,368,251,82.1,0.700,1.4,False
4,lgtbi,2015,2,644,566,93.5,0.878,1.1,False
5,lgtbi,2016,3,183,141,54.1,0.463,2.2,False
6,lgtbi,2019,3,485,362,56.3,0.455,2.2,False
7,lgtbi,2023,3,192,166,47.4,0.410,2.4,False


## 2016 inmigración

In [117]:
# ============================================================
# CELDA 43 — CANDIDATOS INMIGRACIÓN–2016
# ============================================================

nuevos_candidatos_inmigracion_2016 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4856902",
        "titulo_resultado": (
            "Somos un país de emigrantes"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4856902"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "refugiados asilo países receptores 2016"
        ),
        "lote_descubrimiento": "ampliacion_06",
        "criterio_inclusion": (
            "Debate sobre solicitudes de asilo, países "
            "receptores y distribución de refugiados"
        ),
        "subtema_preliminar": (
            "refugiados_asilo_internacional"
        ),
        "requiere_revision_tematica": True,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "5068993",
        "titulo_resultado": (
            "¿Qué opinión tenéis de este artículo "
            "sobre inmigración?"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=5068993"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigración estado del bienestar 2016"
        ),
        "lote_descubrimiento": "ampliacion_06",
        "criterio_inclusion": (
            "Discusión sobre aportaciones fiscales, "
            "prestaciones y percepción social de inmigrantes"
        ),
        "subtema_preliminar": (
            "inmigracion_estado_bienestar"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "5149331",
        "titulo_resultado": (
            'Refugiado sirio: "Esperaba más de España"'
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=5149331"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "refugiado sirio ayudas España 2016"
        ),
        "lote_descubrimiento": "ampliacion_06",
        "criterio_inclusion": (
            "Debate sobre asilo, ayudas públicas, vivienda, "
            "universidad y reunificación familiar"
        ),
        "subtema_preliminar": (
            "refugiados_ayudas_reagrupacion"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "5306998",
        "titulo_resultado": (
            "Valencia: inmigrantes sin papeles empadronados "
            "cobrarán hasta 532 euros"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=5306998"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigrantes sin papeles ayudas Valencia 2016"
        ),
        "lote_descubrimiento": "ampliacion_06",
        "criterio_inclusion": (
            "Discusión sobre irregularidad administrativa, "
            "empadronamiento y renta de inclusión"
        ),
        "subtema_preliminar": (
            "inmigracion_irregular_ayudas"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "5320037",
        "titulo_resultado": (
            "Progres, asumidlo: el nacionalismo identitario "
            "tomará las riendas de Europa"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=5320037"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigración masiva multiculturalismo 2016"
        ),
        "lote_descubrimiento": "ampliacion_06",
        "criterio_inclusion": (
            "La inmigración masiva y el multiculturalismo "
            "son argumentos centrales del mensaje inicial"
        ),
        "subtema_preliminar": (
            "multiculturalismo_identidad_politica"
        ),
        "requiere_revision_tematica": False,
    },
])

# ------------------------------------------------------------
# Normalización y eliminación de duplicados
# ------------------------------------------------------------

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"].astype(str)
)

nuevos_candidatos_inmigracion_2016[
    "id_hilo"
] = (
    nuevos_candidatos_inmigracion_2016[
        "id_hilo"
    ].astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_realmente_nuevos_lote_06 = (
    nuevos_candidatos_inmigracion_2016.loc[
        ~nuevos_candidatos_inmigracion_2016[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos_lote_06,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Preparación del lote
# ------------------------------------------------------------

ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

pendientes_lote_06 = (
    candidatos_realmente_nuevos_lote_06.loc[
        ~candidatos_realmente_nuevos_lote_06[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 06:",
    len(candidatos_realmente_nuevos_lote_06)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos pendientes del lote 06:",
    len(pendientes_lote_06)
)

display(
    pendientes_lote_06[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 06: 5
Candidatos totales en el catálogo: 43
Candidatos pendientes del lote 06: 5


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,4856902,Somos un país de emigrantes,refugiados_asilo_internacional,True
1,5068993,¿Qué opinión tenéis de este artículo sobre inm...,inmigracion_estado_bienestar,False
2,5149331,"Refugiado sirio: ""Esperaba más de España""",refugiados_ayudas_reagrupacion,False
3,5306998,Valencia: inmigrantes sin papeles empadronados...,inmigracion_irregular_ayudas,False
4,5320037,"Progres, asumidlo: el nacionalismo identitario...",multiculturalismo_identidad_politica,False


In [118]:
# ============================================================
# CELDA 44 — DESCARGA DEL LOTE 06
# ============================================================

(
    comentarios_lote_06,
    diagnosticos_lote_06,
    incidencias_lote_06,
) = procesar_lote_forocoches(
    candidatos_pendientes=pendientes_lote_06,
    nombre_lote="ampliacion_06",
    sesion=sesion,
)


[1/5] inmigracion | 2016 | hilo 4856902
Somos un país de emigrantes
Primera página pública: 30 mensajes detectados
Hilo 4856902: 5 página(s)
  Página 1/5: 30 comentarios
  Página 2/5: 30 comentarios
  Página 3/5: 30 comentarios
  Página 4/5: 30 comentarios
  Página 5/5: 12 comentarios
Candidato técnicamente válido: 132 comentarios totales | 132 comentarios de 2016

[2/5] inmigracion | 2016 | hilo 5068993
¿Qué opinión tenéis de este artículo sobre inmigración?
Primera página pública: 14 mensajes detectados
Hilo 5068993: 1 página(s)
  Página 1/1: 14 comentarios
Candidato técnicamente válido: 14 comentarios totales | 14 comentarios de 2016

[3/5] inmigracion | 2016 | hilo 5149331
Refugiado sirio: "Esperaba más de España"
Primera página pública: 30 mensajes detectados
Hilo 5149331: 55 página(s)
  Página 1/55: 30 comentarios
  Página 2/55: 30 comentarios
  Página 3/55: 30 comentarios
  Página 4/55: 30 comentarios
  Página 5/55: 30 comentarios
  Página 6/55: 30 comentarios
  Página 7/55: 30

In [119]:
# ============================================================
# CELDA 45 — REVISIÓN TEMÁTICA DEL HILO 4856902
# ============================================================

id_hilo_revision = "4856902"

comentarios_revision_4856902 = (
    pd.concat(
        comentarios_lote_06,
        ignore_index=True,
        sort=False,
    )
    .loc[
        lambda df:
        df["id_hilo"].astype(str)
        == id_hilo_revision
    ]
    .copy()
    .sort_values(
        "numero_mensaje"
    )
    .reset_index(drop=True)
)

columna_texto_revision = (
    "texto_sin_citas"
    if (
        "texto_sin_citas"
        in comentarios_revision_4856902.columns
    )
    else "texto_original"
)

texto_revision_normalizado = (
    comentarios_revision_4856902[
        columna_texto_revision
    ]
    .fillna("")
    .astype(str)
    .str.lower()
)

# Refugiados, inmigración y asilo
patron_refugiados_inmigracion = (
    r"\brefugiad"
    r"|\binmigr"
    r"|\basilo"
    r"|\bsiri"
    r"|\bmigrant"
    r"|solicitud(?:es)?\s+de\s+asilo"
    r"|país(?:es)?\s+de\s+acogida"
    r"|acoger"
)

# Emigración española
patron_emigracion_espanola = (
    r"\bemigr"
    r"|españoles\s+fuera"
    r"|españoles\s+en\s+el\s+extranjero"
    r"|irse\s+de\s+españa"
    r"|salir\s+de\s+españa"
    r"|trabajar\s+fuera"
    r"|retorn"
    r"|\bvolver"
)

comentarios_revision_4856902[
    "menciona_refugiados_inmigracion"
] = texto_revision_normalizado.str.contains(
    patron_refugiados_inmigracion,
    regex=True,
    na=False,
)

comentarios_revision_4856902[
    "menciona_emigracion_espanola"
] = texto_revision_normalizado.str.contains(
    patron_emigracion_espanola,
    regex=True,
    na=False,
)

resumen_revision_4856902 = pd.DataFrame({
    "indicador": [
        "comentarios_totales",
        "mencionan_refugiados_inmigracion",
        "mencionan_emigracion_espanola",
    ],
    "numero_comentarios": [
        len(comentarios_revision_4856902),
        int(
            comentarios_revision_4856902[
                "menciona_refugiados_inmigracion"
            ].sum()
        ),
        int(
            comentarios_revision_4856902[
                "menciona_emigracion_espanola"
            ].sum()
        ),
    ],
})

display(
    resumen_revision_4856902
)

# Muestra de todos los comentarios que presentan
# indicadores de alguno de los dos encuadres.
muestra_revision_4856902 = (
    comentarios_revision_4856902.loc[
        comentarios_revision_4856902[
            "menciona_refugiados_inmigracion"
        ]
        | comentarios_revision_4856902[
            "menciona_emigracion_espanola"
        ]
    ]
    .copy()
)

with pd.option_context(
    "display.max_colwidth",
    500,
    "display.max_rows",
    150,
):
    display(
        muestra_revision_4856902[
            [
                "numero_mensaje",
                "menciona_refugiados_inmigracion",
                "menciona_emigracion_espanola",
                columna_texto_revision,
            ]
        ]
    )

,indicador,numero_comentarios
0,comentarios_totales,132
1,mencionan_refugiados_inmigracion,40
2,mencionan_emigracion_espanola,17


,numero_mensaje,menciona_refugiados_inmigracion,menciona_emigracion_espanola,texto_sin_citas
0,1,True,True,"Este hilo va para todos aquellos islamofobios, para que entiendan un poco mejor lo que está pasando en nuestro país respecto a los países árabes .\nEn los últimos años se ha reducido mucho el número de inmigrantes que llega a nuestro país y esté bajón está presente en todos los países. Aquí podéis ver la fuente:\nhttp://www.ine.es/jaxi/Datos.htm?pat...px&type=pcaxis\nPor tanto los refugiados no son gente que viene a buscar paguitas sino familias que se van de un país en guerra. Y son muy poc..."
9,10,True,False,"1) Imagen de solicitudes de asilo. Das por hecho que cada Sirio solo ha solicitado asilo en un pais, y no en todos los posibles. Venga pues, vamos a asumir que cada sirio solo ha hecho 1 solicitud de asilo en 1 único pais.\n2) Fijate si eres corto, que me pones una imagen en la que asumes que se han refugiado en Egipto tantos como en Alemania, cuando Alemania esta bastante mas lejos y hay que atravesar más países. Ergo das la razón a mi mensaje.\n3) Imagen en la que en Grecia apenas hay 10.0..."
10,11,True,False,"Me llamas corto cuando ni te has fijado que los azules son solicitudes(por tanto pueden o no estar en el país) Y los morados son Refugiados registrados! Turquía Sola tiene 2 millones que es una barbaridad más de lo que tiene toda Europa.\nHe contestado a tu respuesta irónica de que todos van hacía Europa. Y como ves el 90% o más de los refugiados los acogen los países vecinos, ya se que te jode pero es la realidad\nSe te ve atrasado."
11,12,True,False,"Salen inmigrantes que vinieron antes y algun que otro universitario.\nEntran africanos, moros sudamernicanos y rumanos\nHay mas Ingleses en España que al reves"
13,14,True,False,"Tienes razón en que los datos están desactualizados, por ejemplo en el Líbano ya van camino de los 2 millones. En un país de 4 millones de habitantes. ¿Te imaginas que en España hubiera 20 millones de refugiados? Eso es lo que ocurre en el Líbano, en un país en que la mitad de la población es cristiana y hasta hoy en día no ha habido revuelta alguna ni flipaos hablando de invasión o guerras."
14,15,True,True,"El hecho de que la gente de éste país haya emigrado no hace que los que vayan a venir sean buenas personas, eso lo entiendes, no?\nLo digo porque leyendo lo que has puesto en el segundo párrafo das por hecho de que si alguien se va, otra inmigrante bueno lo reemplazará. Y aquí todos sabemos que no es así; sólo hace falta ver las noticias.\nEstamos creando un nido de parásitos a nivel global; si en Francia nos dieran 2000 pavos a los españoles simplemente por cruzar la frontera no sólo querrí..."
16,17,True,False,"Nuevamente, muestras retraso y comprensión lectora tirando a nula.\nComo sabes si esos refugiados en países limitrofes , no han solicitado asilo tambien en Europa? Esa era la primera idea, y a partir de ahí el resto vienen bajo el supuesto de que cada refugiado sólo ha pedido asilo una vez (resumiendo para los cortos como tú, que no se compute más de una vez a cada refugiado).\nOtra muestra de la comprensión lectora nula: No he dicho que la mayoría van a Europa, sino a Alemania (o eso preten..."
18,19,False,True,Despues del owned que te has comido tienes los santos cojones de volver a seguir arrastrandote?
19,20,True,False,"A ver, sólo el año pasado han entrado en Europa (Alemania y Suecia principalmente) más de UN MILLON de refugiados. En ese mapa no están.\nEl mapa está mal o es que ahora ya no son refugiados sirios?"
30,31,False,True,"¿Lo de tercermundizar zonas de ciudades creando guetos, lo de crear bandas callejeras o mafiosas, lo de imponer sus costumbres bajo la amenaza de llamarte malvado rasista, tambien lo hacian los emigrantes españoles?"


In [120]:
# ============================================================
# CELDA 46 — CONSOLIDACIÓN DEL LOTE 06
# ============================================================

# ------------------------------------------------------------
# 1. Etiquetar la revisión temática
# ------------------------------------------------------------

for dataframe_hilo in comentarios_lote_06:
    dataframe_hilo[
        "revision_tematica"
    ] = "valido_directo"

for diagnostico in diagnosticos_lote_06:
    diagnostico[
        "revision_tematica"
    ] = "valido_directo"

# ------------------------------------------------------------
# 2. Consolidar comentarios
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            *comentarios_lote_06,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Consolidar diagnósticos
# ------------------------------------------------------------

diagnosticos_forocoches_ampliado = (
    pd.concat(
        [
            diagnosticos_forocoches_ampliado,
            pd.DataFrame(
                diagnosticos_lote_06
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Consolidar incidencias
# ------------------------------------------------------------

if incidencias_lote_06:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                pd.DataFrame(
                    incidencias_lote_06
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo", "estado"],
            keep="last",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Reconstruir corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Nueva evaluación metodológica
# ------------------------------------------------------------

def evaluar_cobertura_estrato(
    dataframe_estrato
):
    distribucion = (
        dataframe_estrato
        .groupby(
            "id_hilo",
            as_index=False,
        )
        .agg(
            comentarios=(
                "id_mensaje",
                "nunique",
            )
        )
    )

    total_hilos = int(
        distribucion[
            "id_hilo"
        ].nunique()
    )

    total_comentarios = int(
        distribucion[
            "comentarios"
        ].sum()
    )

    total_usuarios = int(
        dataframe_estrato[
            "usuario_hash"
        ].nunique()
    )

    proporciones = (
        distribucion["comentarios"]
        / total_comentarios
    )

    mayor_hilo_pct = float(
        proporciones.max() * 100
    )

    hhi = float(
        (proporciones ** 2).sum()
    )

    hilos_efectivos = float(
        1 / hhi
    )

    # Cobertura mínima para cerrar la extracción.
    # La concentración no excluye el estrato:
    # determina el tratamiento posterior en NLP.
    cobertura_suficiente = all([
        total_hilos >= 12,
        total_comentarios >= 500,
        total_usuarios >= 300,
    ])

    requiere_equilibrado_nlp = (
        mayor_hilo_pct > 25
    )

    if not cobertura_suficiente:
        estado = "ampliar_extraccion"
    elif requiere_equilibrado_nlp:
        estado = "cerrado_equilibrar_nlp"
    else:
        estado = "cerrado_equilibrado_natural"

    return pd.Series({
        "hilos": total_hilos,
        "comentarios": total_comentarios,
        "usuarios": total_usuarios,
        "mayor_hilo_pct": round(
            mayor_hilo_pct,
            1,
        ),
        "hhi": round(
            hhi,
            3,
        ),
        "hilos_efectivos_raw": round(
            hilos_efectivos,
            1,
        ),
        "cobertura_suficiente": (
            cobertura_suficiente
        ),
        "requiere_equilibrado_nlp": (
            requiere_equilibrado_nlp
        ),
        "estado": estado,
    })

# ------------------------------------------------------------
# 7. Panel de los ocho estratos
# ------------------------------------------------------------

filas_estado_estratos = []

for (
    tema_actual,
    anio_actual,
), dataframe_estrato in (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
):
    metricas = evaluar_cobertura_estrato(
        dataframe_estrato
    )

    filas_estado_estratos.append({
        "tema": tema_actual,
        "anio_objetivo": int(
            anio_actual
        ),
        **metricas.to_dict(),
    })

estado_estratos = (
    pd.DataFrame(
        filas_estado_estratos
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Estratos con cobertura suficiente:",
    int(
        estado_estratos[
            "cobertura_suficiente"
        ].sum()
    ),
    "de",
    len(estado_estratos)
)

display(
    estado_estratos
)

Hilos válidos totales: 41
Comentarios anuales totales: 5473
Estratos con cobertura suficiente: 1 de 8


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos_raw,cobertura_suficiente,requiere_equilibrado_nlp,estado
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,True,False,cerrado_equilibrado_natural
1,inmigracion,2016,8,2341,1531,66.3,0.464,2.2,False,True,ampliar_extraccion
2,inmigracion,2019,3,408,352,55.1,0.415,2.4,False,True,ampliar_extraccion
3,inmigracion,2023,3,368,251,82.1,0.700,1.4,False,True,ampliar_extraccion
4,lgtbi,2015,2,644,566,93.5,0.878,1.1,False,True,ampliar_extraccion
5,lgtbi,2016,3,183,141,54.1,0.463,2.2,False,True,ampliar_extraccion
6,lgtbi,2019,3,485,362,56.3,0.455,2.2,False,True,ampliar_extraccion
7,lgtbi,2023,3,192,166,47.4,0.410,2.4,False,True,ampliar_extraccion


In [122]:
# ============================================================
# CELDA 47 — LOTE 07: DIVERSIDAD DE HILOS INMIGRACIÓN–2016
# ============================================================

nuevos_candidatos_lote_07 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4759444",
        "titulo_resultado": (
            "El Gobierno de Rajoy concedió 4.400 "
            "nacionalidades por decreto; 4.200 a sefardíes"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4759444"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "nacionalidades inmigración enero 2016"
        ),
        "lote_descubrimiento": "ampliacion_07",
        "criterio_inclusion": (
            "Debate sobre adquisición de nacionalidad, "
            "cartas de naturaleza y política migratoria"
        ),
        "subtema_preliminar": (
            "nacionalidad_ciudadania"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4810374",
        "titulo_resultado": (
            "Trabajar sin papeles, ¿regularización? "
            "Tema serio"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4810374"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "trabajo sin papeles regularización 2016"
        ),
        "lote_descubrimiento": "ampliacion_07",
        "criterio_inclusion": (
            "Situación irregular, permiso de trabajo, "
            "residencia y empleo informal"
        ),
        "subtema_preliminar": (
            "trabajo_irregular_regularizacion"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "4814861",
        "titulo_resultado": (
            "Conexión entre inmigración y delincuencia "
            "realmente no existe"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=4814861"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "inmigración delincuencia febrero 2016"
        ),
        "lote_descubrimiento": "ampliacion_07",
        "criterio_inclusion": (
            "Debate explícito sobre percepción social, "
            "inmigración y delincuencia"
        ),
        "subtema_preliminar": (
            "inmigracion_delincuencia_percepcion"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2016,
        "id_hilo": "5234727",
        "titulo_resultado": (
            "Refugees cubanos"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=5234727"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "refugiados cubanos noviembre 2016"
        ),
        "lote_descubrimiento": "ampliacion_07",
        "criterio_inclusion": (
            "Discusión sobre refugiados políticos, "
            "inmigración económica y vías de asilo"
        ),
        "subtema_preliminar": (
            "refugiados_politicos_asilo"
        ),
        "requiere_revision_tematica": False,
    },
])

# ------------------------------------------------------------
# Normalización y catálogo
# ------------------------------------------------------------

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"].astype(str)
)

nuevos_candidatos_lote_07["id_hilo"] = (
    nuevos_candidatos_lote_07[
        "id_hilo"
    ].astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_realmente_nuevos_lote_07 = (
    nuevos_candidatos_lote_07.loc[
        ~nuevos_candidatos_lote_07[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos_lote_07,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Preparar pendientes
# ------------------------------------------------------------

ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

pendientes_lote_07 = (
    candidatos_realmente_nuevos_lote_07.loc[
        ~candidatos_realmente_nuevos_lote_07[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 07:",
    len(candidatos_realmente_nuevos_lote_07)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos pendientes del lote 07:",
    len(pendientes_lote_07)
)

display(
    pendientes_lote_07[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
        ]
    ]
)

Candidatos incorporados en el lote 07: 4
Candidatos totales en el catálogo: 47
Candidatos pendientes del lote 07: 4


,id_hilo,titulo_resultado,subtema_preliminar
0,4759444,El Gobierno de Rajoy concedió 4.400 nacionalid...,nacionalidad_ciudadania
1,4810374,"Trabajar sin papeles, ¿regularización? Tema serio",trabajo_irregular_regularizacion
2,4814861,Conexión entre inmigración y delincuencia real...,inmigracion_delincuencia_percepcion
3,5234727,Refugees cubanos,refugiados_politicos_asilo


In [123]:
# ============================================================
# CELDA 48 — DESCARGA DEL LOTE 07
# ============================================================

(
    comentarios_lote_07,
    diagnosticos_lote_07,
    incidencias_lote_07,
) = procesar_lote_forocoches(
    candidatos_pendientes=pendientes_lote_07,
    nombre_lote="ampliacion_07",
    sesion=sesion,
)


[1/4] inmigracion | 2016 | hilo 4759444
El Gobierno de Rajoy concedió 4.400 nacionalidades por decreto; 4.200 a sefardíes
Primera página pública: 30 mensajes detectados
Hilo 4759444: 1 página(s)
  Página 1/1: 30 comentarios
Candidato técnicamente válido: 30 comentarios totales | 30 comentarios de 2016

[2/4] inmigracion | 2016 | hilo 4810374
Trabajar sin papeles, ¿regularización? Tema serio
Primera página pública: 12 mensajes detectados
Hilo 4810374: 1 página(s)
  Página 1/1: 12 comentarios
Candidato técnicamente válido: 12 comentarios totales | 12 comentarios de 2016

[3/4] inmigracion | 2016 | hilo 4814861
Conexión entre inmigración y delincuencia realmente no existe
Primera página pública: 30 mensajes detectados
Hilo 4814861: 4 página(s)
  Página 1/4: 30 comentarios
  Página 2/4: 30 comentarios
  Página 3/4: 30 comentarios
  Página 4/4: 11 comentarios
Candidato técnicamente válido: 101 comentarios totales | 101 comentarios de 2016

[4/4] inmigracion | 2016 | hilo 5234727
Refugees c

In [124]:
# ============================================================
# CELDA 49 — CONSOLIDACIÓN DEL LOTE 07
# ============================================================

# ------------------------------------------------------------
# 1. Etiquetas de revisión
# ------------------------------------------------------------

for dataframe_hilo in comentarios_lote_07:
    dataframe_hilo[
        "revision_tematica"
    ] = "valido_directo"

for diagnostico in diagnosticos_lote_07:
    diagnostico[
        "revision_tematica"
    ] = "valido_directo"

# ------------------------------------------------------------
# 2. Consolidar comentarios
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            *comentarios_lote_07,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Consolidar diagnósticos
# ------------------------------------------------------------

diagnosticos_forocoches_ampliado = (
    pd.concat(
        [
            diagnosticos_forocoches_ampliado,
            pd.DataFrame(
                diagnosticos_lote_07
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Incidencias, si existieran
# ------------------------------------------------------------

if incidencias_lote_07:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                pd.DataFrame(
                    incidencias_lote_07
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo", "estado"],
            keep="last",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Reconstruir corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Actualizar el panel metodológico
# ------------------------------------------------------------

filas_estado_estratos = []

for (
    tema_actual,
    anio_actual,
), dataframe_estrato in (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
):
    metricas = evaluar_cobertura_estrato(
        dataframe_estrato
    )

    filas_estado_estratos.append({
        "tema": tema_actual,
        "anio_objetivo": int(
            anio_actual
        ),
        **metricas.to_dict(),
    })

estado_estratos = (
    pd.DataFrame(
        filas_estado_estratos
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Estratos con cobertura suficiente:",
    int(
        estado_estratos[
            "cobertura_suficiente"
        ].sum()
    ),
    "de",
    len(estado_estratos)
)

display(
    estado_estratos
)

Hilos válidos totales: 45
Comentarios anuales totales: 5653
Estratos con cobertura suficiente: 2 de 8


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos_raw,cobertura_suficiente,requiere_equilibrado_nlp,estado
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,True,False,cerrado_equilibrado_natural
1,inmigracion,2016,12,2521,1614,61.6,0.402,2.5,True,True,cerrado_equilibrar_nlp
2,inmigracion,2019,3,408,352,55.1,0.415,2.4,False,True,ampliar_extraccion
3,inmigracion,2023,3,368,251,82.1,0.700,1.4,False,True,ampliar_extraccion
4,lgtbi,2015,2,644,566,93.5,0.878,1.1,False,True,ampliar_extraccion
5,lgtbi,2016,3,183,141,54.1,0.463,2.2,False,True,ampliar_extraccion
6,lgtbi,2019,3,485,362,56.3,0.455,2.2,False,True,ampliar_extraccion
7,lgtbi,2023,3,192,166,47.4,0.410,2.4,False,True,ampliar_extraccion


## 2019 Inmigración

In [125]:
# ============================================================
# CELDA 50 — LOTE 08: CANDIDATOS INMIGRACIÓN–2019
# ============================================================

nuevos_candidatos_lote_08 = pd.DataFrame([
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7204675",
        "titulo_resultado": (
            "La Generalitat de Catalunya paga "
            "100 € al día por cada MENA"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7204675"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "MENAS costes acogida 2019"
        ),
        "lote_descubrimiento": "ampliacion_08",
        "criterio_inclusion": (
            "Debate sobre menores migrantes, costes de "
            "acogida, centros y gestión pública"
        ),
        "subtema_preliminar": (
            "menas_costes_acogida"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7285578",
        "titulo_resultado": (
            "Salvini, sobre el Open Arms: "
            "Que España los haga volver o lo haremos nosotros"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7285578"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "Open Arms inmigración julio 2019"
        ),
        "lote_descubrimiento": "ampliacion_08",
        "criterio_inclusion": (
            "Debate sobre rescate marítimo, ONG, fronteras "
            "y devolución de migrantes"
        ),
        "subtema_preliminar": (
            "rescate_maritimo_open_arms"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7292972",
        "titulo_resultado": (
            "La verdad sobre los MENAS"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7292972"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "MENAS estadísticas julio 2019"
        ),
        "lote_descubrimiento": "ampliacion_08",
        "criterio_inclusion": (
            "Discusión sobre número de menores migrantes, "
            "delincuencia y evidencia estadística"
        ),
        "subtema_preliminar": (
            "menas_delincuencia_estadisticas"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7356102",
        "titulo_resultado": (
            "Venezuela se convierte en el mayor éxodo "
            "de la historia: supera a afganos y sirios"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7356102"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "Venezuela éxodo refugiados agosto 2019"
        ),
        "lote_descubrimiento": "ampliacion_08",
        "criterio_inclusion": (
            "Refugiados, solicitudes de asilo, "
            "desplazamiento y migración venezolana"
        ),
        "subtema_preliminar": (
            "exodo_venezolano_asilo"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7365765",
        "titulo_resultado": (
            "¿Solución al tema de la inmigración?"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7365765"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "solución inmigración agosto 2019"
        ),
        "lote_descubrimiento": "ampliacion_08",
        "criterio_inclusion": (
            "Propuestas sobre fronteras, deportación, "
            "visados, rescate marítimo e integración"
        ),
        "subtema_preliminar": (
            "politica_migratoria_soluciones"
        ),
        "requiere_revision_tematica": False,
    },
    {
        "tema": "inmigracion",
        "anio_objetivo": 2019,
        "id_hilo": "7411938",
        "titulo_resultado": (
            "Os dejo mi opinión sobre la inmigración"
        ),
        "url_hilo": (
            "https://forocoches.com/foro/"
            "showthread.php?t=7411938"
        ),
        "fuente_descubrimiento": "buscador_externo",
        "consulta_descubrimiento": (
            "opinión inmigración integración septiembre 2019"
        ),
        "lote_descubrimiento": "ampliacion_08",
        "criterio_inclusion": (
            "Debate sobre envejecimiento, pensiones, "
            "integración, ayudas y delincuencia"
        ),
        "subtema_preliminar": (
            "integracion_demografia_bienestar"
        ),
        "requiere_revision_tematica": False,
    },
])

# ------------------------------------------------------------
# Normalización y catálogo
# ------------------------------------------------------------

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"].astype(str)
)

nuevos_candidatos_lote_08["id_hilo"] = (
    nuevos_candidatos_lote_08[
        "id_hilo"
    ].astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_realmente_nuevos_lote_08 = (
    nuevos_candidatos_lote_08.loc[
        ~nuevos_candidatos_lote_08[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_realmente_nuevos_lote_08,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Preparar pendientes
# ------------------------------------------------------------

ids_hilos_validos_actuales = set(
    comentarios_forocoches_bruto_ampliado[
        "id_hilo"
    ]
    .astype(str)
    .unique()
)

pendientes_lote_08 = (
    candidatos_realmente_nuevos_lote_08.loc[
        ~candidatos_realmente_nuevos_lote_08[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_hilos_validos_actuales)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 08:",
    len(candidatos_realmente_nuevos_lote_08)
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos)
)

print(
    "Candidatos pendientes del lote 08:",
    len(pendientes_lote_08)
)

display(
    pendientes_lote_08[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
        ]
    ]
)

Candidatos incorporados en el lote 08: 6
Candidatos totales en el catálogo: 53
Candidatos pendientes del lote 08: 6


,id_hilo,titulo_resultado,subtema_preliminar
0,7204675,La Generalitat de Catalunya paga 100 € al día ...,menas_costes_acogida
1,7285578,"Salvini, sobre el Open Arms: Que España los ha...",rescate_maritimo_open_arms
2,7292972,La verdad sobre los MENAS,menas_delincuencia_estadisticas
3,7356102,Venezuela se convierte en el mayor éxodo de la...,exodo_venezolano_asilo
4,7365765,¿Solución al tema de la inmigración?,politica_migratoria_soluciones
5,7411938,Os dejo mi opinión sobre la inmigración,integracion_demografia_bienestar


In [126]:
# ============================================================
# CELDA 51 — DESCARGA DEL LOTE 08
# ============================================================

(
    comentarios_lote_08,
    diagnosticos_lote_08,
    incidencias_lote_08,
) = procesar_lote_forocoches(
    candidatos_pendientes=pendientes_lote_08,
    nombre_lote="ampliacion_08",
    sesion=sesion,
)


[1/6] inmigracion | 2019 | hilo 7204675
La Generalitat de Catalunya paga 100 € al día por cada MENA
Primera página pública: 12 mensajes detectados
Hilo 7204675: 1 página(s)
  Página 1/1: 12 comentarios
Candidato técnicamente válido: 12 comentarios totales | 12 comentarios de 2019

[2/6] inmigracion | 2019 | hilo 7285578
Salvini, sobre el Open Arms: Que España los haga volver o lo haremos nosotros
Primera página pública: 30 mensajes detectados
Hilo 7285578: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 30 comentarios
  Página 3/3: 3 comentarios
Candidato técnicamente válido: 63 comentarios totales | 63 comentarios de 2019

[3/6] inmigracion | 2019 | hilo 7292972
La verdad sobre los MENAS
Primera página pública: 30 mensajes detectados
Hilo 7292972: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 29 comentarios
Candidato técnicamente válido: 59 comentarios totales | 52 comentarios de 2019

[4/6] inmigracion | 2019 | hilo 7356102
Venezuela se convierte en el mayor éxodo de

In [127]:
# ============================================================
# CELDA 52 — CONSOLIDACIÓN DEL LOTE 08
# ============================================================

for dataframe_hilo in comentarios_lote_08:
    dataframe_hilo[
        "revision_tematica"
    ] = "valido_directo"

for diagnostico in diagnosticos_lote_08:
    diagnostico[
        "revision_tematica"
    ] = "valido_directo"

# Comentarios
comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            *comentarios_lote_08,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# Diagnósticos
diagnosticos_forocoches_ampliado = (
    pd.concat(
        [
            diagnosticos_forocoches_ampliado,
            pd.DataFrame(
                diagnosticos_lote_08
            ),
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="last",
    )
    .reset_index(drop=True)
)

# Incidencias
if incidencias_lote_08:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                pd.DataFrame(
                    incidencias_lote_08
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates(
            subset=["id_hilo", "estado"],
            keep="last",
        )
        .reset_index(drop=True)
    )

# Corpus anual
comentarios_forocoches_anual_ampliado = (
    comentarios_forocoches_bruto_ampliado.loc[
        comentarios_forocoches_bruto_ampliado[
            "anio_comentario"
        ]
        == comentarios_forocoches_bruto_ampliado[
            "anio_objetivo"
        ]
    ]
    .copy()
    .drop_duplicates(
        subset=["id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

# Actualizar panel
filas_estado_estratos = []

for (
    tema_actual,
    anio_actual,
), dataframe_estrato in (
    comentarios_forocoches_anual_ampliado
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )
):
    metricas = evaluar_cobertura_estrato(
        dataframe_estrato
    )

    filas_estado_estratos.append({
        "tema": tema_actual,
        "anio_objetivo": int(
            anio_actual
        ),
        **metricas.to_dict(),
    })

estado_estratos = (
    pd.DataFrame(
        filas_estado_estratos
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique()
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado)
)

print(
    "Estratos con cobertura suficiente:",
    int(
        estado_estratos[
            "cobertura_suficiente"
        ].sum()
    ),
    "de",
    len(estado_estratos)
)

display(
    estado_estratos
)

Hilos válidos totales: 51
Comentarios anuales totales: 5886
Estratos con cobertura suficiente: 2 de 8


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos_raw,cobertura_suficiente,requiere_equilibrado_nlp,estado
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,True,False,cerrado_equilibrado_natural
1,inmigracion,2016,12,2521,1614,61.6,0.402,2.5,True,True,cerrado_equilibrar_nlp
2,inmigracion,2019,9,641,502,35.1,0.194,5.2,False,True,ampliar_extraccion
3,inmigracion,2023,3,368,251,82.1,0.700,1.4,False,True,ampliar_extraccion
4,lgtbi,2015,2,644,566,93.5,0.878,1.1,False,True,ampliar_extraccion
5,lgtbi,2016,3,183,141,54.1,0.463,2.2,False,True,ampliar_extraccion
6,lgtbi,2019,3,485,362,56.3,0.455,2.2,False,True,ampliar_extraccion
7,lgtbi,2023,3,192,166,47.4,0.410,2.4,False,True,ampliar_extraccion


In [128]:
# ============================================================
# CELDA 53 — Lote 09: últimos candidatos de inmigración–2019
# ============================================================

nuevos_candidatos_lote_09 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2019,
            "id_hilo": "7022487",
            "titulo_resultado": "Conseguir permiso residencia",
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7022487"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"permiso residencia" inmigración 2019'
            ),
            "lote_descubrimiento": "ampliacion_09",
            "criterio_inclusion": (
                "Consulta directa sobre residencia, trabajo, "
                "regularización y llegada a España."
            ),
            "subtema_preliminar": (
                "residencia_regularizacion_trabajo"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2019,
            "id_hilo": "6962744",
            "titulo_resultado": (
                "Los hijos de migrantes tienen más difícil "
                "encontrar un empleo"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=6962744"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"hijos de migrantes" empleo 2019'
            ),
            "lote_descubrimiento": "ampliacion_09",
            "criterio_inclusion": (
                "Debate explícito sobre descendientes de "
                "migrantes, empleo, integración y discriminación."
            ),
            "subtema_preliminar": (
                "segunda_generacion_empleo_integracion"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2019,
            "id_hilo": "7514350",
            "titulo_resultado": (
                "La IZQUIERDA y las MENTIRAS"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7514350"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'inmigrantes ayudas expulsión 2019'
            ),
            "lote_descubrimiento": "ampliacion_09",
            "criterio_inclusion": (
                "El resultado contiene discusión sobre "
                "inmigración, ayudas públicas y expulsión, "
                "pero el título indica un debate político amplio."
            ),
            "subtema_preliminar": (
                "inmigracion_ayudas_discurso_politico"
            ),
            "requiere_revision_tematica": True,
        },
    ]
)

nuevos_candidatos_lote_09["id_hilo"] = (
    nuevos_candidatos_lote_09["id_hilo"].astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"].astype(str)
)

nuevos_candidatos_lote_09 = (
    nuevos_candidatos_lote_09[
        ~nuevos_candidatos_lote_09["id_hilo"].isin(
            ids_catalogados
        )
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevos_candidatos_lote_09,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["tema", "anio_objetivo", "id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_09 = (
    hilos_candidatos[
        (hilos_candidatos["lote_descubrimiento"]
         == "ampliacion_09")
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 09:",
    len(nuevos_candidatos_lote_09),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 09:",
    len(candidatos_pendientes_lote_09),
)

display(
    candidatos_pendientes_lote_09[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 09: 3
Candidatos totales en el catálogo: 56
Candidatos pendientes del lote 09: 3


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,7022487,Conseguir permiso residencia,residencia_regularizacion_trabajo,False
1,6962744,Los hijos de migrantes tienen más difícil enco...,segunda_generacion_empleo_integracion,False
2,7514350,La IZQUIERDA y las MENTIRAS,inmigracion_ayudas_discurso_politico,True


In [129]:
# ============================================================
# CELDA 54 — Procesamiento técnico del lote 09
# ============================================================

(
    comentarios_validos_lote_09,
    diagnosticos_validos_lote_09,
    incidencias_lote_09,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_09,
    nombre_lote="ampliacion_09",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    comentarios_validos_lote_09["id_hilo"]
    .astype(str)
    .nunique()
    if not comentarios_validos_lote_09.empty
    else 0,
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_09),
)

if not incidencias_lote_09.empty:
    display(incidencias_lote_09)


[1/3] inmigracion | 2019 | hilo 7022487
Conseguir permiso residencia
Primera página pública: 15 mensajes detectados
Hilo 7022487: 1 página(s)
  Página 1/1: 15 comentarios
Candidato técnicamente válido: 15 comentarios totales | 15 comentarios de 2019

[2/3] inmigracion | 2019 | hilo 6962744
Los hijos de migrantes tienen más difícil encontrar un empleo
Descartado: no contiene mensajes públicos reconocibles.

[3/3] inmigracion | 2019 | hilo 7514350
La IZQUIERDA y las MENTIRAS
Primera página pública: 30 mensajes detectados
Hilo 7514350: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 6 comentarios
Candidato técnicamente válido: 36 comentarios totales | 36 comentarios de 2019

Hilos técnicamente válidos: 2
Hilos descartados o con error: 1



AttributeError: 'list' object has no attribute 'empty'

In [131]:
# ============================================================
# CELDA 55 — Revisión temática del hilo 7514350
# ============================================================

import re
import pandas as pd

# ------------------------------------------------------------
# 1. Unificar los resultados del lote
# ------------------------------------------------------------

if isinstance(comentarios_validos_lote_09, list):

    dataframes_lote_09 = [
        df
        for df in comentarios_validos_lote_09
        if isinstance(df, pd.DataFrame) and not df.empty
    ]

    if dataframes_lote_09:
        comentarios_lote_09_df = pd.concat(
            dataframes_lote_09,
            ignore_index=True,
        )
    else:
        comentarios_lote_09_df = pd.DataFrame()

elif isinstance(comentarios_validos_lote_09, pd.DataFrame):

    comentarios_lote_09_df = (
        comentarios_validos_lote_09.copy()
    )

else:

    raise TypeError(
        "El resultado comentarios_validos_lote_09 "
        "no tiene un formato reconocido."
    )


# ------------------------------------------------------------
# 2. Seleccionar el hilo que requiere revisión
# ------------------------------------------------------------

if comentarios_lote_09_df.empty:

    raise ValueError(
        "No hay comentarios válidos disponibles "
        "en el lote 09."
    )

comentarios_lote_09_df["id_hilo"] = (
    comentarios_lote_09_df["id_hilo"]
    .astype(str)
)

hilo_revision_7514350 = (
    comentarios_lote_09_df[
        comentarios_lote_09_df["id_hilo"]
        .eq("7514350")
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Comentarios recuperados del hilo 7514350:",
    len(hilo_revision_7514350),
)

if hilo_revision_7514350.empty:

    raise ValueError(
        "No se encontraron comentarios del hilo 7514350."
    )


# ------------------------------------------------------------
# 3. Patrones temáticos
# ------------------------------------------------------------

patron_inmigracion = re.compile(
    r"\b("
    r"inmigr\w*|"
    r"migrant\w*|"
    r"extranj\w*|"
    r"refugiad\w*|"
    r"menas?|"
    r"sin\s+papeles|"
    r"irregular(?:es)?|"
    r"pateras?|"
    r"fronteras?|"
    r"deport\w*|"
    r"expuls\w*|"
    r"asilo"
    r")\b",
    flags=re.IGNORECASE,
)

patron_ayudas = re.compile(
    r"\b("
    r"ayudas?|"
    r"subvencion\w*|"
    r"pagas?|"
    r"prestacion\w*|"
    r"rentas?|"
    r"servicios?\s+sociales?|"
    r"estado\s+del\s+bienestar"
    r")\b",
    flags=re.IGNORECASE,
)


# ------------------------------------------------------------
# 4. Aplicar los indicadores
# ------------------------------------------------------------

texto_revision = (
    hilo_revision_7514350["texto_sin_citas"]
    .fillna("")
    .astype(str)
)

hilo_revision_7514350["menciona_inmigracion"] = (
    texto_revision.str.contains(
        patron_inmigracion,
        na=False,
    )
)

hilo_revision_7514350["menciona_ayudas"] = (
    texto_revision.str.contains(
        patron_ayudas,
        na=False,
    )
)

hilo_revision_7514350["menciona_ambos_encuadres"] = (
    hilo_revision_7514350["menciona_inmigracion"]
    & hilo_revision_7514350["menciona_ayudas"]
)


# ------------------------------------------------------------
# 5. Resumen cuantitativo
# ------------------------------------------------------------

total_comentarios = len(hilo_revision_7514350)

comentarios_inmigracion = int(
    hilo_revision_7514350[
        "menciona_inmigracion"
    ].sum()
)

comentarios_ayudas = int(
    hilo_revision_7514350[
        "menciona_ayudas"
    ].sum()
)

comentarios_ambos = int(
    hilo_revision_7514350[
        "menciona_ambos_encuadres"
    ].sum()
)

resumen_revision_7514350 = pd.DataFrame(
    {
        "indicador": [
            "comentarios_totales",
            "mencionan_inmigracion",
            "mencionan_ayudas",
            "mencionan_ambos_encuadres",
        ],
        "numero_comentarios": [
            total_comentarios,
            comentarios_inmigracion,
            comentarios_ayudas,
            comentarios_ambos,
        ],
        "porcentaje": [
            100.0,
            round(
                comentarios_inmigracion
                / total_comentarios
                * 100,
                1,
            ),
            round(
                comentarios_ayudas
                / total_comentarios
                * 100,
                1,
            ),
            round(
                comentarios_ambos
                / total_comentarios
                * 100,
                1,
            ),
        ],
    }
)

display(resumen_revision_7514350)


# ------------------------------------------------------------
# 6. Mostrar los mensajes relacionados con inmigración
# ------------------------------------------------------------

mensajes_tematicos_7514350 = (
    hilo_revision_7514350.loc[
        hilo_revision_7514350[
            "menciona_inmigracion"
        ],
        [
            "numero_mensaje",
            "menciona_inmigracion",
            "menciona_ayudas",
            "texto_sin_citas",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Mensajes mostrados para revisión temática:",
    len(mensajes_tematicos_7514350),
)

display(mensajes_tematicos_7514350)

Comentarios recuperados del hilo 7514350: 36


C:\Users\herre\AppData\Local\Temp\ipykernel_28460\2556322327.py:126: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  texto_revision.str.contains(
C:\Users\herre\AppData\Local\Temp\ipykernel_28460\2556322327.py:133: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  texto_revision.str.contains(


,indicador,numero_comentarios,porcentaje
0,comentarios_totales,36,100.0
1,mencionan_inmigracion,4,11.1
2,mencionan_ayudas,2,5.6
3,mencionan_ambos_encuadres,2,5.6


Mensajes mostrados para revisión temática: 4


,numero_mensaje,menciona_inmigracion,menciona_ayudas,texto_sin_citas
0,16,True,True,A ti te han dado alguna paga etc? O te roban d...
1,22,True,True,¿Tienen derecho los inmigrantes a las mismas a...
2,24,True,False,Tú lo estás diciendo hijo mío que estás en la ...
3,25,True,False,Tampoco quiero irme al otro extremo como es vo...


In [135]:
# ============================================================
# CELDA 56 — Decisión temática definitiva del lote 09
# ============================================================

import pandas as pd

ID_APROBADO_LOTE_09 = "7022487"
ID_EXCLUIDO_LOTE_09 = "7514350"

# ------------------------------------------------------------
# 1. Comentarios aprobados
# ------------------------------------------------------------

comentarios_aprobados_lote_09 = (
    comentarios_lote_09_df[
        comentarios_lote_09_df["id_hilo"]
        .astype(str)
        .eq(ID_APROBADO_LOTE_09)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Unificar los diagnósticos del lote
# ------------------------------------------------------------

if isinstance(diagnosticos_validos_lote_09, list):

    diagnosticos_lote_09_disponibles = [
        df
        for df in diagnosticos_validos_lote_09
        if isinstance(df, pd.DataFrame) and not df.empty
    ]

    if diagnosticos_lote_09_disponibles:
        diagnosticos_lote_09_df = pd.concat(
            diagnosticos_lote_09_disponibles,
            ignore_index=True,
        )
    else:
        diagnosticos_lote_09_df = pd.DataFrame()

elif isinstance(
    diagnosticos_validos_lote_09,
    pd.DataFrame,
):

    diagnosticos_lote_09_df = (
        diagnosticos_validos_lote_09.copy()
    )

else:

    diagnosticos_lote_09_df = pd.DataFrame()

if (
    not diagnosticos_lote_09_df.empty
    and "id_hilo" in diagnosticos_lote_09_df.columns
):
    diagnosticos_aprobados_lote_09 = (
        diagnosticos_lote_09_df[
            diagnosticos_lote_09_df["id_hilo"]
            .astype(str)
            .eq(ID_APROBADO_LOTE_09)
        ]
        .copy()
        .reset_index(drop=True)
    )
else:
    diagnosticos_aprobados_lote_09 = pd.DataFrame()


# ------------------------------------------------------------
# 3. Registrar la exclusión temática
# ------------------------------------------------------------

registro_exclusion_tematica_7514350 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2019,
            "id_hilo": ID_EXCLUIDO_LOTE_09,
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7514350"
            ),
            "tipo_error": "exclusion_tematica",
            "mensaje_error": (
                "La inmigración aparece únicamente en "
                "4 de 36 comentarios (11,1 %). El hilo "
                "trata principalmente sobre política general."
            ),
            "lote": "ampliacion_09",
        }
    ]
)

# Adaptar el registro a las columnas existentes de incidencias
if isinstance(
    incidencias_forocoches_ampliacion,
    pd.DataFrame,
):

    for columna in (
        set(incidencias_forocoches_ampliacion.columns)
        - set(registro_exclusion_tematica_7514350.columns)
    ):
        registro_exclusion_tematica_7514350[columna] = pd.NA

    for columna in (
        set(registro_exclusion_tematica_7514350.columns)
        - set(incidencias_forocoches_ampliacion.columns)
    ):
        incidencias_forocoches_ampliacion[columna] = pd.NA

    registro_exclusion_tematica_7514350 = (
        registro_exclusion_tematica_7514350[
            incidencias_forocoches_ampliacion.columns
        ]
    )

    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                registro_exclusion_tematica_7514350,
            ],
            ignore_index=True,
        )
        .drop_duplicates(
            subset=["id_hilo", "tipo_error"],
            keep="last",
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print(
    "Hilos aprobados del lote 09:",
    comentarios_aprobados_lote_09[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios anuales aprobados:",
    len(
        comentarios_aprobados_lote_09[
            comentarios_aprobados_lote_09[
                "anio_comentario"
            ].eq(2019)
        ]
    ),
)

print(
    "Hilo excluido temáticamente:",
    ID_EXCLUIDO_LOTE_09,
)

display(
    comentarios_aprobados_lote_09[
        [
            "id_hilo",
            "numero_mensaje",
            "fecha_comentario",
            "texto_sin_citas",
        ]
    ].head()
)

Hilos aprobados del lote 09: 1
Comentarios anuales aprobados: 15
Hilo excluido temáticamente: 7514350


,id_hilo,numero_mensaje,fecha_comentario,texto_sin_citas
0,7022487,1,2019-02-25 19:11:00,"Buenas tardes shurs,\nTengo un primo que viene..."
1,7022487,2,2019-02-25 19:12:00,¿Y como pretendes que le den de alta en la seg...
2,7022487,3,2019-02-25 19:15:00,Que se case contigo por ejemplo
3,7022487,4,2019-02-25 19:16:00,Me ahorro el comentario.
4,7022487,5,2019-02-25 19:17:00,pregunta a los de vox en su twitter a ver qué ...


In [136]:
# ============================================================
# CELDA 57 — Lote 10: sustitutos de inmigración–2019
# ============================================================

nuevos_candidatos_lote_10 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2019,
            "id_hilo": "7582097",
            "titulo_resultado": (
                "Menores extranjeros en el centro de "
                'Hortaleza: "No nos gusta vivir aquí"'
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7582097"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'MENAS menores extranjeros 2019'
            ),
            "lote_descubrimiento": "ampliacion_10",
            "criterio_inclusion": (
                "Debate explícito sobre menores extranjeros "
                "no acompañados, centros de acogida, llegada "
                "migratoria e integración."
            ),
            "subtema_preliminar": (
                "menas_acogida_integracion"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2019,
            "id_hilo": "7386240",
            "titulo_resultado": (
                "Estas es la calidad de noticias y sucesos "
                "en la que estáis cayendo en ForoCoches"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7386240"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'inmigración MENAS noticias 2019'
            ),
            "lote_descubrimiento": "ampliacion_10",
            "criterio_inclusion": (
                "Discusión sobre la representación de la "
                "inmigración, los MENAS, los refugiados "
                "sirios y las noticias publicadas en el foro."
            ),
            "subtema_preliminar": (
                "representacion_mediatica_menas"
            ),
            "requiere_revision_tematica": False,
        },
    ]
)

nuevos_candidatos_lote_10["id_hilo"] = (
    nuevos_candidatos_lote_10["id_hilo"]
    .astype(str)
)

# ------------------------------------------------------------
# Evitar duplicados en el catálogo
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

nuevos_candidatos_lote_10 = (
    nuevos_candidatos_lote_10[
        ~nuevos_candidatos_lote_10["id_hilo"]
        .isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevos_candidatos_lote_10,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Seleccionar los candidatos aún no consolidados
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_10 = (
    hilos_candidatos[
        (
            hilos_candidatos["lote_descubrimiento"]
            .eq("ampliacion_10")
        )
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Comprobación
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 10:",
    len(nuevos_candidatos_lote_10),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 10:",
    len(candidatos_pendientes_lote_10),
)

display(
    candidatos_pendientes_lote_10[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 10: 0
Candidatos totales en el catálogo: 58
Candidatos pendientes del lote 10: 2


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,7582097,Menores extranjeros en el centro de Hortaleza:...,menas_acogida_integracion,False
1,7386240,Estas es la calidad de noticias y sucesos en l...,representacion_mediatica_menas,False


In [137]:
# ============================================================
# CELDA 58 — Procesamiento técnico del lote 10
# ============================================================

(
    comentarios_validos_lote_10,
    diagnosticos_validos_lote_10,
    incidencias_lote_10,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_10,
    nombre_lote="ampliacion_10",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# Contar los hilos técnicamente válidos
# ------------------------------------------------------------

if isinstance(comentarios_validos_lote_10, list):

    ids_validos_lote_10 = set()

    for dataframe_hilo in comentarios_validos_lote_10:

        if (
            isinstance(dataframe_hilo, pd.DataFrame)
            and not dataframe_hilo.empty
            and "id_hilo" in dataframe_hilo.columns
        ):
            ids_validos_lote_10.update(
                dataframe_hilo["id_hilo"]
                .astype(str)
                .unique()
                .tolist()
            )

    numero_hilos_validos_lote_10 = len(
        ids_validos_lote_10
    )

elif isinstance(
    comentarios_validos_lote_10,
    pd.DataFrame,
):

    numero_hilos_validos_lote_10 = (
        comentarios_validos_lote_10["id_hilo"]
        .astype(str)
        .nunique()
        if not comentarios_validos_lote_10.empty
        else 0
    )

else:

    numero_hilos_validos_lote_10 = 0

# ------------------------------------------------------------
# Contar incidencias
# ------------------------------------------------------------

if isinstance(incidencias_lote_10, list):
    numero_incidencias_lote_10 = len(
        incidencias_lote_10
    )
elif isinstance(incidencias_lote_10, pd.DataFrame):
    numero_incidencias_lote_10 = len(
        incidencias_lote_10
    )
else:
    numero_incidencias_lote_10 = 0

print(
    "Hilos técnicamente válidos:",
    numero_hilos_validos_lote_10,
)

print(
    "Hilos descartados o con error:",
    numero_incidencias_lote_10,
)

if (
    isinstance(incidencias_lote_10, pd.DataFrame)
    and not incidencias_lote_10.empty
):
    display(incidencias_lote_10)


[1/2] inmigracion | 2019 | hilo 7582097
Menores extranjeros en el centro de Hortaleza: "No nos gusta vivir aquí"
Primera página pública: 30 mensajes detectados
Hilo 7582097: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 28 comentarios
  Página 3/3: 28 comentarios
Candidato técnicamente válido: 58 comentarios totales | 58 comentarios de 2019

[2/2] inmigracion | 2019 | hilo 7386240
Estas es la calidad de noticias y sucesos en la que estáis cayendo en ForoCoches
Primera página pública: 30 mensajes detectados
Hilo 7386240: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 30 comentarios
  Página 3/3: 30 comentarios
Candidato técnicamente válido: 60 comentarios totales | 60 comentarios de 2019

Hilos técnicamente válidos: 2
Hilos descartados o con error: 0

Hilos técnicamente válidos: 2
Hilos descartados o con error: 0


In [138]:
# ============================================================
# CELDA 59 — Consolidación de los lotes 09 y 10
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Función auxiliar para unificar listas de DataFrames
# ------------------------------------------------------------

def unificar_dataframes(resultado):

    if isinstance(resultado, pd.DataFrame):
        return resultado.copy()

    if isinstance(resultado, list):

        dataframes_validos = [
            elemento
            for elemento in resultado
            if (
                isinstance(elemento, pd.DataFrame)
                and not elemento.empty
            )
        ]

        if dataframes_validos:
            return pd.concat(
                dataframes_validos,
                ignore_index=True,
            )

    return pd.DataFrame()


# ------------------------------------------------------------
# 2. Unificar los comentarios válidos del lote 10
# ------------------------------------------------------------

comentarios_lote_10_df = unificar_dataframes(
    comentarios_validos_lote_10
)

diagnosticos_lote_10_df = unificar_dataframes(
    diagnosticos_validos_lote_10
)

incidencias_lote_09_df = unificar_dataframes(
    incidencias_lote_09
)

incidencias_lote_10_df = unificar_dataframes(
    incidencias_lote_10
)


# ------------------------------------------------------------
# 3. Combinar los hilos aprobados de ambos lotes
# ------------------------------------------------------------

comentarios_aprobados_lotes_09_10 = pd.concat(
    [
        comentarios_aprobados_lote_09,
        comentarios_lote_10_df,
    ],
    ignore_index=True,
)

comentarios_aprobados_lotes_09_10["id_hilo"] = (
    comentarios_aprobados_lotes_09_10["id_hilo"]
    .astype(str)
)

comentarios_aprobados_lotes_09_10 = (
    comentarios_aprobados_lotes_09_10
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Hilos aprobados pendientes de consolidación:",
    comentarios_aprobados_lotes_09_10[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios brutos pendientes:",
    len(comentarios_aprobados_lotes_09_10),
)


# ------------------------------------------------------------
# 4. Consolidar el corpus bruto
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lotes_09_10,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Seleccionar únicamente comentarios del año objetivo
# ------------------------------------------------------------

comentarios_anuales_lotes_09_10 = (
    comentarios_aprobados_lotes_09_10[
        pd.to_numeric(
            comentarios_aprobados_lotes_09_10[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lotes_09_10[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Consolidar el corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lotes_09_10,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Consolidar los diagnósticos
# ------------------------------------------------------------

diagnosticos_nuevos = pd.concat(
    [
        diagnosticos_aprobados_lote_09,
        diagnosticos_lote_10_df,
    ],
    ignore_index=True,
)

if not diagnosticos_nuevos.empty:

    if "id_hilo" in diagnosticos_nuevos.columns:
        diagnosticos_nuevos["id_hilo"] = (
            diagnosticos_nuevos["id_hilo"]
            .astype(str)
        )

    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_nuevos,
            ],
            ignore_index=True,
        )
    )

    columnas_diagnostico_clave = [
        columna
        for columna in [
            "id_hilo",
            "pagina",
            "url_pagina",
        ]
        if columna
        in diagnosticos_forocoches_ampliado.columns
    ]

    if columnas_diagnostico_clave:

        diagnosticos_forocoches_ampliado = (
            diagnosticos_forocoches_ampliado
            .drop_duplicates(
                subset=columnas_diagnostico_clave,
                keep="first",
            )
            .reset_index(drop=True)
        )


# ------------------------------------------------------------
# 8. Incorporar la incidencia técnica del lote 09
# ------------------------------------------------------------

# El lote 10 no produjo incidencias.
# La exclusión temática de 7514350 ya se registró
# en la celda anterior.

if not incidencias_lote_09_df.empty:

    for columna in (
        set(incidencias_forocoches_ampliacion.columns)
        - set(incidencias_lote_09_df.columns)
    ):
        incidencias_lote_09_df[columna] = pd.NA

    for columna in (
        set(incidencias_lote_09_df.columns)
        - set(incidencias_forocoches_ampliacion.columns)
    ):
        incidencias_forocoches_ampliacion[columna] = pd.NA

    incidencias_lote_09_df = incidencias_lote_09_df[
        incidencias_forocoches_ampliacion.columns
    ]

    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                incidencias_lote_09_df,
            ],
            ignore_index=True,
        )
    )

    columnas_incidencia_clave = [
        columna
        for columna in [
            "id_hilo",
            "tipo_error",
        ]
        if columna
        in incidencias_forocoches_ampliacion.columns
    ]

    if columnas_incidencia_clave:

        incidencias_forocoches_ampliacion = (
            incidencias_forocoches_ampliacion
            .drop_duplicates(
                subset=columnas_incidencia_clave,
                keep="last",
            )
            .reset_index(drop=True)
        )


# ------------------------------------------------------------
# 9. Comprobación específica de inmigración–2019
# ------------------------------------------------------------

estrato_inmigracion_2019 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].eq("inmigracion")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2019)
    ]
    .copy()
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado),
)

print(
    "Hilos de inmigración–2019:",
    estrato_inmigracion_2019[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios de inmigración–2019:",
    len(estrato_inmigracion_2019),
)

print(
    "Usuarios de inmigración–2019:",
    estrato_inmigracion_2019[
        "usuario_hash"
    ].nunique(),
)

Hilos aprobados pendientes de consolidación: 3
Comentarios brutos pendientes: 133

Hilos válidos totales: 54
Comentarios anuales totales: 6019
Hilos de inmigración–2019: 12
Comentarios de inmigración–2019: 774
Usuarios de inmigración–2019: 592


In [139]:
# ============================================================
# CELDA 60 — Panel actualizado de cobertura por estrato
# ============================================================

import numpy as np
import pandas as pd

MINIMO_HILOS = 12
MINIMO_COMENTARIOS = 500
MINIMO_USUARIOS = 300
UMBRAL_CONCENTRACION_PCT = 25.0

filas_estado_estratos = []

for (
    tema,
    anio_objetivo,
), grupo in comentarios_forocoches_anual_ampliado.groupby(
    ["tema", "anio_objetivo"],
    dropna=False,
):

    grupo = grupo.copy()

    comentarios_por_hilo = (
        grupo.groupby("id_hilo")
        .size()
        .sort_values(ascending=False)
    )

    numero_hilos = int(
        comentarios_por_hilo.size
    )

    numero_comentarios = int(
        len(grupo)
    )

    numero_usuarios = int(
        grupo["usuario_hash"].nunique()
    )

    if numero_comentarios > 0:

        pesos_hilos = (
            comentarios_por_hilo
            / numero_comentarios
        )

        mayor_hilo_pct = float(
            pesos_hilos.max() * 100
        )

        hhi = float(
            np.square(pesos_hilos).sum()
        )

        hilos_efectivos_raw = float(
            1 / hhi
        ) if hhi > 0 else 0.0

    else:

        mayor_hilo_pct = 0.0
        hhi = 0.0
        hilos_efectivos_raw = 0.0

    cobertura_suficiente = bool(
        numero_hilos >= MINIMO_HILOS
        and numero_comentarios >= MINIMO_COMENTARIOS
        and numero_usuarios >= MINIMO_USUARIOS
    )

    requiere_equilibrado_nlp = bool(
        cobertura_suficiente
        and mayor_hilo_pct > UMBRAL_CONCENTRACION_PCT
    )

    if not cobertura_suficiente:

        estado = "ampliar_extraccion"

    elif requiere_equilibrado_nlp:

        estado = "cerrado_equilibrar_nlp"

    else:

        estado = "cerrado_equilibrado_natural"

    filas_estado_estratos.append(
        {
            "tema": tema,
            "anio_objetivo": int(anio_objetivo),
            "hilos": numero_hilos,
            "comentarios": numero_comentarios,
            "usuarios": numero_usuarios,
            "mayor_hilo_pct": round(
                mayor_hilo_pct,
                1,
            ),
            "hhi": round(
                hhi,
                3,
            ),
            "hilos_efectivos_raw": round(
                hilos_efectivos_raw,
                1,
            ),
            "cobertura_suficiente": (
                cobertura_suficiente
            ),
            "requiere_equilibrado_nlp": (
                requiere_equilibrado_nlp
            ),
            "estado": estado,
        }
    )

estado_estratos = (
    pd.DataFrame(filas_estado_estratos)
    .sort_values(
        ["tema", "anio_objetivo"]
    )
    .reset_index(drop=True)
)

estratos_cerrados = int(
    estado_estratos[
        "cobertura_suficiente"
    ].sum()
)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado),
)

print(
    "Estratos con cobertura suficiente:",
    f"{estratos_cerrados} de {len(estado_estratos)}",
)

display(estado_estratos)

Hilos válidos totales: 54
Comentarios anuales totales: 6019
Estratos con cobertura suficiente: 3 de 8


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos_raw,cobertura_suficiente,requiere_equilibrado_nlp,estado
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,True,False,cerrado_equilibrado_natural
1,inmigracion,2016,12,2521,1614,61.6,0.402,2.5,True,True,cerrado_equilibrar_nlp
2,inmigracion,2019,12,774,592,29.1,0.145,6.9,True,True,cerrado_equilibrar_nlp
3,inmigracion,2023,3,368,251,82.1,0.700,1.4,False,False,ampliar_extraccion
4,lgtbi,2015,2,644,566,93.5,0.878,1.1,False,False,ampliar_extraccion
5,lgtbi,2016,3,183,141,54.1,0.463,2.2,False,False,ampliar_extraccion
6,lgtbi,2019,3,485,362,56.3,0.455,2.2,False,False,ampliar_extraccion
7,lgtbi,2023,3,192,166,47.4,0.410,2.4,False,False,ampliar_extraccion


## 2023 Inmigración

In [140]:
# ============================================================
# CELDA 61 — Lote 11: ampliación de inmigración–2023
# ============================================================

nuevos_candidatos_lote_11 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9577492",
            "titulo_resultado": (
                "ONG roba dinero para MENAS"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9577492"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'MENAS ONG 2023'
            ),
            "lote_descubrimiento": "ampliacion_11",
            "criterio_inclusion": (
                "Debate explícito sobre menores migrantes, "
                "centros de acogida, financiación pública "
                "y gestión de una entidad social."
            ),
            "subtema_preliminar": (
                "menas_acogida_gestion_fondos"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9810310",
            "titulo_resultado": (
                "El Hierro recibió más migrantes en 7 meses "
                "que todo el Mediterráneo español en 2023"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9810310"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"El Hierro" migrantes 2023'
            ),
            "lote_descubrimiento": "ampliacion_11",
            "criterio_inclusion": (
                "Debate central sobre llegadas migratorias "
                "en cayucos, menores migrantes, Canarias "
                "y redistribución territorial."
            ),
            "subtema_preliminar": (
                "ruta_canaria_cayucos_llegadas"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9790242",
            "titulo_resultado": (
                "Delincuencia es igual a inmigración"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9790242"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"Delincuencia es igual a inmigración"'
            ),
            "lote_descubrimiento": "ampliacion_11",
            "criterio_inclusion": (
                "Debate explícito sobre la relación percibida "
                "entre inmigración, delincuencia, barrios "
                "multiculturales y seguridad."
            ),
            "subtema_preliminar": (
                "inmigracion_delincuencia_seguridad"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9726464",
            "titulo_resultado": (
                "Interior traslada a Huesca a 200 migrantes "
                "llegados desde Canarias"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9726464"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'migrantes Canarias Huesca 2023'
            ),
            "lote_descubrimiento": "ampliacion_11",
            "criterio_inclusion": (
                "Debate central sobre traslado de migrantes "
                "desde Canarias, acogida, coordinación "
                "institucional y distribución territorial."
            ),
            "subtema_preliminar": (
                "traslado_acogida_distribucion_territorial"
            ),
            "requiere_revision_tematica": False,
        },
    ]
)

nuevos_candidatos_lote_11["id_hilo"] = (
    nuevos_candidatos_lote_11["id_hilo"]
    .astype(str)
)

# ------------------------------------------------------------
# 1. Eliminar candidatos ya catalogados
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

nuevos_candidatos_lote_11 = (
    nuevos_candidatos_lote_11[
        ~nuevos_candidatos_lote_11["id_hilo"]
        .isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Incorporar al catálogo
# ------------------------------------------------------------

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevos_candidatos_lote_11,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Seleccionar candidatos pendientes
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_11 = (
    hilos_candidatos[
        hilos_candidatos["lote_descubrimiento"]
        .eq("ampliacion_11")
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 11:",
    len(nuevos_candidatos_lote_11),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 11:",
    len(candidatos_pendientes_lote_11),
)

display(
    candidatos_pendientes_lote_11[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 11: 4
Candidatos totales en el catálogo: 62
Candidatos pendientes del lote 11: 4


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,9577492,ONG roba dinero para MENAS,menas_acogida_gestion_fondos,False
1,9810310,El Hierro recibió más migrantes en 7 meses que...,ruta_canaria_cayucos_llegadas,False
2,9790242,Delincuencia es igual a inmigración,inmigracion_delincuencia_seguridad,False
3,9726464,Interior traslada a Huesca a 200 migrantes lle...,traslado_acogida_distribucion_territorial,False


In [141]:
# ============================================================
# CELDA 62 — Procesamiento técnico del lote 11
# ============================================================

(
    comentarios_validos_lote_11,
    diagnosticos_validos_lote_11,
    incidencias_lote_11,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_11,
    nombre_lote="ampliacion_11",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# 1. Contar hilos técnicamente válidos
# ------------------------------------------------------------

comentarios_lote_11_df = unificar_dataframes(
    comentarios_validos_lote_11
)

if (
    not comentarios_lote_11_df.empty
    and "id_hilo" in comentarios_lote_11_df.columns
):

    numero_hilos_validos_lote_11 = (
        comentarios_lote_11_df["id_hilo"]
        .astype(str)
        .nunique()
    )

else:

    numero_hilos_validos_lote_11 = 0

# ------------------------------------------------------------
# 2. Unificar incidencias
# ------------------------------------------------------------

incidencias_lote_11_df = unificar_dataframes(
    incidencias_lote_11
)

numero_incidencias_lote_11 = (
    len(incidencias_lote_11_df)
    if not incidencias_lote_11_df.empty
    else 0
)

# ------------------------------------------------------------
# 3. Resumen
# ------------------------------------------------------------

print(
    "Hilos técnicamente válidos:",
    numero_hilos_validos_lote_11,
)

print(
    "Hilos descartados o con error:",
    numero_incidencias_lote_11,
)

if not incidencias_lote_11_df.empty:
    display(incidencias_lote_11_df)


[1/4] inmigracion | 2023 | hilo 9577492
ONG roba dinero para MENAS
Primera página pública: 30 mensajes detectados
Hilo 9577492: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 19 comentarios
Candidato técnicamente válido: 49 comentarios totales | 49 comentarios de 2023

[2/4] inmigracion | 2023 | hilo 9810310
El Hierro recibió más migrantes en 7 meses que todo el Mediterráneo español en 2023
Primera página pública: 20 mensajes detectados
Hilo 9810310: 1 página(s)
  Página 1/1: 20 comentarios
Candidato técnicamente válido: 20 comentarios totales | 20 comentarios de 2023

[3/4] inmigracion | 2023 | hilo 9790242
Delincuencia es igual a inmigración
Primera página pública: 30 mensajes detectados
Hilo 9790242: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 30 comentarios
  Página 3/3: 14 comentarios
Candidato técnicamente válido: 74 comentarios totales | 73 comentarios de 2023

[4/4] inmigracion | 2023 | hilo 9726464
Interior traslada a Huesca a 200 migrantes llegados desde C

In [144]:
# ============================================================
# CELDA 63 — Consolidación del lote 11
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Unificar resultados
# ------------------------------------------------------------

comentarios_lote_11_df = unificar_dataframes(
    comentarios_validos_lote_11
)

diagnosticos_lote_11_df = unificar_dataframes(
    diagnosticos_validos_lote_11
)

incidencias_lote_11_df = unificar_dataframes(
    incidencias_lote_11
)

if comentarios_lote_11_df.empty:

    raise ValueError(
        "El lote 11 no contiene comentarios válidos."
    )

comentarios_lote_11_df["id_hilo"] = (
    comentarios_lote_11_df["id_hilo"]
    .astype(str)
)

comentarios_lote_11_df = (
    comentarios_lote_11_df
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Hilos pendientes de consolidación:",
    comentarios_lote_11_df[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios brutos pendientes:",
    len(comentarios_lote_11_df),
)


# ------------------------------------------------------------
# 2. Consolidar el corpus bruto
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_lote_11_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Seleccionar los comentarios del año objetivo
# ------------------------------------------------------------

anio_comentario_lote_11 = pd.to_numeric(
    comentarios_lote_11_df["anio_comentario"],
    errors="coerce",
)

anio_objetivo_lote_11 = pd.to_numeric(
    comentarios_lote_11_df["anio_objetivo"],
    errors="coerce",
)

comentarios_anuales_lote_11 = (
    comentarios_lote_11_df[
        anio_comentario_lote_11.eq(
            anio_objetivo_lote_11
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Consolidar el corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_11,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Consolidar los diagnósticos
# ------------------------------------------------------------

if not diagnosticos_lote_11_df.empty:

    if "id_hilo" in diagnosticos_lote_11_df.columns:

        diagnosticos_lote_11_df["id_hilo"] = (
            diagnosticos_lote_11_df["id_hilo"]
            .astype(str)
        )

    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_11_df,
            ],
            ignore_index=True,
        )
    )

    columnas_diagnostico_clave = [
        columna
        for columna in [
            "id_hilo",
            "pagina",
            "url_pagina",
        ]
        if columna
        in diagnosticos_forocoches_ampliado.columns
    ]

    if columnas_diagnostico_clave:

        diagnosticos_forocoches_ampliado = (
            diagnosticos_forocoches_ampliado
            .drop_duplicates(
                subset=columnas_diagnostico_clave,
                keep="first",
            )
            .reset_index(drop=True)
        )


# ------------------------------------------------------------
# 6. Consolidar posibles incidencias
# ------------------------------------------------------------

if not incidencias_lote_11_df.empty:

    for columna in (
        set(incidencias_forocoches_ampliacion.columns)
        - set(incidencias_lote_11_df.columns)
    ):
        incidencias_lote_11_df[columna] = pd.NA

    for columna in (
        set(incidencias_lote_11_df.columns)
        - set(incidencias_forocoches_ampliacion.columns)
    ):
        incidencias_forocoches_ampliacion[columna] = pd.NA

    incidencias_lote_11_df = (
        incidencias_lote_11_df[
            incidencias_forocoches_ampliacion.columns
        ]
    )

    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                incidencias_lote_11_df,
            ],
            ignore_index=True,
        )
    )


# ------------------------------------------------------------
# 7. Comprobar inmigración–2023
# ------------------------------------------------------------

estrato_inmigracion_2023 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].eq("inmigracion")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2023)
    ]
    .copy()
)

comentarios_por_hilo_2023 = (
    estrato_inmigracion_2023
    .groupby("id_hilo")
    .size()
    .sort_values(ascending=False)
)

mayor_hilo_pct_2023 = (
    comentarios_por_hilo_2023.iloc[0]
    / len(estrato_inmigracion_2023)
    * 100
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado),
)

print(
    "Hilos de inmigración–2023:",
    estrato_inmigracion_2023[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios de inmigración–2023:",
    len(estrato_inmigracion_2023),
)

print(
    "Usuarios de inmigración–2023:",
    estrato_inmigracion_2023[
        "usuario_hash"
    ].nunique(),
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2023:.1f}%",
)

print(
    "Hilos todavía necesarios:",
    max(
        12
        - estrato_inmigracion_2023[
            "id_hilo"
        ].astype(str).nunique(),
        0,
    ),
)

Hilos pendientes de consolidación: 4
Comentarios brutos pendientes: 665

Hilos válidos totales: 58
Comentarios anuales totales: 6683
Hilos de inmigración–2023: 7
Comentarios de inmigración–2023: 1032
Usuarios de inmigración–2023: 716
Mayor aportación de un hilo: 50.6%
Hilos todavía necesarios: 5


In [146]:
# ============================================================
# CELDA 64 — Lote 12: cierre de inmigración–2023
# ============================================================

nuevos_candidatos_lote_12 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9712540",
            "titulo_resultado": (
                "¿Por qué llegan ahora los CAYUCOS "
                "a EL HIERRO?"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9712540"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"cayucos" "El Hierro" 2023'
            ),
            "lote_descubrimiento": "ampliacion_12",
            "criterio_inclusion": (
                "Debate explícito sobre rutas migratorias "
                "africanas, cayucos, mafias, devoluciones "
                "y llegadas a Canarias."
            ),
            "subtema_preliminar": (
                "ruta_canaria_cayucos_origenes"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9507733",
            "titulo_resultado": (
                "¿Tan complicado es contratar "
                "trabajadores extranjeros?"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9507733"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"trabajadores extranjeros" 2023'
            ),
            "lote_descubrimiento": "ampliacion_12",
            "criterio_inclusion": (
                "Debate sobre inmigración laboral legal, "
                "permisos de residencia, contratación, "
                "escasez de trabajadores y regularización."
            ),
            "subtema_preliminar": (
                "inmigracion_laboral_contratacion_legal"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9415175",
            "titulo_resultado": (
                "Ucranianos en España"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9415175"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"Ucranianos en España" 2023'
            ),
            "lote_descubrimiento": "ampliacion_12",
            "criterio_inclusion": (
                "Debate explícito sobre refugiados "
                "ucranianos en España, ayudas, vivienda, "
                "trabajo y diferencias socioeconómicas."
            ),
            "subtema_preliminar": (
                "refugiados_ucranianos_acogida"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9484087",
            "titulo_resultado": (
                "Gobierno elimina requisito del "
                "empadronamiento para acceder "
                "a las prestaciones"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9484087"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'empadronamiento prestaciones extranjeros 2023'
            ),
            "lote_descubrimiento": "ampliacion_12",
            "criterio_inclusion": (
                "El mensaje inicial plantea expresamente "
                "el acceso de personas extranjeras a ayudas "
                "sin requisito de empadronamiento."
            ),
            "subtema_preliminar": (
                "extranjeros_prestaciones_empadronamiento"
            ),
            "requiere_revision_tematica": True,
        },
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9423437",
            "titulo_resultado": (
                "En búsqueda de trabajo (arraigo social)"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9423437"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"arraigo social" trabajo 2023'
            ),
            "lote_descubrimiento": "ampliacion_12",
            "criterio_inclusion": (
                "Caso práctico sobre una persona migrante "
                "en situación irregular, búsqueda de empleo, "
                "contrato laboral y arraigo social."
            ),
            "subtema_preliminar": (
                "arraigo_social_empleo_regularizacion"
            ),
            "requiere_revision_tematica": False,
        },
    ]
)

nuevos_candidatos_lote_12["id_hilo"] = (
    nuevos_candidatos_lote_12["id_hilo"]
    .astype(str)
)

# ------------------------------------------------------------
# 1. Excluir candidatos ya catalogados
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

nuevos_candidatos_lote_12 = (
    nuevos_candidatos_lote_12[
        ~nuevos_candidatos_lote_12["id_hilo"]
        .isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Incorporar al catálogo
# ------------------------------------------------------------

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevos_candidatos_lote_12,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Seleccionar únicamente los pendientes
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_12 = (
    hilos_candidatos[
        hilos_candidatos["lote_descubrimiento"]
        .eq("ampliacion_12")
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 12:",
    len(nuevos_candidatos_lote_12),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 12:",
    len(candidatos_pendientes_lote_12),
)

display(
    candidatos_pendientes_lote_12[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 12: 5
Candidatos totales en el catálogo: 67
Candidatos pendientes del lote 12: 5


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,9712540,¿Por qué llegan ahora los CAYUCOS a EL HIERRO?,ruta_canaria_cayucos_origenes,False
1,9507733,¿Tan complicado es contratar trabajadores extr...,inmigracion_laboral_contratacion_legal,False
2,9415175,Ucranianos en España,refugiados_ucranianos_acogida,False
3,9484087,Gobierno elimina requisito del empadronamiento...,extranjeros_prestaciones_empadronamiento,True
4,9423437,En búsqueda de trabajo (arraigo social),arraigo_social_empleo_regularizacion,False


In [147]:
# ============================================================
# CELDA 65 — Procesamiento técnico del lote 12
# ============================================================

(
    comentarios_validos_lote_12,
    diagnosticos_validos_lote_12,
    incidencias_lote_12,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_12,
    nombre_lote="ampliacion_12",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# 1. Unificar los resultados
# ------------------------------------------------------------

comentarios_lote_12_df = unificar_dataframes(
    comentarios_validos_lote_12
)

diagnosticos_lote_12_df = unificar_dataframes(
    diagnosticos_validos_lote_12
)

incidencias_lote_12_df = unificar_dataframes(
    incidencias_lote_12
)

# ------------------------------------------------------------
# 2. Contar los hilos técnicamente válidos
# ------------------------------------------------------------

if (
    not comentarios_lote_12_df.empty
    and "id_hilo" in comentarios_lote_12_df.columns
):

    comentarios_lote_12_df["id_hilo"] = (
        comentarios_lote_12_df["id_hilo"]
        .astype(str)
    )

    numero_hilos_validos_lote_12 = (
        comentarios_lote_12_df["id_hilo"]
        .nunique()
    )

else:

    numero_hilos_validos_lote_12 = 0

# ------------------------------------------------------------
# 3. Resumen
# ------------------------------------------------------------

print(
    "Hilos técnicamente válidos:",
    numero_hilos_validos_lote_12,
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_12_df),
)

if not incidencias_lote_12_df.empty:
    display(incidencias_lote_12_df)


[1/5] inmigracion | 2023 | hilo 9712540
¿Por qué llegan ahora los CAYUCOS a EL HIERRO?
Primera página pública: 6 mensajes detectados
Hilo 9712540: 1 página(s)
  Página 1/1: 6 comentarios
Candidato técnicamente válido: 6 comentarios totales | 6 comentarios de 2023

[2/5] inmigracion | 2023 | hilo 9507733
¿Tan complicado es contratar trabajadores extranjeros?
Primera página pública: 6 mensajes detectados
Hilo 9507733: 1 página(s)
  Página 1/1: 6 comentarios
Candidato técnicamente válido: 6 comentarios totales | 6 comentarios de 2023

[3/5] inmigracion | 2023 | hilo 9415175
Ucranianos en España
Primera página pública: 30 mensajes detectados
Hilo 9415175: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 17 comentarios
Candidato técnicamente válido: 47 comentarios totales | 47 comentarios de 2023

[4/5] inmigracion | 2023 | hilo 9484087
Gobierno elimina requisito del empadronamiento para acceder a las prestaciones
Primera página pública: 30 mensajes detectados
Hilo 9484087: 3 página(

In [149]:
# ============================================================
# CELDA 66 — Revisión temática del hilo 9484087
# ============================================================

import re
import pandas as pd

ID_HILO_REVISION = "9484087"

# ------------------------------------------------------------
# 1. Seleccionar el hilo
# ------------------------------------------------------------

comentarios_lote_12_df["id_hilo"] = (
    comentarios_lote_12_df["id_hilo"]
    .astype(str)
)

hilo_revision_9484087 = (
    comentarios_lote_12_df[
        comentarios_lote_12_df["id_hilo"]
        .eq(ID_HILO_REVISION)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Comentarios recuperados del hilo 9484087:",
    len(hilo_revision_9484087),
)

if hilo_revision_9484087.empty:

    raise ValueError(
        "No se encontraron comentarios del hilo 9484087."
    )


# ------------------------------------------------------------
# 2. Definir patrones temáticos
# ------------------------------------------------------------

patron_inmigracion = re.compile(
    r"\b(?:"
    r"inmigr\w*|"
    r"migrant\w*|"
    r"extranj\w*|"
    r"refugiad\w*|"
    r"menas?|"
    r"sin\s+papeles|"
    r"irregular(?:es)?|"
    r"residentes?\s+extranjer\w*|"
    r"terceros?\s+pa[ií]ses|"
    r"turistas?\s+extranjer\w*"
    r")\b",
    flags=re.IGNORECASE,
)

patron_prestaciones = re.compile(
    r"\b(?:"
    r"prestacion\w*|"
    r"ayudas?|"
    r"paguitas?|"
    r"subvencion\w*|"
    r"servicios?\s+sociales?|"
    r"empadron\w*|"
    r"padr[oó]n|"
    r"rentas?|"
    r"beneficiari\w*|"
    r"asistencia\s+social"
    r")\b",
    flags=re.IGNORECASE,
)


# ------------------------------------------------------------
# 3. Aplicar indicadores
# ------------------------------------------------------------

texto_revision_9484087 = (
    hilo_revision_9484087["texto_sin_citas"]
    .fillna("")
    .astype(str)
)

hilo_revision_9484087["menciona_inmigracion"] = (
    texto_revision_9484087.str.contains(
        patron_inmigracion,
        na=False,
    )
)

hilo_revision_9484087["menciona_prestaciones"] = (
    texto_revision_9484087.str.contains(
        patron_prestaciones,
        na=False,
    )
)

hilo_revision_9484087["menciona_ambos_encuadres"] = (
    hilo_revision_9484087["menciona_inmigracion"]
    & hilo_revision_9484087["menciona_prestaciones"]
)


# ------------------------------------------------------------
# 4. Construir el resumen cuantitativo
# ------------------------------------------------------------

total_comentarios = len(hilo_revision_9484087)

numero_inmigracion = int(
    hilo_revision_9484087[
        "menciona_inmigracion"
    ].sum()
)

numero_prestaciones = int(
    hilo_revision_9484087[
        "menciona_prestaciones"
    ].sum()
)

numero_ambos = int(
    hilo_revision_9484087[
        "menciona_ambos_encuadres"
    ].sum()
)

resumen_revision_9484087 = pd.DataFrame(
    {
        "indicador": [
            "comentarios_totales",
            "mencionan_inmigracion",
            "mencionan_prestaciones",
            "mencionan_ambos_encuadres",
        ],
        "numero_comentarios": [
            total_comentarios,
            numero_inmigracion,
            numero_prestaciones,
            numero_ambos,
        ],
        "porcentaje": [
            100.0,
            round(
                numero_inmigracion
                / total_comentarios
                * 100,
                1,
            ),
            round(
                numero_prestaciones
                / total_comentarios
                * 100,
                1,
            ),
            round(
                numero_ambos
                / total_comentarios
                * 100,
                1,
            ),
        ],
    }
)

display(resumen_revision_9484087)


# ------------------------------------------------------------
# 5. Mostrar todos los mensajes sobre inmigración
# ------------------------------------------------------------

mensajes_inmigracion_9484087 = (
    hilo_revision_9484087.loc[
        hilo_revision_9484087[
            "menciona_inmigracion"
        ],
        [
            "numero_mensaje",
            "menciona_inmigracion",
            "menciona_prestaciones",
            "texto_sin_citas",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Mensajes mostrados para revisión:",
    len(mensajes_inmigracion_9484087),
)

display(mensajes_inmigracion_9484087)


# ------------------------------------------------------------
# 6. Mostrar el inicio del hilo para interpretar el contexto
# ------------------------------------------------------------

print(
    "Primeros mensajes del hilo:"
)

display(
    hilo_revision_9484087[
        [
            "numero_mensaje",
            "texto_sin_citas",
        ]
    ].head(10)
)

Comentarios recuperados del hilo 9484087: 87


,indicador,numero_comentarios,porcentaje
0,comentarios_totales,87,100.0
1,mencionan_inmigracion,8,9.2
2,mencionan_prestaciones,23,26.4
3,mencionan_ambos_encuadres,6,6.9


Mensajes mostrados para revisión: 8


,numero_mensaje,menciona_inmigracion,menciona_prestaciones,texto_sin_citas
0,1,True,True,La ley de servicios sociales elimina el requis...
1,9,True,True,Lo siguiente es enviar ayudas personales y nom...
2,33,True,True,Un ilegal recién llegado en patera tendrá los ...
3,67,True,False,Era el único inconveniente que tenían que solu...
4,68,True,True,Bienvenidos inmigrantes que podéis tener pagui...
5,70,True,True,Si alguien tiene ganas de leerse las 90 pagina...
6,83,True,False,Se puede ir a un país suramericano conseguir l...
7,87,True,True,"pues pasará como en alemania, los pobres inmig..."


Primeros mensajes del hilo:


,numero_mensaje,texto_sin_citas
0,1,La ley de servicios sociales elimina el requis...
1,2,Lo siguiente es votar sin padrón
2,3,se vienen los nomadas de paguitas
3,4,haz que pase
4,5,Ésa es la clave de todo.
5,6,yo flipo en serio.
6,7,ale a pagar impuestos cabrones para dar dinero...
7,8,"Haz que pague\nBorra, soy yo desde otro lao"
8,9,Lo siguiente es enviar ayudas personales y nom...
9,10,"Panchitos,moros,etc están de enhorabuena."


In [150]:
# ============================================================
# CELDA 67 — Decisión temática definitiva del lote 12
# ============================================================

import pandas as pd

IDS_APROBADOS_LOTE_12 = {
    "9712540",
    "9507733",
    "9415175",
    "9423437",
}

ID_EXCLUIDO_LOTE_12 = "9484087"

# ------------------------------------------------------------
# 1. Reservar únicamente los comentarios aprobados
# ------------------------------------------------------------

comentarios_aprobados_lote_12 = (
    comentarios_lote_12_df[
        comentarios_lote_12_df["id_hilo"]
        .astype(str)
        .isin(IDS_APROBADOS_LOTE_12)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_aprobados_lote_12 = (
    comentarios_aprobados_lote_12
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. Reservar los diagnósticos aprobados
# ------------------------------------------------------------

if (
    not diagnosticos_lote_12_df.empty
    and "id_hilo" in diagnosticos_lote_12_df.columns
):

    diagnosticos_lote_12_df["id_hilo"] = (
        diagnosticos_lote_12_df["id_hilo"]
        .astype(str)
    )

    diagnosticos_aprobados_lote_12 = (
        diagnosticos_lote_12_df[
            diagnosticos_lote_12_df["id_hilo"]
            .isin(IDS_APROBADOS_LOTE_12)
        ]
        .copy()
        .reset_index(drop=True)
    )

else:

    diagnosticos_aprobados_lote_12 = pd.DataFrame()


# ------------------------------------------------------------
# 3. Registrar la exclusión temática
# ------------------------------------------------------------

registro_exclusion_9484087 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": ID_EXCLUIDO_LOTE_12,
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9484087"
            ),
            "tipo_error": "exclusion_tematica",
            "mensaje_error": (
                "Solo 8 de 87 comentarios (9,2 %) "
                "mencionan inmigración. El debate se centra "
                "principalmente en prestaciones sociales "
                "y empadronamiento general."
            ),
            "lote": "ampliacion_12",
        }
    ]
)

# ------------------------------------------------------------
# 4. Adaptar las columnas al registro general
# ------------------------------------------------------------

for columna in (
    set(incidencias_forocoches_ampliacion.columns)
    - set(registro_exclusion_9484087.columns)
):
    registro_exclusion_9484087[columna] = pd.NA

for columna in (
    set(registro_exclusion_9484087.columns)
    - set(incidencias_forocoches_ampliacion.columns)
):
    incidencias_forocoches_ampliacion[columna] = pd.NA

registro_exclusion_9484087 = (
    registro_exclusion_9484087[
        incidencias_forocoches_ampliacion.columns
    ]
)

incidencias_forocoches_ampliacion = (
    pd.concat(
        [
            incidencias_forocoches_ampliacion,
            registro_exclusion_9484087,
        ],
        ignore_index=True,
    )
)

columnas_incidencia_clave = [
    columna
    for columna in [
        "id_hilo",
        "tipo_error",
    ]
    if columna
    in incidencias_forocoches_ampliacion.columns
]

if columnas_incidencia_clave:

    incidencias_forocoches_ampliacion = (
        incidencias_forocoches_ampliacion
        .drop_duplicates(
            subset=columnas_incidencia_clave,
            keep="last",
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 5. Comprobación
# ------------------------------------------------------------

comentarios_anuales_aprobados_lote_12 = (
    comentarios_aprobados_lote_12[
        pd.to_numeric(
            comentarios_aprobados_lote_12[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(2023)
    ]
)

print(
    "Hilos aprobados del lote 12:",
    comentarios_aprobados_lote_12[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios brutos aprobados:",
    len(comentarios_aprobados_lote_12),
)

print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_aprobados_lote_12),
)

print(
    "Hilo excluido temáticamente:",
    ID_EXCLUIDO_LOTE_12,
)

display(
    comentarios_aprobados_lote_12.groupby(
        "id_hilo"
    )
    .agg(
        comentarios_brutos=(
            "id_mensaje",
            "count",
        ),
        comentarios_2023=(
            "anio_comentario",
            lambda serie: (
                pd.to_numeric(
                    serie,
                    errors="coerce",
                ).eq(2023).sum()
            ),
        ),
    )
    .reset_index()
)

Hilos aprobados del lote 12: 4
Comentarios brutos aprobados: 96
Comentarios anuales aprobados: 95
Hilo excluido temáticamente: 9484087


,id_hilo,comentarios_brutos,comentarios_2023
0,9415175,47,47
1,9423437,37,36
2,9507733,6,6
3,9712540,6,6


In [151]:
# ============================================================
# CELDA 68 — Lote 13: sustituto final inmigración–2023
# ============================================================

nuevo_candidato_lote_13 = pd.DataFrame(
    [
        {
            "tema": "inmigracion",
            "anio_objetivo": 2023,
            "id_hilo": "9747146",
            "titulo_resultado": (
                "¿Cuál de los siguientes problemas "
                "de España te preocupa más?"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9747146"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"problemas de España" inmigrantes MENAS 2023'
            ),
            "lote_descubrimiento": "ampliacion_13",
            "criterio_inclusion": (
                "El resultado indexado contiene un debate "
                "sustancial sobre inmigración, MENAS, empleo, "
                "vivienda, salarios y acceso a ayudas, pero "
                "el planteamiento inicial es más general."
            ),
            "subtema_preliminar": (
                "inmigracion_problemas_sociales_economia"
            ),
            "requiere_revision_tematica": True,
        }
    ]
)

nuevo_candidato_lote_13["id_hilo"] = (
    nuevo_candidato_lote_13["id_hilo"]
    .astype(str)
)

# ------------------------------------------------------------
# 1. Evitar duplicados
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

nuevo_candidato_lote_13 = (
    nuevo_candidato_lote_13[
        ~nuevo_candidato_lote_13["id_hilo"]
        .isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Incorporar al catálogo
# ------------------------------------------------------------

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevo_candidato_lote_13,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Seleccionar el candidato pendiente
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_13 = (
    hilos_candidatos[
        hilos_candidatos["lote_descubrimiento"]
        .eq("ampliacion_13")
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 13:",
    len(nuevo_candidato_lote_13),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 13:",
    len(candidatos_pendientes_lote_13),
)

display(
    candidatos_pendientes_lote_13[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 13: 1
Candidatos totales en el catálogo: 68
Candidatos pendientes del lote 13: 1


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,9747146,¿Cuál de los siguientes problemas de España te...,inmigracion_problemas_sociales_economia,True


In [152]:
# ============================================================
# CELDA 69 — Procesamiento técnico del lote 13
# ============================================================

(
    comentarios_validos_lote_13,
    diagnosticos_validos_lote_13,
    incidencias_lote_13,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_13,
    nombre_lote="ampliacion_13",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# 1. Unificar los resultados
# ------------------------------------------------------------

comentarios_lote_13_df = unificar_dataframes(
    comentarios_validos_lote_13
)

diagnosticos_lote_13_df = unificar_dataframes(
    diagnosticos_validos_lote_13
)

incidencias_lote_13_df = unificar_dataframes(
    incidencias_lote_13
)

# ------------------------------------------------------------
# 2. Contar hilos técnicamente válidos
# ------------------------------------------------------------

if (
    not comentarios_lote_13_df.empty
    and "id_hilo" in comentarios_lote_13_df.columns
):

    comentarios_lote_13_df["id_hilo"] = (
        comentarios_lote_13_df["id_hilo"]
        .astype(str)
    )

    numero_hilos_validos_lote_13 = (
        comentarios_lote_13_df[
            "id_hilo"
        ].nunique()
    )

else:

    numero_hilos_validos_lote_13 = 0

# ------------------------------------------------------------
# 3. Mostrar resumen
# ------------------------------------------------------------

print(
    "Hilos técnicamente válidos:",
    numero_hilos_validos_lote_13,
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_13_df),
)

if not incidencias_lote_13_df.empty:
    display(incidencias_lote_13_df)


[1/1] inmigracion | 2023 | hilo 9747146
¿Cuál de los siguientes problemas de España te preocupa más?
Primera página pública: 30 mensajes detectados
Hilo 9747146: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 5 comentarios
Candidato técnicamente válido: 35 comentarios totales | 35 comentarios de 2023

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [153]:
# ============================================================
# CELDA 70 — Revisión temática del hilo 9747146
# ============================================================

import re
import pandas as pd

ID_HILO_REVISION = "9747146"

# ------------------------------------------------------------
# 1. Seleccionar el hilo
# ------------------------------------------------------------

hilo_revision_9747146 = (
    comentarios_lote_13_df[
        comentarios_lote_13_df["id_hilo"]
        .astype(str)
        .eq(ID_HILO_REVISION)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Comentarios recuperados del hilo 9747146:",
    len(hilo_revision_9747146),
)

if hilo_revision_9747146.empty:

    raise ValueError(
        "No se encontraron comentarios del hilo 9747146."
    )


# ------------------------------------------------------------
# 2. Patrones relacionados con inmigración
# ------------------------------------------------------------

patron_inmigracion = re.compile(
    r"\b(?:"
    r"inmigr\w*|"
    r"migrant\w*|"
    r"extranj\w*|"
    r"refugiad\w*|"
    r"menas?|"
    r"sin\s+papeles|"
    r"irregular(?:es)?|"
    r"pateras?|"
    r"cayucos?|"
    r"fronteras?|"
    r"deport\w*|"
    r"expuls\w*|"
    r"asilo|"
    r"multicultur\w*|"
    r"marroqu[ií]\w*|"
    r"magreb[ií]\w*|"
    r"latinoamerican\w*"
    r")\b",
    flags=re.IGNORECASE,
)

patron_impacto_social = re.compile(
    r"\b(?:"
    r"vivienda|"
    r"alquiler\w*|"
    r"empleo|"
    r"trabaj\w*|"
    r"salari\w*|"
    r"ayudas?|"
    r"paguitas?|"
    r"delincuencia|"
    r"seguridad|"
    r"sanidad|"
    r"educaci[oó]n|"
    r"prestacion\w*|"
    r"integraci[oó]n|"
    r"econom[ií]a"
    r")\b",
    flags=re.IGNORECASE,
)


# ------------------------------------------------------------
# 3. Aplicar indicadores
# ------------------------------------------------------------

texto_revision_9747146 = (
    hilo_revision_9747146["texto_sin_citas"]
    .fillna("")
    .astype(str)
)

hilo_revision_9747146["menciona_inmigracion"] = (
    texto_revision_9747146.str.contains(
        patron_inmigracion,
        na=False,
    )
)

hilo_revision_9747146["menciona_impacto_social"] = (
    texto_revision_9747146.str.contains(
        patron_impacto_social,
        na=False,
    )
)

hilo_revision_9747146["menciona_ambos_encuadres"] = (
    hilo_revision_9747146["menciona_inmigracion"]
    & hilo_revision_9747146["menciona_impacto_social"]
)


# ------------------------------------------------------------
# 4. Resumen cuantitativo
# ------------------------------------------------------------

total_comentarios = len(hilo_revision_9747146)

numero_inmigracion = int(
    hilo_revision_9747146[
        "menciona_inmigracion"
    ].sum()
)

numero_impacto_social = int(
    hilo_revision_9747146[
        "menciona_impacto_social"
    ].sum()
)

numero_ambos = int(
    hilo_revision_9747146[
        "menciona_ambos_encuadres"
    ].sum()
)

resumen_revision_9747146 = pd.DataFrame(
    {
        "indicador": [
            "comentarios_totales",
            "mencionan_inmigracion",
            "mencionan_impacto_social",
            "mencionan_ambos_encuadres",
        ],
        "numero_comentarios": [
            total_comentarios,
            numero_inmigracion,
            numero_impacto_social,
            numero_ambos,
        ],
        "porcentaje": [
            100.0,
            round(
                numero_inmigracion
                / total_comentarios
                * 100,
                1,
            ),
            round(
                numero_impacto_social
                / total_comentarios
                * 100,
                1,
            ),
            round(
                numero_ambos
                / total_comentarios
                * 100,
                1,
            ),
        ],
    }
)

display(resumen_revision_9747146)


# ------------------------------------------------------------
# 5. Mostrar mensajes relacionados con inmigración
# ------------------------------------------------------------

mensajes_inmigracion_9747146 = (
    hilo_revision_9747146.loc[
        hilo_revision_9747146[
            "menciona_inmigracion"
        ],
        [
            "numero_mensaje",
            "menciona_inmigracion",
            "menciona_impacto_social",
            "texto_sin_citas",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Mensajes relacionados con inmigración:",
    len(mensajes_inmigracion_9747146),
)

display(mensajes_inmigracion_9747146)


# ------------------------------------------------------------
# 6. Mostrar el inicio completo del debate
# ------------------------------------------------------------

print(
    "Primeros 15 mensajes del hilo:"
)

display(
    hilo_revision_9747146[
        [
            "numero_mensaje",
            "texto_sin_citas",
        ]
    ].head(15)
)

Comentarios recuperados del hilo 9747146: 35


,indicador,numero_comentarios,porcentaje
0,comentarios_totales,35,100.0
1,mencionan_inmigracion,8,22.9
2,mencionan_impacto_social,21,60.0
3,mencionan_ambos_encuadres,7,20.0


Mensajes relacionados con inmigración: 8


,numero_mensaje,menciona_inmigracion,menciona_impacto_social,texto_sin_citas
0,1,True,True,"Siempre me ha soprendido, o al menos a través ..."
1,2,True,True,"La inmigración es el mayor problema actual, ya..."
2,5,True,True,seguridad por los inmigrantes.\ninmigrantes po...
3,6,True,False,La inmigración sin duda. Un pais sin fronteras...
4,15,True,True,"No sé que decirte, se me ocurre una etnia en c..."
5,24,True,True,Pues si no lo entiendes no eres muy listo shur...
6,29,True,True,- Que nos sablen a impuestos para montar chiri...
7,31,True,True,"1. Serán lo que son y de donde vendan, pero si..."


Primeros 15 mensajes del hilo:


,numero_mensaje,texto_sin_citas
0,1,"Siempre me ha soprendido, o al menos a través ..."
1,2,"La inmigración es el mayor problema actual, ya..."
2,3,"Bueno eso de que no influye, es tu opinión."
3,4,Pon una encuesta joder
4,5,seguridad por los inmigrantes.\ninmigrantes po...
5,6,La inmigración sin duda. Un pais sin fronteras...
6,7,No veo la alopecia en la encuesta.
7,8,No digo que no influya. Digo que sea el que má...
8,9,Todos esos problemas tienen en común que su or...
9,10,La vivienda y la baja natalidad


In [154]:
# ============================================================
# CELDA 71 — Consolidación de los lotes 12 y 13
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Aprobar el hilo revisado del lote 13
# ------------------------------------------------------------

comentarios_aprobados_lote_13 = (
    comentarios_lote_13_df[
        comentarios_lote_13_df["id_hilo"]
        .astype(str)
        .eq("9747146")
    ]
    .copy()
    .reset_index(drop=True)
)

if (
    not diagnosticos_lote_13_df.empty
    and "id_hilo" in diagnosticos_lote_13_df.columns
):

    diagnosticos_lote_13_df["id_hilo"] = (
        diagnosticos_lote_13_df["id_hilo"]
        .astype(str)
    )

    diagnosticos_aprobados_lote_13 = (
        diagnosticos_lote_13_df[
            diagnosticos_lote_13_df["id_hilo"]
            .eq("9747146")
        ]
        .copy()
        .reset_index(drop=True)
    )

else:

    diagnosticos_aprobados_lote_13 = pd.DataFrame()


# ------------------------------------------------------------
# 2. Combinar comentarios aprobados
# ------------------------------------------------------------

comentarios_aprobados_lotes_12_13 = (
    pd.concat(
        [
            comentarios_aprobados_lote_12,
            comentarios_aprobados_lote_13,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

comentarios_aprobados_lotes_12_13["id_hilo"] = (
    comentarios_aprobados_lotes_12_13[
        "id_hilo"
    ].astype(str)
)

print(
    "Hilos aprobados pendientes de consolidación:",
    comentarios_aprobados_lotes_12_13[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios brutos pendientes:",
    len(comentarios_aprobados_lotes_12_13),
)


# ------------------------------------------------------------
# 3. Consolidar el corpus bruto
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lotes_12_13,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Seleccionar comentarios del año objetivo
# ------------------------------------------------------------

anios_comentario = pd.to_numeric(
    comentarios_aprobados_lotes_12_13[
        "anio_comentario"
    ],
    errors="coerce",
)

anios_objetivo = pd.to_numeric(
    comentarios_aprobados_lotes_12_13[
        "anio_objetivo"
    ],
    errors="coerce",
)

comentarios_anuales_lotes_12_13 = (
    comentarios_aprobados_lotes_12_13[
        anios_comentario.eq(anios_objetivo)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Consolidar el corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lotes_12_13,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Consolidar diagnósticos
# ------------------------------------------------------------

diagnosticos_nuevos_12_13 = pd.concat(
    [
        diagnosticos_aprobados_lote_12,
        diagnosticos_aprobados_lote_13,
    ],
    ignore_index=True,
)

if not diagnosticos_nuevos_12_13.empty:

    if "id_hilo" in diagnosticos_nuevos_12_13.columns:

        diagnosticos_nuevos_12_13["id_hilo"] = (
            diagnosticos_nuevos_12_13[
                "id_hilo"
            ].astype(str)
        )

    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_nuevos_12_13,
            ],
            ignore_index=True,
        )
    )

    columnas_diagnostico_clave = [
        columna
        for columna in [
            "id_hilo",
            "pagina",
            "url_pagina",
        ]
        if columna
        in diagnosticos_forocoches_ampliado.columns
    ]

    if columnas_diagnostico_clave:

        diagnosticos_forocoches_ampliado = (
            diagnosticos_forocoches_ampliado
            .drop_duplicates(
                subset=columnas_diagnostico_clave,
                keep="first",
            )
            .reset_index(drop=True)
        )


# ------------------------------------------------------------
# 7. Comprobar inmigración–2023
# ------------------------------------------------------------

estrato_inmigracion_2023 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].eq("inmigracion")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2023)
    ]
    .copy()
)

comentarios_por_hilo_2023 = (
    estrato_inmigracion_2023
    .groupby("id_hilo")
    .size()
    .sort_values(ascending=False)
)

pesos_hilos_2023 = (
    comentarios_por_hilo_2023
    / len(estrato_inmigracion_2023)
)

mayor_hilo_pct_2023 = float(
    pesos_hilos_2023.max() * 100
)

hhi_2023 = float(
    np.square(pesos_hilos_2023).sum()
)

hilos_efectivos_2023 = (
    1 / hhi_2023
    if hhi_2023 > 0
    else 0
)

cobertura_suficiente_2023 = bool(
    estrato_inmigracion_2023[
        "id_hilo"
    ].astype(str).nunique() >= 12
    and len(estrato_inmigracion_2023) >= 500
    and estrato_inmigracion_2023[
        "usuario_hash"
    ].nunique() >= 300
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado),
)

print(
    "Hilos de inmigración–2023:",
    estrato_inmigracion_2023[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios de inmigración–2023:",
    len(estrato_inmigracion_2023),
)

print(
    "Usuarios de inmigración–2023:",
    estrato_inmigracion_2023[
        "usuario_hash"
    ].nunique(),
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2023:.1f}%",
)

print(
    "Índice HHI:",
    f"{hhi_2023:.3f}",
)

print(
    "Número efectivo de hilos:",
    f"{hilos_efectivos_2023:.1f}",
)

print(
    "Estrato suficiente:",
    cobertura_suficiente_2023,
)

Hilos aprobados pendientes de consolidación: 5
Comentarios brutos pendientes: 131

Hilos válidos totales: 63
Comentarios anuales totales: 6813
Hilos de inmigración–2023: 12
Comentarios de inmigración–2023: 1162
Usuarios de inmigración–2023: 791
Mayor aportación de un hilo: 44.9%
Índice HHI: 0.282
Número efectivo de hilos: 3.6
Estrato suficiente: True


# LGTB

### 2015

In [155]:
# ============================================================
# CELDA 72 — Lote 14: ampliación de LGTBI–2015
# ============================================================

nuevos_candidatos_lote_14 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2015,
            "id_hilo": "4407356",
            "titulo_resultado": (
                "Mañana voy a Chueca"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4407356"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                'Chueca orgullo gay 2015'
            ),
            "lote_descubrimiento": "ampliacion_14",
            "criterio_inclusion": (
                "Debate explícito sobre Chueca, el Orgullo, "
                "ambiente LGTBI, prejuicios, visibilidad "
                "y estereotipos sobre homosexuales."
            ),
            "subtema_preliminar": (
                "orgullo_chueca_visibilidad_estereotipos"
            ),
            "requiere_revision_tematica": False,
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2015,
            "id_hilo": "4532333",
            "titulo_resultado": (
                "Javier Maroto se ha casado esta mañana "
                "con su novio en el Ayuntamiento de Vitoria"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4532333"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"Javier Maroto" novio matrimonio 2015'
            ),
            "lote_descubrimiento": "ampliacion_14",
            "criterio_inclusion": (
                "Debate sobre matrimonio entre personas "
                "del mismo sexo, orientación sexual, "
                "aceptación social y posición política."
            ),
            "subtema_preliminar": (
                "matrimonio_homosexual_politica"
            ),
            "requiere_revision_tematica": False,
        },
    ]
)

nuevos_candidatos_lote_14["id_hilo"] = (
    nuevos_candidatos_lote_14["id_hilo"]
    .astype(str)
)

# ------------------------------------------------------------
# 1. Excluir candidatos ya catalogados
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

nuevos_candidatos_lote_14 = (
    nuevos_candidatos_lote_14[
        ~nuevos_candidatos_lote_14["id_hilo"]
        .isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Incorporar al catálogo
# ------------------------------------------------------------

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevos_candidatos_lote_14,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Seleccionar candidatos pendientes
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_14 = (
    hilos_candidatos[
        hilos_candidatos["lote_descubrimiento"]
        .eq("ampliacion_14")
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 14:",
    len(nuevos_candidatos_lote_14),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 14:",
    len(candidatos_pendientes_lote_14),
)

display(
    candidatos_pendientes_lote_14[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 14: 2
Candidatos totales en el catálogo: 70
Candidatos pendientes del lote 14: 2


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,4407356,Mañana voy a Chueca,orgullo_chueca_visibilidad_estereotipos,False
1,4532333,Javier Maroto se ha casado esta mañana con su ...,matrimonio_homosexual_politica,False


In [156]:
# ============================================================
# CELDA 73 — Procesamiento técnico del lote 14
# ============================================================

(
    comentarios_validos_lote_14,
    diagnosticos_validos_lote_14,
    incidencias_lote_14,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_14,
    nombre_lote="ampliacion_14",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# 1. Unificar los resultados
# ------------------------------------------------------------

comentarios_lote_14_df = unificar_dataframes(
    comentarios_validos_lote_14
)

diagnosticos_lote_14_df = unificar_dataframes(
    diagnosticos_validos_lote_14
)

incidencias_lote_14_df = unificar_dataframes(
    incidencias_lote_14
)

# ------------------------------------------------------------
# 2. Contar hilos técnicamente válidos
# ------------------------------------------------------------

if (
    not comentarios_lote_14_df.empty
    and "id_hilo" in comentarios_lote_14_df.columns
):

    comentarios_lote_14_df["id_hilo"] = (
        comentarios_lote_14_df["id_hilo"]
        .astype(str)
    )

    numero_hilos_validos_lote_14 = (
        comentarios_lote_14_df[
            "id_hilo"
        ].nunique()
    )

else:

    numero_hilos_validos_lote_14 = 0

# ------------------------------------------------------------
# 3. Resumen
# ------------------------------------------------------------

print(
    "Hilos técnicamente válidos:",
    numero_hilos_validos_lote_14,
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_14_df),
)

if not incidencias_lote_14_df.empty:
    display(incidencias_lote_14_df)


[1/2] lgtbi | 2015 | hilo 4407356
Mañana voy a Chueca
Descartado: no contiene mensajes públicos reconocibles.

[2/2] lgtbi | 2015 | hilo 4532333
Javier Maroto se ha casado esta mañana con su novio en el Ayuntamiento de Vitoria
Primera página pública: 30 mensajes detectados
Hilo 4532333: 3 página(s)
  Página 1/3: 30 comentarios
  Página 2/3: 30 comentarios
  Página 3/3: 19 comentarios
Candidato técnicamente válido: 79 comentarios totales | 79 comentarios de 2015

Hilos técnicamente válidos: 1
Hilos descartados o con error: 1

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [157]:
# ============================================================
# CELDA 74 — Consolidación del lote 14
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Función robusta para normalizar incidencias
# ------------------------------------------------------------

def normalizar_registros(resultado):

    if resultado is None:
        return pd.DataFrame()

    if isinstance(resultado, pd.DataFrame):
        return resultado.copy()

    if isinstance(resultado, dict):
        return pd.DataFrame([resultado])

    if isinstance(resultado, list):

        registros = []

        for elemento in resultado:

            if isinstance(elemento, pd.DataFrame):

                if not elemento.empty:
                    registros.extend(
                        elemento.to_dict("records")
                    )

            elif isinstance(elemento, dict):

                registros.append(elemento)

        if registros:
            return pd.DataFrame(registros)

    return pd.DataFrame()


# ------------------------------------------------------------
# 2. Normalizar resultados del lote
# ------------------------------------------------------------

comentarios_aprobados_lote_14 = (
    comentarios_lote_14_df.copy()
)

diagnosticos_aprobados_lote_14 = (
    diagnosticos_lote_14_df.copy()
)

incidencias_lote_14_df = normalizar_registros(
    incidencias_lote_14
)

comentarios_aprobados_lote_14["id_hilo"] = (
    comentarios_aprobados_lote_14[
        "id_hilo"
    ].astype(str)
)

comentarios_aprobados_lote_14 = (
    comentarios_aprobados_lote_14
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Hilos aprobados pendientes:",
    comentarios_aprobados_lote_14[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios brutos pendientes:",
    len(comentarios_aprobados_lote_14),
)

print(
    "Incidencias reales del lote:",
    len(incidencias_lote_14_df),
)


# ------------------------------------------------------------
# 3. Consolidar el corpus bruto
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_14,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Seleccionar comentarios del año objetivo
# ------------------------------------------------------------

anios_comentario_lote_14 = pd.to_numeric(
    comentarios_aprobados_lote_14[
        "anio_comentario"
    ],
    errors="coerce",
)

anios_objetivo_lote_14 = pd.to_numeric(
    comentarios_aprobados_lote_14[
        "anio_objetivo"
    ],
    errors="coerce",
)

comentarios_anuales_lote_14 = (
    comentarios_aprobados_lote_14[
        anios_comentario_lote_14.eq(
            anios_objetivo_lote_14
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Consolidar el corpus anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_14,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["id_hilo", "id_mensaje"],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Consolidar diagnósticos
# ------------------------------------------------------------

if not diagnosticos_aprobados_lote_14.empty:

    if "id_hilo" in diagnosticos_aprobados_lote_14.columns:

        diagnosticos_aprobados_lote_14["id_hilo"] = (
            diagnosticos_aprobados_lote_14[
                "id_hilo"
            ].astype(str)
        )

    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_aprobados_lote_14,
            ],
            ignore_index=True,
        )
    )

    columnas_diagnostico_clave = [
        columna
        for columna in [
            "id_hilo",
            "pagina",
            "url_pagina",
        ]
        if columna
        in diagnosticos_forocoches_ampliado.columns
    ]

    if columnas_diagnostico_clave:

        diagnosticos_forocoches_ampliado = (
            diagnosticos_forocoches_ampliado
            .drop_duplicates(
                subset=columnas_diagnostico_clave,
                keep="first",
            )
            .reset_index(drop=True)
        )


# ------------------------------------------------------------
# 7. Consolidar la incidencia técnica
# ------------------------------------------------------------

if not incidencias_lote_14_df.empty:

    for columna in (
        set(incidencias_forocoches_ampliacion.columns)
        - set(incidencias_lote_14_df.columns)
    ):
        incidencias_lote_14_df[columna] = pd.NA

    for columna in (
        set(incidencias_lote_14_df.columns)
        - set(incidencias_forocoches_ampliacion.columns)
    ):
        incidencias_forocoches_ampliacion[columna] = pd.NA

    incidencias_lote_14_df = (
        incidencias_lote_14_df[
            incidencias_forocoches_ampliacion.columns
        ]
    )

    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                incidencias_lote_14_df,
            ],
            ignore_index=True,
        )
    )

    columnas_incidencia_clave = [
        columna
        for columna in [
            "id_hilo",
            "tipo_error",
        ]
        if columna
        in incidencias_forocoches_ampliacion.columns
    ]

    if columnas_incidencia_clave:

        incidencias_forocoches_ampliacion = (
            incidencias_forocoches_ampliacion
            .drop_duplicates(
                subset=columnas_incidencia_clave,
                keep="last",
            )
            .reset_index(drop=True)
        )


# ------------------------------------------------------------
# 8. Comprobar el estrato LGTBI–2015
# ------------------------------------------------------------

estrato_lgtbi_2015 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].eq("lgtbi")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2015)
    ]
    .copy()
)

comentarios_por_hilo_lgtbi_2015 = (
    estrato_lgtbi_2015
    .groupby("id_hilo")
    .size()
    .sort_values(ascending=False)
)

mayor_hilo_pct_lgtbi_2015 = (
    comentarios_por_hilo_lgtbi_2015.iloc[0]
    / len(estrato_lgtbi_2015)
    * 100
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado),
)

print(
    "Hilos de LGTBI–2015:",
    estrato_lgtbi_2015[
        "id_hilo"
    ].astype(str).nunique(),
)

print(
    "Comentarios de LGTBI–2015:",
    len(estrato_lgtbi_2015),
)

print(
    "Usuarios de LGTBI–2015:",
    estrato_lgtbi_2015[
        "usuario_hash"
    ].nunique(),
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_lgtbi_2015:.1f}%",
)

print(
    "Hilos todavía necesarios:",
    max(
        12
        - estrato_lgtbi_2015[
            "id_hilo"
        ].astype(str).nunique(),
        0,
    ),
)

Hilos aprobados pendientes: 1
Comentarios brutos pendientes: 79
Incidencias reales del lote: 1

Hilos válidos totales: 64
Comentarios anuales totales: 6892
Hilos de LGTBI–2015: 3
Comentarios de LGTBI–2015: 723
Usuarios de LGTBI–2015: 632
Mayor aportación de un hilo: 83.3%
Hilos todavía necesarios: 9


In [158]:
# ============================================================
# CELDA 75 — Lote 15: candidato multitemático LGTBI–2015
# ============================================================

nuevo_candidato_lote_15 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2015,
            "id_hilo": "4377160",
            "titulo_resultado": (
                "El duro desafío de los musulmanes "
                "que abandonan el Islam"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4377160"
            ),
            "fuente_descubrimiento": "buscador_web",
            "consulta_descubrimiento": (
                'site:forocoches.com/foro/showthread.php '
                '"2015" homosexual "salir del armario"'
            ),
            "lote_descubrimiento": "ampliacion_15",
            "criterio_inclusion": (
                "El mensaje inicial incluye el testimonio "
                "de un joven homosexual criado en un entorno "
                "religioso conservador, con rechazo, identidad "
                "y salida del armario. El hilo es multitemático "
                "y requiere comprobar el peso real de LGTBI."
            ),
            "subtema_preliminar": (
                "homosexualidad_religion_rechazo_familiar"
            ),
            "requiere_revision_tematica": True,
        }
    ]
)

nuevo_candidato_lote_15["id_hilo"] = (
    nuevo_candidato_lote_15["id_hilo"]
    .astype(str)
)

# ------------------------------------------------------------
# 1. Evitar duplicados
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

nuevo_candidato_lote_15 = (
    nuevo_candidato_lote_15[
        ~nuevo_candidato_lote_15["id_hilo"]
        .isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Incorporar al catálogo
# ------------------------------------------------------------

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            nuevo_candidato_lote_15,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Seleccionar candidato pendiente
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_15 = (
    hilos_candidatos[
        hilos_candidatos["lote_descubrimiento"]
        .eq("ampliacion_15")
        & (
            ~hilos_candidatos["id_hilo"]
            .astype(str)
            .isin(ids_hilos_validos)
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobación
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 15:",
    len(nuevo_candidato_lote_15),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 15:",
    len(candidatos_pendientes_lote_15),
)

display(
    candidatos_pendientes_lote_15[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 15: 1
Candidatos totales en el catálogo: 71
Candidatos pendientes del lote 15: 1


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,4377160,El duro desafío de los musulmanes que abandona...,homosexualidad_religion_rechazo_familiar,True


In [159]:
# ============================================================
# CELDA 76 — Procesamiento técnico del lote 15
# ============================================================

(
    comentarios_validos_lote_15,
    diagnosticos_validos_lote_15,
    incidencias_lote_15,
) = procesar_lote_forocoches(
    candidatos_pendientes=candidatos_pendientes_lote_15,
    nombre_lote="ampliacion_15",
    sesion=sesion,
    espera_pagina=1.5,
    espera_hilo=2,
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# 1. Unificar resultados
# ------------------------------------------------------------

comentarios_lote_15_df = unificar_dataframes(
    comentarios_validos_lote_15
)

diagnosticos_lote_15_df = unificar_dataframes(
    diagnosticos_validos_lote_15
)

incidencias_lote_15_df = normalizar_registros(
    incidencias_lote_15
)

# ------------------------------------------------------------
# 2. Contar resultados reales
# ------------------------------------------------------------

if (
    not comentarios_lote_15_df.empty
    and "id_hilo" in comentarios_lote_15_df.columns
):

    comentarios_lote_15_df["id_hilo"] = (
        comentarios_lote_15_df["id_hilo"]
        .astype(str)
    )

    numero_hilos_validos_lote_15 = (
        comentarios_lote_15_df[
            "id_hilo"
        ].nunique()
    )

else:

    numero_hilos_validos_lote_15 = 0

# ------------------------------------------------------------
# 3. Resumen
# ------------------------------------------------------------

print(
    "Hilos técnicamente válidos:",
    numero_hilos_validos_lote_15,
)

print(
    "Hilos descartados o con error:",
    len(incidencias_lote_15_df),
)

if not incidencias_lote_15_df.empty:
    display(incidencias_lote_15_df)


[1/1] lgtbi | 2015 | hilo 4377160
El duro desafío de los musulmanes que abandonan el Islam
Primera página pública: 30 mensajes detectados
Hilo 4377160: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 1 comentarios
Candidato técnicamente válido: 31 comentarios totales | 31 comentarios de 2015

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [160]:
# ============================================================
# CELDA 77 — Revisión temática del hilo 4377160
# ============================================================

import re
import pandas as pd

ID_HILO_REVISION = "4377160"

# ------------------------------------------------------------
# 1. Seleccionar el hilo
# ------------------------------------------------------------

hilo_revision_4377160 = (
    comentarios_lote_15_df[
        comentarios_lote_15_df["id_hilo"]
        .astype(str)
        .eq(ID_HILO_REVISION)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Comentarios recuperados del hilo 4377160:",
    len(hilo_revision_4377160),
)

if hilo_revision_4377160.empty:

    raise ValueError(
        "No se encontraron comentarios del hilo 4377160."
    )


# ------------------------------------------------------------
# 2. Definir patrones temáticos
# ------------------------------------------------------------

patron_lgtbi = re.compile(
    r"\b(?:"
    r"lgtbi?\w*|"
    r"homosexual\w*|"
    r"homofob\w*|"
    r"gay(?:s)?|"
    r"lesbian\w*|"
    r"bisexual\w*|"
    r"transexual\w*|"
    r"transg[eé]ner\w*|"
    r"orientaci[oó]n\s+sexual|"
    r"salir\s+del\s+armario|"
    r"mismo\s+sexo|"
    r"pareja\s+homosexual"
    r")\b",
    flags=re.IGNORECASE,
)

patron_religion_apostasia = re.compile(
    r"\b(?:"
    r"islam\w*|"
    r"musulm[aá]n\w*|"
    r"religio\w*|"
    r"ap[oó]stata\w*|"
    r"apostas[ií]a|"
    r"ate[oa]s?|"
    r"cor[aá]n|"
    r"mahoma|"
    r"mezquita\w*|"
    r"salafis\w*"
    r")\b",
    flags=re.IGNORECASE,
)


# ------------------------------------------------------------
# 3. Aplicar indicadores
# ------------------------------------------------------------

texto_revision_4377160 = (
    hilo_revision_4377160["texto_sin_citas"]
    .fillna("")
    .astype(str)
)

hilo_revision_4377160["menciona_lgtbi"] = (
    texto_revision_4377160.str.contains(
        patron_lgtbi,
        na=False,
    )
)

hilo_revision_4377160[
    "menciona_religion_apostasia"
] = texto_revision_4377160.str.contains(
    patron_religion_apostasia,
    na=False,
)

hilo_revision_4377160["menciona_ambos"] = (
    hilo_revision_4377160["menciona_lgtbi"]
    & hilo_revision_4377160[
        "menciona_religion_apostasia"
    ]
)


# ------------------------------------------------------------
# 4. Crear el resumen
# ------------------------------------------------------------

total_comentarios = len(hilo_revision_4377160)

numero_lgtbi = int(
    hilo_revision_4377160[
        "menciona_lgtbi"
    ].sum()
)

numero_religion = int(
    hilo_revision_4377160[
        "menciona_religion_apostasia"
    ].sum()
)

numero_ambos = int(
    hilo_revision_4377160[
        "menciona_ambos"
    ].sum()
)

resumen_revision_4377160 = pd.DataFrame(
    {
        "indicador": [
            "comentarios_totales",
            "mencionan_lgtbi",
            "mencionan_religion_apostasia",
            "mencionan_ambos_encuadres",
        ],
        "numero_comentarios": [
            total_comentarios,
            numero_lgtbi,
            numero_religion,
            numero_ambos,
        ],
        "porcentaje": [
            100.0,
            round(
                numero_lgtbi
                / total_comentarios
                * 100,
                1,
            ),
            round(
                numero_religion
                / total_comentarios
                * 100,
                1,
            ),
            round(
                numero_ambos
                / total_comentarios
                * 100,
                1,
            ),
        ],
    }
)

display(resumen_revision_4377160)


# ------------------------------------------------------------
# 5. Mostrar los mensajes LGTBI detectados
# ------------------------------------------------------------

mensajes_lgtbi_4377160 = (
    hilo_revision_4377160.loc[
        hilo_revision_4377160[
            "menciona_lgtbi"
        ],
        [
            "numero_mensaje",
            "menciona_lgtbi",
            "menciona_religion_apostasia",
            "texto_sin_citas",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Mensajes relacionados con LGTBI:",
    len(mensajes_lgtbi_4377160),
)

display(mensajes_lgtbi_4377160)


# ------------------------------------------------------------
# 6. Mostrar el comienzo del hilo
# ------------------------------------------------------------

print(
    "Primeros 12 mensajes del hilo:"
)

display(
    hilo_revision_4377160[
        [
            "numero_mensaje",
            "texto_sin_citas",
        ]
    ].head(12)
)

Comentarios recuperados del hilo 4377160: 31


,indicador,numero_comentarios,porcentaje
0,comentarios_totales,31,100.0
1,mencionan_lgtbi,1,3.2
2,mencionan_religion_apostasia,10,32.3
3,mencionan_ambos_encuadres,1,3.2


Mensajes relacionados con LGTBI: 1


,numero_mensaje,menciona_lgtbi,menciona_religion_apostasia,texto_sin_citas
0,1,True,True,El duro desafío de los musulmanes que abandona...


Primeros 12 mensajes del hilo:


,numero_mensaje,texto_sin_citas
0,1,El duro desafío de los musulmanes que abandona...
1,2,Debe ser duro porque tienen una cara de moros....
2,3,Mefo a la tía.
3,4,Joder cada día le cojo más asco al Islam.
4,5,Es homo\nY bastante guapilla tengo que reconocer
5,6,Seguirán oliendo a moro.
6,7,Asco de cultura.\nPD: A la tía melafo.
7,8,Todo mi apoyo a esos valientes.
8,9,Estos son los ejemplos del mundo musulmán que ...
9,10,La prima buena de Ozil.


In [161]:
# ============================================================
# CELDA 78 — Exclusión temática del hilo 4377160
# ============================================================

import pandas as pd

registro_exclusion_4377160 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2015,
            "id_hilo": "4377160",
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4377160"
            ),
            "tipo_error": "exclusion_tematica",
            "mensaje_error": (
                "El eje LGTBI aparece como máximo en "
                "2 de 31 comentarios (6,5 %), incluyendo "
                "la expresión no detectada 'es homo'. "
                "La conversación se centra en religión "
                "y apostasía."
            ),
            "lote": "ampliacion_15",
        }
    ]
)

# ------------------------------------------------------------
# 1. Adaptar columnas al registro general
# ------------------------------------------------------------

for columna in (
    set(incidencias_forocoches_ampliacion.columns)
    - set(registro_exclusion_4377160.columns)
):
    registro_exclusion_4377160[columna] = pd.NA

for columna in (
    set(registro_exclusion_4377160.columns)
    - set(incidencias_forocoches_ampliacion.columns)
):
    incidencias_forocoches_ampliacion[columna] = pd.NA

registro_exclusion_4377160 = (
    registro_exclusion_4377160[
        incidencias_forocoches_ampliacion.columns
    ]
)

# ------------------------------------------------------------
# 2. Incorporar y eliminar duplicados
# ------------------------------------------------------------

incidencias_forocoches_ampliacion = (
    pd.concat(
        [
            incidencias_forocoches_ampliacion,
            registro_exclusion_4377160,
        ],
        ignore_index=True,
    )
)

columnas_incidencia_clave = [
    columna
    for columna in [
        "id_hilo",
        "tipo_error",
    ]
    if columna
    in incidencias_forocoches_ampliacion.columns
]

if columnas_incidencia_clave:

    incidencias_forocoches_ampliacion = (
        incidencias_forocoches_ampliacion
        .drop_duplicates(
            subset=columnas_incidencia_clave,
            keep="last",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Comprobación
# ------------------------------------------------------------

print(
    "Hilo excluido temáticamente: 4377160"
)

print(
    "Motivo: el debate es principalmente "
    "sobre religión y apostasía."
)

print(
    "Hilos válidos de LGTBI–2015:",
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].eq("lgtbi")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2015)
    ]["id_hilo"]
    .astype(str)
    .nunique(),
)

Hilo excluido temáticamente: 4377160
Motivo: el debate es principalmente sobre religión y apostasía.
Hilos válidos de LGTBI–2015: 3


In [162]:
# ============================================================
# CELDA 79 — Auditoría de actividad LGTBI durante 2015
# ============================================================

import pandas as pd

comentarios_brutos_auditoria = (
    comentarios_forocoches_bruto_ampliado.copy()
)

comentarios_brutos_auditoria["id_hilo"] = (
    comentarios_brutos_auditoria["id_hilo"]
    .astype(str)
)

comentarios_brutos_auditoria["anio_comentario_num"] = (
    pd.to_numeric(
        comentarios_brutos_auditoria[
            "anio_comentario"
        ],
        errors="coerce",
    )
)

# ------------------------------------------------------------
# 1. Localizar comentarios LGTBI publicados en 2015
# ------------------------------------------------------------

actividad_lgtbi_2015_disponible = (
    comentarios_brutos_auditoria[
        comentarios_brutos_auditoria["tema"]
        .eq("lgtbi")
        & comentarios_brutos_auditoria[
            "anio_comentario_num"
        ].eq(2015)
    ]
    .copy()
)

# ------------------------------------------------------------
# 2. Resumen por hilo
# ------------------------------------------------------------

if actividad_lgtbi_2015_disponible.empty:

    resumen_actividad_lgtbi_2015 = pd.DataFrame(
        columns=[
            "id_hilo",
            "comentarios_2015",
            "usuarios_2015",
            "fecha_minima",
            "fecha_maxima",
        ]
    )

else:

    resumen_actividad_lgtbi_2015 = (
        actividad_lgtbi_2015_disponible
        .groupby("id_hilo", as_index=False)
        .agg(
            comentarios_2015=(
                "id_mensaje",
                "count",
            ),
            usuarios_2015=(
                "usuario_hash",
                "nunique",
            ),
            fecha_minima=(
                "fecha_comentario",
                "min",
            ),
            fecha_maxima=(
                "fecha_comentario",
                "max",
            ),
        )
        .sort_values(
            "comentarios_2015",
            ascending=False,
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Marcar cuáles ya están consolidados en el corpus anual
# ------------------------------------------------------------

ids_lgtbi_2015_consolidados = set(
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].eq("lgtbi")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2015)
    ]["id_hilo"]
    .astype(str)
)

resumen_actividad_lgtbi_2015[
    "ya_consolidado_lgtbi_2015"
] = resumen_actividad_lgtbi_2015[
    "id_hilo"
].astype(str).isin(
    ids_lgtbi_2015_consolidados
)

candidatos_recuperables_lgtbi_2015 = (
    resumen_actividad_lgtbi_2015[
        ~resumen_actividad_lgtbi_2015[
            "ya_consolidado_lgtbi_2015"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Resultados
# ------------------------------------------------------------

print(
    "Hilos LGTBI con comentarios brutos en 2015:",
    len(resumen_actividad_lgtbi_2015),
)

print(
    "Hilos ya consolidados en LGTBI–2015:",
    len(ids_lgtbi_2015_consolidados),
)

print(
    "Hilos potencialmente recuperables:",
    len(candidatos_recuperables_lgtbi_2015),
)

display(resumen_actividad_lgtbi_2015)

if not candidatos_recuperables_lgtbi_2015.empty:

    print(
        "Candidatos que podrían recuperarse "
        "sin nueva extracción:"
    )

    display(candidatos_recuperables_lgtbi_2015)

Hilos LGTBI con comentarios brutos en 2015: 3
Hilos ya consolidados en LGTBI–2015: 3
Hilos potencialmente recuperables: 0


,id_hilo,comentarios_2015,usuarios_2015,fecha_minima,fecha_maxima,ya_consolidado_lgtbi_2015
0,4677567,602,529,2015-12-11 16:00:00,2015-12-19 00:48:00,True
1,4532333,79,66,2015-09-18 13:00:00,2015-09-19 09:31:00,True
2,4323181,42,39,2015-05-20 10:17:00,2015-05-20 11:53:00,True


In [163]:
# ============================================================
# CELDA 80 — Búsqueda RSS de candidatos LGTBI–2015
# ============================================================

import re
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from urllib.parse import quote_plus, urlparse, parse_qs

CONSULTAS_LGTBI_2015 = [
    (
        'site:forocoches.com/foro/showthread.php '
        '"gay" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"gais" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"homosexual" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"homofobia" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"matrimonio homosexual" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"orgullo gay" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"lesbiana" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"bisexual" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"transexual" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"transgénero" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"salir del armario" "2015"'
    ),
    (
        'site:forocoches.com/foro/showthread.php '
        '"pareja del mismo sexo" "2015"'
    ),
]

PATRON_ID_HILO = re.compile(
    r"(?:showthread\.php\?(?:[^#]*&)?t=|"
    r"showthread\.php\?)(\d+)",
    flags=re.IGNORECASE,
)

registros_busqueda_lgtbi_2015 = []

headers_busqueda = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/124 Safari/537.36"
    )
}

for numero_consulta, consulta in enumerate(
    CONSULTAS_LGTBI_2015,
    start=1,
):

    url_rss = (
        "https://www.bing.com/search"
        f"?q={quote_plus(consulta)}"
        "&format=rss"
        "&count=50"
        "&setlang=es"
    )

    print(
        f"[{numero_consulta}/"
        f"{len(CONSULTAS_LGTBI_2015)}] "
        f"{consulta}"
    )

    try:

        respuesta = requests.get(
            url_rss,
            headers=headers_busqueda,
            timeout=30,
        )

        respuesta.raise_for_status()

        raiz_xml = ET.fromstring(
            respuesta.content
        )

        items = raiz_xml.findall(
            ".//item"
        )

        print(
            "  Resultados RSS:",
            len(items),
        )

        for posicion, item in enumerate(
            items,
            start=1,
        ):

            titulo = (
                item.findtext("title")
                or ""
            ).strip()

            enlace = (
                item.findtext("link")
                or ""
            ).strip()

            descripcion = (
                item.findtext("description")
                or ""
            ).strip()

            texto_enlace = (
                enlace
                + " "
                + descripcion
            )

            coincidencia_id = (
                PATRON_ID_HILO.search(
                    texto_enlace
                )
            )

            if not coincidencia_id:
                continue

            id_hilo = coincidencia_id.group(1)

            registros_busqueda_lgtbi_2015.append(
                {
                    "consulta": consulta,
                    "posicion": posicion,
                    "id_hilo": id_hilo,
                    "titulo_resultado": titulo,
                    "url_resultado": enlace,
                    "descripcion_resultado": (
                        descripcion
                    ),
                }
            )

    except Exception as error:

        print(
            "  Error:",
            type(error).__name__,
            str(error)[:200],
        )

    time.sleep(1.5)


# ------------------------------------------------------------
# 1. Construir el catálogo provisional
# ------------------------------------------------------------

resultados_rss_lgtbi_2015 = pd.DataFrame(
    registros_busqueda_lgtbi_2015
)

if resultados_rss_lgtbi_2015.empty:

    candidatos_rss_lgtbi_2015 = pd.DataFrame(
        columns=[
            "id_hilo",
            "titulo_resultado",
            "url_resultado",
            "numero_consultas",
        ]
    )

else:

    candidatos_rss_lgtbi_2015 = (
        resultados_rss_lgtbi_2015
        .groupby(
            [
                "id_hilo",
                "titulo_resultado",
                "url_resultado",
            ],
            as_index=False,
        )
        .agg(
            numero_consultas=(
                "consulta",
                "nunique",
            ),
            consultas_coincidentes=(
                "consulta",
                lambda serie: " | ".join(
                    sorted(set(serie))
                ),
            ),
            descripcion_resultado=(
                "descripcion_resultado",
                "first",
            ),
        )
        .sort_values(
            [
                "numero_consultas",
                "id_hilo",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 2. Excluir IDs ya conocidos
# ------------------------------------------------------------

ids_ya_conocidos = set(
    hilos_candidatos["id_hilo"]
    .astype(str)
)

ids_ya_conocidos.update(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_rss_nuevos_lgtbi_2015 = (
    candidatos_rss_lgtbi_2015[
        ~candidatos_rss_lgtbi_2015[
            "id_hilo"
        ].astype(str).isin(ids_ya_conocidos)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Mostrar resultados
# ------------------------------------------------------------

print("\n" + "=" * 70)

print(
    "Resultados únicos de ForoCoches:",
    len(candidatos_rss_lgtbi_2015),
)

print(
    "Candidatos nuevos no catalogados:",
    len(candidatos_rss_nuevos_lgtbi_2015),
)

display(
    candidatos_rss_nuevos_lgtbi_2015[
        [
            "id_hilo",
            "titulo_resultado",
            "numero_consultas",
            "url_resultado",
        ]
    ].head(30)
)

[1/12] site:forocoches.com/foro/showthread.php "gay" "2015"
  Resultados RSS: 10
[2/12] site:forocoches.com/foro/showthread.php "gais" "2015"
  Resultados RSS: 10
[3/12] site:forocoches.com/foro/showthread.php "homosexual" "2015"
  Resultados RSS: 10
[4/12] site:forocoches.com/foro/showthread.php "homofobia" "2015"
  Resultados RSS: 10
[5/12] site:forocoches.com/foro/showthread.php "matrimonio homosexual" "2015"
  Resultados RSS: 10
[6/12] site:forocoches.com/foro/showthread.php "orgullo gay" "2015"
  Resultados RSS: 10
[7/12] site:forocoches.com/foro/showthread.php "lesbiana" "2015"
  Resultados RSS: 10
[8/12] site:forocoches.com/foro/showthread.php "bisexual" "2015"
  Resultados RSS: 10
[9/12] site:forocoches.com/foro/showthread.php "transexual" "2015"
  Resultados RSS: 10
[10/12] site:forocoches.com/foro/showthread.php "transgénero" "2015"
  Resultados RSS: 10
[11/12] site:forocoches.com/foro/showthread.php "salir del armario" "2015"
  Resultados RSS: 7
[12/12] site:forocoches.com/f

,id_hilo,titulo_resultado,numero_consultas,url_resultado


In [164]:
# ============================================================
# CELDA 81 — Diagnóstico de enlaces devueltos por Bing RSS
# ============================================================

from urllib.parse import quote_plus
import requests
import pandas as pd
import xml.etree.ElementTree as ET

consulta_diagnostico = (
    'site:forocoches.com/foro/showthread.php '
    '"homosexual" "2015"'
)

url_rss_diagnostico = (
    "https://www.bing.com/search"
    f"?q={quote_plus(consulta_diagnostico)}"
    "&format=rss"
    "&count=50"
    "&setlang=es"
)

respuesta_diagnostico = requests.get(
    url_rss_diagnostico,
    headers=headers_busqueda,
    timeout=30,
)

respuesta_diagnostico.raise_for_status()

raiz_diagnostico = ET.fromstring(
    respuesta_diagnostico.content
)

registros_diagnostico = []

for posicion, item in enumerate(
    raiz_diagnostico.findall(".//item"),
    start=1,
):

    registros_diagnostico.append(
        {
            "posicion": posicion,
            "titulo": (
                item.findtext("title")
                or ""
            ).strip(),
            "enlace": (
                item.findtext("link")
                or ""
            ).strip(),
            "descripcion": (
                item.findtext("description")
                or ""
            ).strip(),
        }
    )

diagnostico_enlaces_bing = pd.DataFrame(
    registros_diagnostico
)

print(
    "Resultados inspeccionados:",
    len(diagnostico_enlaces_bing),
)

pd.set_option(
    "display.max_colwidth",
    300,
)

display(
    diagnostico_enlaces_bing[
        [
            "posicion",
            "titulo",
            "enlace",
        ]
    ]
)

print(
    "\nPrimer enlace completo:"
)

if not diagnostico_enlaces_bing.empty:
    print(
        diagnostico_enlaces_bing.loc[
            0,
            "enlace",
        ]
    )

Resultados inspeccionados: 10


,posicion,titulo,enlace
0,1,The best Multi-Asset ETFs,https://www.justetf.com/en/how-to/invest-in-multi-asset-etfs.html
1,2,Multi Asset ETFs - sumgrowth.com,https://www.sumgrowth.com/top-etfs/multi-asset-etfs.html
2,3,BlackRock Multi-Asset Funds | BlackRock,https://www.blackrock.com/us/financial-professionals/investments/products/multi-asset
3,4,ICICI Prudential Multi-Asset Fund Growth - Moneycontrol,https://www.moneycontrol.com/mutual-funds/nav/icici-prudential-multi-asset-fund/MPI038
4,5,Best Multi Asset Allocation Mutual Funds to invest in India 2026,https://groww.in/mutual-funds/category/best-multi-asset-allocation-mutual-funds
5,6,Best Multi Asset Allocation Funds to Invest in India 2026,https://www.etmoney.com/mutual-funds/hybrid/multi-asset-allocation/75
6,7,Franklin Multi-Asset Growth Fund A SCHAX - Morningstar,https://www.morningstar.com/funds/XNAS/SCHAX/quote
7,8,The Best ETFs Of 2026 – Forbes Advisor,https://www.forbes.com/advisor/investing/best-etfs/
8,9,MULTI-ASSET - CI Global Asset Management,https://www.cifinancial.com/ci-gam/ca/en/portfolio-management/multi-asset-strategies.html
9,10,4 Strong Multi-Asset Income Funds - Morningstar,https://www.morningstar.com/portfolios/4-strong-multi-asset-income-funds



Primer enlace completo:
https://www.justetf.com/en/how-to/invest-in-multi-asset-etfs.html


In [166]:
# CELDA 83 — Inspección de robots.txt y posibles sitemaps de ForoCoches

import re
import pandas as pd

urls_infraestructura = [
    "https://forocoches.com/robots.txt",
    "https://www.forocoches.com/robots.txt",
    "https://forocoches.com/sitemap.xml",
    "https://www.forocoches.com/sitemap.xml",
    "https://forocoches.com/foro/sitemap.xml",
    "https://www.forocoches.com/foro/sitemap.xml",
]

resultados_infraestructura = []
sitemaps_detectados = set()

for url in urls_infraestructura:
    try:
        respuesta = sesion.get(
            url,
            timeout=30,
            allow_redirects=True,
        )

        contenido = respuesta.text or ""
        content_type = respuesta.headers.get(
            "Content-Type",
            "",
        )

        titulo = ""
        coincidencia_titulo = re.search(
            r"<title[^>]*>(.*?)</title>",
            contenido,
            flags=re.IGNORECASE | re.DOTALL,
        )

        if coincidencia_titulo:
            titulo = re.sub(
                r"\s+",
                " ",
                coincidencia_titulo.group(1),
            ).strip()

        urls_sitemap = re.findall(
            r"https?://[^\s<>\"']*sitemap[^\s<>\"']*",
            contenido,
            flags=re.IGNORECASE,
        )

        sitemaps_detectados.update(urls_sitemap)

        resultados_infraestructura.append(
            {
                "url_solicitada": url,
                "estado_http": respuesta.status_code,
                "url_final": respuesta.url,
                "content_type": content_type,
                "longitud_respuesta": len(contenido),
                "titulo_html": titulo,
                "sitemaps_encontrados": len(urls_sitemap),
                "vista_previa": re.sub(
                    r"\s+",
                    " ",
                    contenido[:300],
                ).strip(),
            }
        )

    except Exception as error:
        resultados_infraestructura.append(
            {
                "url_solicitada": url,
                "estado_http": None,
                "url_final": None,
                "content_type": None,
                "longitud_respuesta": 0,
                "titulo_html": None,
                "sitemaps_encontrados": 0,
                "vista_previa": (
                    f"{type(error).__name__}: {error}"
                ),
            }
        )

diagnostico_infraestructura = pd.DataFrame(
    resultados_infraestructura
)

print(
    "Recursos inspeccionados:",
    len(diagnostico_infraestructura),
)

print(
    "Sitemaps únicos detectados:",
    len(sitemaps_detectados),
)

display(
    diagnostico_infraestructura[
        [
            "estado_http",
            "url_solicitada",
            "url_final",
            "content_type",
            "longitud_respuesta",
            "titulo_html",
            "sitemaps_encontrados",
            "vista_previa",
        ]
    ]
)

if sitemaps_detectados:
    display(
        pd.DataFrame(
            {
                "url_sitemap": sorted(
                    sitemaps_detectados
                )
            }
        )
    )
else:
    print(
        "No se localizaron direcciones de sitemap "
        "en las respuestas inspeccionadas."
    )

Recursos inspeccionados: 6
Sitemaps únicos detectados: 0


,estado_http,url_solicitada,url_final,content_type,longitud_respuesta,titulo_html,sitemaps_encontrados,vista_previa
0,200,https://forocoches.com/robots.txt,https://forocoches.com/robots.txt,text/plain; charset=utf-8,1731,,0,User-agent: Mediapartners-Google* Allow: / User-agent: archive.org_bot Disallow: / User-agent: BoardReader Disallow: / User-agent: GPTBot Disallow: / User-agent: Google-Extended Disallow: / User-agent: PerplexityBot Disallow: / User-agent: CCBot Disallow: / User-agent: FacebookBot Disallow: / User-
1,200,https://www.forocoches.com/robots.txt,https://forocoches.com/robots.txt,text/plain; charset=utf-8,1731,,0,User-agent: Mediapartners-Google* Allow: / User-agent: archive.org_bot Disallow: / User-agent: BoardReader Disallow: / User-agent: GPTBot Disallow: / User-agent: Google-Extended Disallow: / User-agent: PerplexityBot Disallow: / User-agent: CCBot Disallow: / User-agent: FacebookBot Disallow: / User-
2,200,https://forocoches.com/sitemap.xml,https://forocoches.com/foro/,text/html; charset=UTF-8,146295,Forocoches - Foro de Coches y Motor,0,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 Transitional//EN"" ""http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd""> <html xmlns=""http://www.w3.org/1999/xhtml"" dir=""ltr"" lang=""es""> <head> <!-- Google tag (gtag.js) --> <script> (function () { var ga_registrado = false; var style"
3,200,https://www.forocoches.com/sitemap.xml,https://forocoches.com/foro/,text/html; charset=UTF-8,146295,Forocoches - Foro de Coches y Motor,0,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 Transitional//EN"" ""http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd""> <html xmlns=""http://www.w3.org/1999/xhtml"" dir=""ltr"" lang=""es""> <head> <!-- Google tag (gtag.js) --> <script> (function () { var ga_registrado = false; var style"
4,200,https://forocoches.com/foro/sitemap.xml,https://forocoches.com/foro/,text/html; charset=UTF-8,146295,Forocoches - Foro de Coches y Motor,0,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 Transitional//EN"" ""http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd""> <html xmlns=""http://www.w3.org/1999/xhtml"" dir=""ltr"" lang=""es""> <head> <!-- Google tag (gtag.js) --> <script> (function () { var ga_registrado = false; var style"
5,200,https://www.forocoches.com/foro/sitemap.xml,https://forocoches.com/foro/,text/html; charset=UTF-8,146295,Forocoches - Foro de Coches y Motor,0,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 Transitional//EN"" ""http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd""> <html xmlns=""http://www.w3.org/1999/xhtml"" dir=""ltr"" lang=""es""> <head> <!-- Google tag (gtag.js) --> <script> (function () { var ga_registrado = false; var style"


No se localizaron direcciones de sitemap en las respuestas inspeccionadas.


In [167]:
# CELDA 84 — Mostrar y analizar el robots.txt de ForoCoches

import re
import pandas as pd

url_robots = "https://forocoches.com/robots.txt"

respuesta_robots = sesion.get(
    url_robots,
    timeout=30,
)

respuesta_robots.raise_for_status()

texto_robots = respuesta_robots.text.strip()

print("Estado HTTP:", respuesta_robots.status_code)
print("Caracteres:", len(texto_robots))
print("\nCONTENIDO COMPLETO DE ROBOTS.TXT")
print("=" * 70)
print(texto_robots)
print("=" * 70)

# Extraer las directivas de forma estructurada
directivas_robots = []

agente_actual = None

for linea in texto_robots.splitlines():
    linea = linea.strip()

    if not linea or linea.startswith("#"):
        continue

    if ":" not in linea:
        continue

    directiva, valor = linea.split(":", 1)

    directiva = directiva.strip().lower()
    valor = valor.strip()

    if directiva == "user-agent":
        agente_actual = valor

    directivas_robots.append(
        {
            "user_agent": agente_actual,
            "directiva": directiva,
            "valor": valor,
        }
    )

df_directivas_robots = pd.DataFrame(
    directivas_robots
)

print("\nDirectivas detectadas:", len(df_directivas_robots))

display(df_directivas_robots)

# Buscar cualquier referencia que pueda servir para descubrir contenido
referencias_utiles = df_directivas_robots[
    df_directivas_robots["directiva"].isin(
        [
            "sitemap",
            "allow",
            "disallow",
        ]
    )
].copy()

print("\nReferencias potencialmente útiles:")

display(referencias_utiles)

Estado HTTP: 200
Caracteres: 1730

CONTENIDO COMPLETO DE ROBOTS.TXT
User-agent: Mediapartners-Google* 
Allow: /
User-agent: archive.org_bot
Disallow: /
User-agent: BoardReader
Disallow: /
User-agent: GPTBot
Disallow: /
User-agent: Google-Extended
Disallow: /
User-agent: PerplexityBot
Disallow: /
User-agent: CCBot
Disallow: /
User-agent: FacebookBot
Disallow: /
User-agent: OmgiliBot
Disallow: /
User-agent: anthropic-ai
Disallow: /
User-agent: cohere-ai
Disallow: /
User-agent: *
Disallow: /foro/*styleid=
Disallow: /foro/calendar.php
Disallow: /foro/editpost.php
Disallow: /foro/memberlist.php
Disallow: /foro/misc.php
Disallow: /foro/newreply.php
Disallow: /foro/newthread.php
Disallow: /foro/printthread.php
Disallow: /foro/private.php
Disallow: /foro/register.php
Disallow: /foro/report.php
Disallow: /foro/showgroups.php
Disallow: /foro/usercp.php
Disallow: /foro/admincp/
Disallow: /foro/modcp/
Disallow: /foro/online.php
Disallow: /foro/subscription.php
Disallow: /foro/sendtofriend.php
Disa

,user_agent,directiva,valor
0,Mediapartners-Google*,user-agent,Mediapartners-Google*
1,Mediapartners-Google*,allow,/
2,archive.org_bot,user-agent,archive.org_bot
3,archive.org_bot,disallow,/
4,BoardReader,user-agent,BoardReader
...,...,...,...
61,*,disallow,/foro/forumdisplay.php?*pp=
62,*,disallow,/foro/customavatars/
63,*,disallow,/foro/login.php
64,*,disallow,/redir/



Referencias potencialmente útiles:


,user_agent,directiva,valor
1,Mediapartners-Google*,allow,/
3,archive.org_bot,disallow,/
5,BoardReader,disallow,/
7,GPTBot,disallow,/
9,Google-Extended,disallow,/
11,PerplexityBot,disallow,/
13,CCBot,disallow,/
15,FacebookBot,disallow,/
17,OmgiliBot,disallow,/
19,anthropic-ai,disallow,/


In [168]:
# CELDA 85 — Auditar buscadores y rutas públicas de la portada

from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re

url_portada_foro = "https://forocoches.com/foro/"

respuesta_portada = sesion.get(
    url_portada_foro,
    timeout=30,
)

respuesta_portada.raise_for_status()

soup_portada = BeautifulSoup(
    respuesta_portada.text,
    "html.parser",
)

print("Estado HTTP:", respuesta_portada.status_code)
print("URL final:", respuesta_portada.url)
print(
    "Título:",
    soup_portada.title.get_text(
        " ",
        strip=True,
    )
    if soup_portada.title
    else ""
)

# ------------------------------------------------------------
# 1. Formularios públicos detectados
# ------------------------------------------------------------

formularios_detectados = []

for numero, formulario in enumerate(
    soup_portada.find_all("form"),
    start=1,
):
    accion = formulario.get(
        "action",
        "",
    )

    metodo = formulario.get(
        "method",
        "get",
    ).lower()

    campos = []

    for campo in formulario.find_all(
        ["input", "select", "textarea"]
    ):
        nombre = campo.get("name")

        if nombre:
            campos.append(nombre)

    formularios_detectados.append(
        {
            "numero_formulario": numero,
            "metodo": metodo,
            "accion_original": accion,
            "accion_completa": urljoin(
                respuesta_portada.url,
                accion,
            ),
            "campos": ", ".join(
                sorted(set(campos))
            ),
            "texto_visible": re.sub(
                r"\s+",
                " ",
                formulario.get_text(
                    " ",
                    strip=True,
                ),
            )[:300],
        }
    )

df_formularios_portada = pd.DataFrame(
    formularios_detectados
)

print(
    "\nFormularios detectados:",
    len(df_formularios_portada),
)

if not df_formularios_portada.empty:
    display(df_formularios_portada)
else:
    print("No se detectaron formularios públicos.")

# ------------------------------------------------------------
# 2. Enlaces potencialmente útiles
# ------------------------------------------------------------

palabras_ruta = re.compile(
    r"search|buscar|archive|archivo|"
    r"forumdisplay|showthread|tag|"
    r"subscription|newposts",
    flags=re.IGNORECASE,
)

enlaces_utiles = []

for enlace in soup_portada.find_all(
    "a",
    href=True,
):
    href_original = enlace["href"].strip()

    texto_enlace = re.sub(
        r"\s+",
        " ",
        enlace.get_text(
            " ",
            strip=True,
        ),
    )

    url_completa = urljoin(
        respuesta_portada.url,
        href_original,
    )

    if palabras_ruta.search(
        f"{texto_enlace} {href_original}"
    ):
        enlaces_utiles.append(
            {
                "texto_enlace": texto_enlace,
                "href_original": href_original,
                "url_completa": url_completa,
            }
        )

df_enlaces_utiles = (
    pd.DataFrame(enlaces_utiles)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "\nEnlaces potencialmente útiles:",
    len(df_enlaces_utiles),
)

if not df_enlaces_utiles.empty:
    display(df_enlaces_utiles)
else:
    print(
        "No se encontraron enlaces públicos "
        "de búsqueda, archivo o navegación."
    )

# ------------------------------------------------------------
# 3. Presencia textual de posibles mecanismos
# ------------------------------------------------------------

html_portada_minusculas = (
    respuesta_portada.text.lower()
)

comprobaciones_portada = pd.DataFrame(
    [
        {
            "elemento": termino,
            "presente_en_html": (
                termino
                in html_portada_minusculas
            ),
        }
        for termino in [
            "search.php",
            "forumdisplay.php",
            "showthread.php",
            "archive",
            "buscar",
        ]
    ]
)

display(comprobaciones_portada)

Estado HTTP: 200
URL final: https://forocoches.com/foro/
Título: Forocoches - Foro de Coches y Motor

Formularios detectados: 0
No se detectaron formularios públicos.

Enlaces potencialmente útiles: 48


,texto_enlace,href_original,url_completa
0,¿No tienes invitaciones? Descubre como conseguirlas.,/foro/showthread.php?t=4133485,https://forocoches.com/foro/showthread.php?t=4133485
1,Ver mensajes propios,search.php?searchthreadid=&do=process&exactname=1&userid=0,https://forocoches.com/foro/search.php?searchthreadid=&do=process&exactname=1&userid=0
2,Suscribirse,subscription.php?do=addsubscription&t=,https://forocoches.com/foro/subscription.php?do=addsubscription&t=
3,General Temas no relacionados con el mundo del motor.,forumdisplay.php?f=2,https://forocoches.com/foro/forumdisplay.php?f=2
4,"Electrónica / Informática Tecnología: móviles, video, audio, hardware...",forumdisplay.php?f=17,https://forocoches.com/foro/forumdisplay.php?f=17
5,"Videojuegos PC Master Race, Videoconsolas, Steam y más.",forumdisplay.php?f=43,https://forocoches.com/foro/forumdisplay.php?f=43
6,"Empleo / Emprendimiento Trabajo, empresas y emprendedores.",forumdisplay.php?f=23,https://forocoches.com/foro/forumdisplay.php?f=23
7,"Oposiciones Foro de oposiciones: justicia, guardia civil, policía, prisiones y administrativo...",forumdisplay.php?f=93,https://forocoches.com/foro/forumdisplay.php?f=93
8,"Viajes Para los viajeros: Consultas, experiencias, fotos...",forumdisplay.php?f=27,https://forocoches.com/foro/forumdisplay.php?f=27
9,"Basket Noticias y debate sobre NBA, Euroliga y ACB.",forumdisplay.php?f=95,https://forocoches.com/foro/forumdisplay.php?f=95


,elemento,presente_en_html
0,search.php,True
1,forumdisplay.php,True
2,showthread.php,True
3,archive,False
4,buscar,True


In [169]:
# CELDA 86 — Diagnóstico del índice público del foro General

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs
import pandas as pd
import re

url_foro_general = (
    "https://forocoches.com/foro/"
    "forumdisplay.php?f=2"
)

respuesta_general = sesion.get(
    url_foro_general,
    timeout=30,
)

respuesta_general.raise_for_status()

soup_general = BeautifulSoup(
    respuesta_general.text,
    "html.parser",
)

titulo_pagina_general = (
    soup_general.title.get_text(
        " ",
        strip=True,
    )
    if soup_general.title
    else ""
)

print("Estado HTTP:", respuesta_general.status_code)
print("URL final:", respuesta_general.url)
print("Título:", titulo_pagina_general)
print(
    "Longitud HTML:",
    len(respuesta_general.text),
)

# ------------------------------------------------------------
# 1. Extraer enlaces canónicos de hilos
# ------------------------------------------------------------

hilos_indice = []

for enlace in soup_general.find_all(
    "a",
    href=True,
):
    href = enlace["href"]

    coincidencia_hilo = re.search(
        r"showthread\.php\?t=(\d+)",
        href,
        flags=re.IGNORECASE,
    )

    if not coincidencia_hilo:
        continue

    id_hilo = coincidencia_hilo.group(1)

    titulo = re.sub(
        r"\s+",
        " ",
        enlace.get_text(
            " ",
            strip=True,
        ),
    )

    if not titulo:
        continue

    hilos_indice.append(
        {
            "id_hilo": id_hilo,
            "titulo": titulo,
            "url_hilo": urljoin(
                respuesta_general.url,
                f"showthread.php?t={id_hilo}",
            ),
        }
    )

df_hilos_indice = (
    pd.DataFrame(hilos_indice)
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "\nHilos únicos detectados en la página:",
    len(df_hilos_indice),
)

if not df_hilos_indice.empty:
    display(df_hilos_indice.head(30))
else:
    print(
        "No se detectaron enlaces canónicos "
        "de hilos."
    )

# ------------------------------------------------------------
# 2. Inspeccionar enlaces de paginación
# ------------------------------------------------------------

paginas_detectadas = []

for enlace in soup_general.find_all(
    "a",
    href=True,
):
    href = enlace["href"]

    if "forumdisplay.php" not in href:
        continue

    url_completa = urljoin(
        respuesta_general.url,
        href,
    )

    parametros = parse_qs(
        urlparse(url_completa).query
    )

    if parametros.get("f", [""])[0] != "2":
        continue

    pagina = parametros.get(
        "page",
        [None],
    )[0]

    texto_enlace = re.sub(
        r"\s+",
        " ",
        enlace.get_text(
            " ",
            strip=True,
        ),
    )

    paginas_detectadas.append(
        {
            "texto_enlace": texto_enlace,
            "pagina": pagina,
            "url": url_completa,
        }
    )

df_paginas_general = (
    pd.DataFrame(paginas_detectadas)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "\nEnlaces de navegación detectados:",
    len(df_paginas_general),
)

if not df_paginas_general.empty:
    display(df_paginas_general)

# ------------------------------------------------------------
# 3. Buscar números de página en todo el HTML
# ------------------------------------------------------------

numeros_pagina_html = [
    int(numero)
    for numero in re.findall(
        r"forumdisplay\.php\?"
        r"[^\"']*?"
        r"(?:&amp;|&)page=(\d+)",
        respuesta_general.text,
        flags=re.IGNORECASE,
    )
]

pagina_maxima_visible = (
    max(numeros_pagina_html)
    if numeros_pagina_html
    else None
)

print(
    "\nMayor número de página visible:",
    pagina_maxima_visible,
)

# ------------------------------------------------------------
# 4. Vista textual alrededor del primer hilo
# ------------------------------------------------------------

if not df_hilos_indice.empty:
    primer_id = str(
        df_hilos_indice.loc[0, "id_hilo"]
    )

    nodo_primer_hilo = soup_general.find(
        "a",
        href=re.compile(
            rf"showthread\.php\?t={primer_id}\b",
            flags=re.IGNORECASE,
        ),
    )

    contenedor = (
        nodo_primer_hilo.find_parent("tr")
        if nodo_primer_hilo
        else None
    )

    texto_contenedor = (
        re.sub(
            r"\s+",
            " ",
            contenedor.get_text(
                " ",
                strip=True,
            ),
        )
        if contenedor
        else ""
    )

    print(
        "\nVista previa del registro del primer hilo:"
    )
    print(texto_contenedor[:1000])

Estado HTTP: 200
URL final: https://forocoches.com/foro/forumdisplay.php?f=2
Título: General - Forocoches
Longitud HTML: 317559

Hilos únicos detectados en la página: 44


,id_hilo,titulo,url_hilo
0,4133485,¿No tienes invitaciones? Descubre como conseguirlas.,https://forocoches.com/foro/showthread.php?t=4133485
1,10747911,[URGENTE] XOKAS RESPONDE a BURGER KING,https://forocoches.com/foro/showthread.php?t=10747911
2,10748399,Soy experto en chicas.,https://forocoches.com/foro/showthread.php?t=10748399
3,10748312,Mi familia se hundió tras accidente… ahora necesito comprar coche ¿cuál?,https://forocoches.com/foro/showthread.php?t=10748312
4,10748117,[ATENCION] No entres ⚽️,https://forocoches.com/foro/showthread.php?t=10748117
5,10748315,Puede Cucurella frenar a Messi hoy?,https://forocoches.com/foro/showthread.php?t=10748315
6,10748030,Anteayer tuvo lugar uno de los accidentes más bestias (Camión vs X5),https://forocoches.com/foro/showthread.php?t=10748030
7,10740632,Traigo a todo incluido a amiga a la playa y su larva y me niega la crema del sol xd,https://forocoches.com/foro/showthread.php?t=10740632
8,10748396,Trabajos que se han panchitolizado,https://forocoches.com/foro/showthread.php?t=10748396
9,10748205,"Otra vez los CUENTANEGROS y censores ANTIWOKE han perdido, y van...",https://forocoches.com/foro/showthread.php?t=10748205



Enlaces de navegación detectados: 15


,texto_enlace,pagina,url
0,2,2,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=2
1,3,3,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=3
2,4,4,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=4
3,5,5,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=5
4,6,6,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=6
5,7,7,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=7
6,8,8,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=8
7,9,9,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=9
8,10,10,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=10
9,11,11,https://forocoches.com/foro/forumdisplay.php?f=2&order=desc&page=11



Mayor número de página visible: 38

Vista previa del registro del primer hilo:



In [170]:
# CELDA 87 — Prueba controlada del buscador público interno

from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re

url_busqueda_prueba = (
    "https://forocoches.com/foro/search.php"
)

parametros_busqueda_prueba = {
    "do": "process",
    "query": "homosexual",
    "titleonly": "1",
}

respuesta_busqueda_prueba = sesion.get(
    url_busqueda_prueba,
    params=parametros_busqueda_prueba,
    timeout=30,
    allow_redirects=True,
)

soup_busqueda_prueba = BeautifulSoup(
    respuesta_busqueda_prueba.text,
    "html.parser",
)

titulo_busqueda = (
    soup_busqueda_prueba.title.get_text(
        " ",
        strip=True,
    )
    if soup_busqueda_prueba.title
    else ""
)

texto_pagina_busqueda = re.sub(
    r"\s+",
    " ",
    soup_busqueda_prueba.get_text(
        " ",
        strip=True,
    ),
)

print(
    "Estado HTTP:",
    respuesta_busqueda_prueba.status_code,
)
print(
    "URL solicitada:",
    respuesta_busqueda_prueba.request.url,
)
print(
    "URL final:",
    respuesta_busqueda_prueba.url,
)
print(
    "Título:",
    titulo_busqueda,
)
print(
    "Longitud HTML:",
    len(respuesta_busqueda_prueba.text),
)
print(
    "\nVista previa textual:"
)
print(texto_pagina_busqueda[:1500])

# ------------------------------------------------------------
# Extraer posibles hilos devueltos
# ------------------------------------------------------------

resultados_busqueda_interna = []

for enlace in soup_busqueda_prueba.find_all(
    "a",
    href=True,
):
    href = enlace["href"]

    coincidencia = re.search(
        r"showthread\.php\?"
        r"[^\"']*?\bt=(\d+)",
        href,
        flags=re.IGNORECASE,
    )

    if not coincidencia:
        continue

    id_hilo = coincidencia.group(1)

    titulo = re.sub(
        r"\s+",
        " ",
        enlace.get_text(
            " ",
            strip=True,
        ),
    )

    if not titulo:
        continue

    resultados_busqueda_interna.append(
        {
            "id_hilo": id_hilo,
            "titulo": titulo,
            "url_hilo": urljoin(
                respuesta_busqueda_prueba.url,
                f"showthread.php?t={id_hilo}",
            ),
        }
    )

df_resultados_busqueda_interna = (
    pd.DataFrame(
        resultados_busqueda_interna
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "\nHilos únicos encontrados:",
    len(df_resultados_busqueda_interna),
)

if not df_resultados_busqueda_interna.empty:
    display(df_resultados_busqueda_interna)
else:
    print(
        "La consulta no devolvió hilos "
        "públicos reconocibles."
    )

# ------------------------------------------------------------
# Diagnóstico de posibles restricciones
# ------------------------------------------------------------

indicadores_busqueda = pd.DataFrame(
    [
        {
            "indicador": "solicita_inicio_sesion",
            "detectado": bool(
                re.search(
                    r"iniciar sesión|"
                    r"inicia sesión|"
                    r"login|"
                    r"identific",
                    texto_pagina_busqueda,
                    flags=re.IGNORECASE,
                )
            ),
        },
        {
            "indicador": "mensaje_informacion",
            "detectado": (
                "forocoches - información"
                in titulo_busqueda.lower()
            ),
        },
        {
            "indicador": "mensaje_sin_resultados",
            "detectado": bool(
                re.search(
                    r"no se encontraron|"
                    r"ningún resultado|"
                    r"sin resultados",
                    texto_pagina_busqueda,
                    flags=re.IGNORECASE,
                )
            ),
        },
        {
            "indicador": "contiene_searchid",
            "detectado": (
                "searchid="
                in respuesta_busqueda_prueba.text.lower()
            ),
        },
    ]
)

display(indicadores_busqueda)

Estado HTTP: 200
URL solicitada: https://forocoches.com/foro/search.php?do=process&query=homosexual&titleonly=1
URL final: https://forocoches.com/foro/search.php?do=process&query=homosexual&titleonly=1
Título: Forocoches
Longitud HTML: 112501

Vista previa textual:
Forocoches Registrarse Iniciar sesión Tus notificaciones Mensajes privados Menciones Citas Invitaciones ¿No tienes invitaciones? Descubre como conseguirlas. Modo día Modo noche Modo automático Dar feedback del nuevo diseño Ver mensajes propios Suscribirse Información No has iniciado sesión o no tienes permiso para acceder a esta página. Esto puede deberse a una de varias razones: No has iniciado sesión. Rellena el formulario al pie de esta página e inténtalo de nuevo. Puede ser que no tengas suficientes privilegios para acceder a esta página. ¿Estás tratando de editar mensajes de otros usuarios, acceder a funciones administrativas o algún otro sistema privilegiado? Si estás tratando de crear un mensaje y no es posible, tu cu

,id_hilo,titulo,url_hilo
0,4133485,¿No tienes invitaciones? Descubre como conseguirlas.,https://forocoches.com/foro/showthread.php?t=4133485
1,8241760,Contacto,https://forocoches.com/foro/showthread.php?t=8241760


,indicador,detectado
0,solicita_inicio_sesion,True
1,mensaje_informacion,False
2,mensaje_sin_resultados,False
3,contiene_searchid,False


In [171]:
# CELDA 88 — Incorporar candidato LGTBI–2015 localizado externamente

import pandas as pd

candidatos_lote_16 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2015,
            "id_hilo": "4195631",
            "titulo_resultado": (
                '"LAVADO DE CEREBRO 1" - '
                "La paradoja de la igualdad"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4195631"
            ),
            "subtema_preliminar": (
                "identidad_genero_teoria_queer"
            ),
            "criterio_inclusion": (
                "Debate sobre identidad de género, "
                "teoría queer y transexualidad."
            ),
            "requiere_revision_tematica": True,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_16",
        }
    ]
)

# Normalizar el identificador en ambos DataFrames
hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_16["id_hilo"] = (
    candidatos_lote_16["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_16 = (
    candidatos_lote_16[
        ~candidatos_lote_16["id_hilo"].isin(
            ids_catalogados
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# La concatenación admite las columnas nuevas y conserva
# las existentes del catálogo
hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_16,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_16 = (
    candidatos_nuevos_lote_16[
        ~candidatos_nuevos_lote_16[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 16:",
    len(candidatos_nuevos_lote_16),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 16:",
    len(candidatos_pendientes_lote_16),
)

display(
    candidatos_pendientes_lote_16[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 16: 1
Candidatos totales en el catálogo: 72
Candidatos pendientes del lote 16: 1


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,4195631,"""LAVADO DE CEREBRO 1"" - La paradoja de la igualdad",identidad_genero_teoria_queer,True


In [174]:
# CELDA 89 — Extraer el candidato del lote 16

(
    comentarios_validos_lote_16,
    diagnosticos_lote_16,
    errores_lote_16,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_16,
    "lote_16",
    sesion,
)

comentarios_lote_16_df = unificar_dataframes(
    comentarios_validos_lote_16
)

diagnosticos_lote_16_df = unificar_dataframes(
    diagnosticos_lote_16
)

errores_lote_16_df = normalizar_registros(
    errores_lote_16
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_16_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_16_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_16_df),
)


[1/1] lgtbi | 2015 | hilo 4195631
"LAVADO DE CEREBRO 1" - La paradoja de la igualdad
Primera página pública: 30 mensajes detectados
Hilo 4195631: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 11 comentarios
Candidato técnicamente válido: 41 comentarios totales | 40 comentarios de 2015

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [175]:
# CELDA 90 — Revisión temática del hilo 4195631

import re
import pandas as pd

hilo_revision_4195631 = (
    comentarios_lote_16_df[
        comentarios_lote_16_df[
            "id_hilo"
        ]
        .astype(str)
        .eq("4195631")
    ]
    .copy()
    .reset_index(drop=True)
)

# Utilizamos el texto propio del usuario, sin citas
texto_revision = (
    hilo_revision_4195631[
        "texto_sin_citas"
    ]
    .fillna("")
    .astype(str)
)

# ------------------------------------------------------------
# 1. Vocabulario directamente relacionado con LGTBI
# ------------------------------------------------------------

patron_lgtbi = re.compile(
    r"\b(?:"
    r"lgtbi?\w*|"
    r"gay(?:s)?|"
    r"gais|"
    r"homosexual\w*|"
    r"homofob\w*|"
    r"lesbian\w*|"
    r"bisexual\w*|"
    r"transexual\w*|"
    r"transgener\w*|"
    r"transgéner\w*|"
    r"transfob\w*|"
    r"queer|"
    r"disforia|"
    r"identidad(?:es)?\s+de\s+g[eé]nero|"
    r"orientaci[oó]n(?:es)?\s+sexual(?:es)?|"
    r"cambio\s+de\s+sexo|"
    r"reasignaci[oó]n\s+de\s+sexo"
    r")\b",
    flags=re.IGNORECASE,
)

# ------------------------------------------------------------
# 2. Encuadre general de igualdad, feminismo y roles
# ------------------------------------------------------------

patron_igualdad_genero = re.compile(
    r"\b(?:"
    r"igualdad|"
    r"feminis\w*|"
    r"machis\w*|"
    r"patriarc\w*|"
    r"mujer(?:es)?|"
    r"hombre(?:s)?|"
    r"sexo(?:s)?|"
    r"g[eé]nero(?:s)?|"
    r"roles?\s+de\s+g[eé]nero|"
    r"construcci[oó]n\s+social|"
    r"biolog[ií]\w*"
    r")\b",
    flags=re.IGNORECASE,
)

hilo_revision_4195631[
    "menciona_lgtbi"
] = texto_revision.str.contains(
    patron_lgtbi,
    na=False,
)

hilo_revision_4195631[
    "menciona_igualdad_genero"
] = texto_revision.str.contains(
    patron_igualdad_genero,
    na=False,
)

hilo_revision_4195631[
    "menciona_ambos_encuadres"
] = (
    hilo_revision_4195631[
        "menciona_lgtbi"
    ]
    & hilo_revision_4195631[
        "menciona_igualdad_genero"
    ]
)

total_comentarios = len(
    hilo_revision_4195631
)

numero_lgtbi = int(
    hilo_revision_4195631[
        "menciona_lgtbi"
    ].sum()
)

numero_igualdad = int(
    hilo_revision_4195631[
        "menciona_igualdad_genero"
    ].sum()
)

numero_ambos = int(
    hilo_revision_4195631[
        "menciona_ambos_encuadres"
    ].sum()
)

resumen_revision_4195631 = pd.DataFrame(
    [
        {
            "indicador": "comentarios_totales",
            "numero_comentarios": total_comentarios,
            "porcentaje": 100.0,
        },
        {
            "indicador": "mencionan_lgtbi",
            "numero_comentarios": numero_lgtbi,
            "porcentaje": round(
                numero_lgtbi
                / max(total_comentarios, 1)
                * 100,
                1,
            ),
        },
        {
            "indicador": (
                "mencionan_igualdad_genero"
            ),
            "numero_comentarios": numero_igualdad,
            "porcentaje": round(
                numero_igualdad
                / max(total_comentarios, 1)
                * 100,
                1,
            ),
        },
        {
            "indicador": (
                "mencionan_ambos_encuadres"
            ),
            "numero_comentarios": numero_ambos,
            "porcentaje": round(
                numero_ambos
                / max(total_comentarios, 1)
                * 100,
                1,
            ),
        },
    ]
)

print(
    "Comentarios recuperados del hilo 4195631:",
    total_comentarios,
)

display(resumen_revision_4195631)

mensajes_lgtbi_4195631 = (
    hilo_revision_4195631[
        hilo_revision_4195631[
            "menciona_lgtbi"
        ]
    ][
        [
            "numero_mensaje",
            "menciona_lgtbi",
            "menciona_igualdad_genero",
            "texto_sin_citas",
        ]
    ]
    .reset_index(drop=True)
)

print(
    "\nMensajes relacionados directamente con LGTBI:",
    len(mensajes_lgtbi_4195631),
)

display(mensajes_lgtbi_4195631)

print("\nPrimeros 15 mensajes del hilo:")

display(
    hilo_revision_4195631[
        [
            "numero_mensaje",
            "texto_sin_citas",
        ]
    ].head(15)
)

Comentarios recuperados del hilo 4195631: 41


,indicador,numero_comentarios,porcentaje
0,comentarios_totales,41,100.0
1,mencionan_lgtbi,3,7.3
2,mencionan_igualdad_genero,15,36.6
3,mencionan_ambos_encuadres,2,4.9



Mensajes relacionados directamente con LGTBI: 3


,numero_mensaje,menciona_lgtbi,menciona_igualdad_genero,texto_sin_citas
0,17,True,True,"Busca sobre David Reimer y te darás cuenta de que ser ""hombre"" o ""mujer"" (o sentirse y actuar como tal) tiene una base biológica y neuronal importante. Hay cosas culturales, no las niego, pero la teoría\nqueer\ntan de moda en los sesenta, de que la masculinidad y la feminidad son construcciones ..."
1,24,True,True,"En absoluto. Como te comento, te doy la razón porque biológicamente somos distintos.\nPara ello siempre remito a la gente al caso de David Reimer. En los sesenta se puso de moda la teoría queer, que afirmaba que lo masculino y lo femenino eran construcciones sociales, y que un niño nace sin ello..."
2,36,True,False,Ahora que digan los homosexuales que serlo es un gusto por algo como otro cualquiera y no algo que vaya mal en el coco al nacer..



Primeros 15 mensajes del hilo:


,numero_mensaje,texto_sin_citas
0,1,aquí lo resume un poco:\nhttp://blog.iese.edu/nuriachinchilla...gia-de-genero/
1,2,Voy citando
2,3,empezamos bien!
3,4,
4,5,"Ahora se da cuenta un tonto y venga a darle bola\nCuando, por ejemplo, aceptamos a alguien por ser negro ya le consideramos diferente, si lo considerásemos igual no tendríamos que darle ayuda ni aceptarle porque ya es un igual.\nE dixho"
5,6,@\nNightRaider
6,7,"mirate el documental, y después podrás comprenderlo"
7,8,Pillo sitio para posterior visionado.
8,9,"no te defraudará, no es una chorrada sencionalista; es algo serio y muy importante en la sociedad actual"
9,10,sitiando para verlo mientras como


In [176]:
# CELDA 91 — Excluir temáticamente el hilo 4195631

import pandas as pd

id_hilo_excluido = "4195631"

exclusion_tematica_4195631 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2015,
            "id_hilo": id_hilo_excluido,
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4195631"
            ),
            "tipo_error": "exclusion_tematica",
            "mensaje_error": (
                "Debate principalmente centrado en "
                "igualdad, diferencias sexuales y roles "
                "de género. Solo 3 de 41 comentarios "
                "mencionan directamente cuestiones LGTBI."
            ),
            "lote": "lote_16",
        }
    ]
)

incidencias_forocoches_ampliacion = (
    pd.concat(
        [
            incidencias_forocoches_ampliacion,
            exclusion_tematica_4195631,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "tipo_error",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

hilos_lgtbi_2015_actuales = int(
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ].eq(2015)
        )
    ]["id_hilo"]
    .astype(str)
    .nunique()
)

print(
    "Hilo excluido temáticamente:",
    id_hilo_excluido,
)
print(
    "Motivo: el tema LGTBI es secundario "
    "dentro de un debate general sobre igualdad."
)
print(
    "Hilos válidos de LGTBI–2015:",
    hilos_lgtbi_2015_actuales,
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2015_actuales,
        0,
    ),
)

Hilo excluido temáticamente: 4195631
Motivo: el tema LGTBI es secundario dentro de un debate general sobre igualdad.
Hilos válidos de LGTBI–2015: 3
Hilos todavía necesarios: 9


In [177]:
# CELDA 92 — Incorporar candidatos para LGTBI–2016

import pandas as pd

candidatos_lote_17 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "5029297",
            "titulo_resultado": (
                "Así son los jóvenes gais "
                "de la derecha española"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=5029297"
            ),
            "subtema_preliminar": (
                "homosexualidad_identidad_politica"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre jóvenes gais, "
                "orientación sexual e identidad política."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_17",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "4771850",
            "titulo_resultado": (
                "¿Es ForoCoches heteroflexible?"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4771850"
            ),
            "subtema_preliminar": (
                "orientacion_sexual_heteroflexibilidad"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre bisexualidad, "
                "heteroflexibilidad y prácticas homosexuales."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_17",
        },
    ]
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_17["id_hilo"] = (
    candidatos_lote_17["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_17 = (
    candidatos_lote_17[
        ~candidatos_lote_17[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_17,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_17 = (
    candidatos_nuevos_lote_17[
        ~candidatos_nuevos_lote_17[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 17:",
    len(candidatos_nuevos_lote_17),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 17:",
    len(candidatos_pendientes_lote_17),
)

display(
    candidatos_pendientes_lote_17[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 17: 2
Candidatos totales en el catálogo: 74
Candidatos pendientes del lote 17: 2


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,5029297,Así son los jóvenes gais de la derecha española,homosexualidad_identidad_politica,False
1,4771850,¿Es ForoCoches heteroflexible?,orientacion_sexual_heteroflexibilidad,False


## 2016

In [178]:
# CELDA 93 — Extraer los candidatos del lote 17

(
    comentarios_validos_lote_17,
    diagnosticos_lote_17,
    errores_lote_17,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_17,
    "lote_17",
    sesion,
)

comentarios_lote_17_df = unificar_dataframes(
    comentarios_validos_lote_17
)

diagnosticos_lote_17_df = unificar_dataframes(
    diagnosticos_lote_17
)

errores_lote_17_df = normalizar_registros(
    errores_lote_17
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_17_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_17_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_17_df),
)


[1/2] lgtbi | 2016 | hilo 5029297
Así son los jóvenes gais de la derecha española
Primera página pública: 30 mensajes detectados
Hilo 5029297: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 16 comentarios
Candidato técnicamente válido: 46 comentarios totales | 46 comentarios de 2016

[2/2] lgtbi | 2016 | hilo 4771850
¿Es ForoCoches heteroflexible?
Primera página pública: 6 mensajes detectados
Hilo 4771850: 1 página(s)
  Página 1/1: 6 comentarios
Candidato técnicamente válido: 6 comentarios totales | 6 comentarios de 2016

Hilos técnicamente válidos: 2
Hilos descartados o con error: 0

Hilos técnicamente válidos: 2
Hilos descartados o con error: 0


In [179]:
# CELDA 94 — Aprobar y consolidar el lote 17

import pandas as pd

ids_aprobados_lote_17 = {
    "5029297",
    "4771850",
}

# ------------------------------------------------------------
# 1. Seleccionar los comentarios aprobados
# ------------------------------------------------------------

comentarios_aprobados_lote_17 = (
    comentarios_lote_17_df[
        comentarios_lote_17_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_aprobados_lote_17)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_lote_17 = (
    comentarios_aprobados_lote_17[
        pd.to_numeric(
            comentarios_aprobados_lote_17[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lote_17[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Consolidar comentarios brutos
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_17,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Consolidar comentarios del año objetivo
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_17,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Consolidar diagnósticos e incidencias
# ------------------------------------------------------------

if not diagnosticos_lote_17_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_17_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

if not errores_lote_17_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_17_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Comprobar el estado de LGTBI–2016
# ------------------------------------------------------------

estrato_lgtbi_2016 = (
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            pd.to_numeric(
                comentarios_forocoches_anual_ampliado[
                    "anio_objetivo"
                ],
                errors="coerce",
            ).eq(2016)
        )
    ]
    .copy()
)

hilos_lgtbi_2016 = int(
    estrato_lgtbi_2016[
        "id_hilo"
    ]
    .astype(str)
    .nunique()
)

comentarios_lgtbi_2016 = len(
    estrato_lgtbi_2016
)

usuarios_lgtbi_2016 = int(
    estrato_lgtbi_2016[
        "usuario_hash"
    ].nunique()
)

comentarios_por_hilo_2016 = (
    estrato_lgtbi_2016[
        "id_hilo"
    ]
    .astype(str)
    .value_counts()
)

mayor_hilo_pct_2016 = (
    comentarios_por_hilo_2016.iloc[0]
    / max(comentarios_lgtbi_2016, 1)
    * 100
    if not comentarios_por_hilo_2016.empty
    else 0
)

print(
    "Hilos aprobados del lote 17:",
    len(ids_aprobados_lote_17),
)
print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_17),
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)
print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)
print(
    "Hilos de LGTBI–2016:",
    hilos_lgtbi_2016,
)
print(
    "Comentarios de LGTBI–2016:",
    comentarios_lgtbi_2016,
)
print(
    "Usuarios de LGTBI–2016:",
    usuarios_lgtbi_2016,
)
print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2016:.1f}%",
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2016,
        0,
    ),
)

Hilos aprobados del lote 17: 2
Comentarios anuales aprobados: 52

Hilos válidos totales: 66
Comentarios anuales totales: 6944
Hilos de LGTBI–2016: 5
Comentarios de LGTBI–2016: 235
Usuarios de LGTBI–2016: 178
Mayor aportación de un hilo: 42.1%
Hilos todavía necesarios: 7


In [181]:
# CELDA 95 — Incorporar nuevos candidatos de LGTBI–2016

import pandas as pd

candidatos_lote_18 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "5103831",
            "titulo_resultado": (
                "Comentemos estas noticias "
                "[Tema Serio]"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=5103831"
            ),
            "subtema_preliminar": (
                "violencia_trans_agresion"
            ),
            "criterio_inclusion": (
                "Debate sobre la agresión y el robo "
                "sufridos por una persona transexual."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_18",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "5015696",
            "titulo_resultado": (
                "¿Iríais al orgullo de Madrid?"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=5015696"
            ),
            "subtema_preliminar": (
                "orgullo_lgtbi_visibilidad"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre la celebración "
                "del Orgullo LGTBI de Madrid."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_18",
        },
    ]
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_18["id_hilo"] = (
    candidatos_lote_18["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_18 = (
    candidatos_lote_18[
        ~candidatos_lote_18[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_18,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_18 = (
    candidatos_nuevos_lote_18[
        ~candidatos_nuevos_lote_18[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 18:",
    len(candidatos_nuevos_lote_18),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 18:",
    len(candidatos_pendientes_lote_18),
)

display(
    candidatos_pendientes_lote_18[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 18: 2
Candidatos totales en el catálogo: 76
Candidatos pendientes del lote 18: 2


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,5103831,Comentemos estas noticias [Tema Serio],violencia_trans_agresion,False
1,5015696,¿Iríais al orgullo de Madrid?,orgullo_lgtbi_visibilidad,False


In [182]:
# CELDA 96 — Extraer los candidatos del lote 18

(
    comentarios_validos_lote_18,
    diagnosticos_lote_18,
    errores_lote_18,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_18,
    "lote_18",
    sesion,
)

comentarios_lote_18_df = unificar_dataframes(
    comentarios_validos_lote_18
)

diagnosticos_lote_18_df = unificar_dataframes(
    diagnosticos_lote_18
)

errores_lote_18_df = normalizar_registros(
    errores_lote_18
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_18_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_18_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_18_df),
)


[1/2] lgtbi | 2016 | hilo 5103831
Comentemos estas noticias [Tema Serio]
Primera página pública: 6 mensajes detectados
Hilo 5103831: 1 página(s)
  Página 1/1: 6 comentarios
Candidato técnicamente válido: 6 comentarios totales | 6 comentarios de 2016

[2/2] lgtbi | 2016 | hilo 5015696
¿Iríais al orgullo de Madrid?
Primera página pública: 27 mensajes detectados
Hilo 5015696: 2 página(s)
  Página 1/2: 27 comentarios
  Página 2/2: 27 comentarios
Candidato técnicamente válido: 27 comentarios totales | 27 comentarios de 2016

Hilos técnicamente válidos: 2
Hilos descartados o con error: 0

Hilos técnicamente válidos: 2
Hilos descartados o con error: 0


In [183]:
# CELDA 97 — Aprobar y consolidar el lote 18

import pandas as pd

ids_aprobados_lote_18 = {
    "5103831",
    "5015696",
}

comentarios_aprobados_lote_18 = (
    comentarios_lote_18_df[
        comentarios_lote_18_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_aprobados_lote_18)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_lote_18 = (
    comentarios_aprobados_lote_18[
        pd.to_numeric(
            comentarios_aprobados_lote_18[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lote_18[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Consolidar comentarios brutos
comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_18,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Consolidar comentarios correspondientes a 2016
comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_18,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Consolidar diagnósticos
if not diagnosticos_lote_18_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_18_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# Consolidar posibles incidencias
if not errores_lote_18_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_18_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# Estado actualizado de LGTBI–2016
# ------------------------------------------------------------

estrato_lgtbi_2016 = (
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            pd.to_numeric(
                comentarios_forocoches_anual_ampliado[
                    "anio_objetivo"
                ],
                errors="coerce",
            ).eq(2016)
        )
    ]
    .copy()
)

hilos_lgtbi_2016 = int(
    estrato_lgtbi_2016[
        "id_hilo"
    ]
    .astype(str)
    .nunique()
)

comentarios_lgtbi_2016 = len(
    estrato_lgtbi_2016
)

usuarios_lgtbi_2016 = int(
    estrato_lgtbi_2016[
        "usuario_hash"
    ].nunique()
)

comentarios_por_hilo_2016 = (
    estrato_lgtbi_2016[
        "id_hilo"
    ]
    .astype(str)
    .value_counts()
)

mayor_hilo_pct_2016 = (
    comentarios_por_hilo_2016.iloc[0]
    / max(comentarios_lgtbi_2016, 1)
    * 100
    if not comentarios_por_hilo_2016.empty
    else 0
)

print(
    "Hilos aprobados del lote 18:",
    len(ids_aprobados_lote_18),
)
print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_18),
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)
print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)
print(
    "Hilos de LGTBI–2016:",
    hilos_lgtbi_2016,
)
print(
    "Comentarios de LGTBI–2016:",
    comentarios_lgtbi_2016,
)
print(
    "Usuarios de LGTBI–2016:",
    usuarios_lgtbi_2016,
)
print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2016:.1f}%",
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2016,
        0,
    ),
)

Hilos aprobados del lote 18: 2
Comentarios anuales aprobados: 33

Hilos válidos totales: 68
Comentarios anuales totales: 6977
Hilos de LGTBI–2016: 7
Comentarios de LGTBI–2016: 268
Usuarios de LGTBI–2016: 208
Mayor aportación de un hilo: 36.9%
Hilos todavía necesarios: 5


## 2019

In [184]:
# CELDA 98 — Incorporar primer candidato adicional de LGTBI–2019

import pandas as pd

candidatos_lote_19 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2019,
            "id_hilo": "6993059",
            "titulo_resultado": (
                "Si no te sabes todos los géneros, "
                "eres un FATXA"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=6993059"
            ),
            "subtema_preliminar": (
                "identidades_trans_no_binarias"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre identidades "
                "trans, no binarias y diversidad de género."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_19",
        }
    ]
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_19["id_hilo"] = (
    candidatos_lote_19["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_19 = (
    candidatos_lote_19[
        ~candidatos_lote_19[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_19,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_19 = (
    candidatos_nuevos_lote_19[
        ~candidatos_nuevos_lote_19[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 19:",
    len(candidatos_nuevos_lote_19),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 19:",
    len(candidatos_pendientes_lote_19),
)

display(
    candidatos_pendientes_lote_19[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)#

Candidatos incorporados en el lote 19: 1
Candidatos totales en el catálogo: 77
Candidatos pendientes del lote 19: 1


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,6993059,"Si no te sabes todos los géneros, eres un FATXA",identidades_trans_no_binarias,False


In [185]:
# CELDA 99 — Extraer el candidato del lote 19

(
    comentarios_validos_lote_19,
    diagnosticos_lote_19,
    errores_lote_19,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_19,
    "lote_19",
    sesion,
)

comentarios_lote_19_df = unificar_dataframes(
    comentarios_validos_lote_19
)

diagnosticos_lote_19_df = unificar_dataframes(
    diagnosticos_lote_19
)

errores_lote_19_df = normalizar_registros(
    errores_lote_19
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_19_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_19_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_19_df),
)


[1/1] lgtbi | 2019 | hilo 6993059
Si no te sabes todos los géneros, eres un FATXA
Primera página pública: 26 mensajes detectados
Hilo 6993059: 1 página(s)
  Página 1/1: 26 comentarios
Candidato técnicamente válido: 26 comentarios totales | 26 comentarios de 2019

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [186]:
# CELDA 100 — Aprobar y consolidar el lote 19

import pandas as pd

ids_aprobados_lote_19 = {
    "6993059",
}

comentarios_aprobados_lote_19 = (
    comentarios_lote_19_df[
        comentarios_lote_19_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_aprobados_lote_19)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_lote_19 = (
    comentarios_aprobados_lote_19[
        pd.to_numeric(
            comentarios_aprobados_lote_19[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lote_19[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_19,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_19,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

if not diagnosticos_lote_19_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_19_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

if not errores_lote_19_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_19_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# Estado actualizado de LGTBI–2019
# ------------------------------------------------------------

estrato_lgtbi_2019 = (
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            pd.to_numeric(
                comentarios_forocoches_anual_ampliado[
                    "anio_objetivo"
                ],
                errors="coerce",
            ).eq(2019)
        )
    ]
    .copy()
)

hilos_lgtbi_2019 = int(
    estrato_lgtbi_2019[
        "id_hilo"
    ]
    .astype(str)
    .nunique()
)

comentarios_lgtbi_2019 = len(
    estrato_lgtbi_2019
)

usuarios_lgtbi_2019 = int(
    estrato_lgtbi_2019[
        "usuario_hash"
    ].nunique()
)

comentarios_por_hilo_2019 = (
    estrato_lgtbi_2019[
        "id_hilo"
    ]
    .astype(str)
    .value_counts()
)

mayor_hilo_pct_2019 = (
    comentarios_por_hilo_2019.iloc[0]
    / max(comentarios_lgtbi_2019, 1)
    * 100
    if not comentarios_por_hilo_2019.empty
    else 0
)

print(
    "Hilos aprobados del lote 19:",
    len(ids_aprobados_lote_19),
)
print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_19),
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)
print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)
print(
    "Hilos de LGTBI–2019:",
    hilos_lgtbi_2019,
)
print(
    "Comentarios de LGTBI–2019:",
    comentarios_lgtbi_2019,
)
print(
    "Usuarios de LGTBI–2019:",
    usuarios_lgtbi_2019,
)
print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2019:.1f}%",
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2019,
        0,
    ),
)

Hilos aprobados del lote 19: 1
Comentarios anuales aprobados: 26

Hilos válidos totales: 69
Comentarios anuales totales: 7003
Hilos de LGTBI–2019: 4
Comentarios de LGTBI–2019: 511
Usuarios de LGTBI–2019: 385
Mayor aportación de un hilo: 53.4%
Hilos todavía necesarios: 8


In [187]:
# CELDA 101 — Incorporar candidatos del lote 20 para LGTBI–2019

import pandas as pd

candidatos_lote_20 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2019,
            "id_hilo": "7232702",
            "titulo_resultado": (
                "La NUEVA PELÍCULA de ISABEL COIXET "
                "+NWO +HOMOS +HETEROPATRIARCADO"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7232702"
            ),
            "subtema_preliminar": (
                "representacion_lesbica_cine"
            ),
            "criterio_inclusion": (
                "Debate sobre una película centrada en "
                "una relación lésbica, matrimonio homosexual "
                "y persecución internacional."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_20",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2019,
            "id_hilo": "7540396",
            "titulo_resultado": (
                "Si Vox se moderase y no fuera "
                "ultracatólico sería la fuerza de "
                "derecha más votada"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7540396"
            ),
            "subtema_preliminar": (
                "matrimonio_adopcion_politica"
            ),
            "criterio_inclusion": (
                "Discusión sobre matrimonio homosexual, "
                "adopción y posición política ante los "
                "derechos de gays y lesbianas."
            ),
            "requiere_revision_tematica": True,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_20",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2019,
            "id_hilo": "6916505",
            "titulo_resultado": (
                "La ciencia y sus avances en el "
                "conocimiento de la reencarnación"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=6916505"
            ),
            "subtema_preliminar": (
                "creencias_orientacion_identidad"
            ),
            "criterio_inclusion": (
                "Debate que relaciona homosexualidad, "
                "bisexualidad e identidad sexual con "
                "explicaciones sobre la reencarnación."
            ),
            "requiere_revision_tematica": True,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_20",
        },
    ]
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_20["id_hilo"] = (
    candidatos_lote_20["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_20 = (
    candidatos_lote_20[
        ~candidatos_lote_20[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_20,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_20 = (
    candidatos_nuevos_lote_20[
        ~candidatos_nuevos_lote_20[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 20:",
    len(candidatos_nuevos_lote_20),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 20:",
    len(candidatos_pendientes_lote_20),
)

display(
    candidatos_pendientes_lote_20[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 20: 3
Candidatos totales en el catálogo: 80
Candidatos pendientes del lote 20: 3


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,7232702,La NUEVA PELÍCULA de ISABEL COIXET +NWO +HOMOS +HETEROPATRIARCADO,representacion_lesbica_cine,False
1,7540396,Si Vox se moderase y no fuera ultracatólico sería la fuerza de derecha más votada,matrimonio_adopcion_politica,True
2,6916505,La ciencia y sus avances en el conocimiento de la reencarnación,creencias_orientacion_identidad,True


In [188]:
# CELDA 102 — Extraer los candidatos del lote 20

(
    comentarios_validos_lote_20,
    diagnosticos_lote_20,
    errores_lote_20,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_20,
    "lote_20",
    sesion,
)

comentarios_lote_20_df = unificar_dataframes(
    comentarios_validos_lote_20
)

diagnosticos_lote_20_df = unificar_dataframes(
    diagnosticos_lote_20
)

errores_lote_20_df = normalizar_registros(
    errores_lote_20
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_20_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_20_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_20_df),
)


[1/3] lgtbi | 2019 | hilo 7232702
La NUEVA PELÍCULA de ISABEL COIXET +NWO +HOMOS +HETEROPATRIARCADO
Primera página pública: 18 mensajes detectados
Hilo 7232702: 1 página(s)
  Página 1/1: 18 comentarios
Candidato técnicamente válido: 18 comentarios totales | 18 comentarios de 2019

[2/3] lgtbi | 2019 | hilo 7540396
Si Vox se moderase y no fuera ultracatólico sería la fuerza de derecha más votada
Primera página pública: 30 mensajes detectados
Hilo 7540396: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 15 comentarios
Candidato técnicamente válido: 45 comentarios totales | 45 comentarios de 2019

[3/3] lgtbi | 2019 | hilo 6916505
La ciencia y sus avances en el conocimiento de la reencarnación
Primera página pública: 30 mensajes detectados
Hilo 6916505: 4 página(s)
  Página 1/4: 30 comentarios
  Página 2/4: 30 comentarios
  Página 3/4: 29 comentarios
  Página 4/4: 29 comentarios
Candidato técnicamente válido: 89 comentarios totales | 32 comentarios de 2019

Hilos técnicamente váli

In [189]:
# CELDA 103 — Revisión temática de los hilos 7540396 y 6916505

import re
import pandas as pd

ids_revision_lote_20 = [
    "7540396",
    "6916505",
]

hilos_revision_lote_20 = (
    comentarios_lote_20_df[
        comentarios_lote_20_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_revision_lote_20)
    ]
    .copy()
    .reset_index(drop=True)
)

texto_revision = (
    hilos_revision_lote_20[
        "texto_sin_citas"
    ]
    .fillna("")
    .astype(str)
)

# ------------------------------------------------------------
# 1. Vocabulario directamente relacionado con LGTBI
# ------------------------------------------------------------

patron_lgtbi_revision = re.compile(
    r"\b(?:"
    r"lgtbi?\w*|"
    r"gay(?:s)?|"
    r"gais|"
    r"homosexual\w*|"
    r"homofob\w*|"
    r"lesbian\w*|"
    r"bisexual\w*|"
    r"transexual\w*|"
    r"transgener\w*|"
    r"transgéner\w*|"
    r"transfob\w*|"
    r"queer|"
    r"orientaci[oó]n\s+sexual|"
    r"identidad\s+sexual|"
    r"identidad\s+de\s+g[eé]nero|"
    r"matrimonio\s+(?:gay|homosexual)|"
    r"parejas?\s+del\s+mismo\s+sexo"
    r")\b",
    flags=re.IGNORECASE,
)

# ------------------------------------------------------------
# 2. Encuadre político o religioso
# ------------------------------------------------------------

patron_politica_religion = re.compile(
    r"\b(?:"
    r"vox|"
    r"abascal|"
    r"partido(?:s)?|"
    r"vot(?:o|ar|antes?)|"
    r"derecha|"
    r"izquierda|"
    r"pol[ií]tic\w*|"
    r"cat[oó]lic\w*|"
    r"iglesia|"
    r"religi[oó]n|"
    r"aborto|"
    r"eutanasia|"
    r"inmigraci[oó]n|"
    r"autonom[ií]\w*"
    r")\b",
    flags=re.IGNORECASE,
)

# ------------------------------------------------------------
# 3. Encuadre de reencarnación y espiritualidad
# ------------------------------------------------------------

patron_reencarnacion = re.compile(
    r"\b(?:"
    r"reencarna\w*|"
    r"encarna\w*|"
    r"vidas?\s+anteriores?|"
    r"karma|"
    r"esp[ií]ritu\w*|"
    r"alma(?:s)?|"
    r"espiritual\w*|"
    r"regresi[oó]n|"
    r"muerte|"
    r"nacer|"
    r"renacer|"
    r"ciencia|"
    r"cient[ií]fic\w*"
    r")\b",
    flags=re.IGNORECASE,
)

hilos_revision_lote_20[
    "menciona_lgtbi"
] = texto_revision.str.contains(
    patron_lgtbi_revision,
    na=False,
)

hilos_revision_lote_20[
    "menciona_politica_religion"
] = texto_revision.str.contains(
    patron_politica_religion,
    na=False,
)

hilos_revision_lote_20[
    "menciona_reencarnacion"
] = texto_revision.str.contains(
    patron_reencarnacion,
    na=False,
)

# ------------------------------------------------------------
# 4. Resumen por hilo
# ------------------------------------------------------------

resumen_revision_lote_20 = (
    hilos_revision_lote_20
    .groupby(
        "id_hilo",
        as_index=False,
    )
    .agg(
        comentarios_totales=(
            "id_mensaje",
            "count",
        ),
        mencionan_lgtbi=(
            "menciona_lgtbi",
            "sum",
        ),
        mencionan_politica_religion=(
            "menciona_politica_religion",
            "sum",
        ),
        mencionan_reencarnacion=(
            "menciona_reencarnacion",
            "sum",
        ),
    )
)

for columna in [
    "mencionan_lgtbi",
    "mencionan_politica_religion",
    "mencionan_reencarnacion",
]:
    resumen_revision_lote_20[
        f"{columna}_pct"
    ] = (
        resumen_revision_lote_20[columna]
        / resumen_revision_lote_20[
            "comentarios_totales"
        ]
        * 100
    ).round(1)

display(resumen_revision_lote_20)

# ------------------------------------------------------------
# 5. Mostrar mensajes LGTBI de cada hilo
# ------------------------------------------------------------

for id_hilo in ids_revision_lote_20:
    revision_hilo = (
        hilos_revision_lote_20[
            hilos_revision_lote_20[
                "id_hilo"
            ]
            .astype(str)
            .eq(id_hilo)
        ]
        .copy()
    )

    mensajes_lgtbi = (
        revision_hilo[
            revision_hilo[
                "menciona_lgtbi"
            ]
        ][
            [
                "numero_mensaje",
                "menciona_lgtbi",
                "menciona_politica_religion",
                "menciona_reencarnacion",
                "texto_sin_citas",
            ]
        ]
        .reset_index(drop=True)
    )

    print("\n" + "=" * 70)
    print("Hilo:", id_hilo)
    print(
        "Mensajes relacionados directamente "
        "con LGTBI:",
        len(mensajes_lgtbi),
    )

    display(mensajes_lgtbi.head(25))

    print(
        "Primeros 12 mensajes del hilo:"
    )

    display(
        revision_hilo[
            [
                "numero_mensaje",
                "texto_sin_citas",
            ]
        ].head(12)
    )

,id_hilo,comentarios_totales,mencionan_lgtbi,mencionan_politica_religion,mencionan_reencarnacion,mencionan_lgtbi_pct,mencionan_politica_religion_pct,mencionan_reencarnacion_pct
0,6916505,89,3,5,46,3.4,5.6,51.7
1,7540396,45,3,21,1,6.7,46.7,2.2



Hilo: 7540396
Mensajes relacionados directamente con LGTBI: 3


,numero_mensaje,menciona_lgtbi,menciona_politica_religion,menciona_reencarnacion,texto_sin_citas
0,1,True,True,False,"Y sí señores, eso es así, si Vox quitara de su partido el vivir según las normas de un seguimiento extremista a la religión católica sería el partido más votado de la ""derecha"" Española. En pleno siglo XXI no se puede ir con un discurso contra la eutanasia o el aborto, y mucho menos desprestigia..."
1,29,True,True,False,"Básicamente es lo que todos conocemos como Católico Conservador, viviendo la fé como algo demasiado extremo, adorando a dios de cara a ver la homosexualidad como una enfermedad mental, el aborto o la eutanasia como algo antinatural.\nOjo que todas las opiniones y creencias son respetables, pero ..."
2,36,True,False,False,No soy cristiano pero me parece razonable que los padres adoptivos no sean homosexuales


Primeros 12 mensajes del hilo:


,numero_mensaje,texto_sin_citas
0,1,"Y sí señores, eso es así, si Vox quitara de su partido el vivir según las normas de un seguimiento extremista a la religión católica sería el partido más votado de la ""derecha"" Española. En pleno siglo XXI no se puede ir con un discurso contra la eutanasia o el aborto, y mucho menos desprestigia..."
1,2,Un partido a tu gusto vaya.
2,3,Has descubierto la pólvora
3,4,No sabía que que vivíamos en el 1668
4,5,"¿Qué es ser ultracatólico? Explícate, por favor."
5,6,"Pues estoy de acuerdo, quitando el rollo cristiano, los toros y la caza, perfect!"
6,7,"Tienes razón. Entre los mayores de 70 años, claro"
7,8,Yo no creo que sea ultracatólico.\nYo no me considero católico y me da igual que Abascal vaya a misa de vez en cuando.\nEn qué te basas para decir que es ULTRAcatólico?
8,9,A ver si alguien me indica en que punto de VOX dice que la Iglesia va a entrar en el Gobierno.. por favor.
9,10,Si Vox se moderase sería el PP



Hilo: 6916505
Mensajes relacionados directamente con LGTBI: 3


,numero_mensaje,menciona_lgtbi,menciona_politica_religion,menciona_reencarnacion,texto_sin_citas
0,1,True,False,True,"Estudio científico donde se comprueba que las personas que han experimentado en su vida anterior el sexo contrario al actual, tienen una notable mayor posibilidad de no aceptar su genero actual.\nMarieta Pehlivanova, Monica J. Janke, Jack Lee & Jim B. Tucker (2018): Childhood Gender Nonconformit..."
1,46,True,True,True,"No se que tiene que ver la religión con el mecanismo de la existencia, que yo sepa las religiones dan muy diversas y variopintas versiones sobre la cuestion...no son la cuestión, la existencia y sus mecanismos. No se si percibes la diferencia.\nTu lo llamaras casualidad supongo, otros lo llamamo..."
2,87,True,False,True,"Que las fantasias no se puede demostrar nunca porque son fantasias, y estos recuerdos en multiples ocasiones si, luego no son fantasias, tu te puedes quedar en ese bucle eterno si quieres, mientras no admitas los hechos, que se ha comprobado que es verdad\nQuizás no lo entiendas, pero basta con ..."


Primeros 12 mensajes del hilo:


,numero_mensaje,texto_sin_citas
45,1,"Estudio científico donde se comprueba que las personas que han experimentado en su vida anterior el sexo contrario al actual, tienen una notable mayor posibilidad de no aceptar su genero actual.\nMarieta Pehlivanova, Monica J. Janke, Jack Lee & Jim B. Tucker (2018): Childhood Gender Nonconformit..."
46,2,"El concepto de reencarnación es absurdo, entre otras cosas porque la población mundial va en aumento\nEdit 2021: son reflotes sanos"
47,3,Resumen.\nLa gente que muestra defectos cognitivos asegurando haber vivido otras vidas es mas dada también a sufrir disforia de genero como una secuela mas de la incapacidad física para razonar.\nNada nuevo.
48,4,"O sea, que la peña con problemillas en el cerebro tiene problemillas en el cerebro."
49,5,"Pero recuerda que nuestra galaxia es un pueblo de 400 habitantes en España, y la talla del universo es 3 veces el del planeta tierra\nHay otros mundos más evolucionado que el planeta tierra donde podrias reencarnar, pero esto será según tu grado de evolución. Y tambien hay otros mundos inferiore..."
50,6,Pillo gorro de plata
51,7,"Hay varias formas de que salgan las cuentas\nEn primer lugar la creación de nuevas almas es continua, todo evoluciona desde el mineral pasando por vegetal, animal y ser humano, hasta la perfección. De animales ya muy evolucionados por tanto, se pasa al nivel de espíritu individual consciente y c..."
52,8,Pilla retraso sobre todo. Pero un dia cómo todos los otros lo descubriras.
53,9,¿Puedes explicar como un defecto cognitivo en un niño le permite averiguar cosas imposibles de saber sobre la vida familiar de familias de hace x años en la otra punta del mundo?
54,10,


In [190]:
# CELDA 104 — Consolidar el hilo válido y registrar dos exclusiones

import pandas as pd

ids_aprobados_lote_20 = {
    "7232702",
}

ids_excluidos_lote_20 = {
    "7540396",
    "6916505",
}

# ------------------------------------------------------------
# 1. Comentarios aprobados
# ------------------------------------------------------------

comentarios_aprobados_lote_20 = (
    comentarios_lote_20_df[
        comentarios_lote_20_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_aprobados_lote_20)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_lote_20 = (
    comentarios_aprobados_lote_20[
        pd.to_numeric(
            comentarios_aprobados_lote_20[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lote_20[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Consolidar el hilo aprobado
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_20,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_20,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Añadir solo el diagnóstico del hilo aprobado
if not diagnosticos_lote_20_df.empty:
    if (
        "id_hilo"
        in diagnosticos_lote_20_df.columns
    ):
        diagnosticos_aprobados_lote_20 = (
            diagnosticos_lote_20_df[
                diagnosticos_lote_20_df[
                    "id_hilo"
                ]
                .astype(str)
                .isin(ids_aprobados_lote_20)
            ]
            .copy()
        )
    else:
        diagnosticos_aprobados_lote_20 = (
            diagnosticos_lote_20_df.copy()
        )

    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_aprobados_lote_20,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Registrar las exclusiones temáticas
# ------------------------------------------------------------

exclusiones_tematicas_lote_20 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2019,
            "id_hilo": "7540396",
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=7540396"
            ),
            "tipo_error": "exclusion_tematica",
            "mensaje_error": (
                "El debate se centra en Vox, religión "
                "y estrategia política. Solo 3 de 45 "
                "mensajes mencionan directamente LGTBI."
            ),
            "lote": "lote_20",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2019,
            "id_hilo": "6916505",
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=6916505"
            ),
            "tipo_error": "exclusion_tematica",
            "mensaje_error": (
                "El debate se centra en reencarnación "
                "y espiritualidad. Solo 3 de 89 mensajes "
                "mencionan directamente LGTBI."
            ),
            "lote": "lote_20",
        },
    ]
)

incidencias_forocoches_ampliacion = (
    pd.concat(
        [
            incidencias_forocoches_ampliacion,
            errores_lote_20_df,
            exclusiones_tematicas_lote_20,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "tipo_error",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Estado actualizado de LGTBI–2019
# ------------------------------------------------------------

estrato_lgtbi_2019 = (
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            pd.to_numeric(
                comentarios_forocoches_anual_ampliado[
                    "anio_objetivo"
                ],
                errors="coerce",
            ).eq(2019)
        )
    ]
    .copy()
)

hilos_lgtbi_2019 = int(
    estrato_lgtbi_2019[
        "id_hilo"
    ].astype(str).nunique()
)

comentarios_lgtbi_2019 = len(
    estrato_lgtbi_2019
)

usuarios_lgtbi_2019 = int(
    estrato_lgtbi_2019[
        "usuario_hash"
    ].nunique()
)

comentarios_por_hilo_2019 = (
    estrato_lgtbi_2019[
        "id_hilo"
    ]
    .astype(str)
    .value_counts()
)

mayor_hilo_pct_2019 = (
    comentarios_por_hilo_2019.iloc[0]
    / max(comentarios_lgtbi_2019, 1)
    * 100
)

print(
    "Hilos aprobados del lote 20:",
    len(ids_aprobados_lote_20),
)
print(
    "Hilos excluidos temáticamente:",
    len(ids_excluidos_lote_20),
)
print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_20),
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)
print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)
print(
    "Hilos de LGTBI–2019:",
    hilos_lgtbi_2019,
)
print(
    "Comentarios de LGTBI–2019:",
    comentarios_lgtbi_2019,
)
print(
    "Usuarios de LGTBI–2019:",
    usuarios_lgtbi_2019,
)
print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2019:.1f}%",
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2019,
        0,
    ),
)

Hilos aprobados del lote 20: 1
Hilos excluidos temáticamente: 2
Comentarios anuales aprobados: 18

Hilos válidos totales: 70
Comentarios anuales totales: 7021
Hilos de LGTBI–2019: 5
Comentarios de LGTBI–2019: 529
Usuarios de LGTBI–2019: 397
Mayor aportación de un hilo: 51.6%
Hilos todavía necesarios: 7


## 2023

In [191]:
# CELDA 105 — Incorporar candidatos del lote 21 para LGTBI–2023

import pandas as pd

candidatos_lote_21 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9415357",
            "titulo_resultado": (
                "El niño que fue criado como una niña "
                "- Caso de David Reimer"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9415357"
            ),
            "subtema_preliminar": (
                "identidad_sexual_david_reimer"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre identidad sexual, "
                "transexualidad, intersexualidad y género."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_21",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9547665",
            "titulo_resultado": (
                "Jorgeja y el tucán lo confirman: "
                "si eres hombre no votes a la izquierda"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9547665"
            ),
            "subtema_preliminar": (
                "homosexualidad_voto_matrimonio"
            ),
            "criterio_inclusion": (
                "Debate sobre orientación sexual, voto, "
                "matrimonio homosexual y derechos civiles."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_21",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9644836",
            "titulo_resultado": (
                "Un preso se autodetermina mujer y "
                "embaraza a otra interna tras ser "
                "enviado al módulo femenino"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9644836"
            ),
            "subtema_preliminar": (
                "identidad_trans_sistema_penitenciario"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre identidad trans, "
                "cambio registral y ubicación penitenciaria."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_21",
        },
    ]
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_21["id_hilo"] = (
    candidatos_lote_21["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_21 = (
    candidatos_lote_21[
        ~candidatos_lote_21[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_21,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_21 = (
    candidatos_nuevos_lote_21[
        ~candidatos_nuevos_lote_21[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 21:",
    len(candidatos_nuevos_lote_21),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 21:",
    len(candidatos_pendientes_lote_21),
)

display(
    candidatos_pendientes_lote_21[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 21: 3
Candidatos totales en el catálogo: 83
Candidatos pendientes del lote 21: 3


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,9415357,El niño que fue criado como una niña - Caso de David Reimer,identidad_sexual_david_reimer,False
1,9547665,Jorgeja y el tucán lo confirman: si eres hombre no votes a la izquierda,homosexualidad_voto_matrimonio,False
2,9644836,Un preso se autodetermina mujer y embaraza a otra interna tras ser enviado al módulo femenino,identidad_trans_sistema_penitenciario,False


In [192]:
# CELDA 106 — Extraer los candidatos del lote 21

(
    comentarios_validos_lote_21,
    diagnosticos_lote_21,
    errores_lote_21,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_21,
    "lote_21",
    sesion,
)

comentarios_lote_21_df = unificar_dataframes(
    comentarios_validos_lote_21
)

diagnosticos_lote_21_df = unificar_dataframes(
    diagnosticos_lote_21
)

errores_lote_21_df = normalizar_registros(
    errores_lote_21
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_21_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_21_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_21_df),
)


[1/3] lgtbi | 2023 | hilo 9415357
El niño que fue criado como una niña - Caso de David Reimer
Primera página pública: 30 mensajes detectados
Hilo 9415357: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 8 comentarios
Candidato técnicamente válido: 38 comentarios totales | 38 comentarios de 2023

[2/3] lgtbi | 2023 | hilo 9547665
Jorgeja y el tucán lo confirman: si eres hombre no votes a la izquierda
Primera página pública: 28 mensajes detectados
Hilo 9547665: 2 página(s)
  Página 1/2: 28 comentarios
  Página 2/2: 28 comentarios
Candidato técnicamente válido: 28 comentarios totales | 28 comentarios de 2023

[3/3] lgtbi | 2023 | hilo 9644836
Un preso se autodetermina mujer y embaraza a otra interna tras ser enviado al módulo femenino
Primera página pública: 16 mensajes detectados
Hilo 9644836: 1 página(s)
  Página 1/1: 16 comentarios
Candidato técnicamente válido: 16 comentarios totales | 16 comentarios de 2023

Hilos técnicamente válidos: 3
Hilos descartados o con error: 0

Hilo

In [193]:
# CELDA 107 — Aprobar y consolidar el lote 21

import pandas as pd

ids_aprobados_lote_21 = {
    "9415357",
    "9547665",
    "9644836",
}

comentarios_aprobados_lote_21 = (
    comentarios_lote_21_df[
        comentarios_lote_21_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_aprobados_lote_21)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_lote_21 = (
    comentarios_aprobados_lote_21[
        pd.to_numeric(
            comentarios_aprobados_lote_21[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lote_21[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Consolidar comentarios brutos
comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_21,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Consolidar comentarios de 2023
comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_21,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Consolidar diagnósticos
if not diagnosticos_lote_21_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_21_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# Consolidar posibles incidencias
if not errores_lote_21_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_21_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# Estado actualizado de LGTBI–2023
# ------------------------------------------------------------

estrato_lgtbi_2023 = (
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            pd.to_numeric(
                comentarios_forocoches_anual_ampliado[
                    "anio_objetivo"
                ],
                errors="coerce",
            ).eq(2023)
        )
    ]
    .copy()
)

hilos_lgtbi_2023 = int(
    estrato_lgtbi_2023[
        "id_hilo"
    ].astype(str).nunique()
)

comentarios_lgtbi_2023 = len(
    estrato_lgtbi_2023
)

usuarios_lgtbi_2023 = int(
    estrato_lgtbi_2023[
        "usuario_hash"
    ].nunique()
)

comentarios_por_hilo_2023 = (
    estrato_lgtbi_2023[
        "id_hilo"
    ]
    .astype(str)
    .value_counts()
)

mayor_hilo_pct_2023 = (
    comentarios_por_hilo_2023.iloc[0]
    / max(comentarios_lgtbi_2023, 1)
    * 100
    if not comentarios_por_hilo_2023.empty
    else 0
)

print(
    "Hilos aprobados del lote 21:",
    len(ids_aprobados_lote_21),
)
print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_21),
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)
print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)
print(
    "Hilos de LGTBI–2023:",
    hilos_lgtbi_2023,
)
print(
    "Comentarios de LGTBI–2023:",
    comentarios_lgtbi_2023,
)
print(
    "Usuarios de LGTBI–2023:",
    usuarios_lgtbi_2023,
)
print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2023:.1f}%",
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2023,
        0,
    ),
)

Hilos aprobados del lote 21: 3
Comentarios anuales aprobados: 82

Hilos válidos totales: 73
Comentarios anuales totales: 7103
Hilos de LGTBI–2023: 6
Comentarios de LGTBI–2023: 274
Usuarios de LGTBI–2023: 241
Mayor aportación de un hilo: 33.2%
Hilos todavía necesarios: 6


In [194]:
# CELDA 108 — Incorporar candidatos del lote 22 para LGTBI–2023

import pandas as pd

candidatos_lote_22 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9810719",
            "titulo_resultado": (
                "¿Es transfobia que un hetero "
                "prefiera a una mujer cis?"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9810719"
            ),
            "subtema_preliminar": (
                "transfobia_preferencias_sexuales"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre mujeres trans, "
                "preferencias sexuales, orientación "
                "sexual y transfobia."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_22",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9739698",
            "titulo_resultado": (
                "La Sanidad Canaria condenada a pagar "
                "20.737,10 euros a una persona trans"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9739698"
            ),
            "subtema_preliminar": (
                "sanidad_trans_faloplastia"
            ),
            "criterio_inclusion": (
                "Debate sobre atención sanitaria trans, "
                "faloplastia y financiación pública."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_22",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9755282",
            "titulo_resultado": (
                "Cagada de Ayuso modificando "
                "las leyes Trans"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9755282"
            ),
            "subtema_preliminar": (
                "legislacion_trans_madrid"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre la modificación "
                "de las leyes trans de la Comunidad "
                "de Madrid."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_22",
        },
    ]
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
)

candidatos_lote_22["id_hilo"] = (
    candidatos_lote_22["id_hilo"]
    .astype(str)
)

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
)

candidatos_nuevos_lote_22 = (
    candidatos_lote_22[
        ~candidatos_lote_22[
            "id_hilo"
        ].isin(ids_catalogados)
    ]
    .copy()
    .reset_index(drop=True)
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_22,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

ids_hilos_validos = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str)
)

candidatos_pendientes_lote_22 = (
    candidatos_nuevos_lote_22[
        ~candidatos_nuevos_lote_22[
            "id_hilo"
        ].isin(ids_hilos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidatos incorporados en el lote 22:",
    len(candidatos_nuevos_lote_22),
)
print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)
print(
    "Candidatos pendientes del lote 22:",
    len(candidatos_pendientes_lote_22),
)

display(
    candidatos_pendientes_lote_22[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 22: 3
Candidatos totales en el catálogo: 86
Candidatos pendientes del lote 22: 3


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,9810719,¿Es transfobia que un hetero prefiera a una mujer cis?,transfobia_preferencias_sexuales,False
1,9739698,"La Sanidad Canaria condenada a pagar 20.737,10 euros a una persona trans",sanidad_trans_faloplastia,False
2,9755282,Cagada de Ayuso modificando las leyes Trans,legislacion_trans_madrid,False


In [195]:
# CELDA 109 — Extraer los candidatos del lote 22

(
    comentarios_validos_lote_22,
    diagnosticos_lote_22,
    errores_lote_22,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_22,
    "lote_22",
    sesion,
)

comentarios_lote_22_df = unificar_dataframes(
    comentarios_validos_lote_22
)

diagnosticos_lote_22_df = unificar_dataframes(
    diagnosticos_lote_22
)

errores_lote_22_df = normalizar_registros(
    errores_lote_22
)

print("\n" + "=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_22_df[
            "id_hilo"
        ]
        .astype(str)
        .nunique()
        if not comentarios_lote_22_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_22_df),
)


[1/3] lgtbi | 2023 | hilo 9810719
¿Es transfobia que un hetero prefiera a una mujer cis?
Primera página pública: 30 mensajes detectados
Hilo 9810719: 6 página(s)
  Página 1/6: 30 comentarios
  Página 2/6: 30 comentarios
  Página 3/6: 30 comentarios
  Página 4/6: 30 comentarios
  Página 5/6: 30 comentarios
  Página 6/6: 5 comentarios
Candidato técnicamente válido: 155 comentarios totales | 155 comentarios de 2023

[2/3] lgtbi | 2023 | hilo 9739698
La Sanidad Canaria condenada a pagar 20.737,10 euros a una persona trans
Primera página pública: 16 mensajes detectados
Hilo 9739698: 1 página(s)
  Página 1/1: 16 comentarios
Candidato técnicamente válido: 16 comentarios totales | 16 comentarios de 2023

[3/3] lgtbi | 2023 | hilo 9755282
Cagada de Ayuso modificando las leyes Trans
Primera página pública: 30 mensajes detectados
Hilo 9755282: 6 página(s)
  Página 1/6: 30 comentarios
  Página 2/6: 30 comentarios
  Página 3/6: 30 comentarios
  Página 4/6: 30 comentarios
  Página 5/6: 28 comentari

In [196]:
# CELDA 110 — Aprobar y consolidar el lote 22

import pandas as pd

ids_aprobados_lote_22 = {
    "9810719",
    "9739698",
    "9755282",
}

comentarios_aprobados_lote_22 = (
    comentarios_lote_22_df[
        comentarios_lote_22_df[
            "id_hilo"
        ]
        .astype(str)
        .isin(ids_aprobados_lote_22)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_lote_22 = (
    comentarios_aprobados_lote_22[
        pd.to_numeric(
            comentarios_aprobados_lote_22[
                "anio_comentario"
            ],
            errors="coerce",
        ).eq(
            pd.to_numeric(
                comentarios_aprobados_lote_22[
                    "anio_objetivo"
                ],
                errors="coerce",
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Consolidar comentarios brutos
comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_22,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Consolidar comentarios del año objetivo
comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_anuales_lote_22,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=[
            "id_hilo",
            "id_mensaje",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

# Consolidar diagnósticos
if not diagnosticos_lote_22_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_22_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# Consolidar posibles incidencias
if not errores_lote_22_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_22_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# Estado actualizado de LGTBI–2023
# ------------------------------------------------------------

estrato_lgtbi_2023 = (
    comentarios_forocoches_anual_ampliado[
        (
            comentarios_forocoches_anual_ampliado[
                "tema"
            ].eq("lgtbi")
        )
        & (
            pd.to_numeric(
                comentarios_forocoches_anual_ampliado[
                    "anio_objetivo"
                ],
                errors="coerce",
            ).eq(2023)
        )
    ]
    .copy()
)

hilos_lgtbi_2023 = int(
    estrato_lgtbi_2023[
        "id_hilo"
    ].astype(str).nunique()
)

comentarios_lgtbi_2023 = len(
    estrato_lgtbi_2023
)

usuarios_lgtbi_2023 = int(
    estrato_lgtbi_2023[
        "usuario_hash"
    ].nunique()
)

comentarios_por_hilo_2023 = (
    estrato_lgtbi_2023[
        "id_hilo"
    ]
    .astype(str)
    .value_counts()
)

proporciones_hilos_2023 = (
    comentarios_por_hilo_2023
    / max(comentarios_lgtbi_2023, 1)
)

mayor_hilo_pct_2023 = (
    proporciones_hilos_2023.iloc[0]
    * 100
    if not proporciones_hilos_2023.empty
    else 0
)

hhi_2023 = float(
    (proporciones_hilos_2023 ** 2).sum()
)

hilos_efectivos_2023 = (
    1 / hhi_2023
    if hhi_2023 > 0
    else 0
)

estrato_suficiente_2023 = (
    hilos_lgtbi_2023 >= 12
    and comentarios_lgtbi_2023 >= 500
    and usuarios_lgtbi_2023 >= 300
)

print(
    "Hilos aprobados del lote 22:",
    len(ids_aprobados_lote_22),
)
print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_22),
)

print("\n" + "=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].astype(str).nunique(),
)
print(
    "Comentarios anuales totales:",
    len(
        comentarios_forocoches_anual_ampliado
    ),
)
print(
    "Hilos de LGTBI–2023:",
    hilos_lgtbi_2023,
)
print(
    "Comentarios de LGTBI–2023:",
    comentarios_lgtbi_2023,
)
print(
    "Usuarios de LGTBI–2023:",
    usuarios_lgtbi_2023,
)
print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_2023:.1f}%",
)
print(
    "Índice HHI:",
    f"{hhi_2023:.3f}",
)
print(
    "Número efectivo de hilos:",
    f"{hilos_efectivos_2023:.1f}",
)
print(
    "Estrato suficiente:",
    estrato_suficiente_2023,
)
print(
    "Hilos todavía necesarios:",
    max(
        12 - hilos_lgtbi_2023,
        0,
    ),
)

Hilos aprobados del lote 22: 3
Comentarios anuales aprobados: 319

Hilos válidos totales: 76
Comentarios anuales totales: 7422
Hilos de LGTBI–2023: 9
Comentarios de LGTBI–2023: 593
Usuarios de LGTBI–2023: 488
Mayor aportación de un hilo: 26.1%
Índice HHI: 0.181
Número efectivo de hilos: 5.5
Estrato suficiente: False
Hilos todavía necesarios: 3


In [197]:
# CELDA 111 — Crear el lote 23 para completar LGTBI–2023

import pandas as pd

candidatos_lote_23 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9730181",
            "titulo_resultado": (
                "Si te molesta el lenguaje inclusivo de "
                "Spider-Man 2... (Hobby Consolas)"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9730181"
            ),
            "subtema_preliminar": (
                "lenguaje_inclusivo_identidad_no_binaria"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre personas no binarias, "
                "pronombres, representación homosexual e "
                "identidad de género."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_23",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9424474",
            "titulo_resultado": (
                "Perder la virginidad sin perder "
                "un solo estereotipo"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9424474"
            ),
            "subtema_preliminar": (
                "diversidad_sexual_experiencias_intimas"
            ),
            "criterio_inclusion": (
                "El contenido inicial aborda experiencias "
                "homosexuales, bisexuales, pansexuales y "
                "lesbianas relacionadas con la sexualidad."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_23",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2023,
            "id_hilo": "9488478",
            "titulo_resultado": (
                "6 presos asturianos se hacen mujer "
                "y cambian de módulo"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=9488478"
            ),
            "subtema_preliminar": (
                "ley_trans_prisiones_autodeterminacion"
            ),
            "criterio_inclusion": (
                "Debate central sobre autodeterminación de "
                "género, ley trans y asignación de módulo "
                "penitenciario."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_23",
        },
    ]
)

# ------------------------------------------------------------
# 1. Normalizar identificadores
# ------------------------------------------------------------

candidatos_lote_23["id_hilo"] = (
    candidatos_lote_23["id_hilo"]
    .astype(str)
    .str.strip()
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 2. Evitar duplicados en el catálogo
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .dropna()
    .astype(str)
)

candidatos_nuevos_lote_23 = (
    candidatos_lote_23[
        ~candidatos_lote_23["id_hilo"].isin(
            ids_catalogados
        )
    ]
    .copy()
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_23,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Excluir hilos que ya estén consolidados
# ------------------------------------------------------------

ids_consolidados = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ]
    .dropna()
    .astype(str)
)

candidatos_pendientes_lote_23 = (
    candidatos_lote_23[
        ~candidatos_lote_23["id_hilo"].isin(
            ids_consolidados
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobar el lote
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 23:",
    len(candidatos_nuevos_lote_23),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 23:",
    len(candidatos_pendientes_lote_23),
)

display(
    candidatos_pendientes_lote_23[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 23: 3
Candidatos totales en el catálogo: 89
Candidatos pendientes del lote 23: 3


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,9730181,Si te molesta el lenguaje inclusivo de Spider-Man 2... (Hobby Consolas),lenguaje_inclusivo_identidad_no_binaria,False
1,9424474,Perder la virginidad sin perder un solo estereotipo,diversidad_sexual_experiencias_intimas,False
2,9488478,6 presos asturianos se hacen mujer y cambian de módulo,ley_trans_prisiones_autodeterminacion,False


In [198]:
# CELDA 112 — Extraer los candidatos del lote 23

(
    comentarios_validos_lote_23,
    diagnosticos_lote_23,
    errores_lote_23,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_23,
    "lote_23",
    sesion,
)

# ------------------------------------------------------------
# Unificar los resultados de la extracción
# ------------------------------------------------------------

comentarios_lote_23_df = unificar_dataframes(
    comentarios_validos_lote_23
)

diagnosticos_lote_23_df = unificar_dataframes(
    diagnosticos_lote_23
)

errores_lote_23_df = unificar_dataframes(
    errores_lote_23
)

# ------------------------------------------------------------
# Resumen técnico
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_23_df["id_hilo"].nunique()
        if not comentarios_lote_23_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_23_df),
)

# ------------------------------------------------------------
# Mostrar posibles incidencias
# ------------------------------------------------------------

if not errores_lote_23_df.empty:
    display(errores_lote_23_df)


[1/3] lgtbi | 2023 | hilo 9730181
Si te molesta el lenguaje inclusivo de Spider-Man 2... (Hobby Consolas)
Primera página pública: 30 mensajes detectados
Hilo 9730181: 6 página(s)
  Página 1/6: 30 comentarios
  Página 2/6: 30 comentarios
  Página 3/6: 30 comentarios
  Página 4/6: 30 comentarios
  Página 5/6: 30 comentarios
  Página 6/6: 10 comentarios
Candidato técnicamente válido: 160 comentarios totales | 160 comentarios de 2023

[2/3] lgtbi | 2023 | hilo 9424474
Perder la virginidad sin perder un solo estereotipo
Primera página pública: 5 mensajes detectados
Hilo 9424474: 1 página(s)
  Página 1/1: 5 comentarios
Candidato técnicamente válido: 5 comentarios totales | 5 comentarios de 2023

[3/3] lgtbi | 2023 | hilo 9488478
6 presos asturianos se hacen mujer y cambian de módulo
Primera página pública: 30 mensajes detectados
Hilo 9488478: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 11 comentarios
Candidato técnicamente válido: 41 comentarios totales | 41 comentarios de 2023



In [199]:
# CELDA 113 — Revisar la pertinencia temática del lote 23

import re
import pandas as pd

revision_lote_23 = (
    comentarios_lote_23_df
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 1. Seleccionar el texto disponible
# ------------------------------------------------------------

if "texto_sin_citas" in revision_lote_23.columns:
    texto_revision = (
        revision_lote_23["texto_sin_citas"]
        .fillna("")
        .astype(str)
    )
else:
    texto_revision = (
        revision_lote_23["texto_original"]
        .fillna("")
        .astype(str)
    )

# ------------------------------------------------------------
# 2. Patrones temáticos
# ------------------------------------------------------------

patron_lgtbi = re.compile(
    r"\b(?:"
    r"lgtbi\w*|lgbt\w*|"
    r"gay|gais|lesbiana\w*|"
    r"homosexual\w*|bisexual\w*|pansexual\w*|"
    r"trans|transexual\w*|transgénero\w*|"
    r"no\s+binari[oa]s?|"
    r"género\w*|identidad\s+de\s+género|"
    r"pronombres?|lenguaje\s+inclusivo|"
    r"homofobi\w*|transfobi\w*|"
    r"orientación\s+sexual|"
    r"orgullo"
    r")\b",
    flags=re.IGNORECASE,
)

patron_virginidad_sexualidad = re.compile(
    r"\b(?:"
    r"virginidad|virgen|sexo|sexualidad|"
    r"primera\s+vez|relaciones?\s+sexuales?|"
    r"estereotipos?"
    r")\b",
    flags=re.IGNORECASE,
)

patron_ley_trans_prisiones = re.compile(
    r"\b(?:"
    r"ley\s+trans|autodeterminación|"
    r"cambio\s+de\s+sexo|cambio\s+de\s+género|"
    r"presos?|presas?|prisión|cárcel|"
    r"módulo\s+femenino|módulo\s+de\s+mujeres"
    r")\b",
    flags=re.IGNORECASE,
)

# ------------------------------------------------------------
# 3. Crear indicadores por comentario
# ------------------------------------------------------------

revision_lote_23["menciona_lgtbi"] = (
    texto_revision.str.contains(
        patron_lgtbi,
        na=False,
    )
)

revision_lote_23[
    "menciona_virginidad_sexualidad"
] = texto_revision.str.contains(
    patron_virginidad_sexualidad,
    na=False,
)

revision_lote_23[
    "menciona_ley_trans_prisiones"
] = texto_revision.str.contains(
    patron_ley_trans_prisiones,
    na=False,
)

# ------------------------------------------------------------
# 4. Resumen por hilo
# ------------------------------------------------------------

resumen_revision_lote_23 = (
    revision_lote_23
    .groupby("id_hilo", as_index=False)
    .agg(
        comentarios_totales=(
            "id_hilo",
            "size",
        ),
        mencionan_lgtbi=(
            "menciona_lgtbi",
            "sum",
        ),
        mencionan_virginidad_sexualidad=(
            "menciona_virginidad_sexualidad",
            "sum",
        ),
        mencionan_ley_trans_prisiones=(
            "menciona_ley_trans_prisiones",
            "sum",
        ),
    )
)

for columna in [
    "mencionan_lgtbi",
    "mencionan_virginidad_sexualidad",
    "mencionan_ley_trans_prisiones",
]:
    resumen_revision_lote_23[
        f"{columna}_pct"
    ] = (
        resumen_revision_lote_23[columna]
        / resumen_revision_lote_23[
            "comentarios_totales"
        ]
        * 100
    ).round(1)

display(resumen_revision_lote_23)

# ------------------------------------------------------------
# 5. Mostrar todos los mensajes del hilo pequeño
# ------------------------------------------------------------

hilo_pequeno_9424474 = (
    revision_lote_23[
        revision_lote_23["id_hilo"]
        .astype(str)
        .eq("9424474")
    ]
    [
        [
            "numero_mensaje",
            "menciona_lgtbi",
            "menciona_virginidad_sexualidad",
            "texto_sin_citas",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print()
print(
    "Mensajes recuperados del hilo 9424474:",
    len(hilo_pequeno_9424474),
)

display(hilo_pequeno_9424474)

# ------------------------------------------------------------
# 6. Muestra de mensajes LGTBI de los otros dos hilos
# ------------------------------------------------------------

muestra_lgtbi_lote_23 = (
    revision_lote_23[
        revision_lote_23["menciona_lgtbi"]
        & revision_lote_23["id_hilo"]
        .astype(str)
        .isin(["9730181", "9488478"])
    ]
    [
        [
            "id_hilo",
            "numero_mensaje",
            "menciona_lgtbi",
            "menciona_ley_trans_prisiones",
            "texto_sin_citas",
        ]
    ]
    .groupby(
        "id_hilo",
        group_keys=False,
    )
    .head(10)
    .reset_index(drop=True)
)

print()
print(
    "Muestra de mensajes temáticos:",
    len(muestra_lgtbi_lote_23),
)

display(muestra_lgtbi_lote_23)

,id_hilo,comentarios_totales,mencionan_lgtbi,mencionan_virginidad_sexualidad,mencionan_ley_trans_prisiones,mencionan_lgtbi_pct,mencionan_virginidad_sexualidad_pct,mencionan_ley_trans_prisiones_pct
0,9424474,5,1,1,0,20.0,20.0,0.0
1,9488478,41,3,1,7,7.3,2.4,17.1
2,9730181,160,25,4,0,15.6,2.5,0.0



Mensajes recuperados del hilo 9424474: 5


,numero_mensaje,menciona_lgtbi,menciona_virginidad_sexualidad,texto_sin_citas
0,1,True,True,"Vamos que no quede un solo estereotipo sin describir...Qué coñazo!\nhttps://www.elconfidencial.com/alma-...ion-z_3563296/\nNo se conocen\nni entre ellos ni entre ellas\n. Y, sí, aquí el desdoblamiento de género es necesario. Primero, siete desconocidos se sentaron unos enfrente de otros para hab..."
1,2,False,False,Ni con tus ojos me leo ese tocho mugremita
2,3,False,False,up
3,4,False,False,"Muy interesante shur, cuál fue tu experiencia? Te sientes identificado con algo del texto?"
4,5,False,False,"Gracioso lo de ""desmitificar"".\nY los 13 personajes, también."



Muestra de mensajes temáticos: 13


,id_hilo,numero_mensaje,menciona_lgtbi,menciona_ley_trans_prisiones,texto_sin_citas
0,9730181,1,True,False,"los videojuegos no son para ti\nhttps://www.hobbyconsolas.com/opinio...son-ti-1324182\nMe ha llamado la atención esta parte del artículo.\nPara todos aquellos que se han sentido molestos o han criticado estas decisiones, tengo una mala noticia:\nlas personas no binarias existen, las personas gay..."
1,9730181,16,True,False,"bueno, para ser justos en Spider-man tampoco te alecciona nadie a que le digas elle a nadie, y mira que a mí no me gusta el lenguaje inclusivo\na mí me da que este tema escuece mucho y el problema está en otro lado, se está usando la excusa de que han metido 2 frases inclusivas para soltar todo ..."
2,9730181,18,True,False,"A mí lo del lenguaje inclusivo nunca dejará de parecerme una gilipollez, pero se está haciendo una montaña de un grano de arena. La tía que habla de esa manera en el juego es la del podcast, que yo he tenido desactivado toda mi partida porque ya me sobraba en el juego de Miles. Lo del doctore no..."
3,9730181,19,True,False,"No hablaba del juego.\nQue por cierto , no he podido jugar más , pero el edificio ese con las banderas del lgtbi que todo el mundo ha puesto el grito en el cielo está como en un callejón no ?\ntifa rebirt, menos tetas"
4,9730181,27,True,False,"Un tonto giliprogre ofendidito del estercolero de HobbyConsolas, que seguramente sea afín al lobby lgtbijkmnñopq...., buscando que le hagan casito\n."
5,9730181,32,True,False,"No entiendo cómo ha salido en ese estado. Es injugable.\nPor cierto, en Starfield ayer encontré una secundaria que se refería a un personaje con lenguaje inclusivo (la primera vez que vi la e pensé en una errata y la segunda ya me di cuenta de que no).\nLo digo más que nada por los amantes de lo..."
6,9730181,39,True,False,"A mí me parece muy muy diferente ser discapacitado, homosexual, negro, blanco o chino que no sentirte ni hombre ni mujer, veo hasta peligroso normalizarlo pero bueno"
7,9730181,40,True,False,"¿Qué tendrá que ver el amor a los videojuegos con la política identitaria de género?\nNo señores no estamos en 2015, esto ya no es nuevo. Llevamos casi una década viendo como se comportan estas ideologías que nos llegan desde estados unidos.\n``Son unas simples palabras´´.\nBueno, supongo que si..."
8,9730181,42,True,False,"No, lo pone a la misma altura por que el juego tiene lenguaje para sordomudos. Por eso lo nombra en el articulo, pretendiendo hacer ver, que a los mismos q les molesta el lenguaje inclusivo tb les molestaria los sordomudos. Cuando nadie se ha quejado de eso. Pero tu a lo tuyo eh"
9,9730181,43,True,False,"El revuelo es el que se va a montar cuando se les ocurra hacer un juego entero con lenguaje inclusivo. Ahí sí que será preocupante. Me parece una gilipollez el lenguaje inclusivo. Los tontos están llegando muy lejos y es preocupante. Si, recalco, tontos.\nSon ganas de quejarse por quejarse y de ..."


In [200]:
# CELDA 114 — Consolidar los tres hilos aprobados del lote 23

import pandas as pd

IDS_APROBADOS_LOTE_23 = {
    "9730181",
    "9424474",
    "9488478",
}

ANIO_LOTE_23 = 2023
TEMA_LOTE_23 = "lgtbi"

# ------------------------------------------------------------
# 1. Seleccionar los comentarios aprobados
# ------------------------------------------------------------

comentarios_aprobados_lote_23 = (
    comentarios_lote_23_df[
        comentarios_lote_23_df["id_hilo"]
        .astype(str)
        .isin(IDS_APROBADOS_LOTE_23)
    ]
    .copy()
    .reset_index(drop=True)
)

# Asegurar identificadores normalizados
comentarios_aprobados_lote_23["id_hilo"] = (
    comentarios_aprobados_lote_23["id_hilo"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 2. Obtener únicamente los comentarios de 2023
# ------------------------------------------------------------

if "anio_comentario" in comentarios_aprobados_lote_23.columns:
    comentarios_aprobados_anuales_lote_23 = (
        comentarios_aprobados_lote_23[
            pd.to_numeric(
                comentarios_aprobados_lote_23[
                    "anio_comentario"
                ],
                errors="coerce",
            ).eq(ANIO_LOTE_23)
        ]
        .copy()
        .reset_index(drop=True)
    )

else:
    comentarios_aprobados_lote_23[
        "fecha_comentario"
    ] = pd.to_datetime(
        comentarios_aprobados_lote_23[
            "fecha_comentario"
        ],
        errors="coerce",
    )

    comentarios_aprobados_anuales_lote_23 = (
        comentarios_aprobados_lote_23[
            comentarios_aprobados_lote_23[
                "fecha_comentario"
            ].dt.year.eq(ANIO_LOTE_23)
        ]
        .copy()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Consolidar el conjunto bruto
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_bruto_ampliado,
            comentarios_aprobados_lote_23,
        ],
        ignore_index=True,
        sort=False,
    )
)

# ------------------------------------------------------------
# 4. Consolidar el conjunto anual
# ------------------------------------------------------------

comentarios_forocoches_anual_ampliado = (
    pd.concat(
        [
            comentarios_forocoches_anual_ampliado,
            comentarios_aprobados_anuales_lote_23,
        ],
        ignore_index=True,
        sort=False,
    )
)

# ------------------------------------------------------------
# 5. Eliminar posibles duplicados
# ------------------------------------------------------------

if (
    "id_mensaje"
    in comentarios_forocoches_bruto_ampliado.columns
):
    comentarios_forocoches_bruto_ampliado = (
        comentarios_forocoches_bruto_ampliado
        .drop_duplicates(
            subset=["id_mensaje"],
            keep="first",
        )
        .reset_index(drop=True)
    )
else:
    comentarios_forocoches_bruto_ampliado = (
        comentarios_forocoches_bruto_ampliado
        .drop_duplicates(
            subset=[
                "id_hilo",
                "numero_mensaje",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

if (
    "id_mensaje"
    in comentarios_forocoches_anual_ampliado.columns
):
    comentarios_forocoches_anual_ampliado = (
        comentarios_forocoches_anual_ampliado
        .drop_duplicates(
            subset=["id_mensaje"],
            keep="first",
        )
        .reset_index(drop=True)
    )
else:
    comentarios_forocoches_anual_ampliado = (
        comentarios_forocoches_anual_ampliado
        .drop_duplicates(
            subset=[
                "id_hilo",
                "numero_mensaje",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 6. Consolidar diagnósticos e incidencias
# ------------------------------------------------------------

if not diagnosticos_lote_23_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_23_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

if not errores_lote_23_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_23_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 7. Calcular el estado actualizado de LGTBI–2023
# ------------------------------------------------------------

estrato_lgtbi_2023 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].astype(str).str.lower().eq(TEMA_LOTE_23)
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(ANIO_LOTE_23)
    ]
    .copy()
)

comentarios_por_hilo_lgtbi_2023 = (
    estrato_lgtbi_2023
    .groupby("id_hilo")
    .size()
)

numero_hilos_lgtbi_2023 = int(
    comentarios_por_hilo_lgtbi_2023.size
)

numero_comentarios_lgtbi_2023 = int(
    len(estrato_lgtbi_2023)
)

numero_usuarios_lgtbi_2023 = int(
    estrato_lgtbi_2023["usuario_hash"].nunique()
)

proporciones_hilos = (
    comentarios_por_hilo_lgtbi_2023
    / numero_comentarios_lgtbi_2023
)

mayor_hilo_pct = float(
    proporciones_hilos.max() * 100
)

hhi_lgtbi_2023 = float(
    (proporciones_hilos ** 2).sum()
)

hilos_efectivos_lgtbi_2023 = float(
    1 / hhi_lgtbi_2023
)

estrato_suficiente_lgtbi_2023 = bool(
    numero_hilos_lgtbi_2023 >= 12
    and numero_comentarios_lgtbi_2023 >= 500
    and numero_usuarios_lgtbi_2023 >= 300
)

# ------------------------------------------------------------
# 8. Resumen final
# ------------------------------------------------------------

print(
    "Hilos aprobados del lote 23:",
    comentarios_aprobados_lote_23[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios anuales aprobados:",
    len(comentarios_aprobados_anuales_lote_23),
)

print()
print("=" * 70)

print(
    "Hilos válidos totales:",
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios anuales totales:",
    len(comentarios_forocoches_anual_ampliado),
)

print(
    "Hilos de LGTBI–2023:",
    numero_hilos_lgtbi_2023,
)

print(
    "Comentarios de LGTBI–2023:",
    numero_comentarios_lgtbi_2023,
)

print(
    "Usuarios de LGTBI–2023:",
    numero_usuarios_lgtbi_2023,
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct:.1f}%",
)

print(
    "Índice HHI:",
    f"{hhi_lgtbi_2023:.3f}",
)

print(
    "Número efectivo de hilos:",
    f"{hilos_efectivos_lgtbi_2023:.1f}",
)

print(
    "Estrato suficiente:",
    estrato_suficiente_lgtbi_2023,
)

Hilos aprobados del lote 23: 3
Comentarios anuales aprobados: 206

Hilos válidos totales: 79
Comentarios anuales totales: 7628
Hilos de LGTBI–2023: 12
Comentarios de LGTBI–2023: 799
Usuarios de LGTBI–2023: 607
Mayor aportación de un hilo: 20.0%
Índice HHI: 0.143
Número efectivo de hilos: 7.0
Estrato suficiente: True


In [201]:
# CELDA 115 — Evaluar la cobertura final de los ocho estratos

import numpy as np
import pandas as pd

MIN_HILOS = 12
MIN_COMENTARIOS = 500
MIN_USUARIOS = 300

datos_cobertura = (
    comentarios_forocoches_anual_ampliado
    .copy()
)

datos_cobertura["anio_objetivo"] = (
    pd.to_numeric(
        datos_cobertura["anio_objetivo"],
        errors="coerce",
    )
    .astype("Int64")
)

# ------------------------------------------------------------
# 1. Volumen y usuarios por estrato
# ------------------------------------------------------------

cobertura_global = (
    datos_cobertura
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        hilos=(
            "id_hilo",
            "nunique",
        ),
        comentarios=(
            "id_hilo",
            "size",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
    )
)

# ------------------------------------------------------------
# 2. Concentración de comentarios por hilo
# ------------------------------------------------------------

comentarios_por_hilo = (
    datos_cobertura
    .groupby(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )
    .size()
    .rename("comentarios_hilo")
    .reset_index()
)

comentarios_por_hilo[
    "total_estrato"
] = (
    comentarios_por_hilo
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )["comentarios_hilo"]
    .transform("sum")
)

comentarios_por_hilo["proporcion"] = (
    comentarios_por_hilo["comentarios_hilo"]
    / comentarios_por_hilo["total_estrato"]
)

concentracion_global = (
    comentarios_por_hilo
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        mayor_hilo_pct=(
            "proporcion",
            lambda serie: round(
                serie.max() * 100,
                1,
            ),
        ),
        hhi=(
            "proporcion",
            lambda serie: round(
                (serie ** 2).sum(),
                3,
            ),
        ),
    )
)

concentracion_global[
    "hilos_efectivos"
] = (
    1 / concentracion_global["hhi"]
).round(1)

# ------------------------------------------------------------
# 3. Unir indicadores
# ------------------------------------------------------------

cobertura_global = (
    cobertura_global
    .merge(
        concentracion_global,
        on=[
            "tema",
            "anio_objetivo",
        ],
        how="left",
    )
)

# ------------------------------------------------------------
# 4. Calcular déficits
# ------------------------------------------------------------

cobertura_global["faltan_hilos"] = (
    MIN_HILOS
    - cobertura_global["hilos"]
).clip(lower=0)

cobertura_global["faltan_comentarios"] = (
    MIN_COMENTARIOS
    - cobertura_global["comentarios"]
).clip(lower=0)

cobertura_global["faltan_usuarios"] = (
    MIN_USUARIOS
    - cobertura_global["usuarios"]
).clip(lower=0)

cobertura_global["estrato_suficiente"] = (
    cobertura_global["hilos"].ge(MIN_HILOS)
    & cobertura_global["comentarios"].ge(
        MIN_COMENTARIOS
    )
    & cobertura_global["usuarios"].ge(
        MIN_USUARIOS
    )
)

cobertura_global["estado"] = np.where(
    cobertura_global["estrato_suficiente"],
    "CERRADO",
    "ABIERTO",
)

# ------------------------------------------------------------
# 5. Ordenar y mostrar
# ------------------------------------------------------------

orden_temas = pd.CategoricalDtype(
    categories=[
        "inmigracion",
        "lgtbi",
    ],
    ordered=True,
)

cobertura_global["tema"] = (
    cobertura_global["tema"]
    .astype(orden_temas)
)

cobertura_global = (
    cobertura_global
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

estratos_cerrados = int(
    cobertura_global[
        "estrato_suficiente"
    ].sum()
)

print(
    "Hilos válidos totales:",
    datos_cobertura["id_hilo"].nunique(),
)

print(
    "Comentarios anuales totales:",
    len(datos_cobertura),
)

print(
    "Usuarios únicos globales:",
    datos_cobertura["usuario_hash"].nunique(),
)

print(
    "Estratos cerrados:",
    f"{estratos_cerrados} de {len(cobertura_global)}",
)

display(
    cobertura_global[
        [
            "tema",
            "anio_objetivo",
            "hilos",
            "comentarios",
            "usuarios",
            "mayor_hilo_pct",
            "hhi",
            "hilos_efectivos",
            "faltan_hilos",
            "faltan_comentarios",
            "faltan_usuarios",
            "estado",
        ]
    ]
)

Hilos válidos totales: 79
Comentarios anuales totales: 7628
Usuarios únicos globales: 4648
Estratos cerrados: 5 de 8


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos,faltan_hilos,faltan_comentarios,faltan_usuarios,estado
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,0,0,0,CERRADO
1,inmigracion,2016,12,2521,1614,61.6,0.402,2.5,0,0,0,CERRADO
2,inmigracion,2019,12,774,592,29.1,0.145,6.9,0,0,0,CERRADO
3,inmigracion,2023,12,1162,791,44.9,0.282,3.5,0,0,0,CERRADO
4,lgtbi,2015,3,723,632,83.3,0.709,1.4,9,0,0,ABIERTO
5,lgtbi,2016,7,268,208,36.9,0.257,3.9,5,232,92,ABIERTO
6,lgtbi,2019,5,529,397,51.6,0.386,2.6,7,0,0,ABIERTO
7,lgtbi,2023,12,799,607,20.0,0.143,7.0,0,0,0,CERRADO


In [202]:
# CELDA 116 — Crear el lote 24 para ampliar LGTBI–2016

import pandas as pd

candidatos_lote_24 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "4900779",
            "titulo_resultado": (
                "La que me han liado por decir que "
                "los mariquitas no son normales"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=4900779"
            ),
            "subtema_preliminar": (
                "homosexualidad_integracion_homofobia"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre homosexualidad, "
                "normalidad social, integración y uso de "
                "expresiones consideradas homófobas."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_24",
        },
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "5160233",
            "titulo_resultado": (
                "Dos monjas se casan tras dejar "
                "el convento"
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=5160233"
            ),
            "subtema_preliminar": (
                "lesbianas_matrimonio_religion"
            ),
            "criterio_inclusion": (
                "El hilo trata explícitamente una relación "
                "lesbiana, el matrimonio entre dos mujeres "
                "y su relación con la doctrina religiosa."
            ),
            "requiere_revision_tematica": False,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_24",
        },
    ]
)

# ------------------------------------------------------------
# 1. Normalizar identificadores
# ------------------------------------------------------------

candidatos_lote_24["id_hilo"] = (
    candidatos_lote_24["id_hilo"]
    .astype(str)
    .str.strip()
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 2. Incorporar solamente candidatos nuevos
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .dropna()
    .astype(str)
)

candidatos_nuevos_lote_24 = (
    candidatos_lote_24[
        ~candidatos_lote_24["id_hilo"].isin(
            ids_catalogados
        )
    ]
    .copy()
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_24,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Excluir hilos ya consolidados
# ------------------------------------------------------------

ids_consolidados = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ]
    .dropna()
    .astype(str)
)

candidatos_pendientes_lote_24 = (
    candidatos_lote_24[
        ~candidatos_lote_24["id_hilo"].isin(
            ids_consolidados
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Revisar el lote
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 24:",
    len(candidatos_nuevos_lote_24),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 24:",
    len(candidatos_pendientes_lote_24),
)

display(
    candidatos_pendientes_lote_24[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 24: 1
Candidatos totales en el catálogo: 90
Candidatos pendientes del lote 24: 1


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,5160233,Dos monjas se casan tras dejar el convento,lesbianas_matrimonio_religion,False


In [203]:
# CELDA 117 — Extraer el candidato pendiente del lote 24

(
    comentarios_validos_lote_24,
    diagnosticos_lote_24,
    errores_lote_24,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_24,
    "lote_24",
    sesion,
)

# ------------------------------------------------------------
# 1. Unificar los resultados
# ------------------------------------------------------------

comentarios_lote_24_df = unificar_dataframes(
    comentarios_validos_lote_24
)

diagnosticos_lote_24_df = unificar_dataframes(
    diagnosticos_lote_24
)

errores_lote_24_df = unificar_dataframes(
    errores_lote_24
)

# ------------------------------------------------------------
# 2. Resumen técnico
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_24_df[
            "id_hilo"
        ].nunique()
        if not comentarios_lote_24_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_24_df),
)

# ------------------------------------------------------------
# 3. Mostrar incidencias si existen
# ------------------------------------------------------------

if not errores_lote_24_df.empty:
    display(errores_lote_24_df)


[1/1] lgtbi | 2016 | hilo 5160233
Dos monjas se casan tras dejar el convento
Primera página pública: 12 mensajes detectados
Hilo 5160233: 1 página(s)
  Página 1/1: 12 comentarios
Candidato técnicamente válido: 12 comentarios totales | 12 comentarios de 2016

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [204]:
# CELDA 118 — Aprobar y consolidar el lote 24

import pandas as pd

ID_APROBADO_LOTE_24 = "5160233"
ANIO_LOTE_24 = 2016

# ------------------------------------------------------------
# 1. Seleccionar el hilo aprobado
# ------------------------------------------------------------

comentarios_aprobados_lote_24 = (
    comentarios_lote_24_df[
        comentarios_lote_24_df["id_hilo"]
        .astype(str)
        .eq(ID_APROBADO_LOTE_24)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_aprobados_lote_24["id_hilo"] = (
    comentarios_aprobados_lote_24["id_hilo"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 2. Seleccionar solamente comentarios de 2016
# ------------------------------------------------------------

if "anio_comentario" in comentarios_aprobados_lote_24.columns:
    comentarios_anuales_lote_24 = (
        comentarios_aprobados_lote_24[
            pd.to_numeric(
                comentarios_aprobados_lote_24[
                    "anio_comentario"
                ],
                errors="coerce",
            ).eq(ANIO_LOTE_24)
        ]
        .copy()
        .reset_index(drop=True)
    )
else:
    comentarios_aprobados_lote_24[
        "fecha_comentario"
    ] = pd.to_datetime(
        comentarios_aprobados_lote_24[
            "fecha_comentario"
        ],
        errors="coerce",
    )

    comentarios_anuales_lote_24 = (
        comentarios_aprobados_lote_24[
            comentarios_aprobados_lote_24[
                "fecha_comentario"
            ].dt.year.eq(ANIO_LOTE_24)
        ]
        .copy()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Añadir al conjunto bruto y anual
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = pd.concat(
    [
        comentarios_forocoches_bruto_ampliado,
        comentarios_aprobados_lote_24,
    ],
    ignore_index=True,
    sort=False,
)

comentarios_forocoches_anual_ampliado = pd.concat(
    [
        comentarios_forocoches_anual_ampliado,
        comentarios_anuales_lote_24,
    ],
    ignore_index=True,
    sort=False,
)

# ------------------------------------------------------------
# 4. Eliminar posibles duplicados
# ------------------------------------------------------------

for nombre_df in [
    "comentarios_forocoches_bruto_ampliado",
    "comentarios_forocoches_anual_ampliado",
]:
    dataframe = globals()[nombre_df]

    if "id_mensaje" in dataframe.columns:
        dataframe = dataframe.drop_duplicates(
            subset=["id_mensaje"],
            keep="first",
        )
    else:
        dataframe = dataframe.drop_duplicates(
            subset=[
                "id_hilo",
                "numero_mensaje",
            ],
            keep="first",
        )

    globals()[nombre_df] = (
        dataframe.reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Consolidar diagnósticos e incidencias
# ------------------------------------------------------------

if not diagnosticos_lote_24_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_24_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

if not errores_lote_24_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_24_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 6. Estado actualizado de LGTBI–2016
# ------------------------------------------------------------

estrato_lgtbi_2016 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].astype(str).str.lower().eq("lgtbi")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2016)
    ]
    .copy()
)

comentarios_por_hilo_lgtbi_2016 = (
    estrato_lgtbi_2016
    .groupby("id_hilo")
    .size()
)

numero_hilos_lgtbi_2016 = int(
    comentarios_por_hilo_lgtbi_2016.size
)

numero_comentarios_lgtbi_2016 = int(
    len(estrato_lgtbi_2016)
)

numero_usuarios_lgtbi_2016 = int(
    estrato_lgtbi_2016[
        "usuario_hash"
    ].nunique()
)

proporciones_lgtbi_2016 = (
    comentarios_por_hilo_lgtbi_2016
    / numero_comentarios_lgtbi_2016
)

mayor_hilo_pct_lgtbi_2016 = float(
    proporciones_lgtbi_2016.max() * 100
)

hhi_lgtbi_2016 = float(
    (proporciones_lgtbi_2016 ** 2).sum()
)

hilos_efectivos_lgtbi_2016 = float(
    1 / hhi_lgtbi_2016
)

hilos_pendientes_lgtbi_2016 = max(
    12 - numero_hilos_lgtbi_2016,
    0,
)

comentarios_pendientes_lgtbi_2016 = max(
    500 - numero_comentarios_lgtbi_2016,
    0,
)

usuarios_pendientes_lgtbi_2016 = max(
    300 - numero_usuarios_lgtbi_2016,
    0,
)

# ------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------

print(
    "Hilos aprobados del lote 24:",
    comentarios_aprobados_lote_24[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_24),
)

print()
print("=" * 70)

print(
    "Hilos de LGTBI–2016:",
    numero_hilos_lgtbi_2016,
)

print(
    "Comentarios de LGTBI–2016:",
    numero_comentarios_lgtbi_2016,
)

print(
    "Usuarios de LGTBI–2016:",
    numero_usuarios_lgtbi_2016,
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_lgtbi_2016:.1f}%",
)

print(
    "Índice HHI:",
    f"{hhi_lgtbi_2016:.3f}",
)

print(
    "Número efectivo de hilos:",
    f"{hilos_efectivos_lgtbi_2016:.1f}",
)

print(
    "Hilos todavía necesarios:",
    hilos_pendientes_lgtbi_2016,
)

print(
    "Comentarios todavía necesarios:",
    comentarios_pendientes_lgtbi_2016,
)

print(
    "Usuarios todavía necesarios:",
    usuarios_pendientes_lgtbi_2016,
)

Hilos aprobados del lote 24: 1
Comentarios anuales aprobados: 12

Hilos de LGTBI–2016: 8
Comentarios de LGTBI–2016: 280
Usuarios de LGTBI–2016: 220
Mayor aportación de un hilo: 35.4%
Índice HHI: 0.237
Número efectivo de hilos: 4.2
Hilos todavía necesarios: 4
Comentarios todavía necesarios: 220
Usuarios todavía necesarios: 80


In [205]:
# CELDA 119 — Crear el lote 25 para LGTBI–2016

import pandas as pd

candidatos_lote_25 = pd.DataFrame(
    [
        {
            "tema": "lgtbi",
            "anio_objetivo": 2016,
            "id_hilo": "5135013",
            "titulo_resultado": (
                "Rumor sobre Rajoy..."
            ),
            "url_hilo": (
                "https://forocoches.com/foro/"
                "showthread.php?t=5135013"
            ),
            "subtema_preliminar": (
                "homosexualidad_politica_salida_armario"
            ),
            "criterio_inclusion": (
                "Debate explícito sobre homosexualidad, "
                "salida del armario, matrimonio igualitario "
                "y tratamiento político de la orientación sexual."
            ),
            "requiere_revision_tematica": True,
            "origen_candidato": (
                "busqueda_externa_controlada"
            ),
            "lote": "lote_25",
        },
    ]
)

# ------------------------------------------------------------
# 1. Normalizar identificadores
# ------------------------------------------------------------

candidatos_lote_25["id_hilo"] = (
    candidatos_lote_25["id_hilo"]
    .astype(str)
    .str.strip()
)

hilos_candidatos["id_hilo"] = (
    hilos_candidatos["id_hilo"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 2. Incorporar únicamente candidatos nuevos
# ------------------------------------------------------------

ids_catalogados = set(
    hilos_candidatos["id_hilo"]
    .dropna()
    .astype(str)
)

candidatos_nuevos_lote_25 = (
    candidatos_lote_25[
        ~candidatos_lote_25["id_hilo"].isin(
            ids_catalogados
        )
    ]
    .copy()
)

hilos_candidatos = (
    pd.concat(
        [
            hilos_candidatos,
            candidatos_nuevos_lote_25,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Excluir hilos ya consolidados
# ------------------------------------------------------------

ids_consolidados = set(
    comentarios_forocoches_anual_ampliado[
        "id_hilo"
    ]
    .dropna()
    .astype(str)
)

candidatos_pendientes_lote_25 = (
    candidatos_lote_25[
        ~candidatos_lote_25["id_hilo"].isin(
            ids_consolidados
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Comprobar el lote
# ------------------------------------------------------------

print(
    "Candidatos incorporados en el lote 25:",
    len(candidatos_nuevos_lote_25),
)

print(
    "Candidatos totales en el catálogo:",
    len(hilos_candidatos),
)

print(
    "Candidatos pendientes del lote 25:",
    len(candidatos_pendientes_lote_25),
)

display(
    candidatos_pendientes_lote_25[
        [
            "id_hilo",
            "titulo_resultado",
            "subtema_preliminar",
            "requiere_revision_tematica",
        ]
    ]
)

Candidatos incorporados en el lote 25: 1
Candidatos totales en el catálogo: 91
Candidatos pendientes del lote 25: 1


,id_hilo,titulo_resultado,subtema_preliminar,requiere_revision_tematica
0,5135013,Rumor sobre Rajoy...,homosexualidad_politica_salida_armario,True


In [206]:
# CELDA 120 — Extraer el candidato del lote 25

(
    comentarios_validos_lote_25,
    diagnosticos_lote_25,
    errores_lote_25,
) = procesar_lote_forocoches(
    candidatos_pendientes_lote_25,
    "lote_25",
    sesion,
)

# ------------------------------------------------------------
# 1. Unificar los resultados
# ------------------------------------------------------------

comentarios_lote_25_df = unificar_dataframes(
    comentarios_validos_lote_25
)

diagnosticos_lote_25_df = unificar_dataframes(
    diagnosticos_lote_25
)

errores_lote_25_df = unificar_dataframes(
    errores_lote_25
)

# ------------------------------------------------------------
# 2. Resumen técnico
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    "Hilos técnicamente válidos:",
    (
        comentarios_lote_25_df[
            "id_hilo"
        ].nunique()
        if not comentarios_lote_25_df.empty
        else 0
    ),
)

print(
    "Hilos descartados o con error:",
    len(errores_lote_25_df),
)

# ------------------------------------------------------------
# 3. Mostrar posibles incidencias
# ------------------------------------------------------------

if not errores_lote_25_df.empty:
    display(errores_lote_25_df)


[1/1] lgtbi | 2016 | hilo 5135013
Rumor sobre Rajoy...
Primera página pública: 30 mensajes detectados
Hilo 5135013: 2 página(s)
  Página 1/2: 30 comentarios
  Página 2/2: 19 comentarios
Candidato técnicamente válido: 49 comentarios totales | 49 comentarios de 2016

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0

Hilos técnicamente válidos: 1
Hilos descartados o con error: 0


In [207]:
# CELDA 121 — Evaluar la pertinencia temática del hilo 5135013

import re
import pandas as pd

revision_hilo_5135013 = (
    comentarios_lote_25_df[
        comentarios_lote_25_df["id_hilo"]
        .astype(str)
        .eq("5135013")
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 1. Preparar el texto
# ------------------------------------------------------------

if "texto_sin_citas" in revision_hilo_5135013.columns:
    texto_revision = (
        revision_hilo_5135013[
            "texto_sin_citas"
        ]
        .fillna("")
        .astype(str)
    )
else:
    texto_revision = (
        revision_hilo_5135013[
            "texto_original"
        ]
        .fillna("")
        .astype(str)
    )

# ------------------------------------------------------------
# 2. Patrones de revisión
# ------------------------------------------------------------

patron_homosexualidad = re.compile(
    r"\b(?:"
    r"homosexual\w*|gay|gais|"
    r"maric[oó]n\w*|mariquita\w*|"
    r"salir\s+del\s+armario|"
    r"salida\s+del\s+armario|"
    r"orientaci[oó]n\s+sexual|"
    r"pareja\s+del\s+mismo\s+sexo|"
    r"matrimonio\s+(?:gay|homosexual|igualitario)|"
    r"homofobi\w*|transgender\w*"
    r")\b",
    flags=re.IGNORECASE,
)

patron_politica_rajoy = re.compile(
    r"\b(?:"
    r"rajoy|maroto|pp|psoe|podemos|"
    r"izquierda|derecha|presidente|"
    r"gobierno|partido|pol[ií]tic\w*|"
    r"elecciones?|vot\w*"
    r")\b",
    flags=re.IGNORECASE,
)

revision_hilo_5135013[
    "menciona_homosexualidad"
] = texto_revision.str.contains(
    patron_homosexualidad,
    na=False,
)

revision_hilo_5135013[
    "menciona_politica_rajoy"
] = texto_revision.str.contains(
    patron_politica_rajoy,
    na=False,
)

revision_hilo_5135013[
    "menciona_ambos_encuadres"
] = (
    revision_hilo_5135013[
        "menciona_homosexualidad"
    ]
    & revision_hilo_5135013[
        "menciona_politica_rajoy"
    ]
)

# ------------------------------------------------------------
# 3. Resumen cuantitativo
# ------------------------------------------------------------

total_comentarios = len(
    revision_hilo_5135013
)

numero_homosexualidad = int(
    revision_hilo_5135013[
        "menciona_homosexualidad"
    ].sum()
)

numero_politica = int(
    revision_hilo_5135013[
        "menciona_politica_rajoy"
    ].sum()
)

numero_ambos = int(
    revision_hilo_5135013[
        "menciona_ambos_encuadres"
    ].sum()
)

resumen_revision_5135013 = pd.DataFrame(
    [
        {
            "indicador": "comentarios_totales",
            "numero_comentarios": total_comentarios,
            "porcentaje": 100.0,
        },
        {
            "indicador": "mencionan_homosexualidad",
            "numero_comentarios": numero_homosexualidad,
            "porcentaje": round(
                numero_homosexualidad
                / total_comentarios
                * 100,
                1,
            ),
        },
        {
            "indicador": "mencionan_politica_rajoy",
            "numero_comentarios": numero_politica,
            "porcentaje": round(
                numero_politica
                / total_comentarios
                * 100,
                1,
            ),
        },
        {
            "indicador": "mencionan_ambos_encuadres",
            "numero_comentarios": numero_ambos,
            "porcentaje": round(
                numero_ambos
                / total_comentarios
                * 100,
                1,
            ),
        },
    ]
)

print(
    "Comentarios recuperados del hilo 5135013:",
    total_comentarios,
)

display(resumen_revision_5135013)

# ------------------------------------------------------------
# 4. Mostrar los mensajes temáticamente relevantes
# ------------------------------------------------------------

mensajes_homosexualidad_5135013 = (
    revision_hilo_5135013[
        revision_hilo_5135013[
            "menciona_homosexualidad"
        ]
    ]
    [
        [
            "numero_mensaje",
            "menciona_homosexualidad",
            "menciona_politica_rajoy",
            "texto_sin_citas",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print()
print(
    "Mensajes relacionados con homosexualidad:",
    len(mensajes_homosexualidad_5135013),
)

display(mensajes_homosexualidad_5135013)

# ------------------------------------------------------------
# 5. Mostrar el comienzo del hilo para revisar su contexto
# ------------------------------------------------------------

print()
print("Primeros 15 mensajes del hilo:")

display(
    revision_hilo_5135013[
        [
            "numero_mensaje",
            "texto_sin_citas",
        ]
    ]
    .head(15)
    .reset_index(drop=True)
)

Comentarios recuperados del hilo 5135013: 49


,indicador,numero_comentarios,porcentaje
0,comentarios_totales,49,100.0
1,mencionan_homosexualidad,6,12.2
2,mencionan_politica_rajoy,7,14.3
3,mencionan_ambos_encuadres,3,6.1



Mensajes relacionados con homosexualidad: 6


,numero_mensaje,menciona_homosexualidad,menciona_politica_rajoy,texto_sin_citas
0,5,True,False,Que era gay ya lo sabia hasta su mujer
1,8,True,True,"Es aquí donde la izquierda pide menos machismo, respeto a los transgenders etc y se hacen burlas menospreciando la homosexualidad en cuanto tienen oportunidad?"
2,12,True,False,"Por que crees que no suprimió la ley de matrimonio homosexual y fue a la de su amigo Marotp?\nPero bien por el, a ver si algún día sale"
3,16,True,False,Era ZP joven cuando ya se hablaba de la trotona de Pontevendra ...\nPero ese tío es tan pasmao que ni para maricón sirve ...
4,28,True,True,Para insultar a la derecha ya no hay homofobia eh? Que majos los progres.
5,47,True,True,"¿Rumores de que Rajoy es gay? Novedad, hoyga."



Primeros 15 mensajes del hilo:


,numero_mensaje,texto_sin_citas
0,1,Resulta que en Galicia se comenta que Marianico tiene un novio desde hace mucho.\nAlgún shur puede ampliar??
1,2,Follaviejos
2,3,Dos ojos por miravet
3,4,La trotona de Pontevedra\nNo magufo
4,5,Que era gay ya lo sabia hasta su mujer
5,6,Más viejo que el cagar
6,7,Se les oye trotar...
7,8,"Es aquí donde la izquierda pide menos machismo, respeto a los transgenders etc y se hacen burlas menospreciando la homosexualidad en cuanto tienen oportunidad?"
8,9,Dos ojos por UK shur
9,10,Eso parece pero es un poco


In [208]:
# CELDA 122 — Aprobar y consolidar el hilo 5135013

import pandas as pd

ID_APROBADO_LOTE_25 = "5135013"
ANIO_LOTE_25 = 2016

# ------------------------------------------------------------
# 1. Seleccionar el hilo aprobado
# ------------------------------------------------------------

comentarios_aprobados_lote_25 = (
    comentarios_lote_25_df[
        comentarios_lote_25_df["id_hilo"]
        .astype(str)
        .eq(ID_APROBADO_LOTE_25)
    ]
    .copy()
    .reset_index(drop=True)
)

comentarios_aprobados_lote_25["id_hilo"] = (
    comentarios_aprobados_lote_25["id_hilo"]
    .astype(str)
    .str.strip()
)

# Etiqueta metodológica para evitar una interpretación errónea
comentarios_aprobados_lote_25[
    "subtema_validado"
] = "rumor_homosexualidad_discurso_politico"

comentarios_aprobados_lote_25[
    "nota_validacion_tematica"
] = (
    "Incluido para analizar el discurso generado por un "
    "rumor sobre homosexualidad y su uso político. "
    "No constituye evidencia sobre la orientación sexual "
    "de la persona mencionada."
)

# ------------------------------------------------------------
# 2. Seleccionar comentarios pertenecientes a 2016
# ------------------------------------------------------------

if "anio_comentario" in comentarios_aprobados_lote_25.columns:
    comentarios_anuales_lote_25 = (
        comentarios_aprobados_lote_25[
            pd.to_numeric(
                comentarios_aprobados_lote_25[
                    "anio_comentario"
                ],
                errors="coerce",
            ).eq(ANIO_LOTE_25)
        ]
        .copy()
        .reset_index(drop=True)
    )
else:
    comentarios_aprobados_lote_25[
        "fecha_comentario"
    ] = pd.to_datetime(
        comentarios_aprobados_lote_25[
            "fecha_comentario"
        ],
        errors="coerce",
    )

    comentarios_anuales_lote_25 = (
        comentarios_aprobados_lote_25[
            comentarios_aprobados_lote_25[
                "fecha_comentario"
            ].dt.year.eq(ANIO_LOTE_25)
        ]
        .copy()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 3. Consolidar comentarios brutos y anuales
# ------------------------------------------------------------

comentarios_forocoches_bruto_ampliado = pd.concat(
    [
        comentarios_forocoches_bruto_ampliado,
        comentarios_aprobados_lote_25,
    ],
    ignore_index=True,
    sort=False,
)

comentarios_forocoches_anual_ampliado = pd.concat(
    [
        comentarios_forocoches_anual_ampliado,
        comentarios_anuales_lote_25,
    ],
    ignore_index=True,
    sort=False,
)

# ------------------------------------------------------------
# 4. Eliminar posibles duplicados
# ------------------------------------------------------------

for nombre_df in [
    "comentarios_forocoches_bruto_ampliado",
    "comentarios_forocoches_anual_ampliado",
]:
    dataframe = globals()[nombre_df]

    if "id_mensaje" in dataframe.columns:
        dataframe = dataframe.drop_duplicates(
            subset=["id_mensaje"],
            keep="first",
        )
    else:
        dataframe = dataframe.drop_duplicates(
            subset=[
                "id_hilo",
                "numero_mensaje",
            ],
            keep="first",
        )

    globals()[nombre_df] = (
        dataframe.reset_index(drop=True)
    )

# ------------------------------------------------------------
# 5. Consolidar diagnósticos e incidencias
# ------------------------------------------------------------

if not diagnosticos_lote_25_df.empty:
    diagnosticos_forocoches_ampliado = (
        pd.concat(
            [
                diagnosticos_forocoches_ampliado,
                diagnosticos_lote_25_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

if not errores_lote_25_df.empty:
    incidencias_forocoches_ampliacion = (
        pd.concat(
            [
                incidencias_forocoches_ampliacion,
                errores_lote_25_df,
            ],
            ignore_index=True,
            sort=False,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 6. Recalcular LGTBI–2016
# ------------------------------------------------------------

estrato_lgtbi_2016 = (
    comentarios_forocoches_anual_ampliado[
        comentarios_forocoches_anual_ampliado[
            "tema"
        ].astype(str).str.lower().eq("lgtbi")
        & pd.to_numeric(
            comentarios_forocoches_anual_ampliado[
                "anio_objetivo"
            ],
            errors="coerce",
        ).eq(2016)
    ]
    .copy()
)

comentarios_por_hilo_lgtbi_2016 = (
    estrato_lgtbi_2016
    .groupby("id_hilo")
    .size()
)

numero_hilos_lgtbi_2016 = int(
    comentarios_por_hilo_lgtbi_2016.size
)

numero_comentarios_lgtbi_2016 = int(
    len(estrato_lgtbi_2016)
)

numero_usuarios_lgtbi_2016 = int(
    estrato_lgtbi_2016[
        "usuario_hash"
    ].nunique()
)

proporciones_lgtbi_2016 = (
    comentarios_por_hilo_lgtbi_2016
    / numero_comentarios_lgtbi_2016
)

mayor_hilo_pct_lgtbi_2016 = float(
    proporciones_lgtbi_2016.max() * 100
)

hhi_lgtbi_2016 = float(
    (proporciones_lgtbi_2016 ** 2).sum()
)

hilos_efectivos_lgtbi_2016 = float(
    1 / hhi_lgtbi_2016
)

# ------------------------------------------------------------
# 7. Mostrar el nuevo estado
# ------------------------------------------------------------

print(
    "Hilos aprobados del lote 25:",
    comentarios_aprobados_lote_25[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios anuales aprobados:",
    len(comentarios_anuales_lote_25),
)

print()
print("=" * 70)

print(
    "Hilos de LGTBI–2016:",
    numero_hilos_lgtbi_2016,
)

print(
    "Comentarios de LGTBI–2016:",
    numero_comentarios_lgtbi_2016,
)

print(
    "Usuarios de LGTBI–2016:",
    numero_usuarios_lgtbi_2016,
)

print(
    "Mayor aportación de un hilo:",
    f"{mayor_hilo_pct_lgtbi_2016:.1f}%",
)

print(
    "Índice HHI:",
    f"{hhi_lgtbi_2016:.3f}",
)

print(
    "Número efectivo de hilos:",
    f"{hilos_efectivos_lgtbi_2016:.1f}",
)

print(
    "Hilos todavía necesarios:",
    max(12 - numero_hilos_lgtbi_2016, 0),
)

print(
    "Comentarios todavía necesarios:",
    max(500 - numero_comentarios_lgtbi_2016, 0),
)

print(
    "Usuarios todavía necesarios:",
    max(300 - numero_usuarios_lgtbi_2016, 0),
)

Hilos aprobados del lote 25: 1
Comentarios anuales aprobados: 49

Hilos de LGTBI–2016: 9
Comentarios de LGTBI–2016: 329
Usuarios de LGTBI–2016: 264
Mayor aportación de un hilo: 30.1%
Índice HHI: 0.194
Número efectivo de hilos: 5.2
Hilos todavía necesarios: 3
Comentarios todavía necesarios: 171
Usuarios todavía necesarios: 36


In [209]:
# CELDA 123 — Cerrar la búsqueda y clasificar la cobertura final

import numpy as np
import pandas as pd

BUSQUEDA_FOROCOCHES_CERRADA = True

MOTIVO_CIERRE_BUSQUEDA = (
    "La búsqueda se detiene por rendimientos decrecientes en la "
    "localización de nuevos hilos públicos, ausencia de API y "
    "limitaciones de indexación. No se reducen los umbrales de "
    "suficiencia definidos previamente."
)

MIN_HILOS = 12
MIN_COMENTARIOS = 500
MIN_USUARIOS = 300

datos_finales_forocoches = (
    comentarios_forocoches_anual_ampliado
    .copy()
)

datos_finales_forocoches["anio_objetivo"] = (
    pd.to_numeric(
        datos_finales_forocoches["anio_objetivo"],
        errors="coerce",
    )
    .astype("Int64")
)

# ------------------------------------------------------------
# 1. Métricas básicas por estrato
# ------------------------------------------------------------

cobertura_final_forocoches = (
    datos_finales_forocoches
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        hilos=(
            "id_hilo",
            "nunique",
        ),
        comentarios=(
            "id_hilo",
            "size",
        ),
        usuarios=(
            "usuario_hash",
            "nunique",
        ),
    )
)

# ------------------------------------------------------------
# 2. Concentración por hilo
# ------------------------------------------------------------

comentarios_por_hilo_final = (
    datos_finales_forocoches
    .groupby(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )
    .size()
    .rename("comentarios_hilo")
    .reset_index()
)

comentarios_por_hilo_final[
    "comentarios_estrato"
] = (
    comentarios_por_hilo_final
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ]
    )["comentarios_hilo"]
    .transform("sum")
)

comentarios_por_hilo_final["proporcion"] = (
    comentarios_por_hilo_final["comentarios_hilo"]
    / comentarios_por_hilo_final[
        "comentarios_estrato"
    ]
)

concentracion_final = (
    comentarios_por_hilo_final
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        mayor_hilo_pct=(
            "proporcion",
            lambda serie: round(
                serie.max() * 100,
                1,
            ),
        ),
        hhi=(
            "proporcion",
            lambda serie: round(
                (serie ** 2).sum(),
                3,
            ),
        ),
    )
)

concentracion_final["hilos_efectivos"] = (
    1 / concentracion_final["hhi"]
).round(1)

cobertura_final_forocoches = (
    cobertura_final_forocoches
    .merge(
        concentracion_final,
        on=[
            "tema",
            "anio_objetivo",
        ],
        how="left",
    )
)

# ------------------------------------------------------------
# 3. Evaluar cada requisito por separado
# ------------------------------------------------------------

cobertura_final_forocoches[
    "cumple_hilos"
] = cobertura_final_forocoches["hilos"].ge(
    MIN_HILOS
)

cobertura_final_forocoches[
    "cumple_comentarios"
] = cobertura_final_forocoches[
    "comentarios"
].ge(MIN_COMENTARIOS)

cobertura_final_forocoches[
    "cumple_usuarios"
] = cobertura_final_forocoches["usuarios"].ge(
    MIN_USUARIOS
)

cobertura_final_forocoches[
    "estrato_suficiente"
] = (
    cobertura_final_forocoches["cumple_hilos"]
    & cobertura_final_forocoches[
        "cumple_comentarios"
    ]
    & cobertura_final_forocoches[
        "cumple_usuarios"
    ]
)

# ------------------------------------------------------------
# 4. Clasificación metodológica
# ------------------------------------------------------------

condiciones = [
    cobertura_final_forocoches[
        "estrato_suficiente"
    ],
    (
        cobertura_final_forocoches[
            "cumple_comentarios"
        ]
        & cobertura_final_forocoches[
            "cumple_usuarios"
        ]
        & ~cobertura_final_forocoches[
            "cumple_hilos"
        ]
    ),
]

categorias = [
    "cobertura_suficiente",
    "cobertura_parcial_por_hilos",
]

cobertura_final_forocoches[
    "nivel_cobertura"
] = np.select(
    condiciones,
    categorias,
    default="cobertura_parcial_limitada",
)

cobertura_final_forocoches[
    "uso_recomendado_nlp"
] = np.select(
    [
        cobertura_final_forocoches[
            "nivel_cobertura"
        ].eq("cobertura_suficiente"),
        cobertura_final_forocoches[
            "nivel_cobertura"
        ].eq("cobertura_parcial_por_hilos"),
    ],
    [
        (
            "Análisis principal; presentar también "
            "estimaciones equilibradas por hilo."
        ),
        (
            "Análisis con cautela; priorizar ponderación "
            "equilibrada por hilo y declarar la limitación."
        ),
    ],
    default=(
        "Análisis exploratorio; evitar conclusiones "
        "representativas sobre toda la plataforma."
    ),
)

# ------------------------------------------------------------
# 5. Registrar el cierre de búsqueda
# ------------------------------------------------------------

cobertura_final_forocoches[
    "busqueda_cerrada"
] = BUSQUEDA_FOROCOCHES_CERRADA

cobertura_final_forocoches[
    "motivo_cierre"
] = MOTIVO_CIERRE_BUSQUEDA

# ------------------------------------------------------------
# 6. Ordenar el resultado
# ------------------------------------------------------------

orden_temas = pd.CategoricalDtype(
    categories=[
        "inmigracion",
        "lgtbi",
    ],
    ordered=True,
)

cobertura_final_forocoches["tema"] = (
    cobertura_final_forocoches["tema"]
    .astype(orden_temas)
)

cobertura_final_forocoches = (
    cobertura_final_forocoches
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Resumen de cierre
# ------------------------------------------------------------

estratos_suficientes = int(
    cobertura_final_forocoches[
        "estrato_suficiente"
    ].sum()
)

print("Búsqueda de nuevos hilos cerrada.")
print(
    "Hilos válidos totales:",
    datos_finales_forocoches[
        "id_hilo"
    ].nunique(),
)

print(
    "Comentarios anuales totales:",
    len(datos_finales_forocoches),
)

print(
    "Usuarios únicos globales:",
    datos_finales_forocoches[
        "usuario_hash"
    ].nunique(),
)

print(
    "Estratos con cobertura suficiente:",
    f"{estratos_suficientes} de "
    f"{len(cobertura_final_forocoches)}",
)

print()
print("Motivo del cierre:")
print(MOTIVO_CIERRE_BUSQUEDA)

display(
    cobertura_final_forocoches[
        [
            "tema",
            "anio_objetivo",
            "hilos",
            "comentarios",
            "usuarios",
            "mayor_hilo_pct",
            "hhi",
            "hilos_efectivos",
            "nivel_cobertura",
            "uso_recomendado_nlp",
        ]
    ]
)

Búsqueda de nuevos hilos cerrada.
Hilos válidos totales: 81
Comentarios anuales totales: 7689
Usuarios únicos globales: 4686
Estratos con cobertura suficiente: 5 de 8

Motivo del cierre:
La búsqueda se detiene por rendimientos decrecientes en la localización de nuevos hilos públicos, ausencia de API y limitaciones de indexación. No se reducen los umbrales de suficiencia definidos previamente.


,tema,anio_objetivo,hilos,comentarios,usuarios,mayor_hilo_pct,hhi,hilos_efectivos,nivel_cobertura,uso_recomendado_nlp
0,inmigracion,2015,16,852,565,24.2,0.127,7.9,cobertura_suficiente,Análisis principal; presentar también estimaciones equilibradas por hilo.
1,inmigracion,2016,12,2521,1614,61.6,0.402,2.5,cobertura_suficiente,Análisis principal; presentar también estimaciones equilibradas por hilo.
2,inmigracion,2019,12,774,592,29.1,0.145,6.9,cobertura_suficiente,Análisis principal; presentar también estimaciones equilibradas por hilo.
3,inmigracion,2023,12,1162,791,44.9,0.282,3.5,cobertura_suficiente,Análisis principal; presentar también estimaciones equilibradas por hilo.
4,lgtbi,2015,3,723,632,83.3,0.709,1.4,cobertura_parcial_por_hilos,Análisis con cautela; priorizar ponderación equilibrada por hilo y declarar la limitación.
5,lgtbi,2016,9,329,264,30.1,0.194,5.2,cobertura_parcial_limitada,Análisis exploratorio; evitar conclusiones representativas sobre toda la plataforma.
6,lgtbi,2019,5,529,397,51.6,0.386,2.6,cobertura_parcial_por_hilos,Análisis con cautela; priorizar ponderación equilibrada por hilo y declarar la limitación.
7,lgtbi,2023,12,799,607,20.0,0.143,7.0,cobertura_suficiente,Análisis principal; presentar también estimaciones equilibradas por hilo.


In [ ]:
# Exportar los resultados definitivos de ForoCoches

from pathlib import Path
import hashlib
import pandas as pd

# ------------------------------------------------------------
# 1. Rutas del proyecto
# ------------------------------------------------------------

RUTA_RAW_FOROCOCHES = RUTA_RAW
RUTA_PROCESADOS_FOROCOCHES = RUTA_PROCESADOS

RUTA_RAW_FOROCOCHES.mkdir(
    parents=True,
    exist_ok=True
)

RUTA_PROCESADOS_FOROCOCHES.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Normalizar los conjuntos definitivos
# ------------------------------------------------------------

comentarios_brutos_finales = (
    comentarios_forocoches_bruto_ampliado
    .copy()
    .reset_index(drop=True)
)

comentarios_anuales_finales = (
    comentarios_forocoches_anual_ampliado
    .copy()
    .reset_index(drop=True)
)

diagnosticos_finales = (
    diagnosticos_forocoches_ampliado
    .copy()
    .reset_index(drop=True)
)

incidencias_finales = (
    incidencias_forocoches_ampliacion
    .copy()
    .reset_index(drop=True)
)

catalogo_candidatos_final = (
    hilos_candidatos
    .copy()
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
    .reset_index(drop=True)
)

# Normalizar identificadores
for dataframe in [
    comentarios_brutos_finales,
    comentarios_anuales_finales,
    diagnosticos_finales,
    incidencias_finales,
    catalogo_candidatos_final,
]:
    if "id_hilo" in dataframe.columns:
        dataframe["id_hilo"] = (
            dataframe["id_hilo"]
            .astype(str)
            .str.strip()
        )

# ------------------------------------------------------------
# 3. Crear catálogo exclusivo de hilos válidos
# ------------------------------------------------------------

ids_hilos_validos = set(
    comentarios_anuales_finales[
        "id_hilo"
    ].astype(str)
)

hilos_validos_finales = (
    catalogo_candidatos_final[
        catalogo_candidatos_final[
            "id_hilo"
        ].astype(str).isin(ids_hilos_validos)
    ]
    .copy()
)

resumen_hilos_validos = (
    comentarios_anuales_finales
    .groupby(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ],
        as_index=False,
    )
    .agg(
        comentarios_anuales=(
            "id_hilo",
            "size",
        ),
        usuarios_unicos=(
            "usuario_hash",
            "nunique",
        ),
        fecha_minima=(
            "fecha_comentario",
            "min",
        ),
        fecha_maxima=(
            "fecha_comentario",
            "max",
        ),
    )
)

columnas_metadatos = [
    columna
    for columna in [
        "id_hilo",
        "titulo_resultado",
        "url_hilo",
        "subtema_preliminar",
        "criterio_inclusion",
        "requiere_revision_tematica",
        "origen_candidato",
        "lote",
    ]
    if columna in hilos_validos_finales.columns
]

metadatos_hilos_validos = (
    hilos_validos_finales[
        columnas_metadatos
    ]
    .drop_duplicates(
        subset=["id_hilo"],
        keep="first",
    )
)

hilos_validos_finales = (
    resumen_hilos_validos
    .merge(
        metadatos_hilos_validos,
        on="id_hilo",
        how="left",
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
            "id_hilo",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Definir los archivos de salida
# ------------------------------------------------------------

archivos_salida = {
    "comentarios_brutos": (
        RUTA_RAW_FOROCOCHES
        / "forocoches_comentarios_brutos.csv"
    ),
    "catalogo_candidatos": (
        RUTA_RAW_FOROCOCHES
        / "forocoches_catalogo_candidatos.csv"
    ),
    "comentarios_anuales": (
        RUTA_PROCESADOS_FOROCOCHES
        / "forocoches_comentarios_anuales.csv"
    ),
    "hilos_validos": (
        RUTA_PROCESADOS_FOROCOCHES
        / "forocoches_hilos_validos.csv"
    ),
    "cobertura": (
        RUTA_PROCESADOS_FOROCOCHES
        / "forocoches_cobertura_estratos.csv"
    ),
    "diagnosticos": (
        RUTA_PROCESADOS_FOROCOCHES
        / "forocoches_diagnosticos_extraccion.csv"
    ),
    "incidencias": (
        RUTA_PROCESADOS_FOROCOCHES
        / "forocoches_incidencias_extraccion.csv"
    ),
}

# ------------------------------------------------------------
# 5. Guardar los CSV
# ------------------------------------------------------------

comentarios_brutos_finales.to_csv(
    archivos_salida["comentarios_brutos"],
    index=False,
    encoding="utf-8-sig",
)

catalogo_candidatos_final.to_csv(
    archivos_salida["catalogo_candidatos"],
    index=False,
    encoding="utf-8-sig",
)

comentarios_anuales_finales.to_csv(
    archivos_salida["comentarios_anuales"],
    index=False,
    encoding="utf-8-sig",
)

hilos_validos_finales.to_csv(
    archivos_salida["hilos_validos"],
    index=False,
    encoding="utf-8-sig",
)

cobertura_final_forocoches.to_csv(
    archivos_salida["cobertura"],
    index=False,
    encoding="utf-8-sig",
)

diagnosticos_finales.to_csv(
    archivos_salida["diagnosticos"],
    index=False,
    encoding="utf-8-sig",
)

incidencias_finales.to_csv(
    archivos_salida["incidencias"],
    index=False,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------
# 6. Crear el manifiesto de integridad
# ------------------------------------------------------------

def calcular_sha256(ruta_archivo):
    sha256 = hashlib.sha256()

    with open(ruta_archivo, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024),
            b"",
        ):
            sha256.update(bloque)

    return sha256.hexdigest()


registros_manifiesto = []

for nombre, ruta in archivos_salida.items():
    registros_manifiesto.append(
        {
            "archivo": nombre,
            "ruta": str(ruta),
            "tamano_bytes": ruta.stat().st_size,
            "sha256": calcular_sha256(ruta),
        }
    )

manifiesto_forocoches = pd.DataFrame(
    registros_manifiesto
)

ruta_manifiesto = (
    RUTA_PROCESADOS_FOROCOCHES
    / "forocoches_manifiesto_archivos.csv"
)

manifiesto_forocoches.to_csv(
    ruta_manifiesto,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------
# 7. Verificación final
# ------------------------------------------------------------

print("Extracción de ForoCoches guardada correctamente.")
print()

print(
    "Comentarios brutos guardados:",
    len(comentarios_brutos_finales),
)

print(
    "Comentarios anuales guardados:",
    len(comentarios_anuales_finales),
)

print(
    "Hilos válidos guardados:",
    hilos_validos_finales[
        "id_hilo"
    ].nunique(),
)

print(
    "Candidatos catalogados:",
    catalogo_candidatos_final[
        "id_hilo"
    ].nunique(),
)

print(
    "Estratos documentados:",
    len(cobertura_final_forocoches),
)

print()
print("Archivos generados:")

display(manifiesto_forocoches)

Extracción de ForoCoches guardada correctamente.

Comentarios brutos guardados: 7995
Comentarios anuales guardados: 7689
Hilos válidos guardados: 81
Candidatos catalogados: 91
Estratos documentados: 8

Archivos generados:


,archivo,ruta,tamano_bytes,sha256
0,comentarios_brutos,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\RAW\forocoches_comentarios_brutos.csv,5882688,3dac94929a7a86a44747d70cd2c435b5f9d247c36f22be8ed892658f1d72e98a
1,catalogo_candidatos,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\RAW\forocoches_catalogo_candidatos.csv,26488,02cd99c11f9e054c8f47aa32b9c466a0137b67543cec420a2bb950e798cce729
2,comentarios_anuales,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\PROCESADOS\forocoches_comentarios_anuales.csv,5482073,2b36453065e9449499119c7c5a46cc075a21f022e061e107996856eb46816188
3,hilos_validos,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\PROCESADOS\forocoches_hilos_validos.csv,22916,dba54541a31677c736fecc4e9fd775bd0cd960552477ab6a0a15b2b4dc63e717
4,cobertura,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\PROCESADOS\forocoches_cobertura_estratos.csv,3317,1299934053d15ac1e70b5f97e4093ac2e1e4ea74b340d1d0370f5f4a1ce04a96
5,diagnosticos,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\PROCESADOS\forocoches_diagnosticos_extraccion.csv,8897,8cb3c5fdefcea875ba5a9387f6be19e0465d82b07ebf69eeacaa010d013a5be7
6,incidencias,C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\RRSS\FOROCOCHES\PROCESADOS\forocoches_incidencias_extraccion.csv,2330,6e539b1cfabc85f668fedb44fa155749638d8f603ba737369377fd0d8ee49589


In [211]:
# CELDA 125 — Validar los archivos definitivos leyendo desde disco

import pandas as pd

ruta_comentarios_finales = (
    RUTA_PROCESADOS_FOROCOCHES
    / "forocoches_comentarios_anuales.csv"
)

ruta_hilos_finales = (
    RUTA_PROCESADOS_FOROCOCHES
    / "forocoches_hilos_validos.csv"
)

ruta_cobertura_final = (
    RUTA_PROCESADOS_FOROCOCHES
    / "forocoches_cobertura_estratos.csv"
)

# ------------------------------------------------------------
# 1. Volver a cargar los archivos guardados
# ------------------------------------------------------------

comentarios_verificacion = pd.read_csv(
    ruta_comentarios_finales,
    encoding="utf-8-sig",
    dtype={
        "id_hilo": str,
        "id_mensaje": str,
    },
    low_memory=False,
)

hilos_verificacion = pd.read_csv(
    ruta_hilos_finales,
    encoding="utf-8-sig",
    dtype={
        "id_hilo": str,
    },
)

cobertura_verificacion = pd.read_csv(
    ruta_cobertura_final,
    encoding="utf-8-sig",
)

# ------------------------------------------------------------
# 2. Normalizar campos esenciales
# ------------------------------------------------------------

comentarios_verificacion["id_hilo"] = (
    comentarios_verificacion["id_hilo"]
    .astype(str)
    .str.strip()
)

hilos_verificacion["id_hilo"] = (
    hilos_verificacion["id_hilo"]
    .astype(str)
    .str.strip()
)

comentarios_verificacion[
    "anio_objetivo"
] = pd.to_numeric(
    comentarios_verificacion[
        "anio_objetivo"
    ],
    errors="coerce",
).astype("Int64")

# ------------------------------------------------------------
# 3. Comprobar duplicados
# ------------------------------------------------------------

if "id_mensaje" in comentarios_verificacion.columns:
    duplicados_mensaje = int(
        comentarios_verificacion[
            "id_mensaje"
        ].duplicated().sum()
    )
else:
    duplicados_mensaje = int(
        comentarios_verificacion.duplicated(
            subset=[
                "id_hilo",
                "numero_mensaje",
            ]
        ).sum()
    )

# ------------------------------------------------------------
# 4. Comprobar textos disponibles
# ------------------------------------------------------------

texto_sin_citas = (
    comentarios_verificacion[
        "texto_sin_citas"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

textos_vacios = int(
    texto_sin_citas.eq("").sum()
)

# ------------------------------------------------------------
# 5. Comprobar años y temas
# ------------------------------------------------------------

anios_encontrados = sorted(
    comentarios_verificacion[
        "anio_objetivo"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

temas_encontrados = sorted(
    comentarios_verificacion[
        "tema"
    ]
    .dropna()
    .astype(str)
    .str.lower()
    .unique()
    .tolist()
)

ids_comentarios = set(
    comentarios_verificacion[
        "id_hilo"
    ]
)

ids_catalogo_validos = set(
    hilos_verificacion[
        "id_hilo"
    ]
)

hilos_sin_metadatos = (
    ids_comentarios
    - ids_catalogo_validos
)

hilos_sin_comentarios = (
    ids_catalogo_validos
    - ids_comentarios
)

# ------------------------------------------------------------
# 6. Resultado de las comprobaciones
# ------------------------------------------------------------

control_calidad_final = pd.DataFrame(
    [
        {
            "control": "comentarios_anuales",
            "resultado": len(
                comentarios_verificacion
            ),
            "esperado": 7689,
            "correcto": (
                len(comentarios_verificacion)
                == 7689
            ),
        },
        {
            "control": "hilos_validos",
            "resultado": comentarios_verificacion[
                "id_hilo"
            ].nunique(),
            "esperado": 81,
            "correcto": (
                comentarios_verificacion[
                    "id_hilo"
                ].nunique()
                == 81
            ),
        },
        {
            "control": "duplicados_mensaje",
            "resultado": duplicados_mensaje,
            "esperado": 0,
            "correcto": (
                duplicados_mensaje == 0
            ),
        },
        {
            "control": "textos_sin_citas_vacios",
            "resultado": textos_vacios,
            "esperado": 0,
            "correcto": (
                textos_vacios == 0
            ),
        },
        {
            "control": "hilos_sin_metadatos",
            "resultado": len(
                hilos_sin_metadatos
            ),
            "esperado": 0,
            "correcto": (
                len(hilos_sin_metadatos) == 0
            ),
        },
        {
            "control": "hilos_sin_comentarios",
            "resultado": len(
                hilos_sin_comentarios
            ),
            "esperado": 0,
            "correcto": (
                len(hilos_sin_comentarios) == 0
            ),
        },
        {
            "control": "estratos_documentados",
            "resultado": len(
                cobertura_verificacion
            ),
            "esperado": 8,
            "correcto": (
                len(cobertura_verificacion) == 8
            ),
        },
    ]
)

print("Años encontrados:", anios_encontrados)
print("Temas encontrados:", temas_encontrados)
print()

display(control_calidad_final)

if control_calidad_final["correcto"].all():
    print(
        "CONTROL FINAL SUPERADO: "
        "la extracción de ForoCoches queda cerrada."
    )
else:
    print(
        "ATENCIÓN: existe alguna comprobación "
        "que debe revisarse antes del NLP."
    )

Años encontrados: [2015, 2016, 2019, 2023]
Temas encontrados: ['inmigracion', 'lgtbi']



,control,resultado,esperado,correcto
0,comentarios_anuales,7689,7689,True
1,hilos_validos,81,81,True
2,duplicados_mensaje,0,0,True
3,textos_sin_citas_vacios,269,0,False
4,hilos_sin_metadatos,0,0,True
5,hilos_sin_comentarios,0,0,True
6,estratos_documentados,8,8,True


ATENCIÓN: existe alguna comprobación que debe revisarse antes del NLP.


In [212]:
# CELDA 126 — Analizar los comentarios sin texto propio

import pandas as pd
import re

comentarios_sin_texto_propio = (
    comentarios_verificacion[
        comentarios_verificacion[
            "texto_sin_citas"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 1. Preparar el texto original
# ------------------------------------------------------------

texto_original_vacios = (
    comentarios_sin_texto_propio[
        "texto_original"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

comentarios_sin_texto_propio[
    "texto_original_vacio"
] = texto_original_vacios.eq("")

comentarios_sin_texto_propio[
    "original_contiene_cita"
] = texto_original_vacios.str.contains(
    r"(?:\bCita\s+de\b|Originally\s+Posted\s+by)",
    case=False,
    regex=True,
    na=False,
)

comentarios_sin_texto_propio[
    "original_contiene_url"
] = texto_original_vacios.str.contains(
    r"(?:https?://|www\.)",
    case=False,
    regex=True,
    na=False,
)

comentarios_sin_texto_propio[
    "original_parece_imagen"
] = texto_original_vacios.str.contains(
    r"(?:\bImage\b|\[img\]|\.jpg\b|\.jpeg\b|"
    r"\.png\b|\.gif\b|\.webp\b)",
    case=False,
    regex=True,
    na=False,
)

comentarios_sin_texto_propio[
    "original_muy_corto"
] = texto_original_vacios.str.len().le(3)

# ------------------------------------------------------------
# 2. Clasificar la causa probable
# ------------------------------------------------------------

def clasificar_texto_vacio(fila):
    if fila["texto_original_vacio"]:
        return "sin_texto_original"

    if fila["original_contiene_cita"]:
        return "solo_cita_o_residuo_de_cita"

    if fila["original_parece_imagen"]:
        return "imagen_o_multimedia"

    if fila["original_contiene_url"]:
        return "solo_enlace_o_contenido_web"

    if fila["original_muy_corto"]:
        return "contenido_muy_corto"

    return "requiere_revision"


comentarios_sin_texto_propio[
    "causa_probable"
] = comentarios_sin_texto_propio.apply(
    clasificar_texto_vacio,
    axis=1,
)

# ------------------------------------------------------------
# 3. Resumen de causas
# ------------------------------------------------------------

resumen_textos_vacios = (
    comentarios_sin_texto_propio[
        "causa_probable"
    ]
    .value_counts(dropna=False)
    .rename_axis("causa_probable")
    .reset_index(name="comentarios")
)

resumen_textos_vacios["porcentaje"] = (
    resumen_textos_vacios["comentarios"]
    / len(comentarios_sin_texto_propio)
    * 100
).round(1)

print(
    "Comentarios sin texto propio:",
    len(comentarios_sin_texto_propio),
)

display(resumen_textos_vacios)

# ------------------------------------------------------------
# 4. Distribución por tema y año
# ------------------------------------------------------------

distribucion_vacios = (
    comentarios_sin_texto_propio
    .groupby(
        [
            "tema",
            "anio_objetivo",
        ],
        as_index=False,
    )
    .agg(
        comentarios_sin_texto=(
            "id_hilo",
            "size",
        ),
        hilos_afectados=(
            "id_hilo",
            "nunique",
        ),
    )
    .sort_values(
        [
            "tema",
            "anio_objetivo",
        ]
    )
    .reset_index(drop=True)
)

display(distribucion_vacios)

# ------------------------------------------------------------
# 5. Mostrar los casos que requieren revisión
# ------------------------------------------------------------

casos_revision_textos_vacios = (
    comentarios_sin_texto_propio[
        comentarios_sin_texto_propio[
            "causa_probable"
        ].eq("requiere_revision")
    ]
    .copy()
    .reset_index(drop=True)
)

columnas_revision = [
    columna
    for columna in [
        "tema",
        "anio_objetivo",
        "id_hilo",
        "numero_mensaje",
        "contiene_cita",
        "texto_original",
        "texto_sin_citas",
        "causa_probable",
    ]
    if columna in casos_revision_textos_vacios.columns
]

print(
    "Casos que requieren revisión manual:",
    len(casos_revision_textos_vacios),
)

display(
    casos_revision_textos_vacios[
        columnas_revision
    ].head(30)
)

Comentarios sin texto propio: 269


,causa_probable,comentarios,porcentaje
0,sin_texto_original,169,62.8
1,solo_cita_o_residuo_de_cita,100,37.2


,tema,anio_objetivo,comentarios_sin_texto,hilos_afectados
0,inmigracion,2015,21,10
1,inmigracion,2016,72,11
2,inmigracion,2019,11,7
3,inmigracion,2023,14,5
4,lgtbi,2015,103,3
5,lgtbi,2016,13,6
6,lgtbi,2019,15,4
7,lgtbi,2023,20,5


Casos que requieren revisión manual: 0


,tema,anio_objetivo,id_hilo,numero_mensaje,contiene_cita,texto_original,texto_sin_citas,causa_probable
